In [ ]:

from google.colab import drive
drive.mount("/content/drive")


# PRISM v10 — Qwen3.8-27B → **27B-A3B** (Shadow Execution Edition)

**Mission.** Transform Alibaba's **Qwen3.8-27B** (dense, native vision-language, hybrid Gated-DeltaNet / Gated-Attention, 262K context) into a sparse model with **the same total parameters**, **≤ 3B active parameters** (STRICT touched-weights accounting, one text token), **near-zero degradation**, full context and multimodality preserved, and a **stock-vLLM-servable artifact**.

```
                     THE RESONA EXECUTION STACK — v10 "SHADOW EXECUTION"
   (reference config — Cell 19 re-solves it from the real config, Cell 31 re-counts the built model)

   Principle: NOTHING IS SKIPPED.  y = S(x) + 1[selected]·(exact(x) − S(x)),  S(x) = U (Uᵀ W) x
   — whatever is not computed exactly takes the value of the best rank-r predictor of that site's output.

FFN (17.1B)       ──► PRISM fission: 128 experts, top-4 (4 x 132 neurons) + 512-neuron shared core
                      + affine rank-128 SHADOW of the unselected experts            → 1.174B active
Full attention    ──► rank-256 SHADOW of q / o  +  exact carrier rows + 1 of 64
(16 layers)           residual-routed row-blocks                                    → 0.326B
Gated DeltaNet    ──► rank-256 SHADOWS of q / v / z / out. SHADOW-MoRM: every head
(48 layers)           WRITES (shadow value); the critic picks the 4 EXACT heads     → 1.265B
AntennaMast       ──► phi(h) = silu(R h), S = 192 (router / critic features)        → 0.138B
Head (1.27B)      ──► SPECTRAL HEAD FUNNEL: rank-256 projection + exact shortlist   → 0.086B
                  ───────────────────────────────────────────────────────────────────────────
                  TOTAL ACTIVE (plan == census of the built graph)             ≈ 2.990B ≤ 3.0B ✓
```

**Why v10.** On the real 27B, v9's served student started at a held-out perplexity of ~2×10⁵ (the soft training path at ~2×10³): every routed projection *zeroed* ~90% of its output rows, every unselected DeltaNet head wrote nothing, the DeltaNet q-routers were never calibrated (top-1 always picked block 0), and the FFN chose one coarse group of 528 neurons. v10 replaces "skip" by "shadow" everywhere, routes on the *residual* the shadow leaves, gives the top-1 routers a counterfactual straight-through gradient, trains the served path from step 0, and distils purely from the dense teacher (`LAMBDA_CE = 0`). The surgery cell prints, per family, how much output energy the shadows capture on held-out tokens — that number, and the `[fidelity]`-style PPL line of the teacher-cache cell, tell you immediately whether the idea carries on your model. It is a high-risk design: nothing here has run on the 27B yet.

**Pipeline:** dense model → calibration (reads the spectra) → fission + resonant surgery → Theorem-7 anchor check (exact dense identity at init) → Sinkhorn-OT self-distillation from a **free in-place teacher** (the student's own exact mode) → precision plan → two artifacts: **(A) COMPAT** = stock `Qwen3_5MoeForConditionalGeneration` → stock vLLM, **(B) NATIVE** = the full ≤ 3B-active RESONA execution + `serve_native.py` launcher → SLIPSTREAM disk-native serving.

| RUN_MODE | What it does | Hardware |
|---|---|---|
| `"demo"` | End-to-end on `Qwen/Qwen3.5-0.8B` (same family). Every theorem and both export paths execute. | Any Colab, ~25 min |
| `"full"` | The real `Qwen/Qwen3.8-27B` transformation with the 3B budget solver. | A100-80GB (bf16) |

**Honesty box.**
1. *The 3B number is counted, not asserted.* "Active" = weight elements read for one text token. The solver prints the whole waterfall, the build **consumes its plan**, and after surgery and again after training a census walks the live module graph (every module once) and **stops the run** if it exceeds 3.0B (`ENFORCE_BUDGET`). The sketch map phi(h) = silu(R h) is charged for every evaluation; `MAST_ACCOUNTING="distinct"` restores the cheaper v8 convention. The reference NATIVE launcher is correctness-first and recomputes phi at every routed site — a production kernel must share it per layer, as the budget assumes.
2. *Degradation is managed, not magicked.* At surgery the model is *exactly* the dense model (Theorem 7); sparse execution introduces a *certified* deviation that self-distillation closes. MoRM 2.0 is the most experimental family; its claims are certificate-backed (contracts M1–M10) and `m_bar = n_heads` is the exact-mode safety switch, one config away.
3. *Two artifacts, two numbers.* COMPAT activates ≈ 9.1B (stock vLLM cannot execute resonant attention); NATIVE activates ≈ 2.99B. Cell 31 prints both, never conflated.
4. *Verification status.* Every formula has a CPU contract (C1–C12, M1–M10, **U1–U12**). v9 was assembled and checked without access to the 27B weights — the first full run is the first real-model test, which is why each new formula can fall back to its v8 behaviour on its own. Treat results as a starting point, not a benchmark claim.


# PRISM changelog

## v10.0 — Shadow Execution (control-variate sparsity)

* **Shadow instead of skip** (`fit_shadow_basis`, `ResonantLinear.attach_shadow`, `SectionedProjector.attach_shadow`): every routed projection outputs `U (Uᵀ W) x` on the rows it does not compute exactly (U = principal output directions of the dense site on calibration tokens); exact carrier / block rows *replace* the shadow values. Row plans and routers are fitted on the **residual** energy.
* **Shadow-MoRM**: with a v shadow every DeltaNet head writes; the write gate is disabled and the influence critic only decides which heads are evaluated exactly (4 per token by default).
* **FFN**: 128 experts / top-4 instead of 32 / top-1 (same routed-neuron budget, 4 independent choices per token) plus an affine rank-128 shadow of the unselected experts (reduced-rank kernel ridge, kept per layer only if it lowers held-out error).
* **Routers learn on the served path**: straight-through gate `sel = hard + (soft − soft.detach())` → `dL/dsel_b = <dL/dy_b, exact_b − S_b>` for every block. DeltaNet q/z routers are now calibrated (v9 left them at zero).
* **Training**: served path from step 0 (`TRAIN_PATH="hard"`), pure teacher distillation (`LAMBDA_CE=0`), every loss term logged every 5 steps.
* **Budget**: shadows are charged (U and M of every site); sketch 192, 64 row-blocks, 1% carrier, 4 exact heads pay for them — 2.990B on the reference config. Launcher twins and the sidecar carry the shadows; twin == notebook module is checked by executing the launcher kernels.
* Switches: `SHADOW_RANK_LIN=0`, `SHADOW_RANK_FFN=0`, `SHADOW_V=False`, `ROUTER_ST=False`, `FFN_EXPERTS=32 / FFN_TOPK=1` give back the v9 behaviour piece by piece.

## v9.0 — crash fix, honest budget, calibrated formulas

### 1 · The crash: `NotImplementedError: Int8Tensor dispatch … aten.view … arg_types=(Int8Tensor, list)`

`torchao.quantize_` walks the module tree through `named_children()` and calls the filter on **every visit**. After the MoRM 2.0 surgery the *same* `MoRM2Policy` object is a child of both `SectionedProjector.morm` and `BetaGate.policy`, so its critic `nn.Linear` was visited twice: the first visit replaced the weight by an `Int8Tensor`, the second tried to quantize that `Int8Tensor` again (`from_hp → quantize_affine → weight.view(list)`) and died. Re-running the cell failed the same way. v8 also aimed int8 at the *trainable* critics and at the bit-exact BetaGate anchor.

**Fix (Cell 27, rewritten).** Default `PRECISION_PLAN="bf16"`: after surgery almost every weight is an adopted raw `Parameter` that torchao never touches, so int8 reached only ≈ 0.1–0.2B of 27B (< 1 %) — risk without a memory benefit. The opt-in `"int8_frozen"` plan is guarded: id-deduplicated filter (each module is offered once), frozen plain-bf16 `Linear`s only, protect list, lossless restore from CPU copies, any failure restores bf16 and the run goes on. Contract **U9** reproduces the double visit on a toy module graph and shows the new filter fixes it.

### 2 · Defects found while auditing everything behind the crash

| # | Where | v8 behaviour | v9 |
|---|---|---|---|
| 1 | budget | the solver *planned* shared core 192 / sketch 512 / write heads …, the pipeline *built* something else; the real active count was **3.15B** and hid behind `assert act <= TARGET*1.10` | the plan is **consumed** by the build; census of the built graph; no tolerance band (`ENFORCE_BUDGET`) |
| 2 | budget | the AntennaMast sketch was charged once per distinct matrix although phi(h) must be evaluated at every (layer, input width): **≈ 0.37B undercount** | `MAST_ACCOUNTING="applications"` charges every evaluation; the solver trims sketch size and write heads to fit (`mast_S` 384, `m_write` 12 at the reference config); `"distinct"` reproduces the v8 convention |
| 3 | budget | `measured_active()` summed a hand-picked registry — it never saw dense `k_proj`, the v half, `in_proj_a`, conv taps, norms, or a surgery site that stayed dense | `census_active()` walks the live module graph (shared children once; unclassified 2-D+ parameters charged in full) |
| 4 | calibration | stash `cat(...)[:N]`: after block 0 had filled it, every later block was truncated away — routers, gains and their validation saw ~0.1 % of the mixture | stratified quotas: block *i* of *n* contributes ⌊N(i+1)/n⌋ − ⌊Ni/n⌋ rows |
| 5 | calibration | shared core = largest *signed mean* activation | ranked by output energy E[a²]·‖W_down[:, r]‖² |
| 6 | calibration | gains fitted on *routed output vs the FULL dense output* (shared core counted twice); calibration τ clamp (0.5, 8) ≠ serving clamp (0.25, 4) | closed-form NNLS on `y_dense − y_shared`, consistent with the served module (contract U2); one clamp |
| 7 | head | HeadFunnel applied the singular-value scaling twice | exact projection `U_r = W P_r`; residual-aware ranking certified by Cauchy–Schwarz |
| 8 | training | with k = 1 the serving gate renormalises to 1, so the router received **no** gradient from the output loss | EMRD gives the router a direct oracle signal |
| 9 | export | the reference launcher's `FunnelHead` assumed 2-D input | flattens leading dimensions |

### 3 · New / upgraded formulas

| Name | Formula | Role |
|---|---|---|
| **Oracle group energy** | $e_{t,g}=\sum_{r\in g}a_r(t)^2\,\lVert W_{down}[:,r]\rVert^2$ | the exact output energy every expert owns on every token — labels for the routers and the EMRD loss |
| **Validated router** | prototype router vs bias-free dual-form ridge on the log energy share; the winner is chosen by the oracle energy it captures on **held-out** tokens, then refit on all | a router must beat the baseline on tokens it never saw |
| **Serving-consistent NNLS gains** | $\min_{c\ge 0}\lVert y_{dense}-y_{shared}-\sum_e c_e g_e y_e\rVert^2+\lambda\lVert c-c_0\rVert^2$ (Lawson–Hanson, ridge toward $c_0=k$; τ grid only for k > 1) | the gains are fitted on the function the served module computes |
| **Partition-KL** | $\mathrm{KL}_{part}=P_S\,\mathrm{KL}_{support}+\mathrm{KL}_{Bern}(P_S\Vert Q_S)$ | distillation on {teacher top-64 ids} ∪ {rest}: lower-bounds the full KL, needs no more than the cached top-64, and — unlike v8's renormalised KL — sees the probability mass the student puts outside the support |
| **EMRD** | $-\log\sum_e p_e\min(1,k\cdot \mathrm{share}_e)$ | router distillation straight from the oracle energies; gives k = 1 a gradient |
| **LTD** | $\lVert y-y_{dense}\rVert^2/\lVert y_{dense}\rVert^2$ per layer (hard phase) | every layer gets a local dense target instead of relying on the final-logit KL to reach 64 layers |
| **HeadFunnel v9** | $U_r=WP_r$; rank by $\tilde z+\kappa\, s\,\rho_v$ with $s=\lVert(I-P_rP_r^\top)h\rVert$, $\rho_v=\lVert(I-P_rP_r^\top)w_v\rVert$; Cauchy–Schwarz: $\lvert z-\tilde z\rvert\le s\rho_v$ | the shortlist ranking is certified; κ is fitted for recall on half of the probe blocks and the recall is reported on the other half |
| **Honest solver** | trim ladder (sketch → shared core → shortlist → write heads); plan consumed; infeasible targets are reported | what is built is what was priced |

### 4 · Verification

A new **v9-contracts cell** (U1–U12, CPU, seconds) asserts each formula on toys and owns a fallback switch per formula (see the cell). The module-graph census is checked against the waterfall on a toy transformer built from the *real* classes (U10). All real-model code paths run for the first time in your Colab — that is what the fallbacks are for.

## Earlier releases (condensed)

- **v8.0.4** — surgery cell re-runnable (swaps registered and undone on re-run); CPU/CUDA index slip fixed (`out_z[:, zblk]`) with an auto-heal net; `fla-core` Triton Gated-DeltaNet kernels on GPU runtimes; mixed calibration/distillation corpus (web + Korean wiki + code + chat + wikitext); teacher set up to 1024 blocks; content-hashed teacher cache.
- **v8.0.3** — OOM hardening (no model math changed): export released `state_dict` aliases, vLLM memory fraction sized from measured free VRAM, load-cell capacity check, per-layer temporaries released, per-micro-batch OOM shield, host-RAM aware parity checks.
- **v8.0** — MoRM 2.0: Write Influence Kernel labels (exact first-order counterfactuals), elastic token-adaptive write budgets (E[m] = m_bar), deferred write-back with exact decay compensation, charge ledger + staleness bound, Proposition F (beta-masking is the write lever that is exact in both kernel forms), severed policy-gradient fix; contracts M1–M10.
- **v7.0** — clean-code release (one 12-contract suite, SLIPSTREAM merged into one cell, theory in docstrings).
- **v6.0** — the training OOM root-caused: telemetry side-channels re-ran during checkpoint recompute and pinned every layer's saved graph; now stored detached and only on the trainer's pass.


In [ ]:
# Setup — packages, allocator config, memory helpers (v9.0)
# torchao requirement (user spec): >= 0.16.0 (current docs line: 0.17.x)
get_ipython().system('pip install -q -U "transformers>=5.8.0" "torchao>=0.16.0" accelerate datasets safetensors sentencepiece einops 2>&1 | tail -2')
# vLLM is installed later in the serving section (heavy install).

# v8.0.4 [SPEED]: fla-core = the optimized Triton kernels for the Gated-DeltaNet
# layers (48 of the 64 layers). Without it transformers falls back to a readable
# PyTorch reference that is >10x slower AND keeps far more activations for backward
# — the largest single cost of the calibration, teacher-cache and training phases.
# Installed on GPU runtimes only. If it cannot be imported, transformers silently
# keeps the reference path, so this can only help. (Set USE_FLA = False to opt out.)
USE_FLA = globals().get("USE_FLA", True)
if USE_FLA:
    get_ipython().system('nvidia-smi -L > /dev/null 2>&1 && pip install -q -U fla-core 2>&1 | tail -2')

import importlib, subprocess, sys, os, math, json, time, random

def _sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

gpu_name = _sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null") or "NO GPU (CPU-only)"
print("GPU:", gpu_name)

import os
# Allocator config BEFORE the first CUDA allocation (PyTorch docs:
# expandable_segments must be active before the allocator is created):
# segments grow in place instead of minting un-splittable cudaMalloc blocks,
# which is what fragmented the A100 after the big calibration allocations.
# Forced (not setdefault) so a stale env value can never re-enable the
# fragmenting allocator.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import torch
import transformers
import torchao
print(f"torch {torch.__version__} | transformers {transformers.__version__} | torchao {torchao.__version__}")

from packaging import version
assert version.parse(transformers.__version__) >= version.parse("5.8.0"), \
    "transformers>=5.8.0 required (qwen3_5 / qwen3_5_moe architectures)"
assert version.parse(torchao.__version__) >= version.parse("0.16.0"), \
    "torchao>=0.16.0 required (user spec)"

# v8.0.4: which GDN kernel will the model use? transformers binds it when the
# qwen3_5 modeling module is first imported (= the model load, after this cell).
FLA_ACTIVE = False
try:
    importlib.invalidate_caches()          # fla-core was pip-installed a moment ago
    import fla as _fla
    import fla.ops.gated_delta_rule as _fla_gdr          # what transformers resolves
    FLA_ACTIVE = bool(torch.cuda.is_available())
    print(f"GDN kernel: fla-core {getattr(_fla, '__version__', '?')} (Triton) "
          f"{'- ACTIVE' if FLA_ACTIVE else '- CPU runtime: reference path is used'}")
except Exception as _e:
    print("GDN kernel: PyTorch reference path (much slower) — fla-core not usable: "
          f"{type(_e).__name__}: {str(_e)[:80]}")

if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability()
    print(f"CUDA compute capability: sm_{major}{minor}",
          "(FP8 training eligible)" if major >= 9 else "(FP8 not available; int8/int4 path)")


# ---- memory helpers (used at every phase boundary) ----------
# ============================================================================
# v8.0.3 OOM hardening (nothing in this block changes any model math)
#   * gpu_mem also prints HOST RAM: on Colab the "OOM" is often the VM itself
#     (session dies when host RAM runs out), not the GPU
#   * is_oom / oom_backoff: classify CUDA OOMs and retry with a smaller, purely
#     batching knob after scrubbing the heap OUTSIDE the except block
#   * release_previous_run: pressing Run on a cell twice must not stack a 2nd
#     copy of the model on top of the 1st
# ============================================================================
OOMFIX_VER = "v9.0"
try:
    import psutil
except Exception:
    psutil = None


def host_ram_gib():
    """(available, total) host RAM in GiB; (inf, inf) when psutil is missing."""
    if psutil is None:
        return float("inf"), float("inf")
    vm = psutil.virtual_memory()
    return vm.available / 2**30, vm.total / 2**30


def vram_gib():
    """(free, total) GPU memory in GiB; (0, 0) on a CPU-only runtime."""
    if not torch.cuda.is_available():
        return 0.0, 0.0
    free, total = torch.cuda.mem_get_info()
    return free / 2**30, total / 2**30


def gpu_mem(tag: str = "") -> None:
    """One-line census: VRAM allocated | reserved | truly free  +  host RAM."""
    ram_av, ram_tot = host_ram_gib()
    ram = (f" | host RAM {ram_av:.0f}/{ram_tot:.0f} GiB free"
           if ram_tot != float("inf") else "")
    if not torch.cuda.is_available():
        print(f"[mem{(' ' + tag) if tag else ''}] CPU-only runtime{ram}")
        return
    alloc = torch.cuda.memory_allocated() / 2**30
    res = torch.cuda.memory_reserved() / 2**30
    free, total = torch.cuda.mem_get_info()
    print(f"[mem{(' ' + tag) if tag else ''}] alloc {alloc:6.1f} GiB | "
          f"reserved {res:6.1f} GiB | free {free / 2**30:5.1f} / "
          f"{total / 2**30:.0f} GiB{ram}")


def free_cuda(tag: str = "") -> None:
    """Call-site `del`s happen first; this releases what they left behind:
    gc.collect() (breaks autograd/reference cycles) THEN empty_cache()
    (returns reserved-but-unallocated blocks to the driver so the NEXT
    phase — export reload, vLLM — can actually claim them)."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gpu_mem(tag)


def is_oom(e: BaseException) -> bool:
    """True for a CUDA OOM, including the allocator-failure RuntimeError flavours."""
    if isinstance(e, torch.cuda.OutOfMemoryError):
        return True
    s = str(e).lower()
    return isinstance(e, RuntimeError) and (
        "out of memory" in s or "cublas_status_alloc_failed" in s)


def oom_backoff(fn, sizes, tag: str = ""):
    """Run fn(size), largest size first. On a CUDA OOM: scrub the heap and retry
    with the next (smaller) size. `size` must be a PURE batching knob
    (micro-batch, chunk rows, ...) so the result is identical at every size.
    The scrub happens after the except block has exited, so the failed
    attempt's graph / traceback is already releasable."""
    sizes = list(sizes)
    for i, s in enumerate(sizes):
        failed, msg = False, ""
        try:
            return fn(s)
        except RuntimeError as e:
            if not is_oom(e):
                raise
            failed, msg = True, str(e)[:300]
        if failed:
            gpu_mem(f"OOM in {tag or getattr(fn, '__name__', 'fn')} at size={s}")
            free_cuda()
            if i == len(sizes) - 1:
                raise torch.cuda.OutOfMemoryError(
                    f"{tag}: still OOM at the smallest size ({s}). {msg}")
            print(f"[oom-backoff] {tag}: retrying with size={sizes[i + 1]}")


def release_previous_run(verbose: bool = True) -> None:
    """Re-run guard: drop everything a previous pass of this notebook still
    holds on the GPU (the model, the module lists that ADOPT its weights,
    optimizer / trainer / vLLM handles) before a fresh load allocates another
    ~52 GiB on top of it — the classic 'OOM the second time I press Run'."""
    g = globals()
    held = []
    for nm in ("PRISM_MODULES", "RESONA_LINEARS", "RESONA_SECTIONED", "RESONA_MORM",
               "RESONA_FUNNEL", "RESONA_MORM_GATES", "_FORCED_CKPT", "_PARKED"):
        obj = g.get(nm)
        if isinstance(obj, list) and obj:
            held.append(f"{nm}[{len(obj)}]")
            obj.clear()
    for nm in ("trainer", "optimizer", "scheduler", "reloaded", "llm", "model",
               "funnel", "MAST", "ORIG_LOGITS", "dense_sd", "ref_sd", "STORE"):
        if g.get(nm) is not None:
            held.append(nm)
            g[nm] = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if held and verbose:
        print(f"[re-run guard] released previous-run state: {', '.join(held)}")
        gpu_mem("after release")


# ---- v8.0.4: safety net for CPU/CUDA index mismatches -------------------------
# "indices should be either on cpu or on the same device as the indexed tensor"
# kills a multi-hour run over a one-line device slip (it did, in the surgery cell).
# The only sensible meaning of such an index is "use it where the tensor lives",
# so on exactly that error the index is moved and the op is retried (once per
# call-site a note is printed so the line can still be fixed). It costs nothing
# unless the error fires. Set INDEX_AUTOHEAL = False to opt out.
INDEX_AUTOHEAL = globals().get("INDEX_AUTOHEAL", True)
_IDX_MSG = "indices should be either on cpu or on the same device as the indexed tensor"
_idx_heal_sites = []


def _idx_to(idx, dev):
    if torch.is_tensor(idx):
        return idx if idx.device == dev else idx.to(dev)
    if isinstance(idx, tuple):
        return tuple(_idx_to(i, dev) for i in idx)
    if isinstance(idx, list):
        return [_idx_to(i, dev) for i in idx]
    return idx


def _install_index_autoheal():
    T = torch.Tensor
    if getattr(T, "_prism_idx_heal", False):
        return
    _get, _set = T.__getitem__, T.__setitem__

    def _note(kind):
        try:
            f = sys._getframe(2)
            site = f"{os.path.basename(f.f_code.co_filename)}:{f.f_lineno}"
        except Exception:
            site = "?"
        if site not in _idx_heal_sites and len(_idx_heal_sites) < 8:
            _idx_heal_sites.append(site)
            print(f"[index-heal] {kind}: index tensor was on a different device than the "
                  f"indexed tensor — moved it and retried (first seen at {site})")

    def __getitem__(self, idx):
        try:
            return _get(self, idx)
        except RuntimeError as e:
            if _IDX_MSG not in str(e):
                raise
        _note("getitem")
        return _get(self, _idx_to(idx, self.device))

    def __setitem__(self, idx, val):
        try:
            return _set(self, idx, val)
        except RuntimeError as e:
            if _IDX_MSG not in str(e):
                raise
        _note("setitem")
        return _set(self, _idx_to(idx, self.device), val)

    T.__getitem__, T.__setitem__ = __getitem__, __setitem__
    T._prism_idx_heal = True


if INDEX_AUTOHEAL:
    _install_index_autoheal()


torch.manual_seed(0); random.seed(0)
print("Setup OK.")
_f0, _t0 = vram_gib(); _r0, _rt0 = host_ram_gib()
print(f"[{OOMFIX_VER}] GPU {_t0:.0f} GiB total / {_f0:.0f} free | "
      f"host RAM {_rt0:.0f} GiB total / {_r0:.0f} free")


## Configuration

Everything downstream is config-driven — the same code runs the demo model and the 27B target, and the sparse geometry is **solved from the real config against the 3B budget at runtime**, never hardcoded. Requested sizes (`SHARED_SIZE`, `FUNNEL_SHORTLIST`, `M_WRITE_FRAC`) are *caps*: the strict budget trims what it cannot afford, and the effective values are re-bound after the solve (every trim is printed).

**Quality modes** (all stay ≤ the budget): `"max_budget_use"` (default) spends every scrap of the 3B on shortlist / memory / coverage; `"headroom"` targets 2.75B leaving ~8 % escalation room for confidence-gated dynamic top-k (roadmap); `"safety"` lifts the write-head cap so MoRM uses as many heads as the budget affords.

**v9 switches.** Every new formula is individually switchable; the v9-contracts cell flips a switch back to the v8 behaviour if its CPU check fails.

| Knob | Default | What it switches |
|---|---|---|
| `FFN_CAL_MODE` | `"v9"` | validated router + NNLS gains (`"v8"`: prototype router + Adam gains) |
| `FFN_ROUTER_MODE` | `"auto"` | `"auto"` picks the router by held-out captured energy (`"prototype"` / `"ridge"` force one) |
| `FFN_CAL_TOKENS` | 4096 / 1024 | calibration rows per FFN layer (full / demo), stratified over blocks |
| `KL_MODE` | `"partition"` | Partition-KL (`"support"` = v8's renormalised KL) |
| `LAMBDA_EMRD`, `LAMBDA_LTD` | 0.05, 0.25 | router energy-mass loss; layer-local dense-twin loss (hard phase only); 0 = off |
| `FUNNEL_USE_RHO` | `True` | residual-aware head shortlist (+V reads per token, counted in the budget) |
| `PRECISION_PLAN` | `"bf16"` | `"int8_frozen"` / `"fp8_train"` are guarded opt-ins |
| `ENFORCE_BUDGET` | `True` | stop when the census of the built graph exceeds the target |
| `MAST_ACCOUNTING` | `"applications"` | strict sketch accounting (`"distinct"` = v8 convention, ~0.37B cheaper on paper) |
| `MAST_S_MIN` | 256 | the solver never trims the antenna sketch below this |


In [ ]:
# Configuration — every knob in the pipeline lives here; the same code
# runs the demo model and the 27B target (Drive is mounted in cell 0).

# ------------------------------- RUN MODE -----------------------------------
RUN_MODE = "full"          # "demo" (Qwen3.5-0.8B, runs anywhere) | "full" (Qwen3.8-27B)
QUALITY_MODE = "max_budget_use"   # "max_budget_use" (spend all slack under 3.0B) | "headroom" (target 2.75B) | "safety" (no write-cap request: MoRM may use every head the budget allows)

# One version constant: every banner and label in the notebook derives
# from this single name, so the label can never drift from the code.
__PRISM_VER__ = "v10.0"


MODELS = {
    "demo": "Qwen/Qwen3.5-0.8B",      # same qwen3_5 family, dense, tiny
    "full": "Qwen/Qwen3.8-27B",       # the target: dense VL hybrid 27B, 262K ctx
}
MODEL_ID = MODELS[RUN_MODE]

# --------------------------- PRISM structure (FFN) ---------------------------
# FFN (E, k) candidates; the v2 solver refines these against the ACTIVE budget.
CANDIDATES = {
    "demo": ((16, 2), (16, 4), (8, 2), (8, 4), (32, 2)),
    "full": ((256, 8), (128, 8), (128, 4), (64, 8), (64, 4), (64, 2),
             (32, 4), (32, 2)),
}[RUN_MODE]
SHARED_SIZE = 512 if RUN_MODE == "full" else 32     # REQUESTED always-on shared-core neurons (v9: the budget solver may trim it; the effective value is re-bound after the solve)
FFN_EXPERTS = 128 if RUN_MODE == "full" else 32   # v10: FINER experts (132 neurons each on the 27B) ...
FFN_TOPK    = 4 if RUN_MODE == "full" else 2      # ... and more of them per token: the same routed-neuron budget
#   as v9's top-1-of-32, but the selection adapts to the token in 4 independent pieces instead of 1.
COMPLETION_RANK = 64 if RUN_MODE == "full" else 16  # ZSSI field rank R per layer

# --------------------------- RESONA execution (v2) ---------------------------
TARGET_ACTIVE = {"demo": None,                     # demo: minimize & report
                 "full": 3.0e9}[RUN_MODE]          # user MANDATE (hard assert)
if QUALITY_MODE == "headroom" and RUN_MODE == "full":
    TARGET_ACTIVE = 2.75e9
MAST_S        = 256                 # AntennaMast sketch dim (once per token)
LIN_BLOCKS    = 64 if RUN_MODE == "full" else 16    # resonant row-blocks per routed Linear (v10: fewer exact rows,
#                                     the budget goes to the shadows instead)
# ---- v10 SHADOW EXECUTION: nothing is skipped — what is not computed exactly takes the value of the best
#      rank-r predictor of that site's output (see 'SHADOW EXECUTION' in the core cell). 0 = off (v9).
SHADOW_RANK_LIN = 256 if RUN_MODE == "full" else 32   # routed projections (q / z / o / out) and DeltaNet v
SHADOW_RANK_FFN = 128 if RUN_MODE == "full" else 16   # affine low-rank model of the unselected FFN experts
SHADOW_V        = True     # DeltaNet: every head writes (shadow value); the MoRM critic picks the EXACT heads
M_WRITE_FLOOR   = 4        # exact heads / token the plan starts from when SHADOW_V is on (the solver may add more)
ROUTER_ST       = True     # straight-through counterfactual gradient for the top-k block routers
LIN_TOPK      = 1                   # blocks activated per token
CARRIER_FRAC  = 0.01                # always-on carrier rows (fraction of rows; the highest RESIDUAL rows)
M_WRITE_FRAC  = None if QUALITY_MODE == "safety" else 0.56   # MoRM write-cap REQUEST (fraction of heads); the solver returns the m_write the budget really affords (<= this cap)
FUNNEL_RANK   = 256                 # Spectral Head Funnel proxy rank
FUNNEL_SHORTLIST = 8192             # exact head rows evaluated per token (a CAP: the solver may use fewer)

# ------------------------------ calibration ---------------------------------
CAL_TOKENS      = 65536 if RUN_MODE == "full" else 16384
SKETCH_DIM      = 512
GAIN_CAL_TOKENS = 2048  if RUN_MODE == "full" else 256   # rows per resonant-linear / MoRM stash (spread over blocks)
GAIN_CAL_STEPS  = 300                                     # v8 Adam gains (only used when FFN_CAL_MODE='v8' / as fallback)

# ------------------------- v9 formula switches ------------------------------
# Every v9 upgrade is individually switchable.  The "v9 contracts" cell runs CPU
# self-checks of each formula and, if one fails, flips its switch back to the v8
# behaviour (loudly) — a new formula can never be the reason a long run dies.
FFN_CAL_MODE      = "v9"        # "v9": validated router + serving-consistent NNLS gains | "v8": prototype router + Adam
FFN_ROUTER_MODE   = "auto"      # "auto" (best on held-out tokens) | "prototype" | "ridge"
FFN_CAL_TOKENS    = 4096 if RUN_MODE == "full" else 1024   # FFN calibration rows per layer (stratified over blocks)
KL_MODE           = "partition" # "partition": KL on {teacher top-K ids} + {rest}  |  "support": v8 renormalised KL
LAMBDA_EMRD       = 0.05        # energy-mass router distillation (0 = off)
LAMBDA_LTD        = 0.25        # layer-local dense-twin loss, hard phase only (0 = off)
FUNNEL_USE_RHO    = True        # residual-aware head shortlist (V extra reads/token — counted in the budget)
FUNNEL_FIT_BLOCKS = 8           # blocks x tokens used to fit kappa and to measure shortlist recall
FUNNEL_FIT_TOK    = 256
PRECISION_PLAN    = "bf16"      # "bf16" (default) | "int8_frozen" (opt-in; see the precision cell)
ENFORCE_BUDGET    = True        # stop when the census of the REAL module graph exceeds TARGET_ACTIVE
MAST_ACCOUNTING   = "applications"  # "applications": every phi evaluation is charged (strict) | "distinct": each distinct sketch matrix once (v8 convention)
MAST_S_MIN        = 192         # the budget solver never trims the antenna sketch below this

# ------------------------------- training -----------------------------------
STEPS          = 400 if RUN_MODE == "full" else 120
WARMUP_STEPS   = 24                  # warmup_ratio is DEPRECATED -> warmup_steps
LR_FIELDS      = 3e-4
LR_ROUTER      = 1e-4
LR_TAU         = 1e-3
LAMBDA_CE      = 0.0      # v9.1: hard-label CE weight. 0 = PURE TEACHER DISTILLATION (targets = dense Qwen3.8, not the corpus)
LAMBDA_KL      = 1.0
TRAIN_PATH     = "hard"   # v10 default: train the SERVED function. "auto" (pick by measured held-out PPL) | "hard" (train the served path) | "soft_then_hard" (v8)
LAMBDA_BAL     = 0.01
LAMBDA_Z       = 1e-4
LAMBDA_ZS      = 0.05
# v9 loss terms (LAMBDA_EMRD / LAMBDA_LTD and KL_MODE live in the v9 block above)
SEQ_LEN        = 1024
# micro-batch 1 x accum 16 = the SAME effective batch 16, while every
# T-proportional consumer (GDN scan saves, soft-mode section outputs,
# head chunks) halves at zero training-semantics cost — and with the
# head GEMM chunked INSIDE the loss (_ChunkedHeadLoss), LOSS_CHUNK
# governs the vocab GEMM itself instead of chunking an
# already-materialized tensor.
BATCH_SIZE     = 1 if RUN_MODE == "full" else 4
GRAD_ACCUM     = 16 if RUN_MODE == "full" else 2
LOSS_CHUNK     = 256      # vocab-GEMM rows per fp32 head-loss chunk
TEACHER_TOPK   = 64
ANNEAL_FRAC    = 0.75
VLLM_GPU_FRAC  = 0.80   # fraction of TOTAL VRAM vLLM may claim (the student is parked on CPU RAM first)

# ----------------------- MoRM 2.0 (RESONA v8) --------------------------------
# Elastic write budget: mean writes per token (m_bar) defaults to the
# solver's m_write (same average cost as v7's fixed top-m); m_max caps the
# per-token spike; K bounds deferred-write staleness; the WIK window is the
# influence-label horizon. BETA_LOGIT_FLOOR drives masked heads to
# sigma(-60) ~ 9e-27 (bit-exact 0 when the kernel takes post-sigmoid beta).
MORM2_M_MAX_FRAC  = 0.75     # per-token write cap (fraction of n_heads)
MORM2_STALENESS_K = 64       # deferred-write staleness bound (tokens)
MORM2_WIK_WINDOW  = 1024     # WIK label horizon (tokens)
MORM2_CTRL_LR     = 0.02     # elastic rate-controller lr (per batch)
MORM2_STE_TAU     = 1.0      # straight-through softness
MORM2_LAMBDA_CRITIC = 0.10   # critic label-regression aux weight
MORM2_CRITIC_BIAS = True     # v9: per-head intercept in the influence critic (v8's bias-free warm start lost each head's mean level)
MORM2_BETA_LOGIT_FLOOR = -60.0
MORM2_LABEL_BLOCKS = 16 if RUN_MODE == "full" else 4   # blocks for WIK labels

# ------------------------------ persistence ---------------------------------
DRIVE_DIR = f"/content/drive/MyDrive/PRISM-{RUN_MODE}"
os.makedirs(DRIVE_DIR, exist_ok=True)
EXPORT_DIR = os.path.join(DRIVE_DIR, "PRISM-Qwen3.8-27B-A3B" if RUN_MODE == "full"
                          else "PRISM-demo-A3B")                 # COMPAT artifact
NATIVE_DIR = EXPORT_DIR + "-RESONA-native"                        # NATIVE artifact
CACHE_DIR  = os.path.join(DRIVE_DIR, "cache"); os.makedirs(CACHE_DIR, exist_ok=True)

print(json.dumps({"PRISM_VER": __PRISM_VER__, "RUN_MODE": RUN_MODE,
                  "MODEL_ID": MODEL_ID,
                  "QUALITY_MODE": QUALITY_MODE, "TARGET_ACTIVE": TARGET_ACTIVE,
                  "MAST_S": MAST_S, "LIN_BLOCKS": LIN_BLOCKS,
                  "LIN_TOPK": LIN_TOPK, "CARRIER_FRAC": CARRIER_FRAC,
                  "FUNNEL_RANK": FUNNEL_RANK,
                  "FUNNEL_SHORTLIST": FUNNEL_SHORTLIST,
                  "FFN_CAL_MODE": FFN_CAL_MODE, "FFN_CAL_TOKENS": FFN_CAL_TOKENS,
                  "KL_MODE": KL_MODE, "LAMBDA_EMRD": LAMBDA_EMRD, "LAMBDA_LTD": LAMBDA_LTD,
                  "FUNNEL_USE_RHO": FUNNEL_USE_RHO, "PRECISION_PLAN": PRECISION_PLAN,
                  "ENFORCE_BUDGET": ENFORCE_BUDGET, "MAST_ACCOUNTING": MAST_ACCOUNTING,
                  "STEPS": STEPS, "WARMUP_STEPS": WARMUP_STEPS,
                  "DRIVE_DIR": DRIVE_DIR}, indent=1, default=str))


## The PRISM core — fission math + `PrismMoeMLP`

Theorems 1/2/3 (exact dense identity at init, antisymmetric zero-sum growth, serving parity of the materialized experts) are stated and verified in the docstrings below and the verification suite. Pure PyTorch, self-contained.

The v9 calibration and loss formulas — oracle group energy, importance ranking, validated router, NNLS gains, Partition-KL, EMRD, LTD — sit in this same cell, next to the module they serve.


In [ ]:
# PRISM core — fission math + PrismMoeMLP (pure PyTorch, self-contained)
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as _torch_checkpoint

# ============================================================================
# PRISM CORE — Prototype-Resonant Interleaved-Spectral Mixture
# Canonical math + modules for dense -> param-preserving fine-grained MoE
# ("function-field fission") upcycling. Pure PyTorch, self-contained.
#
# Components
#   1. sinkhorn_balanced                        entropic-OT balanced assignment
#                                              (the router training signal)
#   2. neuron_sketches                         JL co-firing sketches
#                                              (memory-safe fission statistics)
#   3. spectral_seeds / balanced_assign /
#      fission_layer                           spectral-seeded BALANCED fission
#                                              (equal-size experts + shared core)
#   4. zero_sum_gains / zero_sum_penalty       ZSSI completion fields
#                                              (Theorem 2)
#   5. router_prototypes / calibrate_gains     energy-derived router init
#   6. PrismMoeMLP                             uniform / soft / topk modes,
#                                              export materialization
#                                              (Theorems 1 and 3)
#   7. count_params / autotune                 parameter + active accounting
# ============================================================================


def _dm(x: torch.Tensor, ref: torch.Tensor) -> torch.Tensor:
    """Dtype-boundary guard: cast x to ref's dtype (no-op when equal).

    The dense substrate is loaded in bf16 on GPU while the ZSSI completion
    field / router stack stay fp32 (fitting + training precision).  Every
    matmul between the two worlds must pass through this guard — bf16 @ fp32
    raises RuntimeError otherwise.  Activations are upcast into fp32 field
    math; results are cast back into the substrate compute dtype so nothing
    leaks fp32 into the model stream (which would double memory on A100).
    """
    return x if x.dtype == ref.dtype else x.to(ref.dtype)


# ----------------------------------------------------------------------------
# 1. Sinkhorn-OT balanced assignment
# ----------------------------------------------------------------------------
def sinkhorn_balanced(logits: torch.Tensor, n_iter: int = 5, tau: float = 1.0) -> torch.Tensor:
    """Entropic OT between tokens (uniform mass) and experts (uniform capacity).

    Given router logits [T, E], returns a soft plan P [T, E] whose columns are
    (approximately) balanced: sum_t P[t, e] ~= T/E for every e, rows sum to 1.
    Used as the *balanced target* that the top-k router is distilled toward:
    a projection, not a penalty (smoother than Switch-style aux losses).

    Args:
        logits: [T, E] router scores.
        n_iter: Sinkhorn row/col normalization sweeps.
        tau:    temperature applied to logits before OT.
    Returns:
        P: [T, E] balanced soft assignment plan.
    """
    if tau <= 0:
        raise ValueError(f"tau must be > 0, got {tau}")
    logK = logits / tau
    T, E = logK.shape
    f = torch.zeros(T, device=logK.device, dtype=logK.dtype)
    g = torch.zeros(E, device=logK.device, dtype=logK.dtype)
    for _ in range(n_iter):
        g = (math.log(T / E)) - torch.logsumexp(logK + f[:, None], dim=0)  # col marginals
        f = -torch.logsumexp(logK + g[None, :], dim=1)                     # row marginals
    P = torch.exp(logK + f[:, None] + g[None, :])
    return P


# ----------------------------------------------------------------------------
# 2. Neuron co-firing sketches (Johnson-Lindenstrauss)
# ----------------------------------------------------------------------------
def neuron_sketches(A: torch.Tensor, sketch_dim: int = 512,
                    generator: torch.Generator | None = None) -> torch.Tensor:
    """Sketch neuron activation patterns so inner products of sketch rows
    approximate token-averaged co-firing correlations between neurons.

    Given A [T, D] (tokens x neurons), returns S [D, s] with
        S_i . S_j  ~=  corr(a_i, a_j)
    Memory: O(D * s) instead of O(D^2) / O(T * D).

    Args:
        A: [T, D] activation matrix (centered internally).
        sketch_dim: JL projection dimension (>= 256 recommended).
        generator: optional torch.Generator for reproducibility.
    Returns:
        S: [D, sketch_dim] unit-norm sketch rows.
    """
    if A.dim() != 2:
        raise ValueError(f"A must be 2-D [T, D], got shape {tuple(A.shape)}")
    Ac = A - A.mean(dim=0, keepdim=True)
    sd = Ac.std(dim=0, keepdim=True).clamp_min(1e-6)
    Az = Ac / sd
    T = A.shape[0]
    Phi = torch.randn(T, sketch_dim, generator=generator, device=A.device, dtype=A.dtype)
    Phi = Phi / math.sqrt(sketch_dim)
    S = Az.t() @ Phi
    S = S / S.norm(dim=1, keepdim=True).clamp_min(1e-8)
    return S


# ----------------------------------------------------------------------------
# 3. Spectral-seeded BALANCED fission
# ----------------------------------------------------------------------------
def _kmeans(X: torch.Tensor, k: int, iters: int = 25, seed: int = 0) -> torch.Tensor:
    """Plain k-means (torch) returning centroids [k, d]. X: [N, d]."""
    g = torch.Generator(device="cpu").manual_seed(seed)
    idx = torch.randperm(X.shape[0], generator=g)[:k]
    C = X[idx.to(X.device)].clone()
    for _ in range(iters):
        a = torch.cdist(X, C).argmin(dim=1)
        for j in range(k):
            m = a == j
            if m.any():
                C[j] = X[m].mean(dim=0)
    return C


def spectral_seeds(S: torch.Tensor, E: int, subsample: int = 2048, seed: int = 0) -> torch.Tensor:
    """Spectral seeding: eigendecompose the co-firing affinity of a neuron
    subsample, embed in the top modes, k-means -> E prototype seeds in sketch
    space.

    Args:
        S: [D, s] unit-norm neuron sketches.
        E: number of routed experts.
        subsample: neurons used for the spectral stage.
        seed: RNG seed.
    Returns:
        seeds: [E, s] seed rows in sketch space.
    """
    D = S.shape[0]
    g = torch.Generator(device="cpu").manual_seed(seed)
    pick = torch.randperm(D, generator=g)[: min(subsample, D)].to(S.device)
    Ssub = S[pick]
    Aff = Ssub @ Ssub.t()                                        # co-firing affinity
    deg = Aff.sum(dim=1).clamp_min(1e-8)
    L = Aff / deg.sqrt()[:, None] / deg.sqrt()[None, :]          # normalized
    m = min(E, 16)
    evals, evecs = torch.linalg.eigh(L)                          # ascending
    emb = evecs[:, -m:] * evals[-m:].clamp_min(0).sqrt()
    emb = emb / emb.norm(dim=1, keepdim=True).clamp_min(1e-8)
    C_emb = _kmeans(emb, E, seed=seed)
    nearest = torch.cdist(emb, C_emb).argmin(dim=0)              # [E]
    return Ssub[nearest]


def balanced_assign(S: torch.Tensor, seeds: torch.Tensor, iters: int = 40) -> torch.Tensor:
    """BALANCED assignment of neurons to experts (equal cluster sizes),
    by margin-based repair of over-full clusters. Deterministic.

    Args:
        S: [D, s] unit-norm neuron sketches.
        seeds: [E, s] seeds.
        iters: max repair iterations.
    Returns:
        assign: [D] expert ids with sizes equal (max - min <= 1).
    """
    D, E = S.shape[0], seeds.shape[0]
    base = D // E
    rem = D - base * E
    targets = torch.full((E,), base, device=S.device)
    targets[:rem] += 1
    sim = S @ seeds.t()                                          # [D, E]
    assign = sim.argmax(dim=1)
    for _ in range(iters):
        sizes = torch.bincount(assign, minlength=E)
        diff = sizes - targets
        if int(diff.max()) <= 0:
            break
        j = int(diff.argmax())                                   # most over-full
        margin = sim.topk(2, dim=1).values
        slack = margin[:, 0] - margin[:, 1]                      # confidence
        n_move = int(diff[j])
        in_j = assign == j
        cand = torch.where(in_j, slack, torch.full_like(slack, float("inf")))
        movers = cand.topk(n_move, largest=False).indices        # least confident
        # destinations: under-full clusters, best sim for each mover
        room = (targets - sizes).clamp_min(0)
        room[j] = 0
        for mi in movers.tolist():
            sims_m = sim[mi].clone()
            sims_m[room <= 0] = float("-inf")
            dest = int(sims_m.argmax())
            assign[mi] = dest
            sizes[dest] += 1
            room[dest] -= 1
    return assign


def fission_layer(duty: torch.Tensor, S: torch.Tensor, E: int, shared_size: int,
                  seed: int = 0, min_expert_size: int = 64) -> dict:
    """Fission one FFN's intermediate dimension:
      1. shared core  = `shared_size` highest-importance neurons (always-on),
      2. routed experts = spectral-seeded BALANCED clustering of the rest.

    Args:
        duty: [D] neuron importance used to rank the always-on shared core.
              v9 passes E[a^2]*||W_down[:, r]||^2 (output energy, ffn_importance); v8 passed
              the SIGNED mean activation (blind to magnitude, cancels for sign-flipping neurons).
        S: [D, s] unit-norm co-firing sketches (may include all neurons).
        E: number of routed experts.
        shared_size: neurons for the always-on shared expert.
        seed: RNG seed.
        min_expert_size: sanity floor on neurons per routed expert.
    Returns:
        dict(assign [D] long: -1 = shared, 0..E-1 = routed expert,
             duty [D], shared_idx [<=shared_size])
    """
    D = S.shape[0]
    if E <= 0 or D - shared_size < E * min_expert_size:
        raise ValueError("Need (D - shared_size) >= E*min_expert_size neurons")
    shared_order = duty.topk(shared_size).indices
    shared_mask = torch.zeros(D, dtype=torch.bool, device=S.device)
    shared_mask[shared_order] = True
    Sr = S[~shared_mask]
    seeds = spectral_seeds(Sr, E, seed=seed)
    assign_r = balanced_assign(Sr, seeds)
    assign = torch.full((D,), -1, dtype=torch.long, device=S.device)
    assign[~shared_mask] = assign_r
    return {"assign": assign, "duty": duty, "shared_idx": shared_order}


# ----------------------------------------------------------------------------
# 4. Zero-Sum Superposition Init (ZSSI)
# ----------------------------------------------------------------------------
def zero_sum_gains(E: int, R: int, scale: float, seed: int = 0) -> torch.Tensor:
    """Zero-column-sum gain matrix G [E, R] used to warm-start or regularize
    the ZSSI completion field: the *mean expert* carries no completion, and
    capacity grows as antisymmetric deviations around the dense function.

    NOTE (honest math): because experts own disjoint neuron rows, zero column
    sums do NOT make the scattered field vanish row-wise; exactness at init is
    guaranteed instead by initializing gains to exactly 0 (see PrismMoeMLP).
    Zero-sum structure constrains GROWTH during training (Theorem 2).
    """
    g = torch.Generator().manual_seed(seed)
    G = torch.randn(E, R, generator=g) * scale
    G = G - G.mean(dim=0, keepdim=True)
    return G


def zero_sum_penalty(gains: torch.Tensor) -> torch.Tensor:
    """Differentiable penalty enforcing zero column sums on [E, R] gains
    (Theorem 2: antisymmetric capacity growth around the dense function)."""
    return gains.mean(dim=0).pow(2).sum()


# ----------------------------------------------------------------------------
# 5. Prototype-Resonant router init + calibrated gains
# ----------------------------------------------------------------------------
def router_prototypes(H: torch.Tensor, energy: torch.Tensor) -> torch.Tensor:
    """Functional prototypes: energy-weighted centroids of layer inputs.

    Args:
        H: [T, d] pre-FFN hidden states (subsample of calibration tokens).
        energy: [T, E] per-token activation energy owned by each expert.
    Returns:
        W_r: [E, d] unit-norm router rows (prototype directions).
    """
    tot = energy.sum(dim=0).clamp_min(1e-8)
    W_r = (energy.t() @ H) / tot[:, None]
    return W_r / W_r.norm(dim=1, keepdim=True).clamp_min(1e-8)


def calibrate_gains(expert_outputs: torch.Tensor, y_dense: torch.Tensor,
                    topk_idx: torch.Tensor, steps: int = 300, lr: float = 0.05,
                    init: torch.Tensor | None = None) -> torch.Tensor:
    """Fit per-expert scalar gains c >= 0 so the top-k serving mixture best
    reconstructs the dense output at init (projected SGD):

        min_c  sum_t || sum_{i} c_{e_i(t)} y_{e_i(t)}(t) - y_dense(t) ||^2

    Args:
        expert_outputs: [T, E, d] all expert outputs on the calibration tokens.
        y_dense: [T, d] dense targets.
        topk_idx: [T, k] serving top-k selections.
        steps, lr: optimizer settings.
        init: optional [E] starting gains.
    Returns:
        c: [E] nonnegative gains (fused multiplicatively into down_proj).
    """
    device = y_dense.device
    E = expert_outputs.shape[1]
    sel = torch.gather(expert_outputs, 1,
                       topk_idx[:, :, None].expand(-1, -1, expert_outputs.shape[-1]))
    c0 = torch.ones(E, device=device) if init is None else init.clone()
    c = torch.nn.Parameter(c0)
    opt = torch.optim.Adam([c], lr=lr)
    for _ in range(steps):
        y_hat = (sel * c[topk_idx][:, :, None]).sum(dim=1)
        loss = F.mse_loss(y_hat, y_dense.to(y_hat.dtype))   # dtype-boundary guard
        opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            c.clamp_(min=0.0)
    return c.detach()


# ----------------------------------------------------------------------------
# 5b. v9 calibration math — oracle energies, serving-consistent NNLS gains,
#     validated routers, ONE call per fissioned layer
# ----------------------------------------------------------------------------
import numpy as _np


def oracle_group_energy(act: torch.Tensor, dn2: torch.Tensor,
                        row_assign: torch.Tensor, n_groups: int) -> torch.Tensor:
    """Per-token OUTPUT energy owned by every neuron group.

    act [T, D] = silu(g) * u;  dn2 [D] = ||W_down[:, r]||^2;  row_assign [D] long
    in [0, n_groups] (n_groups marks the SHARED rows).  Returns [T, n_groups+1]:
        e[t, g] = sum_{r in g} act[t, r]^2 * dn2[r]
    the Gram-diagonal energy of group g's contribution to y = sum_r act_r W_down[:, r]
    (exact for ranking when the rank-1 terms are incoherent). One elementwise
    pass + one index_add; v8 used the activation energy WITHOUT the ||W_down||^2
    factor, which ranks a neuron by its pre-projection size, not by its effect."""
    a2 = act.float().pow(2) * dn2.float()[None, :]
    return a2.new_zeros(a2.shape[0], n_groups + 1).index_add_(1, row_assign, a2)


def ffn_importance(sq_mean: torch.Tensor, down_w: torch.Tensor) -> torch.Tensor:
    """[D] output-energy importance of every intermediate neuron:
    E[a_r^2] * ||W_down[:, r]||^2 (down_w is the HF [d, D] weight). v8 ranked the
    always-on shared core by the SIGNED mean activation: blind to magnitude and
    cancelling for neurons whose sign flips (a = silu(g)*u is signed)."""
    return sq_mean.float() * down_w.float().pow(2).sum(0)


def nnls_normal(A, b, max_iter=None):
    """min_{c >= 0}  0.5 c'Ac - b'c  for SPD A — Lawson-Hanson active set on the
    normal equations (finite termination; a variable dropped when it hits zero
    can re-enter, unlike the 'clip negatives and refit' shortcut)."""
    A = _np.asarray(A, dtype=_np.float64)
    b = _np.asarray(b, dtype=_np.float64)
    n = b.shape[0]
    c = _np.zeros(n)
    P = _np.zeros(n, dtype=bool)
    tol = 1e-12 * max(float(_np.abs(b).max(initial=0.0)),
                      float(_np.abs(_np.diag(A)).max(initial=0.0)), 1e-300)
    max_iter = max_iter or 60 * n + 60
    it = 0
    w = b - A @ c
    while it < max_iter:
        cand = (~P) & (w > tol)
        if not cand.any():
            break
        P[int(_np.argmax(_np.where(cand, w, -_np.inf)))] = True
        while it < max_iter:
            it += 1
            idx = _np.flatnonzero(P)
            if idx.size == 0:
                c = _np.zeros(n)
                break
            s = _np.zeros(n)
            s[idx] = _np.linalg.solve(A[_np.ix_(idx, idx)], b[idx])
            if s[idx].min() > 0.0:
                c = s
                break
            neg = P & (s <= 0.0)
            a = float(_np.min(c[neg] / (c[neg] - s[neg])))
            c = c + a * (s - c)
            P &= c > 1e-14 * max(1.0, float(c.max()))
            c[~P] = 0.0
        w = b - A @ c
    return c


def gate_select(logits: torch.Tensor, k: int, tau: float):
    """The serving gate: softmax(logits / tau) -> top-k -> renormalise (exactly the
    topk branch of PrismMoeMLP and of the stock Qwen3_5Moe router)."""
    p = torch.softmax(logits / tau, dim=-1)
    topv, topi = p.topk(k, dim=-1)
    return topi, topv / topv.sum(-1, keepdim=True).clamp_min(1e-30)


def gain_stats(Ysel: torch.Tensor, g: torch.Tensor, idx: torch.Tensor,
               target: torch.Tensor, n_experts: int, acc=None):
    """Accumulate the sufficient statistics (A, b, tt) of the SERVING objective
        f(c) = sum_t || target_t - sum_i g[t,i] c[idx[t,i]] Ysel[t,i] ||^2
             = tt - 2 b.c + c'Ac
    Ysel [t,k,d] selected expert outputs, g [t,k] gate weights as served,
    idx [t,k] expert ids, target [t,d] = y_dense - y_shared (the residual the
    routed experts must supply: v8 fitted the routed output to the FULL dense
    output, double-counting the shared core). Chunk-additive, so any number of
    calibration tokens fits in O(E^2) memory. acc = (A_flat, b, tt) to add into."""
    E = n_experts
    Z = Ysel.float() * g.float()[:, :, None]                          # [t,k,d]
    G = torch.einsum("tid,tjd->tij", Z, Z).double()                   # [t,k,k]
    r = torch.einsum("tid,td->ti", Z, target.float()).double()        # [t,k]
    if acc is None:
        acc = (torch.zeros(E * E, dtype=torch.float64, device=Ysel.device),
               torch.zeros(E, dtype=torch.float64, device=Ysel.device), 0.0)
    A, b, tt = acc
    pair = (idx[:, :, None] * E + idx[:, None, :]).reshape(-1)
    A = A.index_add(0, pair, G.reshape(-1))
    b = b.index_add(0, idx.reshape(-1), r.reshape(-1))
    return A, b, tt + float(target.float().pow(2).sum())


def solve_gains_nnls(A, b, c0, ridge_rel: float = 1e-3):
    """c = argmin_{c >= 0}  c'Ac - 2 b.c + lam ||c - c0||^2  (numpy float64).
    The ridge toward the prior c0 keeps experts that were (almost) never
    selected on the calibration set at a sane value instead of 0 or infinity."""
    A = _np.asarray(A, dtype=_np.float64)
    b = _np.asarray(b, dtype=_np.float64)
    c0 = _np.asarray(c0, dtype=_np.float64)
    n = b.shape[0]
    tr = float(_np.trace(A)) / max(n, 1)
    lam = ridge_rel * tr if tr > 0.0 else 1.0
    return nnls_normal(A + lam * _np.eye(n), b + lam * c0)


def gain_error(A, b, tt, c) -> float:
    """tt - 2 b.c + c'Ac : total squared serving error of gains c."""
    c = _np.asarray(c, dtype=_np.float64)
    return max(float(tt) - 2.0 * float(b @ c) + float(c @ A @ c), 0.0)


def router_capture(logits: torch.Tensor, en: torch.Tensor, k: int) -> float:
    """Energy-weighted share of the oracle group energy captured by the router's
    top-k set (1.0 = the selected experts own all of the routed energy)."""
    idx = logits.topk(k, dim=-1).indices
    return float(en.gather(1, idx).sum() / en.sum().clamp_min(1e-30))


def ridge_router_from_gram(K: torch.Tensor, X: torch.Tensor, Y: torch.Tensor,
                           lam_rel: float) -> torch.Tensor:
    """Bias-free dual-form ridge: W = X^T (K + lam I)^-1 Y, returned as the
    router weight [E, d] (a plain stock router has no bias, so none is fitted).
    K = X X^T [T, T]; lam is relative to the mean diagonal of K."""
    T = K.shape[0]
    lam = lam_rel * float(K.diagonal().mean())
    alpha = torch.linalg.solve(K + lam * torch.eye(T, device=K.device, dtype=K.dtype), Y)
    return (X.t() @ alpha).t().contiguous()


@torch.no_grad()
def calibrate_ffn_layer(prism, hs: torch.Tensor, k: int, router_mode: str = "auto",
                        gains_mode: str = "nnls", val_frac: float = 0.25,
                        chunk: int = 256, lam_grid=(1e-3, 1e-2, 1e-1, 1.0, 10.0),
                        tau_mult=(0.5, 1.0, 2.0), ridge_rel: float = 1e-3,
                        seed: int = 0) -> dict:
    """Training-free calibration of ONE fissioned FFN (router, temperature, gains).

    ROUTER  Candidates: the v8 prototype router (energy-weighted centroids) and
            bias-free ridge regressions of log(energy share) on the layer input
            (a small lambda grid). The winner is the candidate whose top-k set
            captures the most ORACLE energy on HELD-OUT tokens — selection is
            validated, never assumed — and is refit on all tokens.
    GAINS   Closed-form non-negative least squares of the SERVED function
            y = y_shared + sum_i g_i c_i y_i  against y_dense (gate weights g,
            temperature clamp and shared core exactly as served), instead of
            Adam on a routed-only MSE. For k > 1 the temperature is chosen from
            a small grid by held-out serving error.
    hs [T, d]: calibration inputs, stratified over documents (any float dtype).
    Returns a report dict; writes prism.router.weight / log_tau / gains."""
    dev = prism.router.weight.device
    E, d = prism.E, prism.d_model
    X = hs.to(dev).float()
    T = X.shape[0]
    perm = torch.randperm(T, generator=torch.Generator().manual_seed(1000 + seed)).to(dev)
    n_val = min(max(1, int(round(val_frac * T))), T - 1)
    va, fi = perm[:n_val], perm[n_val:]
    # ---- oracle group energies (completion-free: fresh fields are exactly 0) ----
    en = torch.empty(T, E, device=dev)
    for i in range(0, T, chunk):
        hc = X[i:i + chunk]
        gm = _dm(hc, prism.W_gate) @ prism.W_gate
        um = _dm(hc, prism.W_up) @ prism.W_up
        act = F.silu(gm.float()) * um.float()
        en[i:i + chunk] = oracle_group_energy(act, prism._dn2, prism._row_assign, E)[:, :E]
        del gm, um, act
    en_fit, en_val, X_fit, X_val = en[fi], en[va], X[fi], X[va]
    # ---- router candidates, validated on held-out tokens ----
    cands = {}
    if router_mode in ("auto", "prototype"):
        cands["prototype"] = router_prototypes(X_fit, en_fit)
    if router_mode in ("auto", "ridge"):
        share = en_fit / en_fit.sum(-1, keepdim=True).clamp_min(1e-30)
        Y_fit, K_fit = torch.log(share + 1e-4), X_fit @ X_fit.t()
        for lam in lam_grid:
            cands[f"ridge{lam:g}"] = ridge_router_from_gram(K_fit, X_fit, Y_fit, lam)
        del K_fit
    cap = {n: router_capture(X_val @ W.t(), en_val, k) for n, W in cands.items()}
    best = max(cap, key=cap.get)
    if "prototype" in cap and cap["prototype"] >= cap[best] - 1e-4:
        best = "prototype"                       # ties -> the parameter-free v8 router
    if best == "prototype":
        W = router_prototypes(X, en)
        lg = X @ W.t()
        tau0 = float((lg.std(dim=-1).mean() / max(0.5 * math.log(E), 1e-3)).clamp(0.25, 4.0))
    else:
        share = en / en.sum(-1, keepdim=True).clamp_min(1e-30)
        W = ridge_router_from_gram(X @ X.t(), X, torch.log(share + 1e-4), float(best[5:]))
        tau0 = 1.0                               # logits are already log-shares
    prism.router.weight.copy_(W.to(prism.router.weight.dtype))
    logit_all = X @ W.t()                                              # [T, E] fp32
    topi_all = logit_all.topk(k, dim=-1).indices
    load = torch.bincount(topi_all.reshape(-1), minlength=E).float()
    rep = {"router": best, "capture": cap[best], "capture_proto": cap.get("prototype"),
           "capture_all": router_capture(logit_all, en, k),
           "oracle": float(en.topk(k, dim=-1).values.sum() / en.sum().clamp_min(1e-30)),
           "load_cv": float(load.std() / load.mean().clamp_min(1e-9)), "tau": tau0,
           "rel_err": None, "rel_err_prior": None, "rel_err_all": None}
    if gains_mode != "nnls":
        prism.log_tau.fill_(math.log(min(max(tau0, 0.25), 4.0)))
        return rep
    # ---- serving-consistent NNLS gains (one pass; every candidate tau shares it) ----
    if k > 1:
        taus = sorted({round(min(max(tau0 * m, 0.25), 4.0), 6) for m in tau_mult})
    else:
        taus = [round(min(max(tau0, 0.25), 4.0), 6)]                  # k=1: gate == 1 for any tau
    is_val = torch.zeros(T, dtype=torch.bool, device=dev)
    is_val[va] = True
    stats = {(t_, s_): None for t_ in taus for s_ in ("fit", "val")}
    for i in range(0, T, chunk):
        hc = X[i:i + chunk]
        Ye = prism.expert_outputs(hc, use_completion=False)           # [t, E, d]
        tgt = prism.dense_forward(hc).float() - prism.shared_forward(hc, use_completion=False).float()
        vm = is_val[i:i + chunk]
        for tau in taus:
            topi, g = gate_select(logit_all[i:i + chunk], k, tau)
            Ysel = Ye.gather(1, topi[:, :, None].expand(-1, -1, Ye.shape[-1]))
            for split, msk in (("fit", ~vm), ("val", vm)):
                if bool(msk.any()):
                    stats[(tau, split)] = gain_stats(Ysel[msk], g[msk], topi[msk], tgt[msk], E,
                                                     stats[(tau, split)])
        del Ye, tgt
    def _np_stats(st):
        return (st[0].cpu().numpy().reshape(E, E), st[1].cpu().numpy(), float(st[2]))
    c0 = _np.full(E, float(k))                    # prior: every selected expert enters at weight 1
    best_tau, best_err = taus[0], None
    for tau in taus:
        Af, bf, ttf = _np_stats(stats[(tau, "fit")])
        Av, bv, ttv = _np_stats(stats[(tau, "val")]) if stats[(tau, "val")] is not None \
            else (Af, bf, ttf)
        c_fit = solve_gains_nnls(Af, bf, c0, ridge_rel)
        err = gain_error(Av, bv, ttv, c_fit) / max(ttv, 1e-30)
        if best_err is None or err < best_err - 1e-9:
            best_tau, best_err = tau, err
            rep["rel_err_prior"] = gain_error(Av, bv, ttv, c0) / max(ttv, 1e-30)
    Af, bf, ttf = _np_stats(stats[(best_tau, "fit")])
    if stats[(best_tau, "val")] is not None:
        Av, bv, ttv = _np_stats(stats[(best_tau, "val")])
        Af, bf, ttf = Af + Av, bf + bv, ttf + ttv
    c = solve_gains_nnls(Af, bf, c0, ridge_rel)                       # refit on ALL tokens
    prism.gains.copy_(torch.as_tensor(c, dtype=prism.gains.dtype, device=prism.gains.device))
    prism.log_tau.fill_(math.log(best_tau))
    # squared error of the SERVED routed part on ALL calibration tokens, relative to ||y_dense - y_shared||^2:
    # equals ||y_served - y_dense||^2 / ||y_dense - y_shared||^2 of the module itself (checked by the v9 contracts)
    rep.update(tau=best_tau, rel_err=best_err,
               rel_err_all=gain_error(Af, bf, ttf, c) / max(ttf, 1e-30))
    return rep


# ----------------------------------------------------------------------------
# 5d. FFN shadow (v10): an affine low-rank model of what the served FFN leaves out
# ----------------------------------------------------------------------------
@torch.no_grad()
def fit_ffn_shadow(prism, hs: torch.Tensor, k: int, rank: int, val_frac: float = 0.25,
                   chunk: int = 256, lam_grid=(1e-3, 1e-2, 1e-1, 1.0), seed: int = 0) -> dict:
    """r(h) = y_dense(h) - y_served(h)  ~  b + U (M h)      (reduced-rank kernel ridge, validated on held-out tokens)

    The served FFN evaluates ~6% of the neurons; the rest of the sum is not zero on average and is partly a
    smooth function of h. Its best affine rank-r predictor is fitted in closed form (dual ridge: one eigh of the
    [n, n] Gram matrix serves the whole lambda grid) and kept ONLY if it lowers the held-out error.
    Writes prism.sh_U [d, r], prism.sh_M [r, d], prism.sh_b [d]; returns a report."""
    dev = prism.router.weight.device
    T, d = hs.shape
    X = hs.to(dev).float()
    R = torch.empty(T, d, device=dev)
    was = getattr(prism, "use_shadow", True)
    prism.use_shadow = False
    for i in range(0, T, chunk):
        hc = hs[i:i + chunk].to(dev, dtype=prism.W_gate.dtype)
        y_s, _ = prism._forward_2d(hc, "topk", k, True, False)
        R[i:i + chunk] = prism.dense_forward(hc).float() - y_s.float()
    prism.use_shadow = was
    prism.last_aux = None
    perm = torch.randperm(T, generator=torch.Generator().manual_seed(2000 + seed)).to(dev)
    n_val = min(max(1, int(round(val_frac * T))), T - 1)
    va, fi = perm[:n_val], perm[n_val:]
    xm, rm = X[fi].mean(0), R[fi].mean(0)
    Xc, Rc = X[fi] - xm, R[fi] - rm
    Xv, Rv = X[va] - xm, R[va]
    n = Xc.shape[0]
    K = Xc @ Xc.t()
    evals, Q = torch.linalg.eigh(K)
    evals = evals.clamp_min(0)
    scale = float(evals.mean().clamp_min(1e-12))
    QtR = Q.t() @ Rc                                              # [n, d]
    KvQ = (Xv @ Xc.t()) @ Q                                       # [n_val, n]
    base = float(Rv.pow(2).sum().clamp_min(1e-30))
    err_bias = float((Rv - rm).pow(2).sum()) / base
    best_lam, best_err = None, None
    for lam in lam_grid:
        pv = rm + KvQ @ (QtR / (evals + lam * scale)[:, None])
        e = float((Rv - pv).pow(2).sum()) / base
        if best_err is None or e < best_err:
            best_lam, best_err = lam, e
    rep = {"rel_before": 1.0, "rel_bias_only": err_bias, "rel_fullrank": best_err, "lam": best_lam, "rank": 0,
           "rel_after": 1.0, "resid_over_dense": None}
    G = Q @ (QtR / (evals + best_lam * scale)[:, None])           # [n, d]:  pred(x) = rm + (x - xm) Xc^T G
    P_fit = K @ G                                                 # fitted predictions [n, d]
    r = int(max(0, min(int(rank), n, d)))
    U = M = None
    err_lr = None
    if r > 0:
        ev2, V = torch.linalg.eigh(P_fit.t() @ P_fit)             # output directions of the fitted map
        U = V[:, -r:].contiguous()                                # [d, r] top-r
        M = (G @ U).t() @ Xc                                      # [r, d]
        pv = rm + (Xv @ M.t()) @ U.t()
        err_lr = float((Rv - pv).pow(2).sum()) / base
    # keep the best of {nothing, bias only, bias + low-rank map} by held-out error
    choice, err = "none", 1.0
    if err_bias < err - 0.005:
        choice, err = "bias", err_bias
    if err_lr is not None and err_lr < err - 0.005:
        choice, err = "lowrank", err_lr
    dt = prism.W_gate.dtype
    if choice == "lowrank":
        prism.sh_U = U.to(dt)
        prism.sh_M = M.to(dt)
        prism.sh_b = (rm - (xm @ M.t()) @ U.t()).to(dt)
        rep["rank"] = r
    elif choice == "bias":
        prism.sh_U = torch.zeros(d, 1, device=dev, dtype=dt)
        prism.sh_M = torch.zeros(1, d, device=dev, dtype=dt)
        prism.sh_b = rm.to(dt)
        rep["rank"] = 0
    rep.update(choice=choice, rel_after=err)
    return rep


# ----------------------------------------------------------------------------
# 5c. Partition-KL — the distillation objective on {teacher top-K} + {rest}
# ----------------------------------------------------------------------------
def log1mexp(x: torch.Tensor) -> torch.Tensor:
    """log(1 - exp(x)) for x < 0, stable on both sides of -ln 2."""
    x = x.clamp(max=-1e-6)
    return torch.where(x > -0.6931471805599453,
                       torch.log(-torch.expm1(x)), torch.log1p(-torch.exp(x)))


def kl_support(t_lp: torch.Tensor, s_lp: torch.Tensor) -> torch.Tensor:
    """v8: KL of the two distributions RENORMALISED inside the teacher's top-K
    support (blind to how much probability mass the student puts elsewhere)."""
    tn = t_lp - t_lp.logsumexp(-1, keepdim=True)
    sn = s_lp - s_lp.logsumexp(-1, keepdim=True)
    return (tn.exp() * (tn - sn)).sum(-1)


def partition_kl(t_lp: torch.Tensor, s_lp: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """KL(P || Q) of the COARSE-GRAINED distributions on the partition
    {each of the teacher's top-K ids} U {everything else}, per position.

    t_lp / s_lp [..., K]: FULL-vocabulary log-probs of teacher / student at the
    teacher's top-K ids. By the chain rule of KL
        KL_part = P_S * KL_support(p_hat || q_hat)  +  KL_Bernoulli(P_S || Q_S)
    (P_S, Q_S = probability mass inside the support): the in-support term is
    weighted by how much of the teacher's mass lives there, and the support mass
    itself is matched — which v8's renormalised KL could not see. By data
    processing it lower-bounds the full-vocabulary KL and needs no more than
    the cached top-K."""
    t_lp = t_lp.float()
    s_lp = s_lp.float()
    p = t_lp.exp()
    t_mass = p.sum(-1).clamp(max=1.0 - eps)          # fp16 cache rounding can overshoot 1
    ls = s_lp.logsumexp(-1).clamp(max=-eps)          # log student mass inside the support
    t_rest = 1.0 - t_mass
    in_support = (p * (t_lp - s_lp)).sum(-1)
    return in_support + t_rest * (t_rest.log() - log1mexp(ls))


def emrd_loss(p: torch.Tensor, en: torch.Tensor, k: int) -> torch.Tensor:
    """Energy-Mass Router Distillation (EMRD).

    p  [T, E]  router probabilities (soft or the full softmax of the top-k path)
    en [T, E]  ORACLE output energy owned by every routed expert on this token
               (oracle_group_energy of the dense activations, no grad)
    With share_e = en_e / sum(en) and the saturating weight w_e = min(1, k * share_e)
    ("expert e is worth selecting"), minimise   -log sum_e p_e * w_e.
    Why a new term: with k = 1 the serving gate renormalises to g = 1, so the
    output loss sends NO gradient to the router's choice (only to the gains of the
    chosen expert). EMRD supplies that missing signal directly from the oracle,
    is scale-free, flat where the oracle has no preference (diffuse energy) and
    never saturates (log of the mass on the good set)."""
    share = en.float() / en.float().sum(-1, keepdim=True).clamp_min(1e-30)
    w = (float(k) * share).clamp(max=1.0)
    cap = (p.float() * w).sum(-1).clamp_min(1e-9)
    return -cap.log().mean()


def ltd_loss(y: torch.Tensor, y_dense: torch.Tensor) -> torch.Tensor:
    """Layer-local dense-twin loss: ||y - y_d||^2 / ||y_d||^2 (ratio of sums over the
    micro-batch; the dense twin y_d is the same layer's own dense FFN on the SAME
    input, so every layer gets a dense, well-conditioned local target instead of
    relying on the final-logit KL alone to reach 64 layers)."""
    yd = y_dense.detach().float()
    return (y.float() - yd).pow(2).sum() / yd.pow(2).sum().clamp_min(1e-12)


# ----------------------------------------------------------------------------
# 6. PrismMoeMLP — training-time module (fragment view of dense SwiGLU FFN)
# ----------------------------------------------------------------------------
class PrismMoeMLP(nn.Module):
    """Function-field fission of a dense SwiGLU FFN layer.

    The dense weights stay resident ONCE (frozen). Experts are equal-size
    index-views over the intermediate dimension; ZSSI completions are
    zero-sum low-rank fields on the experts' gate/up projections (exactly
    fusable into standard expert Linear weights); a shared expert holds the
    highest duty-cycle neurons; per-expert scalar gains c_e are fused into
    down_proj at export.

    Theorems (verified in the suite):
      Thm 1 (exact dense identity): with completion gains alpha=beta=0 the
          union of fragments partitions [D] exactly, so
          sum_e E_e(h) + E_0(h) == f(h) for every h.
      Thm 3 (serving parity): materialize_expert(s) fuse fragment + ZSSI
          completion + gain into standard Linear weights that reproduce the
          expert exactly (the bridge into the stock Qwen3_5Moe format).

    forward modes
      'uniform' : y = dense FFN(h)                  (Theorem 1 reference path)
      'soft'    : y = shared + sum_e p_e y_e        (all-fragments, grouped bmm)
      'topk'    : hard top-k sparse path (mask-based, serving-parity semantics)
    """

    def __init__(self, gate: torch.Tensor, up: torch.Tensor, down: torch.Tensor,
                 assign: torch.Tensor, E: int, rank: int = 64, seed: int = 0,
                 k_topk: int | None = None):
        """gate/up: [d, D], down: [d_out, D] (d_out == d for Qwen FFN),
        assign: [D] long (-1 = shared, else 0..E-1).

        NOTE: gate/up/down are NOT cloned — if Parameters are passed, they are
        ADOPTED (zero-copy surgery on a loaded dense model). Set
        requires_grad=False on them to keep the substrate frozen.
        """
        super().__init__()
        self.k_topk = k_topk                       # serving top-k (None -> caller passes k)
        assert gate.shape == up.shape and down.shape[1] == gate.shape[1]
        d, D = gate.shape
        for nm, t in (("W_gate", gate), ("W_up", up), ("W_down", down)):
            if not isinstance(t, nn.Parameter):
                t = nn.Parameter(t, requires_grad=False)
            else:
                t.requires_grad_(False)
            self.register_parameter(nm, t)
        self.register_buffer("assign", assign.clone())
        self.d_model, self.D, self.E = d, D, E
        shared_mask = assign < 0
        self.register_buffer("shared_mask", shared_mask)
        routed = torch.where(~shared_mask)[0]
        order = routed[torch.argsort(assign[routed])]
        m_e = routed.numel() // E
        if routed.numel() % E != 0:
            raise ValueError("routed neurons not divisible by E; balance the fission")
        self.m_e = m_e
        self.register_buffer("expert_idx", order.reshape(E, m_e))
        self.register_buffer("shared_idx", torch.where(shared_mask)[0])
        # ---- natural-row routing maps (derived; persistent=False -> never
        #      enter state_dict / export) ----
        # _row_assign[r]: expert of row r; E (=out of range) marks SHARED
        #   rows -> indexes the EXACT-ZERO column of the padded w_pad.
        # _row_field[r]: assign clamped to a valid expert id (shared rows
        #   -> 0); their ZSSI correction values are irrelevant because
        #   w_row is EXACTLY zero there (0 * finite = 0).
        self.register_buffer("_row_assign",
                             torch.where(assign < 0,
                                         torch.full_like(assign, E), assign),
                             persistent=False)
        self.register_buffer("_row_field", assign.clamp_min(0),
                             persistent=False)
        # ---- ZSSI completion field (trainable) ----
        # EXACT-INIT GUARANTEE (Theorem 1): gains start at exactly ZERO, so the
        # transformed model at init reproduces the dense function under full
        # (uniform) mixing. Capacity grows from zero during training; the
        # zero_sum_penalty keeps growth antisymmetric around the dense field.
        g = torch.Generator().manual_seed(seed)
        U = torch.randn(d, rank, generator=g) / math.sqrt(d)
        P = torch.randn(D, rank, generator=g) / math.sqrt(D)
        self.rank = rank
        self.U = nn.Parameter(U)                                  # [d, R]
        self.P = nn.Parameter(P)                                  # [D, R]
        self.alpha = nn.Parameter(torch.zeros(E, rank))           # [E, R] gate
        self.beta  = nn.Parameter(torch.zeros(E, rank))           # [E, R] up
        self.alpha_shared = nn.Parameter(torch.zeros(rank))       # shared expert field
        self.beta_shared  = nn.Parameter(torch.zeros(rank))
        self.register_buffer("init_scale", torch.tensor(1.0))     # for warm starts
        # ---- prototype-resonant router ----
        self.router = nn.Linear(d, E, bias=False)
        self.log_tau = nn.Parameter(torch.zeros(1))
        # per-expert output gains (calibrated; fused into down_proj at export)
        self.gains = nn.Parameter(torch.ones(E))
        # deployment path switch: 'soft' (train), 'topk' (serving parity),
        # 'uniform' (exact dense teacher — lives inside the student!)
        self.default_mode = "soft"
        # module-level self-checkpointing switch (probes/calibration can
        # disable it via m.self_ckpt = False)
        self.self_ckpt = True
        # ---- v9: oracle machinery ----
        # ||W_down[:, r]||^2 per intermediate row: the Gram-diagonal weight that turns an
        # activation a_r into its OUTPUT energy a_r^2 ||W_down[:, r]||^2 (derived buffer,
        # never enters the state_dict / export).
        with torch.no_grad():
            _wd = self.W_down.detach()
            _dn2 = torch.cat([_wd[:, i:i + 4096].float().pow(2).sum(0)
                              for i in range(0, D, 4096)])
        self.register_buffer("_dn2", _dn2, persistent=False)
        # side channels (computed under no_grad, detached, returned through aux): the trainer
        # switches them on only for its own forward (see DistillTrainer.compute_loss)
        self.side_en = False       # per-expert oracle energy of the dense activations
        self.side_ltd = False      # dense-twin output + this layer's output (layer-local loss)
        # v10 FFN shadow: served y += b + U (M h)  (fit_ffn_shadow; None = off)
        self.register_buffer("sh_U", None)
        self.register_buffer("sh_M", None)
        self.register_buffer("sh_b", None)
        self.use_shadow = True

    # ---------- completion helpers ----------
    def _corr(self, z: torch.Tensor, gains: torch.Tensor) -> torch.Tensor:
        """Per-expert gate/up corrections from the shared field.

        z: [T, R]; gains: [E, R] (or [R] for shared).
        Returns corr [T, E, m_e] for routed (or [T, m_s] for shared).
        """
        P_e = self.P[self.expert_idx.reshape(-1)].reshape(self.E, self.m_e, self.rank)
        tmp = z[:, None, :] * gains[None, :, :]                   # [T, E, R]
        return torch.einsum("emr,ter->etm", P_e, tmp)

    # ---------- expert paths ----------
    def _fragment_stack(self) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Gather (gate, up, down) fragment weights as [E, m_e, d] stacks."""
        idx = self.expert_idx.reshape(-1)
        Wg = self.W_gate.t()[idx].reshape(self.E, self.m_e, self.d_model)
        Wu = self.W_up.t()[idx].reshape(self.E, self.m_e, self.d_model)
        Wd = self.W_down.t()[idx].reshape(self.E, self.m_e, self.d_model)
        return Wg, Wu, Wd

    def expert_outputs(self, h: torch.Tensor, use_completion: bool = True) -> torch.Tensor:
        """All routed expert outputs y_e on tokens h. Returns [T, E, d].

        Reference implementation: gathers [E, m_e, d] fragment weight copies
        and materializes the all-expert output — fine for no-grad
        calibration, the export parity tests and oracles, but the stacks are
        autograd-SAVED in grad mode (~640 MiB per layer at 27B). The
        training forward paths therefore use _natural_routed, which computes
        the same function with zero materialization.
        """
        Wg, Wu, Wd = self._fragment_stack()
        z = _dm(h, self.U) @ self.U                               # [T, R] fp32 field
        if use_completion:
            ca = self._corr(z, self.alpha)                        # [E, T, m_e]
            cb = self._corr(z, self.beta)
        else:
            ca = cb = 0.0
        gate_p = torch.einsum("td,emd->etm", _dm(h, Wg), Wg) + ca
        up_p = torch.einsum("td,emd->etm", _dm(h, Wu), Wu) + cb
        act = F.silu(gate_p) * up_p                               # [E, T, m_e]
        y = torch.einsum("etm,emd->etd", _dm(act, Wd), Wd)        # [E, T, d]
        return y.transpose(0, 1)                                  # [T, E, d]

    def shared_forward(self, h: torch.Tensor, use_completion: bool = True) -> torch.Tensor:
        s = self.shared_idx
        Wg = self.W_gate[:, s].t()                                # [m_s, d]
        Wu = self.W_up[:, s].t()
        Wd = self.W_down[:, s]                                    # [d, m_s]
        if use_completion and self.rank > 0:
            zs = _dm(h, self.U) @ self.U                          # [T, R] fp32
            corr_a = ((self.P[s] * self.alpha_shared[None, :]) @ zs.t()).t()
            corr_b = ((self.P[s] * self.beta_shared[None, :]) @ zs.t()).t()
            a = F.silu(_dm(h, Wg) @ Wg.t() + corr_a) * (_dm(h, Wu) @ Wu.t() + corr_b)
        else:
            a = F.silu(_dm(h, Wg) @ Wg.t()) * (_dm(h, Wu) @ Wu.t())
        return _dm(a, Wd) @ Wd.t()

    def dense_forward(self, h: torch.Tensor) -> torch.Tensor:
        """Exact dense FFN (Theorem-1 reference path).

        BITWISE-ORIGINAL GUARANTEE (Theorem 7 anchor): W_gate/W_up are
        transposed VIEWS of the adopted HF weights, so W.t() is the original
        HF weight object — calling F.linear on it re-dispatches the EXACT
        original GEMM (same layout, same kernel, same accumulation order).
        A hand-rolled `h @ W_view` matmul is mathematically equal but picks a
        different kernel in bf16 -> ~1e-2 logit drift over many layers.
        """
        a = F.silu(F.linear(_dm(h, self.W_gate), self.W_gate.t())) * \
            F.linear(_dm(h, self.W_up), self.W_up.t())
        return F.linear(_dm(a, self.W_down), self.W_down)

    # ---------- natural-order zero-materialization routed path ----------
    def _natural_routed(self, h: torch.Tensor, z: torch.Tensor | None,
                        w_pad: torch.Tensor, use_completion: bool,
                        side: dict | None = None) -> torch.Tensor:
        """Routed experts for all tokens WITHOUT the [E, T, d] stack.

        Linearity of the expert sum is the escape:
            sum_e w[t,e] * y_e[t]  ==  sum_r act[t,r] * w_row[t,r] * Wd[:,r]
        with r the intermediate rows in NATURAL order and
        w_row[t,r] = w_pad[t, _row_assign[r]] — the padded column E gives
        the shared rows' weight EXACTLY zero. Gate/up run as the ORIGINAL
        full-view GEMMs (the pre-surgery dense kernels), the ZSSI
        corrections as row-aligned einsums, and the down projection as ONE
        (act * w_row) @ W_down.t() GEMM: nothing is copied and [E, T, d]
        never exists. Verified against expert_outputs on values and
        gradients in the suite.
        """
        g = _dm(h, self.W_gate) @ self.W_gate                     # [T, D]
        u = _dm(h, self.W_up) @ self.W_up                         # [T, D]
        if use_completion and self.rank > 0:
            if z is None:
                z = _dm(h, self.U) @ self.U                       # [T, R] fp32
            ca = torch.einsum("mr,tr->tm",
                              self.P * self.alpha[self._row_field], z)
            cb = torch.einsum("mr,tr->tm",
                              self.P * self.beta[self._row_field], z)
            act = F.silu(g + ca) * (u + cb)                       # [T, D]
        else:
            act = F.silu(g) * u
        act_b = _dm(act, self.W_down)                             # substrate dtype
        w_row = w_pad[:, self._row_assign].to(act_b.dtype)        # [T, D]
        if side is not None:
            # v9 side channels: DETACHED values computed under no_grad, so they add nothing to
            # autograd's saved set (checkpoint recompute stays consistent whatever the flags).
            with torch.no_grad():
                a0 = ((F.silu(g.float()) * u.float())             # completion-free = the DENSE neuron activations
                      if (use_completion and self.rank > 0) else act.float())
                if self.side_en:
                    side["en"] = oracle_group_energy(a0, self._dn2, self._row_assign,
                                                     self.E)[:, :self.E]
                if self.side_ltd:
                    side["y_dense"] = _dm(a0, self.W_down) @ self.W_down.t()   # dense twin [T, d]
        return (act_b * w_row) @ self.W_down.t()                  # [T, d]

    # ---------- main forward ----------
    def forward(self, h: torch.Tensor, mode: str | None = None, k: int | None = None,
                use_completion: bool = True, return_aux: bool = False):
        # shape-agnostic entry: decoders call mlp(hidden [B, L, d]); the core
        # math below is [T, d]. Flatten on entry, restore the lead dims on
        # every exit path.
        lead = h.shape[:-1]
        h2 = h.reshape(-1, h.shape[-1]) if len(lead) != 1 else h
        # Module self-checkpointing: bounds the layer's backward footprint
        # to its [T, d] input BY CONSTRUCTION, independent of any
        # Trainer/HF layer checkpointing. use_reentrant=False keeps the
        # side-channel aux (consumed by compute_loss BEFORE backward)
        # fully differentiable and is correct with frozen inputs and
        # trainable parameters (gradients reach params through the
        # recompute).
        if (self.training and torch.is_grad_enabled()
                and getattr(self, "self_ckpt", True)):
            y, aux = _torch_checkpoint(self._forward_2d, h2, mode, k,
                                       use_completion, return_aux,
                                       use_reentrant=False)
        else:
            y, aux = self._forward_2d(h2, mode, k, use_completion, return_aux)
        if len(lead) != 1:
            y = y.reshape(*lead, y.shape[-1])
        return (y, aux) if return_aux else y

    def _forward_2d(self, h: torch.Tensor, mode: str | None = None, k: int = 8,
                    use_completion: bool = True, return_aux: bool = False):
        if mode is None:
            mode = getattr(self, "default_mode", "soft")
        if k is None:
            k = self.k_topk                        # solver-chosen serving top-k
        if k is None:
            raise ValueError("serving top-k not set: pass k= or construct with k_topk=")
        if mode == "uniform":
            y = self.dense_forward(h)
            return ((y, {}) if return_aux else y, {})
        logits = self.router(_dm(h, self.router.weight)) / torch.exp(self.log_tau).clamp(0.25, 4.0)
        aux: dict = {"logits": logits}
        z = _dm(h, self.U) @ self.U if (use_completion and self.rank > 0) else None
        y_shared = self.shared_forward(h, use_completion)
        side = {} if (self.side_en or self.side_ltd) else None    # v9 side channels
        if mode == "soft":
            p = torch.softmax(logits, dim=-1)                     # [T, E]
            w = p * self.gains[None, :]                           # [T, E] fp32
            # padded column E = the shared rows' EXACT-zero weight
            w_pad = torch.cat([w, w.new_zeros(w.shape[0], 1)], dim=1)
            y = y_shared + self._natural_routed(h, z, w_pad, use_completion, side)
            aux["p"] = p
        elif mode == "topk":
            p_full = torch.softmax(logits, dim=-1)
            topv, topi = p_full.topk(k, dim=-1)
            g = topv / topv.sum(dim=-1, keepdim=True)
            w = g * self.gains[topi]                              # [T, k] fp32
            # hard selection as a scattered mask — skipped experts get
            # EXACT zero (same terms as a gather, regrouped)
            w_pad = torch.zeros(h.shape[0], self.E + 1, dtype=w.dtype,
                                device=h.device).scatter(1, topi, w)
            y = y_shared + self._natural_routed(h, z, w_pad, use_completion, side)
            if self.sh_U is not None and getattr(self, "use_shadow", True):
                # v10: what the unselected experts would have added, predicted from h
                hm = _dm(h, self.sh_M)
                y = y + ((hm @ self.sh_M.t()) @ self.sh_U.t() + self.sh_b).to(y.dtype)
            aux["p"] = p_full
            aux["load"] = torch.bincount(topi.reshape(-1), minlength=self.E).float()
        else:
            raise ValueError(f"unknown mode {mode}")
        # Telemetry contract: store aux ONLY on the trainer's forward
        # (via _store_aux) or under no-grad. The checkpoint recomputes
        # that run during backward must never store anything — holding
        # any tensor of a recomputed graph pins that graph's entire
        # saved set, which is how the training OOMs happened.
        y_out = _dm(y, h)         # never leak fp32 field math into the model stream
        if side:
            aux.update(side)      # v9: detached oracle energies / dense twin
            if "y_dense" in side:
                aux["y"] = y_out  # the trainer's LTD term compares THIS output with the dense twin
        if (not torch.is_grad_enabled()) or getattr(self, "_store_aux", False):
            self.last_aux = aux
        return y_out, aux

    # ---------- export: materialize standard expert Linear weights ----------
    def materialize_expert(self, e: int) -> dict:
        """Fuse fragment + ZSSI completion + gain into standard Linear weights.

        Returns dict: gate_proj [m_e, d], up_proj [m_e, d], down_proj [d, m_e].
        Exactly reproduces the expert path (Theorem 3: serving parity, ~2.5e-7).
        """
        idx = self.expert_idx[e]
        Wg = self.W_gate[:, idx].t().clone()
        Wu = self.W_up[:, idx].t().clone()
        P_e = self.P[idx]                                         # [m_e, R]
        A_e = P_e @ torch.diag(self.alpha[e].detach()) @ self.U.t()
        B_e = P_e @ torch.diag(self.beta[e].detach()) @ self.U.t()
        # HF Linear convention: down_proj weight is [d_out, m_e]
        Wd = (self.gains[e].detach() * self.W_down[:, idx]).clone()
        return {"gate_proj": Wg + _dm(A_e, Wg), "up_proj": Wu + _dm(B_e, Wu),
                "down_proj": Wd}

    def materialize_shared(self) -> dict:
        s = self.shared_idx
        Wg = self.W_gate[:, s].t().clone()
        Wu = self.W_up[:, s].t().clone()
        Ps = self.P[s]
        A_s = Ps @ torch.diag(self.alpha_shared.detach()) @ self.U.t()
        B_s = Ps @ torch.diag(self.beta_shared.detach()) @ self.U.t()
        Wd = self.W_down[:, s].clone()  # [d_out, m_s] HF convention
        return {"gate_proj": Wg + _dm(A_s, Wg), "up_proj": Wu + _dm(B_s, Wu),
                "down_proj": Wd}

    def materialize_experts_fused(self, device=None) -> dict:
        """Vectorized export in the REAL Qwen3_5Moe '3D fused' layout:
            gate_up_proj: [E, 2*m_e, d]  (rows 0:m_e = gate, m_e:2m_e = up)
            down_proj:    [E, d_out, m_e]
        (verified against transformers.models.qwen3_5_moe.Qwen3_5MoeExperts,
        which chunks gate_up_proj[e] into (gate, up) and applies
        F.linear(x, gate_up_proj[e]) / F.linear(a, down_proj[e]).)
        Equals stacking materialize_expert(e) over e (Theorem 3, fused form).
        """
        dev = device or self.W_gate.device
        idx = self.expert_idx.reshape(-1)
        E, m_e, d = self.E, self.m_e, self.d_model
        with torch.no_grad():
            P_stack = self.P[idx].reshape(E, m_e, self.rank).to(dev)
            Wg_stack = self.W_gate.t()[idx].reshape(E, m_e, d).to(dev)
            Wu_stack = self.W_up.t()[idx].reshape(E, m_e, d).to(dev)
            gate_stack = Wg_stack + torch.einsum(
                "emr,er,dr->emd", P_stack, self.alpha.detach().to(dev),
                self.U.detach().to(dev)).to(Wg_stack.dtype)
            up_stack = Wu_stack + torch.einsum(
                "emr,er,dr->emd", P_stack, self.beta.detach().to(dev),
                self.U.detach().to(dev)).to(Wu_stack.dtype)
            down_stack = (self.W_down.t()[idx].reshape(E, m_e, -1).to(dev)
                          * self.gains.detach().to(dev).to(Wg_stack.dtype)[:, None, None]
                          ).transpose(1, 2)
        return {"gate_up_proj": torch.cat([gate_stack, up_stack], dim=1).contiguous(),
                "down_proj": down_stack.contiguous()}


# ----------------------------------------------------------------------------
# 7. Param accounting & auto-tuner
# ----------------------------------------------------------------------------
def count_params(d_model: int, D: int, n_layers: int, E: int, k: int,
                 shared_size: int, vocab: int, attn_params: int,
                 rank: int = 64, mtp_params: int = 0, vision_params: int = 0,
                 tie_word_embeddings: bool = False) -> dict:
    """Parameter + active-parameter accounting for PRISM fission.

    Total  = dense-equivalent total + shared low-rank completion fields.
    Active = embeddings (+head) + attention + shared expert + k/E of routed
             FFN + router rows (+ MTP/vision if counted).
    """
    m_e = (D - shared_size) // E
    embed = vocab * d_model * (1 if tie_word_embeddings else 2)
    routed_total = n_layers * E * 3 * d_model * m_e
    shared_total = n_layers * 3 * d_model * shared_size
    field_per_layer = d_model * rank + D * rank + 2 * E * rank + 2 * rank
    total = embed + attn_params + routed_total + shared_total + \
        n_layers * field_per_layer + mtp_params + vision_params
    active = embed + attn_params + n_layers * 3 * d_model * shared_size + \
        n_layers * k * 3 * d_model * m_e + n_layers * E * d_model + \
        mtp_params + vision_params
    return {"total": total, "active": active,
            "ffn_total": routed_total + shared_total,
            "ffn_active": n_layers * (k * 3 * d_model * m_e + 3 * d_model * shared_size),
            "m_e": m_e}


def autotune(d_model: int, D: int, n_layers: int, vocab: int, attn_params: int,
             target_active: float, shared_size: int = 512, rank: int = 64,
             candidates=((256, 8), (128, 8), (128, 4), (64, 8), (64, 4),
                         (64, 2), (32, 4), (32, 2))) -> tuple:
    """Choose (E, k) minimizing active params subject to expert size >= 64;
    reports the architecture floor (attention + embeddings are always on)."""
    rows = []
    for E, k in candidates:
        if (D - shared_size) % E != 0:
            continue          # balanced fission requires exact divisibility
        m_e = (D - shared_size) // E
        if m_e < 64:
            continue
        r = count_params(d_model, D, n_layers, E, k, shared_size, vocab,
                         attn_params, rank)
        rows.append({"E": E, "k": k, **r})
    rows.sort(key=lambda x: x["active"])
    floor = count_params(d_model, D, n_layers, candidates[0][0], 1,
                         shared_size, vocab, attn_params, rank)
    floor["active"] -= n_layers * 3 * d_model * ((D - shared_size) // candidates[0][0])
    return rows[0], rows

PRISM_MODULES: list = []   # (name, PrismMoeMLP) — populated by the surgery cell
print("PRISM core loaded.")


## The RESONA core — resonant linears, MoRM, sectioned projector, head funnel

Theorem 4 (entry-exact row partition), Theorem 5 (per-token tail certificate), the MoRM definition + Proposition 6 (skipped-write certificate), the HeadFunnel definition and Theorem 7 (whole-model exactness) live in the class docstrings. Dependency order: AntennaMast → ResonantLinear → MoRM → SectionedProjector → HeadFunnel → waterfall/solver → `set_world`.

**v9:** `HeadFunnel` uses the exact rank-r projection `U_r = W P_r` and a residual-aware ranking certified by Cauchy–Schwarz (κ fitted on probe tokens); the active-parameter waterfall charges the sketch per (layer, input width).


In [ ]:
# RESONA core — resonant linears, MoRM, sectioned projector, head funnel,
# waterfall + the 3B budget solver
# ============================================================================
# RESONA CORE — the three execution families that break the 10.5B floor
# of FFN-only sparsity, plus the whole-model budget solver:
#
#   8.  AntennaMast        — one global random-feature sketch shared by every
#                            router in the model (one phi(h) per token).
#   9.  ResonantLinear     — row-mode routing of ANY whole Linear: always-on
#                            carrier rows + top-k resonant row-blocks.
#                            Entry-exact partition (Theorem 4): full mode ==
#                            dense exactly; per-token tail certificate (Thm 5).
#   10. MoRM               — Mixture of Recurrent Memories: write-sparse
#                            DeltaNet value-heads (top-m writes per token,
#                            read-dense). Masking v or beta zeroes the rank-1
#                            write exactly -> installs inside the stock GDN
#                            with zero kernel surgery (Prop 6 certificate).
#   11. SectionedProjector — fused DeltaNet in_proj split into contiguous
#                            sections [q|k|v|z]: q/z routed rows, k dense,
#                            v head-routed (MoRM), all zero-copy on the
#                            fused weight (total params preserved).
#   12. HeadFunnel         — Spectral Head Funnel: rank-r randomized-SVD
#                            proxy of lm_head + exact top-V_s shortlist rows.
#   13. active_waterfall / solve_budget_v2 — STRICT touched-weights
#                            accounting + coordinate solver for the budget.
#
# Conventions (STRICT, honest): "active params" = weight elements actually
# READ/EVALUATED to compute one text token's forward pass. Embedding = 1
# row. lm_head = funnel proxy + shortlist rows. Vision tower & MTP draft
# excluded from text-token active (reported separately).
# ============================================================================
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as _torch_checkpoint




# ----------------------------------------------------------------------------
# 8. AntennaMast — one shared random-feature sketch for all routers
# ----------------------------------------------------------------------------
class AntennaMast(nn.Module):
    """phi(h) = silu(R h) with frozen random R [S, d].

    One mast serves EVERY sparse site in the model: each site owns only a tiny
    linear head A [n_modes, S]. Cost per token: S x d (once per token, not per
    site). Random features are distribution-agnostic; heads are ridge- or
    distillation-trained per site.
    """

    def __init__(self, d_model: int, sketch_dim: int = 512, seed: int = 0):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        R = torch.randn(sketch_dim, d_model, generator=g) / math.sqrt(d_model)
        self.register_buffer("R", R)
        self.S = sketch_dim
        self.d_model = d_model
        self.seed = seed

    def _R_for(self, d_in: int, device: torch.device) -> torch.Tensor:
        """Sketch matrix for input width d_in. Widths != d_model (e.g. the
        DeltaNet out_proj input, whose width is value_dim) get a DETERMINISTIC
        sibling sketch derived from the master seed — so calibration, serving
        and the exported kernel all generate identical R without serializing
        every width."""
        if d_in == self.d_model:
            return self.R
        key = f"R{d_in}"
        R = getattr(self, key, None)
        if R is None or R.device != device:
            g = torch.Generator().manual_seed(self.seed * 1000003 + d_in)
            R = torch.randn(self.S, d_in, generator=g) / math.sqrt(d_in)
            self.register_buffer(key, R.to(device=device, dtype=self.R.dtype))
        return getattr(self, key)

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        """h [..., d] -> phi [..., S]  (any input width d)"""
        R = self._R_for(h.shape[-1], h.device)
        return F.silu(h.to(R.dtype) @ R.t())


# ----------------------------------------------------------------------------
# 9. ResonantLinear — row-mode routing for a WHOLE Linear (Theorem 4/5)
# ----------------------------------------------------------------------------
def plan_row_modes(out_rows: int, in_rows: int, n_blocks: int,
                   carrier_frac: float,
                   row_energy: torch.Tensor | None = None) -> dict:
    """Partition `out_rows` output rows into carrier + equal row-blocks.

    carrier absorbs the division remainder so blocks are EXACTLY equal
    (regular grouped-GEMM export). Carrier rows = highest total energy mass
    (row_energy[i] = E_calib[(w_i . x)^2]; uniform if None).

    Returns dict(carrier_idx [c], block_idx [B, r]) — indices are sorted
    (cache-friendly) and cover EVERY row exactly once (entry partition).
    """
    if row_energy is None:
        row_energy = torch.ones(out_rows)
    device = row_energy.device
    n_carrier0 = max(0, min(int(round(carrier_frac * out_rows)), out_rows // 4))
    n_blocks = max(1, min(n_blocks, (out_rows - n_carrier0)))
    r = (out_rows - n_carrier0) // n_blocks
    if r == 0:
        n_blocks, r = 1, out_rows - max(0, out_rows // 4)
    n_carrier = out_rows - n_blocks * r          # carrier absorbs remainder
    order = torch.argsort(row_energy.to(device), descending=True)
    carrier_idx = torch.sort(order[:n_carrier]).values
    rest_idx = torch.sort(order[n_carrier:]).values
    block_idx = rest_idx.reshape(n_blocks, r)
    assert int(block_idx.numel() + carrier_idx.numel()) == out_rows
    return {"carrier_idx": carrier_idx, "block_idx": block_idx}


# ----------------------------------------------------------------------------
# 8b. SHADOW EXECUTION (v10) — control-variate sparsity
# ----------------------------------------------------------------------------
# Up to v9 everything that was not computed exactly was ZEROED: ~90% of the output rows of every routed
# projection, every unselected value head. The served student therefore started at a held-out PPL of ~2e5.
# Shadow execution never skips:
#       y  =  S(x)  +  1[selected] * (exact(x) - S(x)),          S(x) = U (U^T W) x
# S is the best rank-r predictor of THIS SITE'S OUTPUT under the calibration distribution (U = top principal
# directions of the dense outputs), so the error of an unselected row falls from its full energy to its residual
# energy. The router is calibrated on the RESIDUAL (it picks the block the shadow explains worst), and in grad
# mode it gets the exact counterfactual gradient  dL/dw_b = <dL/dy_b, exact_b - S_b>  for EVERY block through a
# straight-through gate (v9's top-1 routers received no gradient at all on the served path).
@torch.no_grad()
def fit_shadow_basis(Y: torch.Tensor, rank: int, device=None, val_frac: float = 0.2):
    """Y [N, out]: stash of DENSE outputs of one site.
    Returns (U [out, r] orthonormal fp32 | None, held-out energy fraction captured, residual [N, out] fp32)."""
    Yd = Y.to(device).float() if device is not None else Y.float()
    N, out = Yd.shape
    n_val = max(1, int(round(val_frac * N))) if N >= 8 else 0
    n_fit = N - n_val
    r = int(max(0, min(int(rank), n_fit, out)))
    if r == 0:
        return None, 0.0, Yd
    _, _, Vh = torch.linalg.svd(Yd[:n_fit], full_matrices=False)
    U = Vh[:r].t().contiguous()                                   # [out, r]
    res = Yd - (Yd @ U) @ U.t()
    sl = slice(n_fit, N) if n_val else slice(0, N)
    cap = 1.0 - float(res[sl].pow(2).sum() / Yd[sl].pow(2).sum().clamp_min(1e-30))
    return U, cap, res


@torch.no_grad()
def ridge_router_init(feats: torch.Tensor, energy: torch.Tensor) -> torch.Tensor:
    """UNCENTERED ridge of log(block energy) on the mast features -> router weight [B, S].
    (The routers have no bias; a centered fit — v9 — drops every block's mean level, and the top-1 choice is then
    made on deviations only. Uncentered, the mean level rides on the mean feature direction.)"""
    X = feats.to(torch.float32)
    y = torch.log(energy.to(torch.float32).clamp_min(1e-12))
    lam = 1e-2 * (X * X).mean().clamp_min(1e-12) * X.shape[0]
    A = torch.linalg.solve(X.t() @ X + lam * torch.eye(X.shape[1], device=X.device), X.t() @ y)   # [S, B]
    return A.t().contiguous()


def _block_energy(res: torch.Tensor, block_idx: torch.Tensor) -> torch.Tensor:
    """res [N, rows] residual (or raw) outputs, block_idx [B, r] -> per-token block energy [N, B]."""
    B = block_idx.shape[0]
    cols = block_idx.reshape(-1).to(res.device)
    return res.index_select(1, cols).float().pow(2).view(res.shape[0], B, -1).sum(-1)


class ResonantLinear(nn.Module):
    """Executes y = W x with row-mode resonance:

        y = W[carrier] x  +  sum_{b in topk(phi)} c_b * W[block_b] x

    * ENTRY-EXACT PARTITION (Theorem 4): carrier ∪ ⋃blocks = every row exactly
      once, so evaluating all blocks reproduces y = W x EXACTLY (fp
      reassociation aside). The dense weight is ADOPTED (zero copy).
    * TAIL CERTIFICATE (Theorem 5): per-token error of a selected set S obeys
        || W x - y_S ||  <=  sqrt( sum_{b not in S} ||W[b] x||^2 )
      — measurable during calibration (one dense matmul yields all row dots).
    * Router: A [B, S] head on the shared AntennaMast (phi computed once per
      token per layer and shared by all sites of that layer).
    * Per-block gains c_b >= 0 (calibrated at init; fused into W at export).
    """

    def __init__(self, weight: torch.Tensor, plan: dict, mast: AntennaMast,
                 calib_feats: torch.Tensor | None = None,
                 calib_energy: torch.Tensor | None = None,
                 topk: int = 1):
        """weight: [out, in] (HF Linear .weight, adopted zero-copy, frozen).
        calib_feats: [T, S] mast features on calibration tokens (optional).
        calib_energy: [T, B] per-token per-block output energy (optional).
        """
        super().__init__()
        if not isinstance(weight, nn.Parameter):
            weight = nn.Parameter(weight, requires_grad=False)
        else:
            weight.requires_grad_(False)
        self.register_parameter("weight", weight)
        self.register_buffer("carrier_idx", plan["carrier_idx"].clone())
        self.register_buffer("block_idx", plan["block_idx"].clone())
        self.out_f, self.in_f = weight.shape
        self.B, self.r = plan["block_idx"].shape
        self.topk = min(topk, self.B)
        self.mast = mast                                   # shared, not owned
        self.router = nn.Linear(mast.S, self.B, bias=False)
        if calib_feats is not None and calib_energy is not None:
            self._ridge_init_router(calib_feats, calib_energy)
        else:
            nn.init.zeros_(self.router.weight)
        self.gains = nn.Parameter(torch.ones(self.B))
        self.default_mode = "topk"
        self.last_cert: dict | None = None
        # v10 shadow: U [out, r] (orthonormal output basis) and M = U^T W [r, in]; None = v9 behaviour
        self.register_buffer("sh_U", None)
        self.register_buffer("sh_M", None)
        self.router_st = True          # straight-through router gradient on the served path
        # module self-checkpointing switch (same contract as PrismMoeMLP):
        # in grad mode the whole row-mode computation is recomputed at
        # backward from the [T, in] input, so the soft/topk saved set
        # collapses to the [T, in] boundary BY CONSTRUCTION.
        self.self_ckpt = True

    def _ridge_init_router(self, feats: torch.Tensor, energy: torch.Tensor):
        """Ridge-regress log(energy + eps) on mast features -> warm router."""
        with torch.no_grad():
            X = feats.to(torch.float32)
            y = torch.log(energy.to(torch.float32).clamp_min(1e-12))
            Xc = X - X.mean(0, keepdim=True)
            yc = y - y.mean(0, keepdim=True)
            lam = 1e-2 * (Xc * Xc).mean().clamp_min(1e-12)
            eye = torch.eye(X.shape[1], device=X.device)
            A = torch.linalg.solve(Xc.t() @ Xc + lam * eye, Xc.t() @ yc)  # [S, B]
            self.router.weight.data.copy_(A.t().to(self.router.weight.dtype))

    # ---- v10 shadow ----
    @torch.no_grad()
    def attach_shadow(self, U: torch.Tensor):
        """U [out, r]: orthonormal basis of the principal OUTPUT directions (fit_shadow_basis)."""
        U = U.to(self.weight.device, torch.float32)
        M = torch.cat([U[:, i:i + 64].t() @ self.weight.float() for i in range(0, U.shape[1], 64)], 0)
        self.sh_U = U.to(self.weight.dtype).contiguous()
        self.sh_M = M.to(self.weight.dtype).contiguous()

    def _shadow(self, x: torch.Tensor) -> torch.Tensor:
        return (x @ self.sh_M.t()) @ self.sh_U.t()

    # ---- execution ----
    def _carrier_out(self, x: torch.Tensor) -> torch.Tensor:
        if self.carrier_idx.numel() == 0:
            return x.new_zeros(x.shape[0], self.out_f)
        return F.linear(x, self.weight[self.carrier_idx])

    def block_outputs(self, x: torch.Tensor) -> torch.Tensor:
        """ALL block outputs stacked [T, B, r] (calibration / full mode).
        Column-gather of ONE whole-module GEMM: the weight operand is the
        parameter VIEW (no [B*r, d] weight-row copy is formed); the output
        rows are the same either way, so values and certificates are
        unchanged."""
        y_all = F.linear(x, self.weight)                   # [T, out_f]
        return y_all[:, self.block_idx.reshape(-1)] \
            .view(x.shape[0], self.B, self.r)              # [T, B, r]

    def forward(self, x: torch.Tensor, mode: str | None = None,
                return_cert: bool = False):
        mode = mode or self.default_mode
        lead = x.shape[:-1]                                # support [..., in]
        T = int(x.numel() // self.in_f)
        x2 = x.reshape(T, self.in_f)
        # Module self-checkpointing: in grad mode (training) the module's
        # backward footprint is its [T, in] input BY CONSTRUCTION. The
        # dense and no-grad/serving paths are untouched (honest active
        # accounting). use_reentrant=False keeps the side-channel cert
        # differentiable and is correct with frozen inputs + trainable
        # params (gradients reach params through the recompute).
        if (self.training and torch.is_grad_enabled()
                and mode != "dense"
                and getattr(self, "self_ckpt", True)):
            y = _torch_checkpoint(self._forward_2d, x2, mode,
                                  use_reentrant=False)
        else:
            y = self._forward_2d(x2, mode)
        y = y.reshape(*lead, self.out_f) if len(lead) != 1 else y
        return (y, self.last_cert) if return_cert else y

    def _forward_2d(self, x: torch.Tensor, mode: str) -> torch.Tensor:
        T = x.shape[0]
        if mode == "dense":                                # literal dense path
            y = F.linear(x, self.weight)
            self.last_cert = {"rel": 0.0}
            return y
        if mode == "soft":                                 # differentiable all-blocks
            # w = softmax(logits)*B*gains: at init (zero router, gains=1) this is
            # EXACTLY dense (each disjoint block weighted 1.0) — training starts
            # at the Theorem-7 anchor and deforms continuously.
            logits = self.router(self.mast(x))             # [T, B]
            p = torch.softmax(logits, dim=-1)              # [T, B]
            yb = self.block_outputs(x)                     # [T, B, r]
            w = p * (float(self.B) * self.gains[None, :])
            # blocks are DISJOINT row-groups: scale each block by its own weight
            # and CONCATENATE (an einsum over b would wrongly sum the blocks)
            rest_out = (w.unsqueeze(-1) * yb).reshape(T, self.B * self.r)
            y = x.new_zeros(T, self.out_f)
            if self.carrier_idx.numel() > 0:
                y.index_copy_(1, self.carrier_idx, self._carrier_out(x))
            rest_all = self.block_idx.reshape(-1)
            y.index_copy_(1, rest_all, rest_out.to(y.dtype))
            # diagnostics only: store detached values so the stored
            # copy can never pin a (re)computed graph; the differentiable
            # p/logits flow through the RETURNED y, exactly as before.
            self.last_cert = {"p": p.detach(), "logits": logits.detach()}
            return y
        if mode == "full":                                 # EXACT resonant path
            rest_out = self.block_outputs(x).reshape(T, -1)  # [T, B*r] concat
            y = x.new_zeros(T, self.out_f)
            if self.carrier_idx.numel() > 0:
                y.index_copy_(1, self.carrier_idx, self._carrier_out(x))
            rest_all = self.block_idx.reshape(-1)
            y.index_copy_(1, rest_all, rest_out.to(y.dtype))
            self.last_cert = {"rel": 0.0}
            return y
        # ---- sparse resonant path (top-k) ----
        if torch.is_grad_enabled():
            # Grad-mode backprop supergraph: ONE whole-module GEMM whose
            # weight operand is the parameter VIEW, with the exact sparse
            # block weights applied AFTER the GEMM. A skipped block
            # contributes exactly zero to forward AND backward, so this
            # branch and the true-sparse chunked gather below are the SAME
            # function with the SAME gradients — while the [T*k, r, in]
            # weight copies the gather would save for grad_x never exist.
            # Serving (no-grad) keeps the honest sparse gather.
            phi = self.mast(x)
            logits = self.router(phi)
            topv, topi = logits.topk(self.topk, dim=-1)
            g = self.gains[topi]
            y_all = F.linear(x, self.weight)               # [T, out_f]
            w = torch.zeros(T, self.B, dtype=y_all.dtype, device=x.device)
            w = w.scatter(1, topi, g.to(y_all.dtype))      # differentiable
            factor = torch.zeros(T, self.out_f, dtype=y_all.dtype,
                                 device=x.device)
            if self.carrier_idx.numel() > 0:
                factor[:, self.carrier_idx] = 1.0
            factor[:, self.block_idx.reshape(-1)] = w.unsqueeze(-1) \
                .expand(T, self.B, self.r).reshape(T, self.B * self.r)
            self.last_cert = {"topi": topi.detach(), "logits": logits.detach()}
            if self.sh_U is None:
                return y_all * factor
            # v10 shadow: y = S + sel * (g * exact - S); carrier rows exact. `sel` is the hard 0/1 block
            # choice plus a straight-through soft part, so the router receives
            #   dL/dsel_b = <dL/dy_b, g_b exact_b - S_b>   for EVERY block (the value is unchanged).
            sel = torch.zeros(T, self.B, dtype=y_all.dtype, device=x.device).scatter(
                1, topi, torch.ones_like(topi, dtype=y_all.dtype))
            if self.training and getattr(self, "router_st", True):
                ps = torch.softmax(logits.float(), dim=-1).to(y_all.dtype)
                sel = sel + (ps - ps.detach())
            rest = self.block_idx.reshape(-1)
            gate = torch.zeros(T, self.out_f, dtype=y_all.dtype, device=x.device)
            if self.carrier_idx.numel() > 0:
                gate[:, self.carrier_idx] = 1.0
            gate[:, rest] = sel.unsqueeze(-1).expand(T, self.B, self.r).reshape(T, self.B * self.r)
            gfull = torch.ones(self.out_f, dtype=y_all.dtype, device=x.device).scatter(
                0, rest, self.gains.to(y_all.dtype).repeat_interleave(self.r))
            S = self._shadow(x)
            return S + gate * (y_all * gfull[None, :] - S)
        phi = self.mast(x)                                 # [T, S]
        logits = self.router(phi)                          # [T, B]
        topv, topi = logits.topk(self.topk, dim=-1)        # [T, k]
        g = self.gains[topi]                               # [T, k]
        # Chunked over T: the Wsel gather is [T*k, r, in] — unbounded, OOMs at
        # 27B scale. Scoring is per-token, so chunking is mathematically identical.
        c = _gather_chunk(self.topk * self.r * self.in_f * self.weight.element_size())
        ys_parts, ti_parts = [], []
        for i in range(0, T, c):
            ti = topi[i:i + c]                             # [t, k]
            Ws = self.weight[self.block_idx[ti.reshape(-1)]].reshape(-1, self.r, self.in_f)
            xr = x[i:i + c].repeat_interleave(self.topk, 0).unsqueeze(-1)  # [t*k, in, 1]
            ys_parts.append(torch.bmm(Ws, xr).squeeze(-1).reshape(-1, self.topk, self.r))
            ti_parts.append(ti)
        topi = torch.cat(ti_parts, 0)
        ys = torch.cat(ys_parts, 0) * g.unsqueeze(-1)
        shadow = self.sh_U is not None
        y = self._shadow(x) if shadow else x.new_zeros(T, self.out_f)     # v10: unselected rows = shadow
        if self.carrier_idx.numel() > 0:
            y.index_copy_(1, self.carrier_idx, self._carrier_out(x))
        rows = self.block_idx[topi]                        # [T, k, r]
        if shadow:                                         # exact rows REPLACE the shadow values
            y.scatter_(1, rows.reshape(T, -1), ys.reshape(T, -1).to(y.dtype))
        else:
            y.scatter_add_(1, rows.reshape(T, -1), ys.reshape(T, -1).to(y.dtype))
        self.last_cert = {"topi": topi.detach(), "logits": logits.detach()}
        return y

    def materialize_dense(self) -> torch.Tensor:
        """Recover the EXACT original weight (entry partition inverse, with
        per-block gains folded in — gains==1 recovers the original bitwise)."""
        W = torch.zeros_like(self.weight.data)
        if self.carrier_idx.numel() > 0:
            W[self.carrier_idx] = self.weight.data[self.carrier_idx]
        rows = self.block_idx.reshape(-1)
        g = self.gains.detach()[:, None].expand(self.B, self.r).reshape(-1, 1)
        W[rows] = self.weight.data[rows] * g.to(self.weight.dtype)
        return W

    def active_params(self) -> int:
        c = self.carrier_idx.numel()
        sh = (self.sh_U.numel() + self.sh_M.numel()) if self.sh_U is not None else 0
        return c * self.in_f + min(self.topk, self.B) * self.r * self.in_f + sh


# ----------------------------------------------------------------------------
# 10. MoRM — Mixture of Recurrent Memories (write-sparse value heads)
# ----------------------------------------------------------------------------
class MoRMWritePolicy(nn.Module):
    """Per-token top-m write mask over H recurrent memory slabs.

    Ideal write energy of head h at token t: e_h = beta_h * ||k_h|| * ||v_h||.
    Computing it requires v — the projection we want to SKIP. So the policy
    predicts log e_h from cheap signals only: mast features phi(h) (trained
    head A [H, S]) plus the observable beta_h * ||k_h|| (k is needed anyway
    for the decay path; beta comes from the tiny in_proj_ba).

    Gradients: straight-through top-m (hard forward, soft backward), so the
    self-teacher (dense writes) teaches the policy WHICH heads deserved the
    write. m == H reduces to dense writes (exact).
    """

    def __init__(self, n_heads: int, mast: AntennaMast, m_write: int):
        super().__init__()
        self.H = n_heads
        self.m = min(m_write, n_heads)
        self.router = nn.Linear(mast.S, n_heads, bias=False)
        nn.init.zeros_(self.router.weight)
        self.last_mask: torch.Tensor | None = None

    def forward(self, phi: torch.Tensor,
                obs: torch.Tensor | None = None,
                m: int | None = None) -> torch.Tensor:
        """phi [T, S]; obs [T, H] = log(beta*||k|| + eps). Returns [T, H] mask.

        Forward value is the HARD top-m mask; when training, a soft softmax
        over the selected heads is added in straight-through fashion so
        gradients reach the router."""
        m = m or self.m
        s = self.router(phi)
        if obs is not None:
            s = s + obs
        topv, topi = s.topk(m, dim=-1)
        hard = torch.zeros_like(s).scatter_(-1, topi, 1.0)
        if self.training:
            # bf16-autocast trap: under autocast,
            # softmax ALWAYS runs in fp32 while zeros_like(s) follows s (bf16);
            # scatter_'s TENSOR overload demands an exact dtype match.
            # Explicit fp32 softmax + cast back: identical math on every device
            # (softmax over m<=32 logits costs nothing in fp32).
            soft = torch.zeros_like(s).scatter_(
                -1, topi, torch.softmax(topv.float(), dim=-1).to(s.dtype))
            hard = hard + (soft - soft.detach())          # straight-through
        # store the VALUE only — the straight-through gradient flows
        # through the RETURNED mask.
        self.last_mask = hard.detach()
        return hard


def morm_state_update(S_prev: torch.Tensor, k: torch.Tensor, v: torch.Tensor,
                      beta: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Math-canonical write-sparse recurrent update.

    S_prev: [T, H, dk, dv]; k [T, H, dk]; v [T, H, dv]; beta [T, H];
    mask [T, H] in {0,1}. Returns S_prev + mask_h * beta_h * k_h v_h^T.

    (The real GatedDeltaNet interleaves decay/normalizer bookkeeping; the
    masked rank-1 write term is identical — masking v_h or beta_h zeroes the
    write exactly, which is what SectionedProjector exploits in-place.)"""
    outer = torch.einsum("thd,the->thde", k, v) * beta[:, :, None, None]
    return S_prev + outer * mask[:, :, None, None]


def morm_error_bound(S_dense: torch.Tensor, S_sparse: torch.Tensor,
                     k: torch.Tensor, v: torch.Tensor, beta: torch.Tensor,
                     mask: torch.Tensor):
    """Per-token certificate for skipped writes:
        ||S_dense - S_sparse||_F  <=  sum_{h skipped} beta_h ||k_h|| ||v_h||.
    Returns (gap, bound), each [T]."""
    gap = (S_dense - S_sparse).flatten(1).norm(dim=1)
    skipped = (k.norm(dim=-1) * v.norm(dim=-1) * beta.abs()) \
        .masked_fill(mask.bool(), 0.0).sum(dim=1)
    return gap, skipped


# ----------------------------------------------------------------------------
# 11. SectionedProjector — fused DeltaNet in_proj_qkvz under RESONA execution
# ----------------------------------------------------------------------------
class SectionedProjector(nn.Module):
    """Zero-copy split of a fused [q|k|v|z] weight (contiguous row sections):
         q -> routed rows (carrier + top-k blocks; per-token)
         k -> dense rows   (every position's decay/state path needs them)
         v -> MoRM head-routed gather (ONLY selected heads' rows evaluated)
         z -> routed rows  (output gate; per-token)

    The fused weight stays registered ONCE here (sections are views).
    mode='dense' reproduces the stock fused projection exactly; mode='full'
    reproduces it through the resonant path (Theorem 4); mode='topk' is the
    sparse serving path (v zero-padded to full width so the downstream
    DeltaNet code is UNCHANGED). Under autograd every sparse family
    executes as its masked-dense SUPERGRAPH (exact values, exact gradients,
    zero weight materialization); under no-grad it executes the true sparse
    path (honest active-parameter accounting)."""

    def __init__(self, weight: torch.Tensor, section_sizes, v_head_rows: int,
                 plan_q=None, plan_z=None, mast: AntennaMast | None = None,
                 morm: MoRMWritePolicy | None = None, topk: int = 1):
        super().__init__()
        if not isinstance(weight, nn.Parameter):
            weight = nn.Parameter(weight, requires_grad=False)
        else:
            weight.requires_grad_(False)
        self.register_parameter("weight", weight)
        q_r, k_r, v_r, z_r = section_sizes
        assert v_r % v_head_rows == 0, "v rows must divide into heads"
        self.q_rows, self.k_rows, self.v_rows, self.z_rows = q_r, k_r, v_r, z_r
        self.n_v_heads = v_r // v_head_rows
        self.v_head_rows = v_head_rows
        bounds = [0, q_r, q_r + k_r, q_r + k_r + v_r, q_r + k_r + v_r + z_r]
        self.register_buffer("sec_bounds", torch.tensor(bounds))
        self.mast = mast
        self.morm = morm
        self.topk = topk
        self.default_mode = "topk"
        self.register_parameter("v_gains", nn.Parameter(torch.ones(self.n_v_heads)))
        # routed-row machinery for q and z sections (inline, zero-copy views)
        for tag, n_rows, plan in (("q", q_r, plan_q), ("z", z_r, plan_z)):
            if plan is None:
                continue
            self.register_buffer(f"{tag}_carrier", plan["carrier_idx"].clone())
            self.register_buffer(f"{tag}_blocks", plan["block_idx"].clone())
            B, r = plan["block_idx"].shape
            setattr(self, f"{tag}_B", B)
            setattr(self, f"{tag}_r", r)
            setattr(self, f"{tag}_router",
                    nn.Linear(mast.S, B, bias=False) if mast is not None else None)
            self.register_parameter(f"{tag}_gains",
                                    nn.Parameter(torch.ones(B)))
            if mast is not None:
                getattr(self, f"{tag}_router").weight.data.zero_()
        self.last_mask: torch.Tensor | None = None
        # telemetry switch: default True stores last_out (probes read it);
        # the trainer sets it False for the training phase only (the dict
        # pins [T, q+k+v+z] tensors per GDN layer across fwd+bwd) and
        # restores it after.
        self.telemetry = True
        # v10 shadows of the q / v / z sections (None = v9 behaviour). With a v shadow EVERY head writes:
        # the MoRM critic no longer decides which writes exist, only which writes are EXACT.
        for _t in ("q", "v", "z"):
            self.register_buffer(f"{_t}_sh_U", None)
            self.register_buffer(f"{_t}_sh_M", None)
        self.router_st = True
        # module self-checkpointing switch (bounds the fused soft-mode
        # saved set to the [T, d] input; see ResonantLinear).
        self.self_ckpt = True

    # ---- v10 shadow ----
    def _sec_bounds(self, tag: str):
        b = [int(v) for v in self.sec_bounds]
        return {"q": (b[0], b[1]), "k": (b[1], b[2]), "v": (b[2], b[3]), "z": (b[3], b[4])}[tag]

    @torch.no_grad()
    def attach_shadow(self, tag: str, U: torch.Tensor):
        lo, hi = self._sec_bounds(tag)
        U = U.to(self.weight.device, torch.float32)
        W = self.weight[lo:hi].float()
        M = torch.cat([U[:, i:i + 64].t() @ W for i in range(0, U.shape[1], 64)], 0)
        setattr(self, f"{tag}_sh_U", U.to(self.weight.dtype).contiguous())
        setattr(self, f"{tag}_sh_M", M.to(self.weight.dtype).contiguous())

    def _shadow(self, x: torch.Tensor, tag: str) -> torch.Tensor:
        return (x @ getattr(self, f"{tag}_sh_M").t()) @ getattr(self, f"{tag}_sh_U").t()

    # ---- routed-rows helper (shared logic for q / z sections) ----
    def _routed_section(self, x: torch.Tensor, tag: str, lo: int, hi: int,
                        mode: str) -> torch.Tensor:
        W = self.weight[lo:hi]                             # section view
        T = x.shape[0]
        if mode == "dense":
            return F.linear(x, W)
        carrier = getattr(self, f"{tag}_carrier")
        blocks = getattr(self, f"{tag}_blocks")
        B, r = getattr(self, f"{tag}_B"), getattr(self, f"{tag}_r")
        if mode == "full":
            y = x.new_zeros(T, hi - lo)
            if carrier.numel() > 0:
                y.index_copy_(1, carrier, F.linear(x, W[carrier]))
            # post-GEMM column gather of the ONE section GEMM (weight VIEW
            # — no [B*r, d] weight copy).
            y.index_copy_(1, blocks.reshape(-1),
                          F.linear(x, W)[:, blocks.reshape(-1)])
            return y
        if mode == "soft":
            # softmax(logits)*B*gains: exact dense at init (see ResonantLinear).
            # Disjoint row-groups: per-block scale + concat (NOT a b-summing einsum).
            logits = getattr(self, f"{tag}_router")(self.mast(x))
            p = torch.softmax(logits, dim=-1)                     # [T, B]
            # post-GEMM column gather — no [B*r, d] weight copy.
            yb = F.linear(x, W)[:, blocks.reshape(-1)].view(T, B, r)
            w = p * (float(B) * getattr(self, f"{tag}_gains")[None, :])
            rest_out = (w.unsqueeze(-1) * yb).reshape(T, B * r)
            y = x.new_zeros(T, hi - lo)
            if carrier.numel() > 0:
                y.index_copy_(1, carrier, F.linear(x, W[carrier]))
            y.index_copy_(1, blocks.reshape(-1), rest_out.to(y.dtype))
            return y
        if torch.is_grad_enabled():
            # Grad-mode backprop supergraph: ONE whole-module GEMM whose
            # weight operand is the parameter VIEW, with the exact sparse
            # block weights applied AFTER the GEMM. A skipped block
            # contributes exactly zero to forward AND backward, so this
            # branch and the true-sparse chunked gather below are the SAME
            # function with the SAME gradients — while the [T*k, r, in]
            # weight copies the gather would save for grad_x never exist.
            # Serving (no-grad) keeps the honest sparse gather.
            logits = getattr(self, f"{tag}_router")(self.mast(x))
            topv, topi = logits.topk(min(self.topk, B), dim=-1)
            g = getattr(self, f"{tag}_gains")[topi]              # [T, k]
            y_all = F.linear(x, W)                               # [T, rows]
            w = torch.zeros(T, B, dtype=y_all.dtype, device=x.device)
            w = w.scatter(1, topi, g.to(y_all.dtype))            # differentiable
            factor = torch.zeros(T, hi - lo, dtype=y_all.dtype, device=x.device)
            if carrier.numel() > 0:
                factor[:, carrier] = 1.0
            factor[:, blocks.reshape(-1)] = w.unsqueeze(-1) \
                .expand(T, B, r).reshape(T, B * r)
            if getattr(self, f"{tag}_sh_U") is None:
                return y_all * factor
            # v10 shadow (see ResonantLinear): y = S + sel * (g * exact - S), straight-through router gradient
            sel = torch.zeros(T, B, dtype=y_all.dtype, device=x.device).scatter(
                1, topi, torch.ones_like(topi, dtype=y_all.dtype))
            if self.training and getattr(self, "router_st", True):
                ps = torch.softmax(logits.float(), dim=-1).to(y_all.dtype)
                sel = sel + (ps - ps.detach())
            rest = blocks.reshape(-1)
            gate = torch.zeros(T, hi - lo, dtype=y_all.dtype, device=x.device)
            if carrier.numel() > 0:
                gate[:, carrier] = 1.0
            gate[:, rest] = sel.unsqueeze(-1).expand(T, B, r).reshape(T, B * r)
            gfull = torch.ones(hi - lo, dtype=y_all.dtype, device=x.device).scatter(
                0, rest, getattr(self, f"{tag}_gains").to(y_all.dtype).repeat_interleave(r))
            S = self._shadow(x, tag)
            return S + gate * (y_all * gfull[None, :] - S)
        logits = getattr(self, f"{tag}_router")(self.mast(x))     # [T, B]
        topv, topi = logits.topk(min(self.topk, B), dim=-1)
        g = getattr(self, f"{tag}_gains")[topi]
        # Chunked over T: the Wsel gather is [T*k, r, d] — unbounded, OOMs at
        # 27B scale. Scoring is per-token, so chunking is mathematically identical.
        k_sel = topi.shape[1]
        c = _gather_chunk(k_sel * r * self.weight.shape[1] * self.weight.element_size())
        ys_parts, ti_parts = [], []
        for i in range(0, T, c):
            ti = topi[i:i + c]                             # [t, k]
            Ws = W[blocks[ti.reshape(-1)]].reshape(-1, r, self.weight.shape[1])
            xr = x[i:i + c].repeat_interleave(k_sel, 0).unsqueeze(-1)      # [t*k, d, 1]
            ys_parts.append(torch.bmm(Ws, xr).squeeze(-1).reshape(-1, k_sel, r))
            ti_parts.append(ti)
        topi = torch.cat(ti_parts, 0)
        ys = torch.cat(ys_parts, 0) * g.unsqueeze(-1)
        shadow = getattr(self, f"{tag}_sh_U") is not None
        y = self._shadow(x, tag) if shadow else x.new_zeros(T, hi - lo)
        if carrier.numel() > 0:
            y.index_copy_(1, carrier, F.linear(x, W[carrier]))
        if shadow:
            y.scatter_(1, blocks[topi].reshape(T, -1), ys.reshape(T, -1).to(y.dtype))
        else:
            y.scatter_add_(1, blocks[topi].reshape(T, -1), ys.reshape(T, -1).to(y.dtype))
        return y

    def _sp_core(self, x: torch.Tensor, mode: str,
                 force_mask: torch.Tensor | None) -> torch.Tensor:
        """The 2-D fused [q|k|v|z] computation (checkpoint-wrapped by
        forward())."""
        T = x.shape[0]
        b0, b1, b2, b3, b4 = (int(self.sec_bounds[0]), int(self.sec_bounds[1]),
                              int(self.sec_bounds[2]), int(self.sec_bounds[3]),
                              int(self.sec_bounds[4]))
        if mode == "dense":
            self.last_out = None
            return F.linear(x, self.weight)
        # q / k / z sections
        q = self._routed_section(x, "q", b0, b1, mode) if hasattr(self, "q_B") \
            else F.linear(x, self.weight[b0:b1])
        k = F.linear(x, self.weight[b1:b2])                # dense by design
        z = self._routed_section(x, "z", b3, b4, mode) if hasattr(self, "z_B") \
            else F.linear(x, self.weight[b3:b4])
        # ---- MoRM: pick heads, evaluate ONLY their rows ----
        if self.morm is not None and mode != "full":
            mask = self.morm(self.mast(x))                 # [T, H] in (0,1]
        else:
            mask = torch.ones(T, self.n_v_heads, device=x.device, dtype=x.dtype)
        if force_mask is not None:
            mask = mask * force_mask
        self.last_mask = mask.detach()
        Wv = self.weight[b2:b3].reshape(self.n_v_heads, self.v_head_rows,
                                        self.weight.shape[1])
        sel = (mask > 0.5).nonzero(as_tuple=False)         # [N, 2] (t, h)
        v_shadow = self.v_sh_U is not None and mode != "full"
        v = self._shadow(x, "v") if v_shadow else x.new_zeros(T, self.v_rows)
        if sel.shape[0] > 0:
            # ZERO-MATERIALIZATION MoRM execution. A skipped (t, h) write
            # contributes EXACTLY zero downstream (the zero-padded v IS
            # the skip), so the sparse v equals the DENSE v masked by the
            # hard selection:
            #   grad ON  (training):  v = (x @ W_v^T) * hard_mask * v_gains
            #     — identical values and gradients to the sparse gather
            #       (skipped entries carry exactly zero gradient through
            #       the 0-mask product); nothing is saved for backward
            #       beyond [T, v_rows] tensors (~32 GiB -> ~48 MiB at 27B).
            #   grad OFF (serving):  segmented per-head GEMM — tokens
            #       grouped by head, ONE bmm over weight VIEWS — reads ONLY
            #       the selected heads' rows (the honest m x hr active
            #       accounting), never forms [N, hr, d], and moves ~60x
            #       less weight traffic than a chunked gather loop.
            # Both branches are verified against the reference oracle in
            # the suite.
            if torch.is_grad_enabled():
                v_dense = F.linear(x, self.weight[b2:b3])          # [T, v_rows]
                # v8: use the (STE) mask DIRECTLY — forward values are
                # exactly the hard 0/1 set, and the straight-through
                # gradient reaches the MoRM policy. v7 thresholded the
                # mask here ((mask > 0.5) is non-differentiable), which
                # silently severed the policy's training signal.
                m2 = mask.to(v_dense.dtype)                       # [T, H]
                gains_row = self.v_gains.to(v_dense.dtype) \
                    .repeat_interleave(self.v_head_rows)           # [v_rows]
                if v_shadow:       # v10: shadow write for every head, exact (x gain) where the critic says so
                    v = v + m2.repeat_interleave(self.v_head_rows, dim=1) \
                        * (v_dense * gains_row[None, :] - v)
                else:
                    v = v_dense * (m2.repeat_interleave(self.v_head_rows, dim=1)
                                   * gains_row[None, :])
            else:
                hidx, tidx = sel[:, 1], sel[:, 0]
                order = torch.argsort(hidx, stable=True)           # group by head
                h_s, t_s = hidx[order], tidx[order]
                counts = torch.bincount(h_s, minlength=self.n_v_heads)
                starts = torch.cumsum(counts, 0) - counts
                pos = torch.arange(sel.shape[0], device=x.device) - starts[h_s]
                max_n = int(counts.max())
                pad_t = torch.zeros(self.n_v_heads, max_n, dtype=torch.long,
                                    device=x.device)
                pad_t[h_s, pos] = t_s
                X_pad = x[pad_t]                                   # [H, max_n, d]
                V_pad = torch.bmm(X_pad, Wv.transpose(1, 2))       # [H, max_n, hr]
                v_sel = V_pad[h_s, pos]                            # [N, hr]
                v_sel = v_sel * self.v_gains.to(v_sel.dtype)[h_s][:, None]
                hr = self.v_head_rows
                rows = (h_s[:, None] * hr
                        + torch.arange(hr, device=x.device)[None, :])
                v[t_s[:, None].expand(-1, hr).reshape(-1),
                  rows.reshape(-1)] = v_sel.reshape(-1).to(v.dtype)
        # Telemetry: `telemetry=False` (set by the trainer for the training
        # phase) skips the dict — it pins [T, q+k+v+z] per GDN layer across
        # fwd+bwd and the trainer never reads it.
        if getattr(self, "telemetry", True):
            self.last_out = {"q": q, "k": k, "v": v, "z": z,
                             "beta_mask": mask}
        else:
            self.last_out = None
        return torch.cat([q, k, v, z], dim=-1)

    def forward(self, x: torch.Tensor, mode: str | None = None,
                force_mask: torch.Tensor | None = None):
        """Returns the FUSED output tensor [..., q+k+v+z] — the exact interface
        of the stock in_proj_qkvz Linear, so the downstream DeltaNet code runs
        UNCHANGED (it splits sections itself). Telemetry in self.last_out.
        Accepts any lead dims ([T,d] or [B,L,d], as HF decoders pass).

        Module self-checkpointing: in grad mode the whole fused computation
        is recomputed at backward from the [T, d] input, so the soft-mode
        saved set ([T, rows] outputs per section, ~0.44 GiB per DeltaNet
        layer at T=2048 — ~21 GiB across 48 layers) collapses to the
        [T, d] boundary BY CONSTRUCTION. MoRM semantics are untouched (the
        straight-through mask recomputes identically). The dense and
        no-grad/serving paths stay untouched (honest sparse accounting)."""
        lead = x.shape[:-1]
        x = x.reshape(-1, self.weight.shape[1])            # flatten to [T, d]
        mode = mode or self.default_mode
        # no input-requires-grad gate: non-reentrant checkpoint is correct
        # with frozen inputs + trainable params (the VL wrapper feeds
        # inputs_embeds and the embedding-side hook can silently miss).
        if (self.training and torch.is_grad_enabled()
                and mode != "dense"
                and getattr(self, "self_ckpt", True)):
            y = _torch_checkpoint(self._sp_core, x, mode, force_mask,
                                  use_reentrant=False)
        else:
            y = self._sp_core(x, mode, force_mask)
        return y.reshape(*lead, -1) if len(lead) != 1 else y


def _shortlist_chunk(shortlist: int, d: int, esize: int,
                     budget: int = 512 << 20) -> int:
    """Tokens per shortlist-scoring chunk so the gathered rows weight[sl]
    ([t, V_s, d]) stay under `budget` bytes of transient memory.

    27B head (d=5120, V_s=8192, bf16): 84 MB PER TOKEN -> 6 tokens = 503 MB.
    Unchunked, a 256-token batch tries to allocate 20 GiB and OOMs an A100.
    """
    return max(1, budget // max(1, shortlist * max(1, d) * max(1, esize)))


def _gather_chunk(per_row_bytes: int, budget: int = 256 << 20) -> int:
    """Rows per chunked-gather step so the gathered block stays under
    `budget` bytes of transient memory (256 MiB keeps the routed-linear
    Wsel and MoRM v-head gathers alive even in degraded-headroom runs;
    scoring is per-token, so chunking is mathematically exact)."""
    return max(1, budget // max(1, per_row_bytes))


# ----------------------------------------------------------------------------
# 12. HeadFunnel — Spectral Head Funnel (rank-r proxy + residual-aware ranking + exact shortlist)
# ----------------------------------------------------------------------------
class HeadFunnel(nn.Module):
    """Two-stage lm_head execution:

        stage 1 (proxy):  z~ = (h P_r) U_r^T          U_r = W P_r  (rank-r projection of the head)
                          score = z~ + kappa * s(h) * rho
        stage 2 (exact):  shortlist = top-V_s rows of the score;
                          logits[sl] = lm_head[sl] h          (EXACT rows)

    RESIDUAL-AWARE RANKING. With Q = I - P_r P_r^T, the proxy error of row i is
        z_i - z~_i = <Q w_i, Q h>,   |z_i - z~_i| <= rho_i * s,
        rho_i = ||Q w_i|| (stored, V floats),  s = ||Q h|| (one norm per token)
    (Cauchy-Schwarz), so z~ + s*rho is a certified upper bound on EVERY logit.
    The worst case is far looser than the typical alignment, therefore
    kappa in [0, 1] interpolates between the plain proxy (0) and the certified
    upper-confidence ranking (1) and is FITTED on calibration states
    (fit_kappa; kappa = 0 is always a candidate, so the fit never loses recall).
    (v8 additionally multiplied U_r by the singular values — a second sigma that
    ranked rows by sigma^2-weighted scores; the proxy is now the exact rank-r
    projection.)

    Returns (values [T, V_s], indices [T, V_s]). mode='dense' gives the full
    exact head. Active params: V*r + d*r + V (rho) + V_s*d  (vs V*d dense). The dense
    head is KEPT as the shortlist source — total parameters preserved exactly.
    """

    def __init__(self, weight: torch.Tensor, rank: int = 256,
                 shortlist: int = 8192, use_rho: bool = True):
        super().__init__()
        if not isinstance(weight, nn.Parameter):
            weight = nn.Parameter(weight, requires_grad=False)
        else:
            weight.requires_grad_(False)
        self.register_parameter("weight", weight)          # [V, d]
        self.V, self.d = weight.shape
        self.rank = max(8, min(rank, min(self.V, self.d) - 1))
        self.shortlist = min(shortlist, self.V)
        # Chunked spectral factors: torch.svd_lowrank would cast the FULL
        # [248320, 5120] head to fp32 (5.1 GiB on top of the resident
        # model). Instead eigendecompose the Gram matrix G = W^T W =
        # sum_chunks W_c^T W_c (fp32, [d, d] = 105 MiB), take P_r = top-r
        # eigenvectors, and build U_r = W @ P_r chunk-wise — the exact
        # rank-r projection, peak transient < 1 GiB, deterministic. The
        # per-row residual norm rho_i = sqrt(||w_i||^2 - ||U_r[i]||^2) falls out
        # of the same pass.
        with torch.no_grad():
            d = self.d
            G = torch.zeros(d, d, device=weight.device, dtype=torch.float32)
            for i in range(0, self.V, 32768):              # Gram accumulation
                Wc = weight[i:i + 32768].to(torch.float32)
                G.add_(Wc.t() @ Wc)
            evals, evecs = torch.linalg.eigh(G)            # ascending
            P_r = evecs[:, -self.rank:].flip(1)                      # [d, r]
            U_r = torch.empty(self.V, self.rank, device=weight.device,
                              dtype=weight.dtype)
            rho = torch.empty(self.V, device=weight.device, dtype=torch.float32)
            for i in range(0, self.V, 32768):              # U = W @ P_r
                Wc = weight[i:i + 32768].to(torch.float32)
                Uc = Wc @ P_r
                U_r[i:i + 32768] = Uc.to(weight.dtype)
                rho[i:i + 32768] = (Wc.pow(2).sum(1) - Uc.pow(2).sum(1)).clamp_min(0).sqrt()
        self.register_buffer("U_r", U_r.to(weight.dtype))
        self.register_buffer("P_r", P_r.to(weight.dtype))
        self.register_buffer("rho", rho if use_rho else None)
        self.kappa = 0.0                                   # fitted by fit_kappa()
        self.default_mode = "funnel"
        self.last_recall: float | None = None

    def _proxy_scores(self, hc: torch.Tensor, kappa: float | None = None) -> torch.Tensor:
        """[t, V] shortlist scores of tokens hc [t, d] (already in the head dtype)."""
        kappa = self.kappa if kappa is None else kappa
        hp = hc @ self.P_r                                 # [t, r]
        pc = hp @ self.U_r.t()                             # [t, V]  rank-r proxy
        if kappa and self.rho is not None:
            s = (hc.float().pow(2).sum(-1) - hp.float().pow(2).sum(-1)).clamp_min(0).sqrt()
            pc = pc + (float(kappa) * s)[:, None].to(pc.dtype) * self.rho[None, :].to(pc.dtype)
        return pc

    def forward(self, h: torch.Tensor, mode: str | None = None,
                ref_logits: torch.Tensor | None = None):
        """h [T, d] -> (values [T, V_s], indices [T, V_s]) | full logits."""
        mode = mode or self.default_mode
        if mode == "dense":
            return h @ self.weight.t(), None
        # Chunked over T (scoring is per-token, so chunking is exact):
        # bounds the [t, V_s, d] gather to ~512 MiB.
        c = _shortlist_chunk(self.shortlist, self.d, self.weight.element_size())
        sl_parts, ex_parts = [], []
        for i in range(0, h.shape[0], c):
            hc = h[i:i + c].to(self.weight.dtype)          # dtype-boundary guard
            slc = self._proxy_scores(hc).topk(self.shortlist, dim=-1).indices  # [t, V_s]
            rows = self.weight[slc]                        # [t, V_s, d] bounded
            ex_parts.append(torch.bmm(rows, hc.unsqueeze(-1))
                            .squeeze(-1).to(h.dtype))      # [t, V_s] exact
            sl_parts.append(slc)
        exact = (torch.cat(ex_parts, 0) if ex_parts
                 else h.new_empty((0, self.shortlist)))
        sl = (torch.cat(sl_parts, 0) if sl_parts
              else h.new_empty((0, self.shortlist), dtype=torch.long))
        if ref_logits is not None:
            true_top = ref_logits.topk(min(64, self.V), dim=-1).indices
            hit = (true_top.unsqueeze(-1) == sl.unsqueeze(1)).any(-1).float()
            self.last_recall = float(hit.mean())
        return exact, sl

    @torch.no_grad()
    def shortlist_recall(self, h: torch.Tensor, true_top: torch.Tensor,
                         kappa: float | None = None) -> float:
        """Mean fraction of each token's true top-n ids inside the shortlist.
        h [N, d] head inputs, true_top [N, n] ids of the exact top-n logits."""
        hit, tot = 0.0, 0
        for i in range(0, h.shape[0], 64):
            hc = h[i:i + 64].to(self.weight.dtype)
            sl = self._proxy_scores(hc, kappa).topk(self.shortlist, dim=-1).indices
            tt = true_top[i:i + 64]
            hit += float((tt.unsqueeze(-1) == sl.unsqueeze(1)).any(-1).float().sum())
            tot += tt.numel()
        return hit / max(tot, 1)

    @torch.no_grad()
    def fit_kappa(self, h: torch.Tensor, true_top: torch.Tensor,
                  grid=(0.0, 0.1, 0.2, 0.35, 0.5, 0.75, 1.0)) -> dict:
        """Pick kappa by shortlist recall on calibration states (ties -> smaller
        kappa). Returns {kappa: recall}; sets self.kappa."""
        if self.rho is None:
            self.kappa = 0.0
            return {0.0: self.shortlist_recall(h, true_top, 0.0)}
        res = {float(k_): self.shortlist_recall(h, true_top, k_) for k_ in grid}
        self.kappa = float(max(res, key=lambda k_: (round(res[k_], 6), -k_)))
        return res

    def active_params(self) -> int:
        """V*r (proxy rows) + d*r (the projection h P_r) + V_s*d (exact rows) + V (rho reads)."""
        return (self.V * self.rank + self.d * self.rank + self.shortlist * self.d
                + (self.V if self.rho is not None else 0))


# ----------------------------------------------------------------------------
# 13. active_waterfall / solve_budget_v2   (v9: mast evaluations charged per layer, trim ladder,
#     plan consumed by the pipeline)
# ----------------------------------------------------------------------------
def _rows_active(n_rows: int, n_blocks: int, topk: int,
                 carrier_frac: float) -> int:
    n_carrier0 = max(0, min(int(round(carrier_frac * n_rows)), n_rows // 4))
    n_blocks = max(1, min(n_blocks, n_rows - n_carrier0))
    r = (n_rows - n_carrier0) // n_blocks
    n_carrier = n_rows - n_blocks * r
    return n_carrier + min(topk, n_blocks) * r


def morm_nheads_probe(v_rows: int, head_rows: int) -> int:
    return v_rows // head_rows


def mast_reads(d: int, n_layers: int, n_linear_attn: int, n_full_attn: int,
               ss: dict, mast_S: int, mode: str = "applications") -> int:
    """Weight elements the AntennaMast sketches touch per token.

    phi(h) = silu(R_w h) is evaluated once per (layer, input width): the attention-side
    input of every layer (width d: q / gate / z / b-gate / critic of that layer share it)
    and the output-projection input of every layer (o_proj: o_in, DeltaNet out_proj:
    dn_out_in; a SIBLING sketch of that width).
      mode='applications' (strict, default): every evaluation is charged — what the
            compute / bandwidth of a '3B-active' model really pays.
      mode='distinct': every distinct sketch MATRIX is charged once (R and its siblings are
            shared by all layers) — the v8 convention, which additionally forgot the sibling."""
    if mode == "distinct":
        widths = {d}
        if n_full_attn and ss.get("o_in"):
            widths.add(ss["o_in"])
        if n_linear_attn and ss.get("dn_out_in"):
            widths.add(ss["dn_out_in"])
        return mast_S * sum(widths)
    return mast_S * (n_layers * d + n_full_attn * ss.get("o_in", 0)
                     + n_linear_attn * ss.get("dn_out_in", 0))


def active_waterfall(d: int, D_ffn: int, n_layers: int, n_linear_attn: int,
                     n_full_attn: int, vocab: int, ss: dict,
                     E_ffn: int, k_ffn: int, shared_ffn: int, rank_zssi: int,
                     mast_S: int, lin_blocks: int, lin_topk: int,
                     carrier_frac: float, m_write: int, funnel_rank: int,
                     funnel_shortlist: int, funnel_rho: bool = True,
                     mast_mode: str = "applications", shadow_lin: int = 0,
                     shadow_ffn: int = 0, shadow_v: bool = False) -> dict:
    """STRICT touched-weights accounting for one text token under RESONA.

    ss (section sizes): q_rows, k_rows, v_rows, gate_rows (full-attn fused
    v+gate => v half dense / gate half routed), o_in, dn_q_rows, dn_k_rows,
    dn_v_rows, dn_z_rows, dn_ba_rows, dn_v_head_rows, dn_out_in.
    v9 corrections vs v8 (all small except the first):
      * the mast sketches are charged per (layer, width) evaluation, see mast_reads
      * FFN: the shared core's ZSSI rows P[s] and the alpha/beta vectors are read too; the
        'E*mast_S' term of v8 was a phantom (the FFN router reads h directly: E*d)
      * DeltaNet conv: the taps of the selected v channels are read as well
      * head funnel: the d x r projection h @ P_r and (optionally) the V residual norms rho
    """
    a = {}
    a["embedding_lookup"] = d
    a["antenna_mast"] = mast_reads(d, n_layers, n_linear_attn, n_full_attn, ss, mast_S,
                                   mast_mode)
    m_e = max(1, (D_ffn - shared_ffn) // E_ffn)
    # v10 shadows: a rank-r shadow of an [n_out, n_in] site reads U [n_out, r] and M [r, n_in]
    def _shadow_cost(n_out, n_in):
        return int(shadow_lin) * (n_out + n_in) if (shadow_lin and n_out > 0) else 0
    a["ffn"] = n_layers * (k_ffn * 3 * d * m_e + 3 * d * shared_ffn
                           + E_ffn * d + d * rank_zssi
                           + (k_ffn * m_e + shared_ffn) * rank_zssi
                           + 2 * (k_ffn + 1) * rank_zssi
                           + ((2 * d * int(shadow_ffn) + d) if shadow_ffn else 0))
    fa = (_rows_active(ss["q_rows"], lin_blocks, lin_topk, carrier_frac) * d
          + ss["k_rows"] * d                        # dense (KV cache)
          + ss["v_rows"] * d                        # dense (KV cache)
          + _rows_active(ss["gate_rows"], lin_blocks, lin_topk, carrier_frac) * d
          + _rows_active(d, lin_blocks, lin_topk, carrier_frac) * ss["o_in"]
          + 3 * lin_blocks * mast_S                 # q/gate/o probe heads
          + _shadow_cost(ss["q_rows"], d) + _shadow_cost(ss["gate_rows"], d)
          + (_shadow_cost(d, ss["o_in"]) if ss["o_in"] else 0))
    a["full_attention"] = fa * n_full_attn
    dn = (_rows_active(ss["dn_q_rows"], lin_blocks, lin_topk, carrier_frac) * d
          + ss["dn_k_rows"] * d                     # dense
          + m_write * ss["dn_v_head_rows"] * d      # MoRM heads evaluated EXACTLY
          + (_shadow_cost(ss["dn_v_rows"], d) if shadow_v else 0)        # v10: shadow writes for all heads
          + _shadow_cost(ss["dn_q_rows"], d) + _shadow_cost(ss["dn_z_rows"], d)
          + (_shadow_cost(d, ss["dn_out_in"]) if ss["dn_out_in"] else 0)
          + _rows_active(ss["dn_z_rows"], lin_blocks, lin_topk, carrier_frac) * d
          + ss["dn_ba_rows"] * d
          + (ss["dn_q_rows"] + ss["dn_k_rows"]
             + (ss["dn_v_rows"] if (shadow_v and shadow_lin and ss["dn_q_rows"])
                else m_write * ss["dn_v_head_rows"])) * 4   # conv1d taps: q, k and the non-zero v channels
          + _rows_active(d, lin_blocks, lin_topk, carrier_frac) * ss["dn_out_in"]
          + (3 * lin_blocks + morm_nheads_probe(ss["dn_v_rows"],
                                                ss["dn_v_head_rows"])) * mast_S
          + morm_nheads_probe(ss["dn_v_rows"], ss["dn_v_head_rows"]))   # v9: the critic's per-head intercept
    a["linear_attention"] = dn * n_linear_attn
    a["lm_head_funnel"] = (vocab * funnel_rank + funnel_shortlist * d
                           + d * funnel_rank + (vocab if funnel_rho else 0))
    a["norms_misc"] = n_layers * 2 * d + d
    a["TOTAL_ACTIVE"] = sum(a.values())
    return a


def solve_budget_v2(d: int, D_ffn: int, n_layers: int, n_linear_attn: int,
                    n_full_attn: int, vocab: int, ss: dict,
                    target_active: float, mast_S: int = 512,
                    rank_zssi: int = 64, verbose: bool = True,
                    shared_ffn: int | None = None, e_ffn: int = 32,
                    min_shared: int = 64, funnel_rank: int = 256,
                    m_write_max: int | None = None,
                    shortlist_max: int | None = None,
                    mast_S_min: int = 256, mast_mode: str = "applications",
                    funnel_rho: bool = True, k_ffn0: int = 1, lin_blocks0: int = 16,
                    carrier0: float = 0.04, m_write0: int | None = None,
                    shadow_lin: int = 0, shadow_ffn: int = 0, shadow_v: bool = False,
                    shadow_lin_min: int = 96) -> dict:
    """Honest budget solver (v9).

    1. FLOOR plan: every optional quality knob at its minimum, the FFN shared core and
       the mast sketch dimension at the REQUESTED size (v8 silently planned a 192-neuron
       core while the pipeline built 512 — a ~0.2B overshoot hidden behind a x1.10 band).
    2. TRIM ladder, in this order, each lever only as far as the budget demands:
         mast_S (down to mast_S_min) -> m_write (down to 12) -> shared core (down to 256,
         steps of E keep (D - S) divisible by E) -> head shortlist (2048) -> m_write (8)
         -> shared core (down to min_shared).
    3. SPEND the slack in quality-per-parameter order (head shortlist -> m_write ->
       lin_topk -> carrier_frac -> k_ffn), exactly as v8; mast_S / shared core are never
       raised above the request.
    `m_write_max` / `shortlist_max` are the user's REQUESTS (config caps): the spend step
    never exceeds them. The returned plan carries `feasible` and the list of `trims`; the
    pipeline CONSUMES the plan (v8 only advertised it).
    """
    n_v_heads = max(1, ss["dn_v_rows"] // max(1, ss["dn_v_head_rows"]))
    S0 = 192 if shared_ffn is None else int(shared_ffn)
    E = int(e_ffn)
    S0 = S0 - ((S0 - D_ffn) % E)              # largest S <= S0 with (D - S) % E == 0
    m_cap = n_v_heads if m_write_max is None else max(1, min(int(m_write_max), n_v_heads))
    sl_cap = 16384 if shortlist_max is None else max(256, int(shortlist_max))
    _m0 = min(max(8, n_v_heads // 4), m_cap) if m_write0 is None else max(1, min(int(m_write0), m_cap))
    plan = dict(E_ffn=E, k_ffn=int(k_ffn0), shared_ffn=S0, funnel_rank=funnel_rank,
                funnel_shortlist=min(4096, sl_cap),
                m_write=_m0,
                lin_blocks=int(lin_blocks0), lin_topk=1, carrier_frac=float(carrier0), mast_S=int(mast_S),
                shadow_lin=int(shadow_lin), shadow_ffn=int(shadow_ffn))

    def eval_plan(p):
        return active_waterfall(d, D_ffn, n_layers, n_linear_attn, n_full_attn,
                                vocab, ss, E_ffn=p["E_ffn"], k_ffn=p["k_ffn"],
                                shared_ffn=p["shared_ffn"], rank_zssi=rank_zssi,
                                mast_S=p["mast_S"], lin_blocks=p["lin_blocks"],
                                lin_topk=p["lin_topk"],
                                carrier_frac=p["carrier_frac"],
                                m_write=p["m_write"], funnel_rank=p["funnel_rank"],
                                funnel_shortlist=p["funnel_shortlist"],
                                funnel_rho=funnel_rho, mast_mode=mast_mode,
                                shadow_lin=p["shadow_lin"], shadow_ffn=p["shadow_ffn"],
                                shadow_v=shadow_v)

    trims = []
    w = eval_plan(plan)
    requested = dict(plan)

    def descend(name, values):
        nonlocal w
        for v in values:
            if w["TOTAL_ACTIVE"] <= target_active:
                return
            if v >= plan[name]:
                continue
            plan[name] = v
            w = eval_plan(plan)
    mast_opts = [v for v in (384, 256, 192, 128) if mast_S_min <= v < plan["mast_S"]]
    m_lo = min(12, m_cap)
    sh_opts = list(range(plan["shadow_lin"] - 32, max(int(shadow_lin_min), 1) - 1, -32)) if plan["shadow_lin"] else []
    ladder = [("mast_S", mast_opts),
              ("shadow_lin", sh_opts),
              ("m_write", list(range(plan["m_write"] - 2, m_lo - 1, -2))),
              ("shared_ffn", list(range(S0 - E, 255, -E))),
              ("funnel_shortlist", [min(2048, sl_cap)]),
              ("m_write", [min(8, m_cap)]),
              ("shared_ffn", list(range(min(S0, 256) - E, min_shared - 1, -E)))]
    for name, vals in ladder:
        before = plan[name]
        descend(name, vals)
        if plan[name] != before:
            trims.append(f"{name} {before} -> {plan[name]}")
    m_vals = list(range(plan["m_write"], m_cap + 1, 2))
    if not m_vals or m_vals[-1] != m_cap:
        m_vals.append(m_cap)
    grid = [("funnel_shortlist", [v for v in (4096, 8192, 12288, 16384) if v <= sl_cap]),
            ("m_write", m_vals),
            ("lin_topk", [1, 2]),
            ("carrier_frac", [c_ for c_ in (0.01, 0.02, 0.04, 0.06, 0.08, 0.10) if c_ >= plan["carrier_frac"]]),
            ("k_ffn", [k_ for k_ in (1, 2, 4, 8, 16) if plan["k_ffn"] <= k_ <= 2 * plan["k_ffn"]])]
    for name, vals in grid:
        for val in vals:
            if val == plan[name]:
                continue
            trial = dict(plan)
            trial[name] = val
            tw = eval_plan(trial)
            if tw["TOTAL_ACTIVE"] <= target_active:
                plan, w = trial, tw
            else:
                break
    plan["feasible"] = bool(w["TOTAL_ACTIVE"] <= target_active)
    plan["trims"] = trims
    plan["requested"] = {k_: requested[k_] for k_ in ("mast_S", "shared_ffn", "m_write",
                                                      "funnel_shortlist", "shadow_lin")}
    plan["shadow_v"] = bool(shadow_v)
    plan["mast_mode"] = mast_mode
    if verbose:
        print(f"{'family':<18}{'active':>14}")
        for kk, vv in w.items():
            print(f"{kk:<18}{vv:>14,}")
        print(f"\nsolved plan: { {k: v for k, v in plan.items() if k not in ('trims', 'requested')} }")
        for t in trims:
            print(f"  [budget] trimmed: {t}")
        print(f"TOTAL ACTIVE = {w['TOTAL_ACTIVE']/1e9:.3f}B "
              f"(target {target_active/1e9:.2f}B, mast accounting: {mast_mode}) -> "
              f"{'OK' if plan['feasible'] else 'OVER BUDGET'}")
    return {"plan": plan, "waterfall": w}

RESONA_LINEARS: list = []      # (name, ResonantLinear) — whole-Linears swapped
RESONA_SECTIONED: list = []    # (name, SectionedProjector) — DeltaNet in_proj
RESONA_MORM: list = []         # (name, MoRMWritePolicy)
RESONA_FUNNEL: list = []       # (name, HeadFunnel) — the lm_head
MAST = None                    # the shared AntennaMast

# ---- global mode presets (one call re-aims the whole model) ----------------
def set_world(mode: str):
    """mode: 'dense_teacher'  = exact original dense model (teacher / oracle)
             'exact_resonant' = full resonant path, exact by Theorem 7
             'soft_train'     = differentiable all-expert paths (training)
             'compat_parity'  = FFN sparse, attention+head dense (COMPAT export
                                semantics — matches the stock-vLLM artifact)
             'serving'        = everything sparse (NATIVE artifact semantics)"""
    if mode == "dense_teacher":
        for _, m in PRISM_MODULES: m.default_mode = "uniform"
        for _, m in RESONA_LINEARS: m.default_mode = "dense"
        for _, m in RESONA_SECTIONED: m.default_mode = "dense"
        for _, m in RESONA_FUNNEL: m.default_mode = "dense"
    elif mode == "exact_resonant":
        for _, m in PRISM_MODULES: m.default_mode = "uniform"
        for _, m in RESONA_LINEARS: m.default_mode = "full"
        for _, m in RESONA_SECTIONED: m.default_mode = "full"
        for _, m in RESONA_FUNNEL: m.default_mode = "dense"
    elif mode == "soft_train":
        for _, m in PRISM_MODULES: m.default_mode = "soft"
        for _, m in RESONA_LINEARS: m.default_mode = "soft"
        for _, m in RESONA_SECTIONED: m.default_mode = "soft"
        for _, m in RESONA_FUNNEL: m.default_mode = "dense"
    elif mode == "compat_parity":
        for _, m in PRISM_MODULES: m.default_mode = "topk"
        for _, m in RESONA_LINEARS: m.default_mode = "dense"
        for _, m in RESONA_SECTIONED: m.default_mode = "dense"
        for _, m in RESONA_FUNNEL: m.default_mode = "dense"
    elif mode == "serving":
        for _, m in PRISM_MODULES: m.default_mode = "topk"
        for _, m in RESONA_LINEARS: m.default_mode = "topk"
        for _, m in RESONA_SECTIONED: m.default_mode = "topk"
        for _, m in RESONA_FUNNEL: m.default_mode = "funnel"
    else:
        raise ValueError(mode)
    # v8.0.1 [F4]: the BetaGate write levers (RESONA_MORM_GATES, filled by
    # the surgery cell) follow the world mode: dense_teacher / exact_resonant
    # / compat_parity need the stock BITWISE in_proj_ba (Theorem-7 anchor,
    # the free dense teacher, the COMPAT parity export), while soft_train /
    # serving run the write-sparsity. Pre-surgery the list is absent ->
    # this is a no-op.
    _gates = globals().get("RESONA_MORM_GATES") or []
    _gate_on = mode in ("soft_train", "serving")
    for _, _g in _gates:
        # v10: with a v shadow every head writes (the gate would delete the shadow writes)
        _g.enabled = _gate_on and not getattr(_g, "shadow_writes", False)
    for _, _m in PRISM_MODULES:            # the FFN shadow belongs to the NATIVE serving path only
        _m.use_shadow = (mode == "serving")
    return mode

print("RESONA core loaded (zero-materialization execution, module "
      "self-checkpointing, leak-free telemetry).")


## MoRM 2.0 — the oracle-scheduled write stack (v8)

MoRM 1.0 (v7) made the DeltaNet value-head memories **write-sparse** with a per-token top-m mask guided by write *energy*. MoRM 2.0 replaces the whole write path with a **rate-distortion scheduler** — the same four ideas that make a cache hierarchy fast, applied to recurrent memory:

| mechanism | what it replaces | certificate |
|---|---|---|
| **Write Influence Kernel** — the exact first-order counterfactual value of every write, closed-form from one captured pass (`write_influence_labels`) | the v-energy heuristic (its no-read-overlap special case) | Theorem B: single-write perturbation is EXACT (contract M2) |
| **Elastic budgets** — token-adaptive write counts, `E[m] = m_bar` via an adaptive rate threshold | fixed per-token top-m | Theorem D (contract M6) |
| **Deferred write-back** — losing writes park in a one-slot per-head buffer and commit later with exact decay compensation | irreversible skips | Theorem C: compensated commit == on-time from the commit point (contract M5) |
| **Charge ledger + staleness bound** — dropped writes charge their decaying debt; nothing waits > K tokens | nothing | Theorem E (contract M7) |

Plus the kernel-level correctness fix (**Proposition F**): FLA's WY form computes `u = A^{-1} (beta (.) v)` — so **beta-masking zeroes a write exactly in both kernel forms**, while v-masking (the v7 lever) leaks *deletion writes* in the delta-corrected form. `BetaGate` masks beta; the v-row gather stays as a pure compute saving. And the v7 gradient fix: the policy's straight-through signal now actually reaches the critic at both mask sites.

The unit of memory management is no longer "does head h write at token t" but "**what is the cheapest multiset of write operations whose removal the reads cannot see**" — writes as a rate-distortion problem with an exact oracle. `MoRM2Policy` is a drop-in replacement for `MoRMWritePolicy`; `MoRM2Decoder` drives decode; `verify_morm2()` (next cell) proves every claim above numerically.

At the end of the next cell lives `census_active` (v9): a walk over the live module graph that charges what one serving token touches — the budget checks in the surgery and evaluation cells use it.


In [ ]:
# ============================================================================
# MoRM 2.0 CORE (v8) — the standalone module, embedded verbatim
# (morm2.py ships alongside this notebook for local use; the cell and the
# file are generated from the same source and are byte-identical minus
# this header). Pure PyTorch, self-contained, CPU-verifiable.
# ============================================================================
"""
morm2.py — MoRM 2.0: Rate-Distortion Scheduling of Recurrent Memory Writes
===========================================================================

Upgrades MoRM 1.0 (Mixture of Recurrent Memories, PRISM/RESONA <= v7) from a
reactive per-token top-m write mask into a certified, anticipatory,
recoverable WRITE SCHEDULER for the value-head memories of Gated DeltaNet
(GDN) layers. Pure PyTorch, self-contained, CPU-verifiable.

--------------------------------------------------------------------------
MoRM 1.0 (v7) recap — and its two structural limits
--------------------------------------------------------------------------
MoRM 1.0 made the H per-head memory slabs of each GDN layer write-sparse:
a per-token policy picks m of H heads whose rank-1 state writes are actually
committed; the rest are skipped. Two structural limits:

  (L1) MYOPIC, ENERGY-GUIDED, IRREVERSIBLE WRITES. The policy scores heads
       by write ENERGY  e_h = beta_h * ||k_h|| * ||v_h||  — an immediate-
       magnitude proxy. But the value of a memory is its FUTURE RETRIEVAL:
       a write that is never read again is worthless at any energy, and a
       small write that a query 500 tokens later depends on is precious.
       And the skip is irreversible: information skipped at token t never
       enters the state, and nothing downstream can recover it.

  (L2) A FIXED WRITE RATE. Every token writes exactly m heads: whitespace,
       boilerplate and function words spend the same memory bandwidth as
       entity-dense tokens. The rate is a hyperparameter, never a
       controlled resource.

--------------------------------------------------------------------------
MoRM 2.0 — the four mechanisms (and their certificates)
--------------------------------------------------------------------------
  (A) WRITE INFLUENCE KERNEL (WIK) — exact counterfactual write values.
      The GDN recurrence is LINEAR in its writes (Theorem A):
          S_t = D_t S_{t-1} + w_t,      w_s := beta_s k_s delta_s^T,
          D_t = diag(lam_t),  delta_s = v_s                    (plain form)
                                 or v_s - S~_{s-1}^T k_s  (delta-corrected)
      so the perturbation of ANY future read from removing ONE write is a
      CLOSED-FORM quantity (Theorem B):
          ||delta o_t||_2 = scale * beta_s * |q_t^T (p(s,t) (*) k_s)|
                            * ||delta_s||_2,
          p(s,t) = prod_{tau=s+1..t} lam_tau   (per key dim).
      `write_influence_labels` computes, from ONE pass of trajectories
      (q, k, delta, beta, lam), the exact first-order counterfactual value
      of EVERY (token, head) write — zero ablation runs. MoRM 1.0's energy
      score is the degenerate special case with all read-overlaps set to 1.

  (B) ELASTIC WRITE RATE — token-adaptive budgets with E[m] = m_bar.
      A multiplicative rate controller (rate-distortion threshold): write
      head h at token t iff predicted influence L_hat_h(t) > lambda_t,
      with lambda_t adapting so the AVERAGE write count converges to the
      budget m_bar (Theorem D). Filler tokens write nothing; salient
      tokens write up to m_max heads. Same average cost as MoRM 1.0,
      strictly better allocation.

  (C) DEFERRED WRITE-BACK — skipped != lost.
      A write that loses the slot race is not dropped: it is PARKED in a
      one-slot per-head buffer and may COMMIT later with EXACT decay
      compensation (Theorem C): committing at t' the operands
          k~ = (prod_{tau=s+1..t'} lam_tau) (*) k_s,   v_s, beta_s
      gives, for every read at tau >= t', EXACTLY the state contribution
      the on-time write would have made (writes commute — the recurrence
      is linear in writes; verified numerically against the oracle). The
      deferral distortion is exactly the windowed influence
      I_h(s, [s, t')) — measurable, and precisely what the scheduler
      minimizes.

  (D) CHARGE LEDGER + STALENESS BOUND.
      Every dropped write charges its predicted remaining influence to a
      per-head ledger that AGES WITH THE HEAD'S OWN DECAY (the debt is
      forgiven exactly as fast as the information would have faded). The
      ledger is a policy input (the policy sees its own debt) and forces
      resolution: no pending write waits longer than K tokens (Theorem E)
      — it commits (pre-empting the fresh write, which inherits the slot
      and the buffer) or is dropped with its remaining value charged and
      certified.

  Plus a kernel-level CORRECTNESS FIX (Proposition F): the only write-
  masking lever that is exact in BOTH the plain outer-product form and the
  delta-corrected WY form of the stock FLA kernel is BETA-masking. FLA's
  `recompute_w_u_fwd_kernel` computes u = A^{-1} (beta (*) v) and
  w = A^{-1} (beta (*) k): beta = 0 turns row s of the unit-triangular
  factor into the identity row, hence u_s = w_s = 0 EXACTLY, while v = 0
  (MoRM 1.0's lever) leaks intra-chunk corrections — a *deletion write*
  -beta k k^T S~ instead of no write. MoRM 2.0 masks beta (and keeps the
  v-row gather purely as a compute saving, semantically inert once
  beta = 0).

--------------------------------------------------------------------------
Theorems (stated here; proven in MoRM2_THEORY.md; verified by verify_morm2)
--------------------------------------------------------------------------
  Theorem A (write superposition / commutation).  The state of the plain
    recurrence is an affine function of the multiset of writes; per-write
    contributions commute and never interact.
  Theorem B (influence exactness).  For a single removed write the output
    perturbation norm above is EXACT (no inequality); for a SET of removed
    writes the triangle inequality gives a certified bound, tight in the
    single-write case.
  Theorem C (deferred-commit exactness).  Plain form: the compensated
    commit reproduces the on-time state trajectory from the commit point
    onward EXACTLY. Delta-corrected form: exact for the carried write
    term; the correction term re-evaluates against the current state and
    is bounded by the same windowed-influence certificate.
  Theorem D (rate convergence).  lambda_{t+1} = lambda_t *
    exp(eta * (m_t - m_bar)) (under-budget -> lower the bar) converges in
    the stochastic-approximation sense to E[m_t] = m_bar for stationary
    score streams (eta -> 0 schedule supported).
  Theorem E (staleness bound).  With force-commit age K, every deferred
    write enters memory within K tokens of its origin or is dropped with
    its remaining influence charged (bounded by its WIK label).
  Proposition F (beta-mask exactness, kernel-level).  As above — verified
    against FLA's WY kernel algebra and numerically against a
    delta-corrected reference oracle.

Conventions: "influence" is an output-space L2 quantity (optionally
weighted per read); all internals compute in fp32 regardless of substrate
dtype; every function is batched over heads [H] and sequential over tokens
[T]. Shapes: q, k [T,H,dk]; v, delta [T,H,dv]; beta [T,H]; lam [T,H] or
[T,H,dk] in (0,1].
"""

from __future__ import annotations

import math
from dataclasses import dataclass, field
from typing import Callable, Optional, Union

import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================================
# Section 0 — Canonical GDN math + reference oracle
# ============================================================================
# Recurrence (per head h), S in R^{dk x dv}, o the read:
#   plain : S_t = diag(lam_t) S_{t-1} + beta_t k_t v_t^T
#   delta : S_t = diag(lam_t) S_{t-1} + beta_t k_t (v_t - S~^T k_t)^T,
#           S~ = diag(lam_t) S_{t-1}
#   read  : o_t = scale * S_t^T q_t        (write-then-read within a token)


def lam_to_full(lam: torch.Tensor, T: int, H: int, dk: int) -> torch.Tensor:
    """Broadcast lam (scalar / [H] / [T,H] / [T,H,dk]) to [T,H,dk] fp32."""
    if not torch.is_tensor(lam):
        return torch.full((T, H, dk), float(lam))
    if lam.dim() == 0:
        return lam.reshape(1, 1, 1).expand(T, H, dk).float()
    if lam.dim() == 1:
        assert lam.shape[0] == H
        return lam[None, :, None].expand(T, H, dk).float()
    if lam.dim() == 2:
        assert lam.shape[0] == T and lam.shape[1] == H
        return lam[:, :, None].expand(T, H, dk).float()
    if lam.dim() == 3:
        assert lam.shape == (T, H, dk)
        return lam.float()
    raise ValueError(f"lam must be 0/1/2/3-D, got {tuple(lam.shape)}")


def gdn_sequential(
    q: torch.Tensor, k: torch.Tensor, v: torch.Tensor,
    beta: torch.Tensor, lam: torch.Tensor,
    form: str = "delta", scale: float = 1.0,
    S0: Optional[torch.Tensor] = None,
    return_deltas: bool = False,
):
    """Pure-torch GDN reference oracle (sequential over T, batched over H).

    Returns (o [T,H,dv], S_final [H,dk,dv]) or, with return_deltas=True,
    (o, S_final, delta [T,H,dv]) where delta is the ACTUAL deposited value
    of each (token, head) rank-1 update — `v` for the plain form, the
    delta-corrected value for the delta form. delta is exactly the quantity
    the influence kernel needs: it is what a skipped write would actually
    have deposited.
    """
    assert form in ("plain", "delta"), form
    T, H, dk = q.shape
    dv = v.shape[-1]
    qf, kf, vf, bf = q.float(), k.float(), v.float(), beta.float()
    lamf = lam_to_full(lam, T, H, dk)
    S = (S0.float().clone() if S0 is not None
         else torch.zeros(H, dk, dv, device=q.device))
    o = torch.empty(T, H, dv, device=q.device, dtype=torch.float32)
    deltas = (torch.empty(T, H, dv, device=q.device, dtype=torch.float32)
              if return_deltas else None)
    for t in range(T):
        S = S * lamf[t][:, :, None]                      # diag(lam) S
        d = vf[t]
        if form == "delta":
            d = d - torch.einsum("hij,hi->hj", S, kf[t])  # v - S^T k
        if return_deltas:
            deltas[t] = d
        S = S + bf[t][:, None, None] * kf[t][:, :, None] * d[:, None, :]
        o[t] = torch.einsum("hi,hij->hj", qf[t], S) * scale
    if return_deltas:
        return o, S, deltas
    return o, S


def gdn_beta_masked_gap(
    q, k, v, beta, lam, mask, form="delta", scale=1.0, S0=None
) -> float:
    """|| o_dense - o_masked ||_F under the beta-mask lever (Prop F):
    masked heads write beta*mask = 0 — exact in both kernel forms."""
    o_dense, _ = gdn_sequential(q, k, v, beta, lam, form, scale, S0)
    o_sparse, _ = gdn_sequential(q, k, v, beta * mask, lam, form, scale, S0)
    return float((o_dense - o_sparse).norm())


# ============================================================================
# Section 1 — Write Influence Kernel (WIK): exact per-write labels
# ============================================================================
def write_influence_labels(
    q: torch.Tensor, k: torch.Tensor, delta: torch.Tensor,
    beta: torch.Tensor, lam: Union[torch.Tensor, float],
    scale: float = 1.0,
    read_weight: Optional[torch.Tensor] = None,
    window: int = 512,
    chunk: int = 64,
    coverage_horizon: Optional[int] = None,
) -> dict:
    """Exact first-order counterfactual value of every (token, head) write.

    For each write (h, s) with actual deposited value delta_s (Theorem B):

        I_h(s) = sum_{t=s}^{s+W-1}  w_t * scale * beta_s
                 * | q_t^T (p(s,t) (*) k_s) | * || delta_s ||_2

    with p(s,t) = prod_{tau=s+1..t} lam_tau (per key dim). This is the
    perturbation norm of read t when write (h, s) ALONE is removed — exact,
    with zero ablation runs (the read side is fully observed offline).

    Streaming windowed algorithm — memory O(H * W * dk), fp32 throughout:
      * a rolling buffer of decayed keys kt_s = p(s, b) (*) k_s for the
        window ending at block start b; each block ages the buffer by its
        decay product (the exponent is <= 0 everywhere: underflow-safe,
        never overflow);
      * block reads score against the buffer by one batched matmul
        (q (*) block_prefix_decay) @ kt^T;
      * intra-block (s in the same block) pairs score through the masked
        relative-decay attention exp(Gc_t - Gc_s), t >= s (exponent <= 0).

    Args:
      q, k  : [T,H,dk] kernel-space q, k (apply the kernel's L2 norm here
              if it normalizes; fold its attn-scale into `scale`).
      delta : [T,H,dv] actual written values (gdn_sequential with
              return_deltas=True; or v for the plain form).
      beta  : [T,H];  lam : [T,H] / [T,H,dk] / scalar.
      read_weight : [T] or [T,H] optional per-read weights (e.g. |dL/do_t|).
      window : influence horizon W (labels exact within it).
      chunk  : processing block C (memory knob; math is chunk-invariant).
      coverage_horizon : if set (default 2*window), also accumulate labels
              to this longer horizon and report the windowed coverage.

    Returns dict:
      labels       [T,H]  windowed influence I_h(s)
      labels_long  [T,H]  labels to `coverage_horizon` (if requested)
      coverage     float  mean fraction of the long-horizon influence
                          captured inside `window` (1.0 => window covers
                          essentially everything)
    """
    T, H, dk = q.shape
    W = max(1, min(window, T))
    C = max(1, min(chunk, W))
    qf, kf = q.float(), k.float()
    df, bf = delta.float(), beta.float()
    lamf = lam_to_full(lam, T, H, dk)
    loglam = lamf.clamp_min(1e-12).log()                 # <= 0

    rw = None
    if read_weight is not None:
        rw = read_weight.float()
        rw = rw[:, None].expand(T, H) if rw.dim() == 1 else rw

    dnorm = df.norm(dim=-1)                              # [T,H]
    acc = torch.zeros(T, H, device=q.device)             # short-horizon
    want_long = coverage_horizon is not None
    Wl = max(1, min(coverage_horizon or W, T))
    acc_l = torch.zeros(T, H, device=q.device) if want_long else None
    Wb = Wl if want_long else W                          # buffer length

    kt_buf = torch.zeros(H, Wb, dk, device=q.device)     # decayed keys
    idx_buf = torch.full((Wb,), -1, dtype=torch.long, device=q.device)

    for b in range(0, T, C):
        e = min(b + C, T)
        c = e - b
        g_blk = loglam[b:e]                              # [c,H,dk]
        Gc = torch.cumsum(g_blk, dim=0)                  # inclusive prefix
        qb = qf[b:e]                                     # [c,H,dk]

        # ---- (1) intra-block pairs: s, t in [b, e), t >= s ---------------
        # score(b+i, b+j) = q^T ( exp(Gc[i] - Gc[j]) (*) k_j ),  i >= j
        RC = max(1, (1 << 22) // max(1, H * dk * c))     # row sub-chunk
        rows = torch.arange(c, device=q.device)
        w_blk = rw[b:e] if rw is not None else None      # [c,H] or None
        GcT = Gc.permute(1, 0, 2)                         # [H, c, dk]
        kT = kf[b:e].permute(1, 0, 2)                    # [H, c, dk]
        for r0 in range(0, c, RC):
            r1 = min(r0 + RC, c)
            # rel[i,h,j,d] = exp(Gc[r0+i,h,d] - Gc[b+j,h,d]).
            # v8.0.2 [G8]: the exponent is <= 0 ONLY on the causal side
            # (i >= j); the masked-out FUTURE side goes positive and, with
            # real GDN decays (loglam down to ~-11 per token), a 64-token
            # block overflows exp to inf — then inf * 0 from the causal
            # mask is NaN and poisons every label downstream (ridge warm
            # start, critic, certificates). This never fired before
            # because the WIK capture itself was broken and the label
            # kernel never saw real data. Clamp the exponent to <= 0
            # BEFORE exp: a no-op on the causal side (exact), and the
            # masked side becomes finite garbage the mask zeroes out.
            rel = (Gc[r0:r1, :, None, :] - GcT[None, :, :, :]).clamp(
                max=0.0).exp()
            sc = (qb[r0:r1, :, None, :] * kT[None, :, :, :]
                  * rel).sum(-1)                         # [r,H,c]
            val = sc.abs().float()
            val = val * (rows[None, None, :] <=
                         (r0 + torch.arange(r1 - r0, device=q.device))[
                             :, None, None]).float()
            if w_blk is not None:
                val = val * w_blk[r0:r1, :, None]
            add = val.sum(0).t()                         # [c,H] over reads
            acc.index_add_(0, torch.arange(b, e, device=q.device), add)
            if want_long:
                acc_l.index_add_(0, torch.arange(b, e, device=q.device), add)

        # ---- (2) cross-buffer pairs: s < b, t in [b, e) -------------------
        valid = idx_buf >= 0
        n_val = int(valid.sum())
        if n_val > 0:
            # score(b+i, s_j) = (q_{b+i} (*) exp(Gc[i]))^T kt_j
            qP = qb * Gc.exp()                           # [c,H,dk]
            sc = torch.einsum("chd,hwd->chw", qP, kt_buf[:, :n_val])
            val = sc.abs()
            if w_blk is not None:
                val = val * w_blk[:, :, None]
            add_long = val.sum(0).t()                    # [n_val,H]
            s_ids = idx_buf[:n_val]
            # short-window mask: only reads with t - s < W count for `acc`;
            # every pair counts for the long accumulator.
            ages = (b - s_ids).float()                   # s_j distance to b
            tpos = torch.arange(c, device=q.device).float()
            in_win = (ages[None, :] + tpos[:, None]) < W  # [c, n_val]
            add_short = (val * in_win[:, None, :]).sum(0).t()
            acc.index_add_(0, s_ids, add_short)
            if want_long:
                acc_l.index_add_(0, s_ids, add_long)

        # ---- (3) age buffer by the block decay; append this block --------
        if n_val > 0:
            kt_buf[:, :n_val] = kt_buf[:, :n_val] * Gc[-1].exp()[:, None, :]
        new_idx = torch.arange(b, e, device=q.device)
        # decay each new token to the END of the block (so it is consistent
        # with the aged old entries at the next block boundary)
        tail = (Gc[-1][None, :, :] - Gc).exp()           # [c,H,dk], <= 1
        new_kt = kf[b:e] * tail                          # [c,H,dk]
        all_idx = torch.cat([idx_buf[:n_val], new_idx])
        all_kt = torch.cat([kt_buf[:, :n_val], new_kt.permute(1, 0, 2)],
                           dim=1)                        # [H, n_val+c, dk]
        keep = max(0, all_idx.shape[0] - Wb)
        m = all_idx.shape[0] - keep
        idx_buf = torch.full((Wb,), -1, dtype=torch.long, device=q.device)
        idx_buf[:m] = all_idx[keep:]
        kt_buf = torch.zeros(H, Wb, dk, device=q.device)
        kt_buf[:, :m] = all_kt[:, keep:]

    labels = acc * bf * dnorm * scale                    # [T,H]
    out = {"labels": labels}
    if want_long:
        labels_l = acc_l * bf * dnorm * scale
        # coverage measurable only where the LONG horizon is fully realized:
        # s <= T - Wl
        n_cov = T - Wl
        if n_cov > 8:
            sl = labels[:n_cov].sum()
            ll = labels_l[:n_cov].sum().clamp_min(1e-30)
            out["labels_long"] = labels_l
            out["coverage"] = float((sl / ll).clamp(0, 1))
        else:
            out["labels_long"] = labels_l
            out["coverage"] = 1.0
    return out


# ============================================================================
# Section 2 — Elastic write rate: the rate-distortion controller (Theorem D)
# ============================================================================
class ElasticBudgetController:
    """Multiplicative rate controller for token-adaptive write budgets.

    Write rule (per token): head h writes iff its content score exceeds the
    running threshold lambda (capped at m_max heads per token). After each
    token the threshold moves toward the budget:

        lambda <- lambda * exp(eta * (m_t - m_bar))

    (too few writes this token -> lower the bar, and vice versa) — a
    stochastic-approximation update whose equilibrium is E[m_t] = m_bar
    for stationary score streams (Theorem D). Supports the Robbins-Monro
    eta -> 0 schedule (`lr_decay_tokens`) for exact convergence, and the
    exact-mode switch m_bar = H (threshold driven below every score ->
    every head writes -> bit-exact dense behavior).
    """

    def __init__(self, m_bar: int, m_max: Optional[int] = None,
                 lr: float = 0.08, lr_decay_tokens: Optional[int] = None,
                 init_threshold: Optional[float] = None):
        self.m_bar = int(m_bar)
        self.m_max = None if m_max is None else int(max(m_max, m_bar))
        self.lr = float(lr)
        self.lr_decay_tokens = lr_decay_tokens
        self._steps = 0
        # positive seed: the update is MULTIPLICATIVE, so 0.0 would be a
        # fixed point — callers warm-start from a score quantile (see
        # schedule_writes / MoRM2Policy.forward) for fast convergence
        self.threshold = (float(init_threshold) if init_threshold is not None
                          else 1e-6)

    @property
    def exact_mode(self) -> bool:
        """True when every head always writes (bit-exact dense behavior)."""
        return self.m_max is None and self.m_bar >= 10**9

    def update(self, m_t: int) -> float:
        """One controller step; returns the new threshold.

        m_t < m_bar (under-budget) -> threshold decreases (admit more);
        m_t > m_bar -> threshold increases (admit fewer)."""
        eta = self.lr
        if self.lr_decay_tokens is not None:
            eta = self.lr / (1.0 + self._steps / self.lr_decay_tokens)
        self.threshold *= math.exp(eta * (m_t - self.m_bar))
        self.threshold = min(self.threshold, 1e30)      # numeric guard
        self._steps += 1
        return self.threshold

    def state_dict(self) -> dict:
        return {"threshold": self.threshold, "steps": self._steps,
                "m_bar": self.m_bar, "m_max": self.m_max}

    def load_state_dict(self, d: dict):
        self.threshold = float(d["threshold"])
        self._steps = int(d["steps"])


# ============================================================================
# Section 3 — The write scheduler: deferral, charge ledger, staleness
# ============================================================================
@dataclass
class ScheduleStats:
    """Telemetry of one scheduling run (all counts over T tokens)."""
    m_trace: torch.Tensor            # [T] writes per token
    writes: int = 0                  # fresh writes executed
    commits: int = 0                 # deferred writes committed
    defers: int = 0                  # fresh writes parked in the buffer
    drops: int = 0                   # writes dropped (charged to the ledger)
    max_staleness: int = 0           # max age at commit (<= K by Theorem E)
    mean_charge: float = 0.0
    mean_m: float = 0.0

    def summary(self) -> str:
        return (f"mean m = {self.mean_m:.2f} | fresh {self.writes} | "
                f"commits {self.commits} | defers {self.defers} | "
                f"drops {self.drops} | max staleness {self.max_staleness}")


class WriteScheduler:
    """The MoRM 2.0 causal write scheduler (one instance per GDN layer).

    Per token, for every head (vectorized over [H]):

      1. AGE      pendings decay by the CURRENT token's lam — so a pending
                  committed at t carries exactly the Theorem-C compensation
                  prod_{tau=s+1..t} lam_tau; the charge ledger ages the same
                  way (debt is forgiven as fast as the information would
                  have faded).
      2. DECIDE   content per head = pending (forced if age >= K, else the
                  larger of remaining value R and fresh score F); the head
                  writes iff content score > lambda_rate, capped at m_max.
      3. RESOLVE  writers emit fresh writes or compensated commits; losers
                  follow the deferral ladder:
                    fresh above threshold, slot lost -> buffer (one slot per
                      head; a weaker incumbent is evicted, dropped, charged)
                    fresh below threshold             -> dropped, charged
                    pending decayed below threshold   -> dropped, charged
                  a forced commit pre-empts a fresh write: the pending takes
                  the slot and the fresh write inherits the buffer.
      4. ADAPT    the rate controller sees m_t.

    All decisions are causal (scores/observables at or before t). The
    emitted (k_eff, v_eff, beta_eff) feed the STOCK kernel unchanged
    (input rewriting only — zero kernel surgery). Skipped heads carry
    beta_eff = 0: the Proposition-F exact mask.
    """

    def __init__(self, n_heads: int, dk: int, dv: int,
                 m_bar: int, staleness_k: int = 64,
                 m_max: Optional[int] = None,
                 controller_lr: float = 0.08,
                 init_threshold: Optional[float] = None,
                 v_provider: Optional[Callable] = None):
        self.H, self.dk, self.dv = n_heads, dk, dv
        self.K = int(staleness_k)
        self.v_provider = v_provider
        self.controller = ElasticBudgetController(
            m_bar=m_bar, m_max=m_max, lr=controller_lr,
            init_threshold=init_threshold)
        self.reset()

    def reset(self):
        self.pend_k = torch.zeros(self.H, self.dk)
        self.pend_v = torch.zeros(self.H, self.dv)
        self.pend_beta = torch.zeros(self.H)
        self.pend_R = torch.zeros(self.H)               # remaining value
        self.pend_age = torch.zeros(self.H, dtype=torch.long)
        self.pend_valid = torch.zeros(self.H, dtype=torch.bool)
        self.pend_t = torch.zeros(self.H, dtype=torch.long)  # origin token
        self.charge = torch.zeros(self.H)
        self.stats = ScheduleStats(m_trace=torch.zeros(0, dtype=torch.long))

    def to(self, device):
        for name in ("pend_k", "pend_v", "pend_beta", "pend_R",
                     "pend_age", "pend_valid", "pend_t", "charge"):
            setattr(self, name, getattr(self, name).to(device))
        return self

    # ------------------------------------------------------------------ step
    def step(self, scores_t: torch.Tensor, k_t: torch.Tensor,
             v_t: Optional[torch.Tensor], beta_t: torch.Tensor,
             lam_t: torch.Tensor, t: int = 0) -> dict:
        """Advance one token.

        scores_t [H] predicted influence (>= 0); k_t [H,dk]; v_t [H,dv] or
        None (then `v_provider(t_orig, heads)` must be set on the scheduler
        to materialize deferred values at commit time); beta_t [H];
        lam_t [H] or [H,dk] — this token's decay.
        """
        H, dev = self.H, scores_t.device
        if self.pend_k.device != dev:
            self.to(dev)
        # control plane: the scheduler is non-differentiable by design (the
        # policy's STE handles training) — detach everything it stores
        F = scores_t.detach().float().to(dev).clamp_min(0.0)   # fresh scores
        # per-key-dim decay of THIS token, [H, dk]
        if lam_t.dim() == 2:
            lam_full = lam_t.detach().float().to(dev)
        elif lam_t.dim() == 1:
            lam_full = lam_t.detach().float().to(dev)[:, None].expand(H, self.dk)
        else:
            lam_full = lam_t.detach().float().to(dev).reshape(1, 1).expand(H, self.dk)
        lam_scalar = lam_full.mean(-1).clamp(1e-6, 1.0)  # [H] aging rate

        # ---- 1. AGE (compensation clock + debt forgiveness) ---------------
        self.pend_k = self.pend_k * lam_full
        self.pend_R = self.pend_R * lam_scalar
        self.charge = self.charge * lam_scalar
        self.pend_age = self.pend_age + self.pend_valid.long()

        # ---- 2. DECIDE ----------------------------------------------------
        thr = self.controller.threshold
        R = torch.where(self.pend_valid, self.pend_R,
                        torch.full_like(self.pend_R, -1.0))
        force = self.pend_valid & (self.pend_age >= self.K)  # Theorem E
        commit_content = self.pend_valid & (force | (R >= F))
        content = torch.where(commit_content, R, F)
        wants = content > thr
        # Staleness guarantee (Theorem E): a pending that reached age K and
        # is still worth a slot (R above the rate line) commits NOW,
        # pre-empting the threshold/cap (its m_t is still counted, so the
        # controller adapts). Below the line it is dropped + charged (iv).
        forced_commit = force & (self.pend_R > thr)
        if self.controller.m_max is not None and self.controller.m_max < H:
            over = int((wants & ~forced_commit).sum()) - self.controller.m_max
            if over > 0:
                sc = torch.where(wants & ~forced_commit, content,
                                 torch.full_like(content, -1.0))
                drop_idx = sc.topk(over, largest=False).indices
                wants[drop_idx] = False
        wants = wants | forced_commit
        m_t = int(wants.sum())

        write = wants
        commit = write & commit_content
        fresh = write & ~commit_content

        # ---- 3a. EMIT (fresh writes + compensated commits) ----------------
        k_eff = k_t.detach().float().to(dev).clone()
        beta_eff = torch.where(write, beta_t.detach().float().to(dev),
                               torch.zeros(H, device=dev))
        v_eff = (v_t.detach().float().to(dev).clone() if v_t is not None
                 else torch.zeros(H, self.dv, device=dev))
        commit_heads = commit.nonzero(as_tuple=False).squeeze(-1)
        if commit_heads.numel() > 0:
            k_eff[commit_heads] = self.pend_k[commit_heads]
            beta_eff[commit_heads] = self.pend_beta[commit_heads]
            if v_t is not None:
                v_eff[commit_heads] = self.pend_v[commit_heads]
            elif self.v_provider is not None:
                v_eff[commit_heads] = self.v_provider(
                    self.pend_t[commit_heads], commit_heads)
            else:
                raise RuntimeError(
                    "deferred commit with neither v nor v_provider — "
                    "a commit must carry its buffered value")
            self.stats.max_staleness = max(
                self.stats.max_staleness,
                int(self.pend_age[commit_heads].max()))
        v_eff = torch.where(write[:, None], v_eff, torch.zeros_like(v_eff))

        # ---- 3b. RESOLVE the losers ---------------------------------------
        # (i) fresh above threshold but no slot: park into the buffer;
        #     evict (drop + charge) a weaker incumbent if occupied
        fresh_park = (~write) & (F > thr)
        incumbent = torch.where(self.pend_valid, self.pend_R,
                                torch.full_like(self.pend_R, -1.0))
        evict = fresh_park & self.pend_valid & (F > incumbent)
        park = fresh_park & (~self.pend_valid | evict)
        evicted_R = torch.where(evict, self.pend_R,
                                torch.zeros_like(self.pend_R))
        self.charge += evicted_R                           # evictions charged
        # (ii) a forced commit pre-empts a fresh write above threshold:
        #      the fresh write inherits the freed buffer slot
        preempt_park = commit & (F > thr)
        # (iii) fresh below threshold (or lost to a stronger incumbent):
        #      dropped, charged
        fresh_drop = ((~write) & (F <= thr)) | \
                     (fresh_park & self.pend_valid & ~evict)
        # (iv) pending decayed below the threshold: dropped, charged
        pend_drop = self.pend_valid & (~write) & (self.pend_R <= thr)

        # writers' pending bookkeeping: commits free their slot
        if commit_heads.numel() > 0:
            self.pend_valid.index_fill_(0, commit_heads, False)

        # park deferred fresh writes (store the RAW operands — never the
        # masked emit values, and never the commit-overridden k_eff of a
        # pre-empting head)
        park_idx = (park | preempt_park).nonzero(as_tuple=False).squeeze(-1)
        if park_idx.numel() > 0:
            self.pend_k[park_idx] = k_t.detach().float().to(dev)[park_idx]
            if v_t is not None:
                self.pend_v[park_idx] = v_t.detach().float().to(dev)[park_idx]
            self.pend_beta[park_idx] = beta_t.detach().float().to(dev)[park_idx]
            self.pend_R[park_idx] = F[park_idx]
            self.pend_t[park_idx] = t
            self.pend_age[park_idx] = 0
            self.pend_valid[park_idx] = True
            self.stats.defers += int(park_idx.numel())

        # charged drops
        fdi = fresh_drop.nonzero(as_tuple=False).squeeze(-1)
        if fdi.numel() > 0:
            self.charge[fdi] += F[fdi]
        pdi = pend_drop.nonzero(as_tuple=False).squeeze(-1)
        if pdi.numel() > 0:
            self.charge[pdi] += self.pend_R[pdi]
            self.pend_valid.index_fill_(0, pdi, False)
        self.stats.drops += int(fdi.numel() + pdi.numel() + evict.sum())

        # ---- 4. TELEMETRY + controller ------------------------------------
        self.stats.writes += int(fresh.sum())
        self.stats.commits += int(commit.sum())
        self.stats.m_trace = torch.cat(
            [self.stats.m_trace, torch.tensor([m_t])])
        self.stats.mean_charge = float(self.charge.detach().mean())
        self.stats.mean_m = float(self.stats.m_trace.float().mean())
        self.controller.update(m_t)

        return {"mask": write, "commit": commit, "k_eff": k_eff,
                "v_eff": v_eff, "beta_eff": beta_eff, "m_t": m_t}


def schedule_writes(
    scores: torch.Tensor, k: torch.Tensor,
    v: Optional[torch.Tensor], beta: torch.Tensor, lam: torch.Tensor,
    m_bar: int, staleness_k: int = 64, m_max: Optional[int] = None,
    controller_lr: float = 0.08,
    controller_lr_decay_tokens: Optional[int] = None,
    init_threshold: Optional[float] = None,
) -> dict:
    """Offline (prefill-style) causal scheduling over a full sequence.

    Args: scores [T,H] predicted influences (causal); k [T,H,dk]; v
      [T,H,dv] or None (deferral without v buffers (k, beta) and requires
      the caller's own value source at commit — pass v in tests/eval);
      beta [T,H]; lam [T,H] or [T,H,dk].
    Returns dict: mask [T,H] (1 = write), commit [T,H] (1 = deferred
      commit), k_eff / beta_eff [T,...] (beta_eff == 0 exactly on skipped
      heads — the Proposition-F lever; commits carry the Theorem-C
      compensated key), v_eff (or None), stats (ScheduleStats), charge [H].
    """
    T, H, dk = k.shape
    # threshold init: explicit > exact mode > warm start from the first
    # tokens' score distribution (the warm start OVER-ADMITS deliberately —
    # half the target quantile — so the controller only ever tightens)
    init_thr = init_threshold
    w0 = min(64, T)
    if init_thr is None:
        if m_bar >= H:
            init_thr = -1.0
        elif w0 > 0:
            q = max(0.0, 1.0 - m_bar / H)
            init_thr = 0.5 * float(scores[:w0].flatten().float()
                                   .quantile(q).clamp_min(1e-9))
    sched = WriteScheduler(H, dk, v.shape[-1] if v is not None else 1,
                           m_bar=m_bar, staleness_k=staleness_k,
                           m_max=m_max, controller_lr=controller_lr,
                           init_threshold=init_thr)
    if controller_lr_decay_tokens is not None:
        sched.controller.lr_decay_tokens = controller_lr_decay_tokens
    sched.to(scores.device)
    masks = torch.zeros(T, H, device=scores.device)
    commits = torch.zeros(T, H, device=scores.device)
    k_eff = k.float().clone()
    beta_eff = beta.float().clone()
    v_eff = (v.float().clone() if v is not None else None)
    lam_full = lam_to_full(lam, T, H, dk)
    for t in range(T):
        out = sched.step(scores[t], k[t], v[t] if v is not None else None,
                         beta[t], lam_full[t], t=t)
        masks[t] = out["mask"]
        commits[t] = out["commit"]
        k_eff[t] = out["k_eff"]
        beta_eff[t] = out["beta_eff"]
        if v is not None:
            v_eff[t] = out["v_eff"]
    return {"mask": masks, "commit": commits, "k_eff": k_eff,
            "v_eff": v_eff, "beta_eff": beta_eff,
            "stats": sched.stats, "charge": sched.charge}


# ============================================================================
# Section 4 — MoRM2Policy: the trainable, drop-in write policy
# ============================================================================
class MoRM2Policy(nn.Module):
    """Influence-predicting write policy with an elastic budget.

    Drop-in replacement for MoRM 1.0's `MoRMWritePolicy` (same forward
    contract: `policy(mast(x)) -> [T, H] mask`), upgraded to:

      * an INFLUENCE CRITIC — per-head linear heads over the shared
        AntennaMast features phi(h) PLUS per-head observable features
        (log write energy, log key norm, log(1+charge), pending-age/K),
        predicting the WIK influence label in log space;
      * an ELASTIC budget — the write set is threshold(L_hat > lambda_t)
        with the rate controller adapting lambda_t so E[m_t] = m_bar,
        instead of a fixed top-m;
      * a straight-through soft threshold (sigmoid((L_hat - lambda)/tau))
        so distillation gradients reach the critic;
      * the EXACT-MODE safety switch: m_bar = n_heads => mask == 1
        identically (bit-exact dense writes, one config away — inherited
        from MoRM 1.0).

    Charge/pending features are decode-time scheduler state; during
    training (batched, stateless across batches) they are zero and the
    critic still sees phi + energy observables.
    """

    N_OBS = 4   # [log(beta*||k||+eps), log(1+||k||), log(1+charge), age/K]

    def __init__(self, n_heads: int, mast, m_bar: int,
                 m_max: Optional[int] = None,
                 controller_lr: float = 0.08,
                 ste_tau: float = 1.0,
                 use_obs: bool = False,
                 critic_bias: bool = True):
        """use_obs=False (default): the critic consumes phi only — the
        convention the notebook's mask sites run (beta/knorm observables are
        not uniformly available at every call site). use_obs=True: the
        caller guarantees per-head obs at every scores()/forward() call
        (e.g. the decode scheduler, which has beta, k-norm, charge and
        pending-age) — then ridge_init(obs=...) fits both critic blocks."""
        super().__init__()
        self.H = n_heads
        self.mast = mast                                # shared, not owned
        self.m_bar = int(m_bar)
        self.m_max = m_max
        self.ste_tau = ste_tau
        self.use_obs = bool(use_obs)
        # v9: the critic carries a per-head INTERCEPT (the log-influence LEVEL of each head). v8's bias-free
        # critic + mean-removed ridge fit kept the slope and lost every head's mean level: exp(score) was right
        # only up to an arbitrary per-head factor, and the elastic threshold is shared by all heads
        # (pooled log-R^2 -5..-8 on the calibration labels; ~7.5 points less influence mass at an equal budget).
        self.critic_phi = nn.Linear(mast.S, n_heads, bias=bool(critic_bias))
        self.critic_obs = nn.Linear(self.N_OBS, n_heads, bias=False)
        nn.init.zeros_(self.critic_phi.weight)
        if self.critic_phi.bias is not None:
            nn.init.zeros_(self.critic_phi.bias)
        nn.init.zeros_(self.critic_obs.weight)
        # the live rate threshold (controller state, NOT gradient-trained)
        self.register_buffer("rate_threshold", torch.tensor(0.0))
        self.controller = ElasticBudgetController(
            m_bar=m_bar, m_max=m_max, lr=controller_lr)
        self.last_mask: Optional[torch.Tensor] = None
        self.last_m: Optional[torch.Tensor] = None
        self.last_scores: Optional[torch.Tensor] = None

    # -- exact mode -----------------------------------------------------------
    @property
    def exact_mode(self) -> bool:
        return self.m_bar >= self.H

    def set_budget(self, m_bar: int, m_max: Optional[int] = None):
        """Retarget the elastic budget (used by the training annealer)."""
        self.m_bar = int(m_bar)
        self.controller.m_bar = int(m_bar)
        self.controller.m_max = None if m_max is None else \
            int(max(m_max, m_bar))

    # -- critic ----------------------------------------------------------------
    def scores(self, phi: torch.Tensor,
               obs: Optional[torch.Tensor] = None) -> torch.Tensor:
        """phi [T,S], obs [T,H,N_OBS] -> predicted influence [T,H] (>=0)."""
        s = self.critic_phi(phi)
        if obs is not None and self.use_obs:
            s = s + torch.einsum("thf,hf->th", obs.float(),
                                 self.critic_obs.weight.float())
        return s.float().exp()

    def ridge_init(self, phi: torch.Tensor, obs: torch.Tensor,
                   labels: torch.Tensor, ridge: float = 1e-2):
        """Per-head ridge fit of log-influence on [phi || obs] features (+ v9: the per-head intercept).

        phi [N,S]; obs [N,H,N_OBS]; labels [N,H]. Solves H independent
        small least-squares systems (one per head) — the exact joint
        supervised warm start for the critic (MoRM 1.0 warm-started on
        v-ENERGY; MoRM 2.0 warm-starts on the exact influence labels).
        """
        with torch.no_grad():
            X_phi = phi.float()
            y = labels.float().clamp_min(1e-12).log()    # [N,H]
            Xo = obs.float() if (obs is not None and self.use_obs) \
                else torch.zeros(phi.shape[0], self.H, self.N_OBS,
                                 device=phi.device)      # [N,H,N_OBS]
            mu = X_phi.mean(0, keepdim=True)
            Xc = X_phi - mu
            for h in range(self.H):
                Xh = torch.cat([Xc, Xo[:, h, :]], dim=1)  # [N, S+N_OBS]
                yh = y[:, h] - y[:, h].mean()
                Xc2 = Xh - Xh.mean(0, keepdim=True)
                lam = ridge * Xc2.pow(2).mean().clamp_min(1e-12)
                A = torch.linalg.solve(
                    Xc2.t() @ Xc2 + lam * torch.eye(Xc2.shape[1],
                                                    device=Xc2.device),
                    Xc2.t() @ yh)
                self.critic_phi.weight.data[h] = A[:self.mast.S].to(
                    self.critic_phi.weight.dtype)
                self.critic_obs.weight.data[h] = A[self.mast.S:].to(
                    self.critic_obs.weight.dtype)
                if self.critic_phi.bias is not None:
                    # v9 INTERCEPT. The fit is on mean-removed y and features, i.e. the prediction at a raw
                    # input is  ybar_h + (phi - mu).A_phi + (obs - obsbar_h).A_obs ; the constant part
                    #   b_h = ybar_h - mu.A_phi - obsbar_h.A_obs
                    # has to live in the critic (v8 dropped it).
                    b_h = (y[:, h].mean() - mu[0] @ A[:self.mast.S]
                           - Xo[:, h, :].mean(0) @ A[self.mast.S:])
                    self.critic_phi.bias.data[h] = b_h.to(self.critic_phi.bias.dtype)

    # -- v1-compatible forward ---------------------------------------------------
    def forward(self, phi: torch.Tensor,
                obs: Optional[torch.Tensor] = None,
                m: Optional[int] = None) -> torch.Tensor:
        """phi [T,S] (obs [T,H,N_OBS] optional) -> hard write mask [T,H].

        Elastic mode (default): threshold(L_hat > rate_threshold), capped
        at m_max, straight-through-smoothed in training. If `m` is given,
        falls back to MoRM 1.0 fixed top-m semantics (compat hook for the
        v1 verification contracts).
        """
        T = phi.shape[0]
        if self.exact_mode and m is None:
            mask = phi.new_ones(T, self.H)
            self.last_mask, self.last_m = mask.detach(), None
            self.last_scores = None
            return mask
        L = self.scores(phi, obs)                        # [T,H] predicted
        if m is not None:                                # v1 compat: top-m
            mm = min(m, self.H)
            tv, ti = L.topk(mm, dim=-1)
            hard = torch.zeros_like(L).scatter_(-1, ti, 1.0)
            if self.training:
                soft = torch.zeros_like(L).scatter_(
                    -1, ti, torch.softmax(tv.float(), -1).to(L.dtype))
                hard = hard + (soft - soft.detach())
            self.last_mask, self.last_m = hard.detach(), hard.sum(-1).detach()
            self.last_scores = L.detach()
            return hard
        # lazy warm start: multiplicative control cannot leave 0.0, so the
        # first forward seeds the threshold from this batch's score
        # distribution (deliberately over-admitting: half the target
        # quantile — the controller only ever tightens from here)
        if float(self.rate_threshold) <= 0.0:
            with torch.no_grad():
                q0 = max(0.0, 1.0 - self.m_bar / self.H)
                thr0 = 0.5 * float(L.detach().flatten().quantile(q0)
                                   .clamp_min(1e-12))
                self.rate_threshold.fill_(thr0)
                self.controller.threshold = thr0
        thr = float(self.rate_threshold)
        hard = (L > thr).to(L.dtype)
        if (self.controller.m_max is not None
                and self.controller.m_max < self.H):
            over = hard.sum(-1, keepdim=True) > self.controller.m_max
            if bool(over.any()):
                ti = L.topk(self.controller.m_max, dim=-1).indices
                capped = torch.zeros_like(L).scatter_(-1, ti, 1.0)
                hard = torch.where(over, capped * (L > thr).to(L.dtype),
                                   hard)
        if self.training:                                # smooth-threshold STE
            soft = torch.sigmoid((L - thr) / max(self.ste_tau, 1e-6))
            hard = hard + (soft - soft.detach())
        self.last_mask = hard.detach()
        self.last_m = (hard.detach() > 0.5).sum(-1)
        self.last_scores = L.detach()
        return hard

    def controller_update(self, m_mean: float) -> float:
        """Feed the observed mean write count to the rate controller."""
        if not self.exact_mode:
            self.controller.update(m_mean)
            self.rate_threshold.fill_(self.controller.threshold)
        return float(self.rate_threshold)

    def obs_features(self, beta: torch.Tensor, knorm: torch.Tensor,
                     charge: Optional[torch.Tensor] = None,
                     age: Optional[torch.Tensor] = None) -> torch.Tensor:
        """Build the per-head observable features [T,H,N_OBS].

        beta/knorm: [T,H]; charge [H] or [T,H]; age [H] or [T,H] (in
        units of K). Charge/age default to 0 (training / prefill).
        """
        T = beta.shape[0]
        z = beta.new_zeros(T, self.H)
        ch = charge if (charge is not None and charge.dim() == 2) \
            else (charge[None, :].expand(T, self.H)
                  if charge is not None else z)
        ag = age if (age is not None and age.dim() == 2) \
            else (age[None, :].expand(T, self.H)
                  if age is not None else z)
        return torch.stack([
            (beta.float() * knorm.float()).clamp_min(1e-12).log(),
            knorm.float().clamp_min(1e-12).log1p(),
            ch.float().clamp_min(0).log1p(),
            ag.float().clamp(0, 1.5)], dim=-1)


# ============================================================================
# Section 5 — Certificates + baselines
# ============================================================================
def energy_scores(k: torch.Tensor, v: torch.Tensor,
                  beta: torch.Tensor) -> torch.Tensor:
    """MoRM 1.0's heuristic write score: beta * ||k|| * ||v||  [T,H]."""
    return beta.float() * k.float().norm(dim=-1) * v.float().norm(dim=-1)


def morm2_skip_certificate(
    q, k, delta, beta, lam, mask, scale: float = 1.0,
    read_weight=None, window: int = 512,
) -> dict:
    """Certified output deviation of a write mask (Theorem B + triangle).

    The bound is the sum over SKIPPED writes of their exact per-write
    influence labels:

        || o_dense - o_masked ||_{1,weighted} <= sum_{skipped} I_h(s)

    (triangle over exact single-write perturbations). MoRM 1.0's
    Proposition-6 energy bound is the special case with read overlaps
    replaced by 1 — always looser. Returns the bound, the labels, and the
    measured true gap when the caller passes `true_gap`.
    """
    out = write_influence_labels(q, k, delta, beta, lam, scale=scale,
                                 read_weight=read_weight, window=window,
                                 coverage_horizon=None)
    labels = out["labels"]
    skipped = (mask.float() <= 0.5)
    bound = float((labels * skipped).sum())
    return {"bound": bound, "labels": labels,
            "n_skipped": int(skipped.sum())}


def influence_of_writes(q, k, delta, beta, lam, s_idx, h_idx, horizons,
                        scale: float = 1.0, read_weight=None) -> torch.Tensor:
    """Exact Theorem-B influence of specific writes over custom horizons.

    For write list (s_i, h_i) with horizon W_i: I_i = sum_{t=s_i}^{s_i+W_i-1}
    w_t * scale * beta * |q_t^T (p(s,t) (*) k_s)| * ||delta_s||. This is the
    deferral-distortion certificate: a write deferred from s and committed
    at c loses exactly I_h(s, [s, c)) (Theorem C's measurable twin).
    """
    T, H, dk = q.shape
    qf, kf, df = q.float(), k.float(), delta.float()
    loglam = lam_to_full(lam, T, H, dk).clamp_min(1e-12).log()
    rw = read_weight.float() if read_weight is not None else None
    res = torch.zeros(len(s_idx), device=q.device)
    for i, (s, h, W) in enumerate(zip(s_idx, h_idx, horizons)):
        e = min(s + W, T)
        if e <= s:
            continue
        # p(s, t) for t in [s, e): p[s]=1 (empty product), then the
        # inclusive cumulative products of loglam[s+1..t]
        gl = loglam[s + 1:e, h, :]                       # [L-1, dk]
        p = torch.cat([torch.ones(1, dk, device=q.device),
                       torch.cumsum(gl, dim=0).exp()], 0)  # [L, dk]
        sc = (qf[s:e, h, :] * kf[s, h, :] * p).sum(-1)   # [L]
        w = rw[s:e] if rw is not None else torch.ones(e - s,
                                                      device=q.device)
        if w.dim() == 2:
            w = w[:, h]
        res[i] = (w * sc.abs()).sum() * scale * float(beta[s, h]) * float(
            df[s, h, :].norm())
    return res

# ============================================================================
# Section 4b — BetaGate: the write-enable lever on the fused [b|a] projection
# ============================================================================
class BetaGate(nn.Module):
    """MoRM 2.0's write-enable mask on a stock fused [b|a] projection
    (Proposition F), plus the per-head decay the scheduler needs.

    Wraps (ADOPTS, never copies) the model's `in_proj_ba` Linear. The stock
    forward computes `out = Linear(x)` with the b-slice (per-head write
    gates, H values) first and the a-slice (per-head log-decay) second
    (the qwen3_5 convention; `b_first=False` flips it). The gate rewrites
    the b-slice in LOGIT space:

        b_masked = where(mask, b, logit_floor)     (floor -60 -> sigma ~ 9e-27)

    so that whatever downstream transform the stock code applies (sigmoid
    in-kernel or outside), skipped heads write ~0 — bit-exact when the
    kernel consumes post-sigmoid beta, and 26 orders of magnitude below
    signal otherwise (asserted < 1e-6 by the notebook's runtime probe).
    With mask == 1 (exact mode) the output is BITWISE the stock tensor.

    `lam_of(a_slice)` computes the per-head decay for the decode scheduler:
    the GDN formula exp(-exp(A_log) * softplus(a + dt_bias)) when the
    parent module carries A_log/dt_bias (Qwen3-Next-style), else the GLA
    formula sigmoid(a) — the notebook's calibration probe selects and
    validates the formula against the stock kernel's outputs.
    """

    def __init__(self, inner: nn.Module, n_heads: int,
                 policy: "MoRM2Policy",
                 b_first: bool = True, logit_floor: float = -60.0,
                 A_log: Optional[torch.Tensor] = None,
                 dt_bias: Optional[torch.Tensor] = None,
                 gate_formula: str = "auto",
                 b_only: Optional[bool] = None):
        super().__init__()
        self.inner = inner                        # adopted stock Linear
        # v8.0.1 [F5a]: re-register the adopted weight at the STOCK key too.
        # Wrapping must NOT rename the state-dict entry: '...in_proj_ba.weight'
        # is what the COMPAT export reader and the checkpoints load; the
        # inner submodule's duplicate '.inner.weight' is harmless (same
        # object, zero-copy — VGateShim follows the same convention).
        self.register_parameter("weight", inner.weight)
        # v8.0.1 [F5b]: world-mode lever — set_world() flips this with the
        # execution family (dense_teacher / exact_resonant / compat_parity
        # need the bitwise stock tensor; soft_train / serving run the gate).
        self.enabled = True
        # v8.0.2 [P3]: qwen3_5 stores b and a as SEPARATE Linears
        # (in_proj_b [H,d] / in_proj_a [H,d]) — there is no fused [2H]
        # module to slice. When the adopted module emits exactly H rows it
        # IS the b-projection alone: gate the whole output (no a-slice to
        # preserve). b_only can also be forced explicitly.
        self.b_only = bool(b_only) if b_only is not None else             bool(getattr(inner, "out_features", 0) == n_heads)
        self.H = int(n_heads)
        self.policy = policy
        self.b_first = bool(b_first)
        self.logit_floor = float(logit_floor)
        self.gate_formula = gate_formula
        # gate parameters are owned by the parent attention module — keep
        # references (not copies) so training moves stay in sync
        if A_log is not None:
            self.register_buffer("A_log", A_log.detach().clone(),
                                 persistent=False)
        else:
            self.A_log = None
        if dt_bias is not None:
            self.register_buffer("dt_bias", dt_bias.detach().clone(),
                                 persistent=False)
        else:
            self.dt_bias = None
        self.last_mask: Optional[torch.Tensor] = None

    # -- stock-compatible forward --------------------------------------------
    def forward(self, x: torch.Tensor):
        out = self.inner(x)                       # [..., 2H] (or [..., H])
        if not getattr(self, "enabled", True):
            return out        # v8.0.1 [F5b]: stock bitwise passthrough
        if getattr(self, "b_only", False):
            # v8.0.2 [P3]: SPLIT layout — inner IS in_proj_b; gate the b
            # logits alone (the a sibling stays dense outside this wrap).
            # Same logit-space blend as the fused path: hard mask -> the
            # values are exactly {b, floor}; mask == 1 -> bitwise stock.
            lead = out.shape[:-1]
            b = out.reshape(-1, out.shape[-1])
            phi = self.policy.mast(x.reshape(-1, x.shape[-1]))
            mask = self.policy(phi)               # [T, H] (STE in training)
            self.last_mask = mask.detach() if hasattr(mask, "detach") \
                else mask
            mf = mask.to(b.dtype).reshape(-1, self.H) if mask.dim() == 2 \
                else mask.to(b.dtype)
            b = (mf * b + (1.0 - mf) * self.logit_floor).to(out.dtype)
            return b.reshape(*lead, -1)
        lead = out.shape[:-1]
        out2 = out.reshape(-1, out.shape[-1])
        H = self.H
        if self.b_first:
            b, a = out2[:, :H], out2[:, H:]
        else:
            a, b = out2[:, :H], out2[:, H:]
        phi = self.policy.mast(x.reshape(-1, x.shape[-1]))
        mask = self.policy(phi)                   # [T, H] (STE in training)
        self.last_mask = mask.detach() if hasattr(mask, "detach") \
            else mask
        # blend (NOT torch.where): forward values are exactly {b, floor}
        # for the hard 0/1 mask, and the straight-through gradient from
        # the policy reaches this site — a `where(mask > 0.5, ...)` here
        # would silently sever the policy's training signal (the v7 bug,
        # fixed at both mask sites in v8)
        mf = mask.to(b.dtype).reshape(-1, H) if mask.dim() == 2 else \
            mask.to(b.dtype)
        b = mf * b + (1.0 - mf) * self.logit_floor
        b = b.to(out2.dtype)
        out2 = torch.cat([b, a], dim=-1) if self.b_first \
            else torch.cat([a, b], dim=-1)
        return out2.reshape(*lead, -1)

    # -- per-head decay for the scheduler -------------------------------------
    def lam_of(self, a_slice: torch.Tensor) -> torch.Tensor:
        """a_slice [..., H] (the a-part of the fused output) -> lam in (0,1).

        formula "gdn":  lam = exp(-exp(A_log) * softplus(a + dt_bias))
        formula "gla":  lam = sigmoid(a)
        formula "auto": gdn when A_log/dt_bias are wired, else gla
        """
        f = self.gate_formula
        if f == "auto":
            f = "gdn" if (self.A_log is not None
                          and self.dt_bias is not None) else "gla"
        if f == "gdn":
            A = self.A_log if self.A_log is not None \
                else torch.zeros_like(a_slice)
            db = self.dt_bias if self.dt_bias is not None \
                else torch.zeros_like(a_slice)
            return torch.exp(-torch.exp(A) * F.softplus(a_slice + db))
        return torch.sigmoid(a_slice)



# ============================================================================
# Section 6 — MoRM2Decoder: stateful decode-time execution (NATIVE serving)
# ============================================================================
class MoRM2Decoder:
    """Decode-time MoRM 2.0 execution for one GDN layer (one token per call).

    Owns the per-layer scheduler (deferral buffer, charge ledger, rate
    controller) and drives the trained policy one token at a time:

        step(phi_t [S], k_t [H,dk], beta_t [H], lam_t, v_t [H,dv] or None)
            -> dict(mask [H], k_eff, beta_eff, v_eff, m_t)

    The caller (the NATIVE serving loop) feeds the emitted (k_eff, v_eff,
    beta_eff) into the stock kernel for the generated token — input
    rewriting only. Charge/age features are live scheduler state, so at
    decode time the critic sees the debt it owes each head (during
    training those features are zero: batches are stateless).
    """

    def __init__(self, policy: MoRM2Policy, dk: int, dv: int,
                 staleness_k: int = 64, m_max: Optional[int] = None,
                 v_provider: Optional[Callable] = None):
        self.policy = policy
        self.dk, self.dv = dk, dv
        self.v_provider = v_provider
        self.sched = WriteScheduler(policy.H, dk, dv,
                                    m_bar=policy.m_bar,
                                    staleness_k=staleness_k,
                                    m_max=m_max or policy.m_max,
                                    init_threshold=-1.0
                                    if policy.exact_mode else None,
                                    v_provider=v_provider)
        self.t = 0

    def reset(self):
        self.sched = WriteScheduler(self.policy.H, self.dk, self.dv,
                                    m_bar=self.policy.m_bar,
                                    staleness_k=self.sched.K,
                                    m_max=self.policy.m_max,
                                    init_threshold=float(
                                        self.policy.rate_threshold),
                                    v_provider=self.v_provider)
        self.t = 0

    def step(self, phi_t: torch.Tensor, k_t: torch.Tensor,
             beta_t: torch.Tensor, lam_t: torch.Tensor,
             v_t: Optional[torch.Tensor] = None) -> dict:
        age_feat = (self.sched.pend_age.float() / max(1, self.sched.K)
                    ).clamp(0, 1.5)
        obs = self.policy.obs_features(
            beta_t.reshape(1, -1), k_t.float().norm(dim=-1).reshape(1, -1),
            charge=self.sched.charge.reshape(1, -1),
            age=age_feat.reshape(1, -1))[0]              # [H, N_OBS]
        scores = self.policy.scores(phi_t.reshape(1, -1),
                                    obs.reshape(1, *obs.shape))[0]
        out = self.sched.step(scores, k_t, v_t, beta_t, lam_t, t=self.t)
        self.t += 1
        return out


# ============================================================================
# Section 7 — The verification suite (every claim above is executable)
# ============================================================================
def verify_morm2(verbose: bool = True, seed: int = 0,
                 device: str = "cpu") -> dict:
    """Numerically verify the MoRM 2.0 contracts M1-M10. Returns a report
    dict; raises AssertionError on any violated contract.

      M1  oracle correctness (both forms vs a naive per-head loop)
      M2  WIK single-write labels are EXACT vs brute-force ablation
          (plain form; delta-form agreement reported — frozen correction)
      M2c label chunk-invariance + cross-implementation agreement
      M3  multi-skip certified bound holds (and its tightness)
      M4  Proposition F: beta-mask is the exact lever in BOTH kernel forms;
          v-masking (MoRM 1.0) leaks deletion writes in the delta form
      M5  Theorem C: deferred commit reproduces the on-time trajectory
          from the commit point onward (plain form EXACT; delta reported)
      M6  Theorem D: the elastic controller converges to E[m] = m_bar
      M7  Theorem E: staleness <= K with exact decay compensation at every
          commit (deterministic pre-emption scenario)
      M8  the rate-distortion win: at EQUAL average budget, influence
          scheduling beats energy top-m (MoRM 1.0) on a synthetic task
          whose energy/influence structure is exactly anti-correlated
      M9  exact mode is bit-exact dense (all writes, no rewrites)
      M10 charge-ledger invariants (drops charge, debt ages with decay)
    """
    torch.manual_seed(seed)
    rep = {}

    def log(*a):
        if verbose:
            print(*a)

    # ---- shared synthetic stream -------------------------------------------
    T, H, dk, dv = 192, 6, 16, 12
    scale = dk ** -0.5
    q = F.normalize(torch.randn(T, H, dk, device=device), dim=-1)
    k = F.normalize(torch.randn(T, H, dk, device=device), dim=-1)
    v = torch.randn(T, H, dv, device=device)
    beta = torch.sigmoid(torch.randn(T, H, device=device) * 0.8 + 1.0)
    lam_s = 1 - 0.06 * torch.rand(H, device=device)      # [H] scalar decay
    lam = lam_s[None, :].expand(T, H)

    # ---- M1: oracle vs naive loops ------------------------------------------
    def naive(q_, k_, v_, beta_, lam_full, form):
        Tn, Hn, dkn = q_.shape
        dvn = v_.shape[-1]
        o = torch.zeros(Tn, Hn, dvn)
        for h in range(Hn):
            S = torch.zeros(dkn, dvn)
            for t in range(Tn):
                S = torch.diag(lam_full[t, h, :]) @ S
                d = v_[t, h]
                if form == "delta":
                    d = d - S.t() @ k_[t, h]
                S = S + beta_[t, h] * torch.outer(k_[t, h], d)
                o[t, h] = q_[t, h] @ S
        return o * (q_.shape[-1] ** -0.5)

    lam_full = lam_to_full(lam, T, H, dk)
    o_p, _, deltas_p = gdn_sequential(q, k, v, beta, lam, "plain", scale,
                                      return_deltas=True)
    o_d, _, deltas_d = gdn_sequential(q, k, v, beta, lam, "delta", scale,
                                      return_deltas=True)
    for form in ("plain", "delta"):
        o_ref = o_p if form == "plain" else o_d
        o_nai = naive(q, k, v, beta, lam_full, form)
        err = float((o_ref - o_nai).norm() / o_nai.norm())
        assert err < 1e-5, f"M1 {form}: oracle != naive ({err:.2e})"
        rep[f"M1_{form}"] = err
    log(f"M1  oracle == naive reference (plain {rep['M1_plain']:.1e}, "
        f"delta {rep['M1_delta']:.1e})")

    # ---- M2: WIK exactness vs brute-force single-write ablation -------------
    labels = write_influence_labels(q, k, deltas_p, beta, lam, scale=scale,
                                    window=T, chunk=64)["labels"]
    smp = [(s, h) for s in range(8, T - 8, 17) for h in range(H)][:24]
    errs_plain = []
    o_dense = o_p
    for (s, h) in smp:
        b2 = beta.clone()
        b2[s, h] = 0.0
        o_abl, _ = gdn_sequential(q, k, v, b2, lam, "plain", scale)
        true_total = float((o_dense - o_abl).norm(dim=-1).sum())
        errs_plain.append(abs(true_total - float(labels[s, h]))
                          / max(float(labels[s, h]), 1e-9))
    rep["M2_plain_max_rel_err"] = max(errs_plain)
    assert max(errs_plain) < 1e-4, \
        f"M2: WIK labels not exact ({max(errs_plain):.2e})"
    labels_d = write_influence_labels(q, k, deltas_d, beta, lam, scale=scale,
                                      window=T, chunk=64)["labels"]
    errs_delta = []
    o_dense_d = o_d
    for (s, h) in smp[:12]:
        b2 = beta.clone()
        b2[s, h] = 0.0
        o_abl, _ = gdn_sequential(q, k, v, b2, lam, "delta", scale)
        true_total = float((o_dense_d - o_abl).norm(dim=-1).sum())
        errs_delta.append(abs(true_total - float(labels_d[s, h]))
                          / max(true_total, 1e-9))
    rep["M2_delta_med_rel_err"] = sorted(errs_delta)[len(errs_delta) // 2]
    log(f"M2  WIK single-write labels EXACT vs ablation "
        f"(plain max rel {max(errs_plain):.2e}; delta-form median agreement "
        f"{rep['M2_delta_med_rel_err']:.1%} — frozen-correction response)")

    # ---- M2c: chunk-invariance + cross-implementation -----------------------
    l16 = write_influence_labels(q, k, deltas_p, beta, lam, scale=scale,
                                 window=T, chunk=16)["labels"]
    inv = float((l16 - labels).norm() / labels.norm())
    assert inv < 1e-5, f"M2c: labels chunk-dependent ({inv:.2e})"
    rep["M2c_chunk_invariance"] = inv
    xval = []
    for (s, h) in smp[:8]:
        got = influence_of_writes(q, k, deltas_p, beta, lam, [s], [h],
                                  [T - s], scale=scale)
        xval.append(abs(float(got[0]) - float(labels[s, h]))
                    / max(float(labels[s, h]), 1e-9))
    rep["M2c_cross_impl"] = max(xval)
    assert max(xval) < 1e-4, \
        f"M2c: implementations disagree ({max(xval):.2e})"
    log(f"M2c labels chunk-invariant ({inv:.1e}) with cross-implementation "
        f"agreement ({max(xval):.1e})")

    # ---- M3: multi-skip certified bound --------------------------------------
    mask = (torch.rand(T, H, device=device) > 0.6).float()
    gap = gdn_beta_masked_gap(q, k, v, beta, lam, mask, "plain", scale)
    cert = morm2_skip_certificate(q, k, deltas_p, beta, lam, mask,
                                  scale=scale, window=T)
    assert gap <= cert["bound"] + 1e-4, \
        f"M3: true gap {gap:.3e} exceeds certified bound {cert['bound']:.3e}"
    rep["M3_tightness"] = gap / max(cert["bound"], 1e-9)
    log(f"M3  multi-skip bound holds: gap {gap:.3f} <= bound "
        f"{cert['bound']:.3f} (tightness {rep['M3_tightness']:.2f})")

    # ---- M4: Proposition F — the exact masking lever -------------------------
    # beta-masking IS skip semantics (the write is exactly absent). In the
    # plain form the v-mask coincides with it; in the delta-corrected form
    # (the stock FLA WY kernel) v-masking deposits -beta k k^T S~ deletion
    # writes instead — a different, UNCERTIFIED operation, not a skip.
    o_bm_d, _ = gdn_sequential(q, k, v, beta * mask, lam, "delta", scale)
    o_vm_d, _ = gdn_sequential(q, k, v * mask[..., None], beta, lam, "delta",
                               scale)
    leak = float((o_vm_d - o_bm_d).norm())
    assert leak > 1e-2, (
        f"M4: in the delta form v-masking must diverge from skip semantics "
        f"(leak {leak:.4f}) — it deposits deletion writes")
    o_bm_p, _ = gdn_sequential(q, k, v, beta * mask, lam, "plain", scale)
    o_vm_p, _ = gdn_sequential(q, k, v * mask[..., None], beta, lam, "plain",
                               scale)
    coincide = float((o_vm_p - o_bm_p).norm())
    assert coincide < 1e-4, (
        f"M4: in the plain form v-mask and beta-mask must coincide exactly "
        f"({coincide:.2e})")
    skip_gap_d = float((o_d - o_bm_d).norm())
    rep["M4_leak_vs_skip"] = leak / max(skip_gap_d, 1e-9)
    rep["M4_plain_coincide"] = coincide
    log(f"M4  beta-mask == skip semantics in both forms; the MoRM 1.0 "
        f"v-mask lever diverges from skip semantics by {leak:.3f} in the "
        f"delta form ({rep['M4_leak_vs_skip']:.2f}x the certified skip gap; "
        f"plain-form coincidence {coincide:.1e})")

    # ---- M5: Theorem C — deferred-commit exactness ---------------------------
    # Contention-free construction: head 0 has NO fresh write of its own at
    # the commit token (beta[c0, 0] = 0 in BOTH runs), so the commit slot
    # carries exactly the deferred write and nothing else changes.
    s0, c0 = 20, 60
    beta5 = beta.clone()
    beta5[c0, 0] = 0.0
    comp = lam[s0 + 1:c0 + 1, 0].prod()                  # scalar-per-head lam
    k_def = k.clone()
    k_def[c0, 0] = k[s0, 0] * comp
    b_def = beta5.clone()
    b_def[s0, 0] = 0.0
    b_def[c0, 0] = beta5[s0, 0]
    v_def = v.clone()
    v_def[c0, 0] = v[s0, 0]
    o_on, _ = gdn_sequential(q, k, v, beta5, lam, "plain", scale)
    o_df, _ = gdn_sequential(q, k_def, v_def, b_def, lam, "plain", scale)
    after = slice(c0, T)
    err_after = float((o_on[after] - o_df[after]).norm()
                      / o_on[after].norm())
    before = slice(s0, c0)
    loss_before = float((o_on[before] - o_df[before]).norm())
    assert err_after < 1e-5 and loss_before > 1e-3, \
        (f"M5: deferred commit must be exact from the commit point "
         f"(after: {err_after:.2e}, window loss: {loss_before:.3f})")
    rep["M5_exact_after_commit"] = err_after
    rep["M5_window_loss"] = loss_before
    # window loss must equal the windowed influence certificate
    win_cert = float(influence_of_writes(q, k, deltas_p, beta, lam, [s0],
                                         [0], [c0 - s0], scale=scale)[0])
    rep["M5_window_cert_ratio"] = loss_before / max(win_cert, 1e-9)
    o_df_d, _ = gdn_sequential(q, k_def, v_def, b_def, lam, "delta", scale)
    rep["M5_delta_after"] = float((o_d[after] - o_df_d[after]).norm()
                                  / o_d[after].norm())
    log(f"M5  deferred commit EXACT after commit (rel {err_after:.1e}); "
        f"window loss {loss_before:.3f} vs influence certificate "
        f"{win_cert:.3f}; delta-form after-commit rel "
        f"{rep['M5_delta_after']:.1e}")

    # ---- M6: Theorem D — rate convergence ------------------------------------
    T2, H2 = 4000, 8
    sc2 = torch.rand(T2, H2, device=device) ** 2
    m_bar = 3
    out6 = schedule_writes(sc2, torch.randn(T2, H2, 4, device=device),
                           torch.randn(T2, H2, 3, device=device),
                           torch.ones(T2, H2, device=device) * 0.5,
                           torch.full((T2, H2), 0.98, device=device),
                           m_bar=m_bar, staleness_k=10 ** 9, m_max=H2)
    burn = 500
    mm = float(out6["stats"].m_trace[burn:].float().mean())
    assert abs(mm - m_bar) < 0.25 * m_bar + 0.15, \
        f"M6: E[m]={mm:.2f} did not converge to m_bar={m_bar}"
    rep["M6_mean_m"] = mm
    log(f"M6  elastic rate converges: E[m] = {mm:.2f} (target {m_bar})")

    # ---- M7: Theorem E — staleness + compensation (deterministic) ------------
    # head 0 wins the single slot every token (F=2); heads 1,2 park at t=0
    # and are capped out until the staleness deadline forces their commit.
    K = 12
    sc3 = torch.zeros(T, H, device=device)
    sc3[:, 0] = 2.0
    sc3[0, 1:] = 1.0
    lam3 = torch.full((T, H), 0.97, device=device)
    out7 = schedule_writes(sc3, k, v, beta, lam3, m_bar=1, staleness_k=K,
                           m_max=1, init_threshold=0.5)
    st = out7["stats"]
    assert st.max_staleness <= K, \
        f"M7: staleness {st.max_staleness} exceeded K={K}"
    assert st.defers >= 2 and st.commits >= 2, \
        f"M7: scenario must exercise deferral+commit ({st.summary()})"
    commits = out7["commit"].nonzero(as_tuple=False)
    for t_cm, h_cm in commits.tolist():
        assert t_cm == K, \
            f"M7: forced commit must fire exactly at age K ({t_cm} != {K})"
        comp7 = lam3[1:K + 1, h_cm].prod()
        assert torch.allclose(out7["k_eff"][t_cm, h_cm],
                              k[0, h_cm] * comp7, atol=1e-5), \
            "M7: commit key is not the exact decay compensation"
        assert abs(float(out7["beta_eff"][t_cm, h_cm])
                   - float(beta[0, h_cm])) < 1e-6, \
            "M7: commit beta must be the parked write's beta"
        assert torch.allclose(out7["v_eff"][t_cm, h_cm], v[0, h_cm]), \
            "M7: commit value must be the parked write's value"
        assert float(out7["mask"][0, h_cm]) == 0.0, \
            "M7: the parked write must have been skipped at its origin"
    rep["M7_max_staleness"] = st.max_staleness
    log(f"M7  staleness bound respected (max age {st.max_staleness} <= "
        f"K={K}); commits carry exact compensation | {st.summary()}")

    # ---- M8: the rate-distortion win -----------------------------------------
    # Anti-correlated construction: LOUD writes (||v|| x6, keys orthogonal
    # to every query — provably zero influence) vs QUIET writes (||v|| x1,
    # each key is exactly what one later query retrieves). Energy
    # scheduling buys loud; influence scheduling buys quiet.
    T3, H3 = 256, 4
    gap_ahead = 24
    q3 = torch.zeros(T3, H3, dk, device=device)
    Qr = F.normalize(torch.randn(H3, dk, device=device), dim=-1)
    ret_pos = torch.arange(H3, T3, 8, device=device)     # retrieval tokens
    # each retrieval token r reads head (r % H3) with Qr[r % H3]
    q3[ret_pos, ret_pos % H3] = Qr[ret_pos % H3]
    role = (torch.rand(T3, H3, device=device) < 0.5)     # loud per (t, h)
    k3 = torch.randn(T3, H3, dk, device=device)
    # loud keys: projected into the orthogonal complement of the H3 query
    # directions of their head -> EXACTLY zero influence
    for h in range(H3):
        basis = Qr[h:h + 1]                              # [1, dk]
        P = torch.eye(dk, device=device) - basis.t() @ basis
        cols = P @ torch.randn(dk, T3, device=device)
        cols = F.normalize(cols, dim=0)
        m_h = role[:, h]
        k3[m_h, h] = cols.t()[m_h]
        # quiet keys that are due for retrieval: key == future query
        for r in range(h, T3 - gap_ahead):
            if not bool(role[r, h]) and (r % H3) == h and \
                    (r + gap_ahead) in ret_pos.tolist():
                k3[r, h] = Qr[(r + gap_ahead) % H3]
    k3 = F.normalize(k3, dim=-1)
    v3 = torch.randn(T3, H3, dv, device=device) \
        * torch.where(role, 6.0, 1.0)[..., None]
    beta3 = torch.sigmoid(torch.randn(T3, H3, device=device) + 1.0)
    lam3b = torch.full((T3, H3), 0.97, device=device)
    o3, _, d3 = gdn_sequential(q3, k3, v3, beta3, lam3b, "plain",
                               dk ** -0.5, return_deltas=True)
    lab3 = write_influence_labels(q3, k3, d3, beta3, lam3b,
                                  scale=dk ** -0.5, window=T3)["labels"]
    m_budget = 2
    en = energy_scores(k3, v3, beta3)
    top_e = en.topk(m_budget, dim=-1).indices
    mask_e = torch.zeros_like(en).scatter_(-1, top_e, 1.0)
    gap_e = gdn_beta_masked_gap(q3, k3, v3, beta3, lam3b, mask_e, "plain",
                                dk ** -0.5)
    top_i = lab3.topk(m_budget, dim=-1).indices
    mask_i = torch.zeros_like(lab3).scatter_(-1, top_i, 1.0)
    gap_i = gdn_beta_masked_gap(q3, k3, v3, beta3, lam3b, mask_i, "plain",
                                dk ** -0.5)
    noisy = lab3 * torch.exp(torch.randn_like(lab3) * 0.35)
    out8 = schedule_writes(noisy, k3, v3, beta3, lam3b, m_bar=m_budget,
                           staleness_k=16, m_max=H3)
    mm8 = float(out8["stats"].m_trace.float().mean())
    gap_2 = gdn_beta_masked_gap(q3, k3, v3, beta3, lam3b, out8["mask"],
                                "plain", dk ** -0.5)
    assert gap_2 < 0.85 * gap_e, \
        (f"M8: MoRM 2.0 ({gap_2:.3f}) must beat energy top-m ({gap_e:.3f}) "
         f"at the same budget (mean m {mm8:.2f} vs {m_budget})")
    rep["M8_gap_energy"] = gap_e
    rep["M8_gap_morm2"] = gap_2
    rep["M8_gap_oracle"] = gap_i
    log(f"M8  equal-budget output deviation: energy top-m {gap_e:.3f} -> "
        f"MoRM 2.0 {gap_2:.2e} (oracle {gap_i:.2e}) — "
        f"{gap_e / max(gap_2, 1e-9):.0f}x better")

    # ---- M9: exact mode is bit-exact -----------------------------------------
    out9 = schedule_writes(torch.rand(T, H, device=device), k, v, beta, lam,
                           m_bar=H, staleness_k=8, m_max=None)
    assert bool((out9["mask"] == 1).all()) and out9["stats"].commits == 0
    o9, _ = gdn_sequential(q, out9["k_eff"], v, out9["beta_eff"], lam,
                           "plain", scale)
    err9 = float((o9 - o_p).norm() / o_p.norm())
    assert err9 < 1e-6, f"M9: exact mode not bit-exact ({err9:.2e})"
    rep["M9_exact_rel"] = err9
    log(f"M9  exact mode (m_bar=H) is bit-exact dense (rel {err9:.1e})")

    # ---- M10: charge-ledger invariants ---------------------------------------
    sched = WriteScheduler(H, dk, dv, m_bar=1, staleness_k=64,
                           init_threshold=1e9)   # nothing is ever eligible
    sched.step(torch.full((H,), 0.1), k[0], v[0], beta[0], lam[0], t=0)
    c1 = float(sched.charge.sum())
    for t in range(1, 40):
        sched.step(torch.zeros(H), k[t], v[t], beta[t], lam[t], t=t)
    c40 = float(sched.charge.sum())
    assert c1 > 0 and c40 < c1, (
        f"M10: charge must accumulate then age with decay ({c1:.3f} -> "
        f"{c40:.3f})")
    assert (sched.charge >= 0).all(), "M10: negative charge"
    rep["M10_charge_aged"] = (c1, c40)
    log(f"M10 charge ledger: drops charge the ledger ({c1:.3f}), then the "
        f"debt ages with the head decay ({c40:.3f})")

    log("verify_morm2: ALL CONTRACTS PASS")
    return rep


# ============================================================================
# 14. census_active — touched-weights census of a LIVE module graph (v9)
# ============================================================================
# v8's measured_active() summed a hand-picked list of registries and so never saw
# the dense k_proj / v half / in_proj_a / conv taps / norms, nor a surgery site that
# was skipped (kept dense). This walk visits EVERY module of the text decoder once
# (a module hanging under two parents — the MoRM policy — is counted once) and
# charges what one serving token touches; any 2-D+ parameter it cannot classify is
# charged in full (never silently free). It mirrors active_waterfall() family by
# family, so a plan can be compared with the graph that was actually built.
import re as _re_census

_CENSUS_SKIP = _re_census.compile(
    r"(visual|vision|merger|mtp|embed|lm_head|patch_embed|multi_modal)", _re_census.I)
_LAYER_RX = _re_census.compile(r"^(.*?\blayers\.\d+)\b")


def _prism_active(m) -> int:
    k = int(m.k_topk or 1)
    S = int(m.shared_idx.numel())
    sh = (int(m.sh_U.numel() + m.sh_M.numel() + m.sh_b.numel())
          if getattr(m, "sh_U", None) is not None else 0)            # v10 FFN shadow
    return (k * 3 * m.d_model * m.m_e + 3 * m.d_model * S + m.E * m.d_model
            + m.d_model * m.rank + (k * m.m_e + S) * m.rank + 2 * (k + 1) * m.rank + sh)


def _morm_heads(sp) -> int:
    pol = sp.morm
    mb = None if pol is None else getattr(pol, "m_bar", getattr(pol, "m", None))
    return sp.n_v_heads if mb is None else min(int(mb), sp.n_v_heads)


def _sectioned_active(m) -> int:
    din = int(m.weight.shape[1])
    qr, zr = getattr(m, "q_router", None), getattr(m, "z_router", None)
    q = ((m.q_carrier.numel() + min(m.topk, m.q_B) * m.q_r) * din
         + (qr.weight.numel() if qr is not None else 0)) if hasattr(m, "q_B") \
        else m.q_rows * din
    z = ((m.z_carrier.numel() + min(m.topk, m.z_B) * m.z_r) * din
         + (zr.weight.numel() if zr is not None else 0)) if hasattr(m, "z_B") \
        else m.z_rows * din
    sh = sum(int(getattr(m, f"{t}_sh_U").numel() + getattr(m, f"{t}_sh_M").numel())
             for t in ("q", "v", "z") if getattr(m, f"{t}_sh_U", None) is not None)   # v10 shadows
    return q + z + m.k_rows * din + _morm_heads(m) * m.v_head_rows * din + sh


def _morm_active(p) -> int:
    if hasattr(p, "critic_phi"):
        return int(p.critic_phi.weight.numel()
                   + (p.critic_phi.bias.numel() if p.critic_phi.bias is not None else 0)
                   + (p.critic_obs.weight.numel() if getattr(p, "use_obs", False) else 0))
    return int(p.router.weight.numel())


def census_active(model, hid: int, n_layers: int, funnel=None, mast=None,
                  mast_mode: str = "applications") -> dict:
    """Touched weights per serving text token, by family (see active_waterfall).
    Side results: census_active.last = {unclassified, mast_applications, mast_distinct}."""
    fam = {"embedding_lookup": int(hid), "antenna_mast": 0, "ffn": 0,
           "full_attention": 0, "linear_attention": 0, "lm_head_funnel": 0,
           "norms_misc": int(n_layers) * 2 * int(hid) + int(hid), "other": 0}
    seen, phi_evals, unclassified = set(), set(), []

    def family_of(path):
        if "linear_attn" in path:
            return "linear_attention"
        if "self_attn" in path:
            return "full_attention"
        if "mlp" in path:
            return "ffn"
        return "other"

    def need_phi(path, width):
        mm = _LAYER_RX.match(path)
        phi_evals.add((mm.group(1) if mm else path, int(width)))

    def walk(mod, path):
        if id(mod) in seen:
            return
        seen.add(id(mod))
        if path and _CENSUS_SKIP.search(path):
            return
        fam_n = family_of(path)
        if isinstance(mod, PrismMoeMLP):
            fam["ffn"] += _prism_active(mod)
            return
        if isinstance(mod, ResonantLinear):
            fam[fam_n] += mod.active_params() + mod.router.weight.numel()
            need_phi(path, mod.in_f)
            return
        if isinstance(mod, SectionedProjector):
            fam[fam_n] += _sectioned_active(mod)
            if hasattr(mod, "q_B") or hasattr(mod, "z_B") or mod.morm is not None:
                need_phi(path, mod.weight.shape[1])
            if mod.morm is not None:
                walk(mod.morm, path + ".morm")
            return
        if isinstance(mod, BetaGate):
            fam[fam_n] += int(mod.inner.weight.numel())      # b (and a) logits: dense reads
            need_phi(path, mod.inner.weight.shape[1])
            walk(mod.policy, path + ".policy")               # shared child: counted once
            return
        if isinstance(mod, (MoRM2Policy, MoRMWritePolicy)):
            fam[fam_n] += _morm_active(mod)
            return
        if hasattr(mod, "rl_gate") and hasattr(mod, "v_w"):  # VGateShim: dense v half + routed gate
            fam[fam_n] += int(mod.v_w.numel())
            walk(mod.rl_gate, path + ".rl_gate")
            return
        if isinstance(mod, nn.Linear):
            fam[fam_n] += int(mod.weight.numel())
            return
        conv = getattr(mod, "conv1d", None)
        if isinstance(conv, nn.Conv1d) and id(conv) not in seen:
            sp = next((c for c in mod.children() if isinstance(c, SectionedProjector)), None)
            n_ch = (sp.q_rows + sp.k_rows
                    + (sp.v_rows if getattr(sp, "v_sh_U", None) is not None      # v10: every v channel is non-zero
                       else _morm_heads(sp) * sp.v_head_rows)
                    if sp is not None else int(conv.weight.shape[0]))
            fam["linear_attention"] += n_ch * int(conv.weight.shape[-1])
            seen.add(id(conv))
        kids = list(mod.named_children())
        if not kids:
            for n_, p_ in mod.named_parameters(recurse=False):
                if p_.ndim >= 2:
                    fam[fam_n] += int(p_.numel())
                    unclassified.append(f"{path}.{n_}")
            return
        for n_, c_ in kids:
            walk(c_, f"{path}.{n_}" if path else n_)

    walk(model, "")
    S = int(mast.S) if mast is not None else 0
    app = S * sum(w_ for _, w_ in phi_evals)
    dis = S * sum({w_ for _, w_ in phi_evals})
    fam["antenna_mast"] = app if mast_mode == "applications" else dis
    fam["lm_head_funnel"] = int(funnel.active_params()) if funnel is not None else 0
    fam["TOTAL_ACTIVE"] = sum(fam.values())
    census_active.last = {"unclassified": unclassified, "mast_applications": app,
                          "mast_distinct": dis}
    return fam


census_active.last = {}


## Verification suite — the 12 load-bearing contracts

Balanced fission + Theorem 1 identity, Theorem 3 materialization parity, gradient flow, Theorem 4/5 resonant-linear contracts, SectionedProjector exactness (fused + split layouts), MoRM mask/certificate/supergraph, HeadFunnel exactness/recall, the 3B solver on the real Qwen3.8-27B config (v9: honest plan — shared core 512, strict sketch accounting — plus a second check that an impossible target is reported infeasible), the Theorem-7 mini-model anchor, and the self-checkpointing + telemetry-diet memory contracts. If any check fails, the notebook **stops** — by design.


In [ ]:
# ============================================================================
# Verification suite — the 12 load-bearing contracts, asserted numerically.
# Every theorem the pipeline relies on is checked here on CPU toys; the real
# model is re-anchored end-to-end by the surgery cell (Theorem 7 on real
# logits). If any check fails, the notebook STOPS — by design.
#   C1  balanced fission + Theorem 1 (exact dense identity at init)
#   C2  Theorem 3 (serving parity: materialization == module, fused layout)
#   C3  gradient flow: every trainable reached, dense substrate frozen
#   C4  Theorem 4: ResonantLinear full == dense; materialize_dense bitwise
#   C5  Theorem 5: per-token tail certificate holds on 100% of tokens
#   C6  SectionedProjector: dense == stock projection; full exact; split
#   C7  MoRM: exactly m writes; m=H bit-exact; m<H under the Prop-6 bound
#   C8  grad-mode supergraph == sparse oracle (MoRM v-path + ResonantLinear)
#   C9  HeadFunnel: exact shortlist rows, recall, chunked == one-shot
#   C10 3B budget solver on the REAL Qwen3.8-27B config (+ C10b: the plan re-evaluates to its own total,
#       strict per-(layer,width) sketch accounting)
#   C11 Theorem 7 anchor: mini-model surgery, full AND soft == dense
#   C12 self-checkpointing + telemetry diet: ON == OFF (values/grads)
# ============================================================================
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
RESULTS = []


def check(name, cond, detail=""):
    RESULTS.append((name, bool(cond), detail))
    print(f"[{'PASS' if cond else 'FAIL'}] {name} {detail}")


# ---------------------------------------------------------------- shared toys
d, D, E, T, RANK, SHARED, K = 64, 256, 8, 128, 16, 16, 3
IN = 256
gate = torch.randn(d, D) / math.sqrt(d)
up = torch.randn(d, D) / math.sqrt(d)
down = torch.randn(d, D) / math.sqrt(d)
h = torch.randn(T, d) / math.sqrt(d)
prog = torch.randint(0, E, (D,))
tok = torch.randn(T, E).softmax(-1)
A = F.silu(tok[:, prog] * torch.randn(T, D).abs() + 0.05 * torch.randn(T, D))
duty = A.mean(0)
S = neuron_sketches(A, sketch_dim=256)

# C1 — balanced fission + Theorem 1 -----------------------------------------
fus = fission_layer(duty, S, E, SHARED, seed=0, min_expert_size=8)
assign = fus["assign"]
sizes = torch.bincount(assign[assign >= 0], minlength=E)
ok1 = (bool((sizes == sizes[0]).all())
       and int(sizes[0]) == (D - SHARED) // E
       and int((assign < 0).sum()) == SHARED
       and torch.equal(torch.where(assign < 0)[0].sort().values,
                       duty.topk(SHARED).indices.sort().values))
moe = PrismMoeMLP(gate, up, down, assign, E, rank=RANK, seed=0)
with torch.no_grad():
    y_dense = moe.dense_forward(h)
    y_sum = (moe.expert_outputs(h, use_completion=False).sum(1)
             + moe.shared_forward(h, use_completion=False))
    rel = float((y_sum - y_dense).norm() / y_dense.norm())
    y_sum_c = (moe.expert_outputs(h, use_completion=True).sum(1)
               + moe.shared_forward(h, use_completion=True))
    rel_c = float((y_sum_c - y_dense).norm() / y_dense.norm())
    alpha0 = float(moe.alpha.abs().max())
check("C1 balanced fission + Theorem 1 exact dense identity at init",
      ok1 and rel < 1e-5 and rel_c < 1e-5 and alpha0 == 0.0,
      f"sizes equal & top-duty shared={ok1}, rel {rel:.2e}, "
      f"zero-gain completions {rel_c:.2e}")

# C2 — Theorem 3 serving parity ---------------------------------------------
moe2 = PrismMoeMLP(gate, up, down, assign, E, rank=RANK, seed=0)
with torch.no_grad():          # nonzero fields make parity non-trivial
    moe2.alpha.add_(zero_sum_gains(E, RANK, 1e-2, seed=3))
    moe2.beta.add_(zero_sum_gains(E, RANK, 1e-2, seed=4))
    moe2.alpha_shared.add_(torch.randn(RANK) * 1e-2)
    moe2.beta_shared.add_(torch.randn(RANK) * 1e-2)
    for e in range(E):
        moe2.gains[e] = 1.37 + 0.11 * e   # gains fuse into down_proj
worst = 0.0
with torch.no_grad():
    for e in range(E):
        W = moe2.materialize_expert(e)
        y_ref = moe2.expert_outputs(h, True)[:, e, :] * moe2.gains[e]
        y_mat = (F.silu(h @ W["gate_proj"].t())
                 * (h @ W["up_proj"].t())) @ W["down_proj"].t()
        worst = max(worst, float((y_mat - y_ref).norm() / y_ref.norm()))
    W = moe2.materialize_shared()
    y_ref = moe2.shared_forward(h, True)
    y_mat = (F.silu(h @ W["gate_proj"].t())
             * (h @ W["up_proj"].t())) @ W["down_proj"].t()
    worst = max(worst, float((y_mat - y_ref).norm() / y_ref.norm()))
    FU = moe2.materialize_experts_fused()     # the real Qwen3_5Moe 3D layout
    for e in (0, E - 1):
        W = moe2.materialize_expert(e)
        y_ref = moe2.expert_outputs(h, True)[:, e, :] * moe2.gains[e]
        gu, dn = FU["gate_up_proj"][e], FU["down_proj"][e]
        ym = (F.silu(h @ gu[:moe2.m_e].t())
              * (h @ gu[moe2.m_e:].t())) @ dn.t()
        worst = max(worst, float((ym - y_ref).norm() / y_ref.norm()))
check("C2 Theorem 3 serving parity (experts+shared+ fused layout)",
      worst < 1e-5, f"worst rel {worst:.2e}")

# C3 — gradient flow ---------------------------------------------------------
moe.zero_grad()
y, aux = moe(h, mode="soft", k=K, use_completion=True, return_aux=True)
loss = (y.square().mean() + aux["p"].logsumexp(-1).mean()
        + 0.01 * zero_sum_penalty(moe.alpha) + 0.01 * zero_sum_penalty(moe.beta))
loss.backward()
check("C3 gradients reach router/ZSSI/field/gains; substrate frozen",
      moe.router.weight.grad is not None and moe.alpha.grad is not None
      and moe.P.grad is not None and moe.U.grad is not None
      and moe.gains.grad is not None
      and moe.W_gate.grad is None and moe.W_up.grad is None
      and moe.W_down.grad is None)

# C4 — Theorem 4 -------------------------------------------------------------
OUT = 384
W = torch.randn(OUT, IN) / math.sqrt(IN)
mast = AntennaMast(IN, sketch_dim=64, seed=0)
plan = plan_row_modes(OUT, IN, n_blocks=12, carrier_frac=0.05)
rl = ResonantLinear(W, plan, mast, topk=2)
x = torch.randn(128, IN)
with torch.no_grad():
    y_dense_l = F.linear(x, W)
    y_full = rl(x, mode="full")
rel4 = float((y_dense_l - y_full).norm() / y_dense_l.norm())
Wm = rl.materialize_dense()
check("C4 Theorem 4: full-mode == dense; materialize_dense bitwise",
      rel4 < 1e-5 and torch.equal(Wm, W),
      f"rel {rel4:.2e}, max|dW| {float((Wm - W).abs().max()):.1e}")

# C5 — Theorem 5 -------------------------------------------------------------
with torch.no_grad():
    topi_c = rl.router(mast(x)).topk(rl.topk, dim=-1).indices
    blk_energy = rl.block_outputs(x).pow(2).sum(-1)     # [T, B]
    y_sp = rl(x, mode="topk")
    err = (y_dense_l - y_sp).norm(dim=-1)
    dropped = blk_energy.clone()
    dropped.scatter_(1, topi_c, 0.0)
    bound = dropped.sum(-1).sqrt()
frac_ok = float((err <= bound * 1.001 + 1e-6).float().mean())
check("C5 Theorem 5: tail certificate holds on 100% of tokens",
      frac_ok == 1.0,
      f"worst ratio {float((err / (bound + 1e-9)).max()):.3f}")

# C6 — SectionedProjector (fused + split layouts) ----------------------------
q_r, k_r, v_r, z_r = 64, 64, 96, 96          # v: 3 heads x 32 rows
mast2 = AntennaMast(IN, sketch_dim=48, seed=1)
xs = torch.randn(64, IN)
Wf = torch.randn(q_r + k_r + v_r + z_r, IN) / math.sqrt(IN)
sp = SectionedProjector(Wf, (q_r, k_r, v_r, z_r), v_head_rows=32,
                        plan_q=plan_row_modes(q_r, IN, 4, 0.08),
                        plan_z=plan_row_modes(z_r, IN, 4, 0.08),
                        mast=mast2, morm=MoRMWritePolicy(3, mast2, 2), topk=2)
with torch.no_grad():
    y_ref_fused = F.linear(xs, Wf)
    d_err = float((sp(xs, mode="dense") - y_ref_fused).abs().max())
    out_f = sp(xs, mode="full")
    qf, kf, vf, zf = torch.split(y_ref_fused, [q_r, k_r, v_r, z_r], -1)
    fq, fk, fv, fz = torch.split(out_f, [q_r, k_r, v_r, z_r], -1)
    f_err = max(float((fq - qf).norm() / qf.norm()),
                float((fz - zf).norm() / zf.norm()), float((fk - kf).norm()))
# split layout (qwen3_5 family): in_proj_qkv [q|k|v] + separate in_proj_z
Wqkv = torch.randn(q_r + k_r + v_r, IN) / math.sqrt(IN)
sp2 = SectionedProjector(Wqkv, (q_r, k_r, v_r, 0), v_head_rows=32,
                         plan_q=plan_row_modes(q_r, IN, 4, 0.08),
                         plan_z=None, mast=mast2,
                         morm=MoRMWritePolicy(3, mast2, 2), topk=2)
with torch.no_grad():
    y_split = F.linear(xs, Wqkv)
    d_err2 = float((sp2(xs, mode="dense") - y_split).abs().max())
    out_f2 = sp2(xs, mode="full")
    q2, k2, v2 = torch.split(y_split, [q_r, k_r, v_r], -1)
    g2, h2, _ = torch.split(out_f2, [q_r, k_r, v_r], -1)
    f_err2 = max(float((g2 - q2).norm() / q2.norm()), float((h2 - k2).norm()))
check("C6 SectionedProjector: dense == stock projection; full exact; split ok",
      d_err < 1e-6 and f_err < 1e-5 and d_err2 < 1e-6 and f_err2 < 1e-5
      and tuple(out_f2.shape) == (64, q_r + k_r + v_r),
      f"dense {d_err:.2e}/{d_err2:.2e}, full {f_err:.2e}/{f_err2:.2e}")

# C7 — MoRM mask + certificates ---------------------------------------------
Tm, H, DK, DV = 32, 6, 16, 16
S_prev = torch.randn(Tm, H, DK, DV)
k_ = torch.randn(Tm, H, DK)
v_ = torch.randn(Tm, H, DV)
beta = torch.rand(Tm, H)
policy = MoRMWritePolicy(H, mast2, m_write=3)
with torch.no_grad():
    phi_t = mast2(torch.randn(Tm, IN))
    mask = policy(phi_t)
    S_dense_w = morm_state_update(S_prev, k_, v_, beta, torch.ones_like(mask))
    S_sparse = morm_state_update(S_prev, k_, v_, beta, mask)
    gap, bnd = morm_error_bound(S_dense_w, S_sparse, k_, v_, beta,
                                (mask > 0.5).float())
    m_exact = (morm_state_update(S_prev, k_, v_, beta, torch.ones_like(mask))
               - S_dense_w).abs().max()
check("C7 MoRM: exact m writes; m=H bit-exact; m<H under the Prop-6 bound",
      bool(((mask > 0.5).sum(-1) == 3).all())
      and bool((gap <= bnd + 1e-4).all())
      and bool(m_exact == 0),
      f"worst ratio {float((gap / (bnd + 1e-9)).max()):.3f}")

# C8 — grad-mode supergraph == sparse oracle ---------------------------------
def _v_reference(sp_, x_, mask_):
    """The chunked-gather MoRM v computation (the mathematically sparse
    path) — the reference oracle for the masked-dense supergraph."""
    b2_, b3_ = int(sp_.sec_bounds[2]), int(sp_.sec_bounds[3])
    Wv_ = sp_.weight[b2_:b3_].reshape(sp_.n_v_heads, sp_.v_head_rows,
                                      sp_.weight.shape[1])
    sel_ = (mask_ > 0.5).nonzero(as_tuple=False)
    v_out = x_.new_zeros(x_.shape[0], sp_.v_rows)
    if sel_.shape[0] > 0:
        c_ = _gather_chunk(sp_.v_head_rows * sp_.weight.shape[1]
                           * sp_.weight.element_size())
        parts_ = []
        for i_ in range(0, sel_.shape[0], c_):
            Wc_ = Wv_[sel_[i_:i_ + c_, 1]]
            xs_ = x_[sel_[i_:i_ + c_, 0]]
            parts_.append(torch.bmm(Wc_, xs_.unsqueeze(-1)).squeeze(-1))
        v_sel_ = torch.cat(parts_, 0)
        v_sel_ = v_sel_ * sp_.v_gains.to(v_sel_.dtype)[sel_[:, 1]][:, None]
        hr_ = sp_.v_head_rows
        rows_ = (sel_[:, 1][:, None] * hr_
                 + torch.arange(hr_, device=x_.device)[None, :])
        v_out[sel_[:, 0][:, None].expand(-1, hr_).reshape(-1),
              rows_.reshape(-1)] = v_sel_.reshape(-1).to(v_out.dtype)
    return v_out

b2_, b3_ = int(sp.sec_bounds[2]), int(sp.sec_bounds[3])
with torch.no_grad():
    out_ng = sp(xs, mode="topk")                 # segmented serving path
    mask_sp = sp.last_out["beta_mask"].clone()
    v_ref = _v_reference(sp, xs, mask_sp)
    v_ng_err = float((out_ng[:, b2_:b3_] - v_ref).abs().max())
out_g = sp(xs.clone(), mode="topk")              # masked-dense grad path
v_g = sp.last_out["v"].detach()
v_grad_err = float((v_g - v_ref).abs().max())
zeros_ok = (bool((v_g.reshape(64, 3, 32)[mask_sp <= 0.5] == 0).all())
            if (mask_sp <= 0.5).any() else True)
rl2 = ResonantLinear(torch.randn(128, IN) / math.sqrt(IN),
                     plan_row_modes(128, IN, 4, 0.08),
                     AntennaMast(IN, sketch_dim=48, seed=3), topk=2)
with torch.no_grad():
    y_rl_ng = rl2(xs, mode="topk")
y_rl_g = rl2(xs.clone(), mode="topk").detach()
check("C8 grad-mode supergraph == sparse oracle (MoRM v + ResonantLinear)",
      v_ng_err < 1e-5 and v_grad_err < 1e-5 and zeros_ok
      and float((y_rl_g - y_rl_ng).abs().max()) < 1e-5,
      f"v {v_ng_err:.2e}/{v_grad_err:.2e}, skipped-zeros={zeros_ok}, "
      f"rl {float((y_rl_g - y_rl_ng).abs().max()):.2e}")

# C9 — HeadFunnel ------------------------------------------------------------
Vh, d_head = 4096, 128
U0, _ = torch.linalg.qr(torch.randn(Vh, 64))
V0, _ = torch.linalg.qr(torch.randn(d_head, 64))
V0 = V0.t()                              # [64, d_head] orthonormal rows
svals = torch.linspace(8, 0.01, 64)
Wh = U0 @ torch.diag(svals) @ V0 + 0.001 * torch.randn(Vh, d_head)
hf = HeadFunnel(Wh.clone(), rank=32, shortlist=256)
xh = torch.randn(64, d_head)
with torch.no_grad():
    ref = xh @ Wh.t()
    got, sl = hf(xh, mode="funnel", ref_logits=ref)
    exact_err = float((got - ref.gather(1, sl)).abs().max())
    hit1 = float((sl == ref.argmax(-1)[:, None]).any(-1).float().mean())
    # chunked == one-shot: force a 3-token chunk and compare
    _saved_sc = _shortlist_chunk
    _shortlist_chunk = lambda *a, **k: 3        # noqa: E731 (monkeypatch)
    got_c, sl_c = hf(xh, mode="funnel")
    _shortlist_chunk = _saved_sc
    inter = (sl_c.unsqueeze(-1) == sl.unsqueeze(1)).any(-1).sum(-1).float()
    exact_c = float((got_c - ref.gather(1, sl_c)).abs().max())
check("C9 HeadFunnel: exact shortlist rows; recall; chunked == one-shot",
      exact_err < 1e-4 and hit1 > 0.95 and bool((inter >= 254).all())
      and exact_c < 1e-5 and hf.active_params() < Vh * d_head * 0.5,
      f"exact {exact_err:.2e}, recall {hit1:.3f}, "
      f"min overlap {float(inter.min()):.0f}/256")

# C10 — 3B budget solver on the REAL Qwen3.8-27B config ----------------------
# d=5120, L=64 (48 linear + 16 full), d_ff=17408, V=248320; full attn
# q 24*256, k 4*256, v(+gate fused) 2*4*256, o_in 6144; deltanet q/k 16*128,
# v/z 48*128, ba 2*48, head_rows 128, out_proj in 48*128.
ss = dict(q_rows=6144, k_rows=1024, v_rows=1024, gate_rows=1024, o_in=6144,
          dn_q_rows=2048, dn_k_rows=2048, dn_v_rows=6144, dn_z_rows=6144,
          dn_ba_rows=96, dn_v_head_rows=128, dn_out_in=6144)
sol = solve_budget_v2(d=5120, D_ffn=17408, n_layers=64, n_linear_attn=48,
                      n_full_attn=16, vocab=248320, ss=ss,
                      target_active=3.0e9, verbose=False,
                      shared_ffn=512, m_write_max=26, shortlist_max=8192)   # the REQUESTED real geometry
w_ = sol["waterfall"]
ffn_dense_b = 64 * 3 * 5120 * 17408
attn_dense_b = (48 * ((2048 + 2048 + 6144 + 6144) * 5120 + 96 * 5120
                      + 6144 * 5120)
                + 16 * ((6144 + 1024 + 2048 + 5120) * 5120))
total_dense_b = ffn_dense_b + attn_dense_b + 2 * 248320 * 5120
check("C10 3B mandate on the real config: solver <= 3.0B; ~27B; >8x",
      w_["TOTAL_ACTIVE"] <= 3.0e9 and 2.4e9 <= w_["TOTAL_ACTIVE"] and sol["plan"]["feasible"]
      and 25e9 < total_dense_b < 28.5e9
      and total_dense_b / w_["TOTAL_ACTIVE"] > 8.0,
      f"{w_['TOTAL_ACTIVE'] / 1e9:.3f}B active / {total_dense_b / 1e9:.2f}B "
      f"dense ({total_dense_b / w_['TOTAL_ACTIVE']:.1f}x)")

# C10b — the plan is CONSUMED, so it must re-evaluate to its own total; sketch reads are strict ------
_p = sol["plan"]
w_chk = active_waterfall(5120, 17408, 64, 48, 16, 248320, ss, E_ffn=_p["E_ffn"], k_ffn=_p["k_ffn"],
                         shared_ffn=_p["shared_ffn"], rank_zssi=64, mast_S=_p["mast_S"],
                         lin_blocks=_p["lin_blocks"], lin_topk=_p["lin_topk"],
                         carrier_frac=_p["carrier_frac"], m_write=_p["m_write"],
                         funnel_rank=_p["funnel_rank"], funnel_shortlist=_p["funnel_shortlist"])
_app = mast_reads(5120, 64, 48, 16, ss, _p["mast_S"], "applications")
_dis = mast_reads(5120, 64, 48, 16, ss, _p["mast_S"], "distinct")
check("C10b plan consistency + strict sketch accounting (per layer & width, not per matrix)",
      w_chk["TOTAL_ACTIVE"] == w_["TOTAL_ACTIVE"] and _app > 20 * _dis and _p["shared_ffn"] % 32 == 0
      and (17408 - _p["shared_ffn"]) % _p["E_ffn"] == 0,
      f"plan {_p['shared_ffn']} shared / mast {_p['mast_S']} / m_write {_p['m_write']} / V_s "
      f"{_p['funnel_shortlist']}; sketch reads {_app / 1e6:.0f}M (applications) vs {_dis / 1e6:.1f}M (distinct)")

# C11 — Theorem 7 anchor on a mini model -------------------------------------
class MiniMLP(nn.Module):
    def __init__(self, d_, D_):
        super().__init__()
        self.gate_proj = nn.Linear(d_, D_, bias=False)
        self.up_proj = nn.Linear(d_, D_, bias=False)
        self.down_proj = nn.Linear(D_, d_, bias=False)

    def forward(self, h_):
        return self.down_proj(F.silu(self.gate_proj(h_)) * self.up_proj(h_))


class MiniBlock(nn.Module):
    def __init__(self, d_, D_):
        super().__init__()
        self.norm = nn.RMSNorm(d_)
        self.mlp = MiniMLP(d_, D_)
        self.attn_q = nn.Linear(d_, d_, bias=False)
        self.attn_o = nn.Linear(d_, d_, bias=False)

    def forward(self, h_):
        a_ = self.attn_o(self.attn_q(self.norm(h_)))
        return h_ + a_ + self.mlp(h_)


class MiniModel(nn.Module):
    def __init__(self, d_=128, D_=256, L_=3, V_=99):
        super().__init__()
        self.embed = nn.Embedding(V_, d_)
        self.layers = nn.ModuleList([MiniBlock(d_, D_) for _ in range(L_)])
        self.head = nn.Linear(d_, V_, bias=False)

    def forward(self, ids_):
        h_ = self.embed(ids_)
        for l_ in self.layers:
            h_ = l_(h_)
        return self.head(h_)


mm = MiniModel()
ids = torch.randint(0, 99, (8, 16))
with torch.no_grad():
    ref_logits = mm(ids)
mast3 = AntennaMast(128, 32, seed=3)
for blk in mm.layers:
    for nm_ in ("attn_q", "attn_o"):
        W0 = getattr(blk, nm_).weight.data
        pl_ = plan_row_modes(W0.shape[0], W0.shape[1], 4, 0.06)
        rl3 = ResonantLinear(W0, pl_, mast3, topk=1)
        rl3.default_mode = "full"
        setattr(blk, nm_, rl3)
with torch.no_grad():
    full_logits = mm(ids)
    rel_f = float((ref_logits - full_logits).norm() / ref_logits.norm())
    for blk in mm.layers:          # soft at init == dense (the training anchor)
        blk.attn_q.default_mode = "soft"
        blk.attn_o.default_mode = "soft"
    soft_logits = mm(ids)
    rel_s = float((ref_logits - soft_logits).norm() / ref_logits.norm())
check("C11 Theorem 7 anchor: surgery full-mode AND soft-at-init == dense",
      rel_f < 1e-5 and rel_s < 1e-5, f"full {rel_f:.2e}, soft {rel_s:.2e}")

# C12 — self-checkpointing + telemetry contracts (the memory guarantees) -----
def _bw(mod_, x_, mode_):
    """Train-mode fwd+bwd with a fixed random upstream grad and a FROZEN
    input; returns (values, {param name: grad})."""
    for p_ in mod_.parameters():
        p_.grad = None
    mod_.train()
    x_ = x_.clone()                       # requires_grad deliberately False
    y_ = mod_(x_, mode=mode_)
    torch.manual_seed(12)
    w_ = torch.randn_like(y_)
    (y_ * w_).sum().backward()
    return y_.detach(), {n_: p_.grad for n_, p_ in mod_.named_parameters()
                         if p_.requires_grad}


def _ckpt_eq(mod_, x_, mode_):
    """self-checkpoint ON == OFF: values bitwise, same reached param set,
    grads allclose (atol+rtol floor around exact zeros)."""
    mod_.self_ckpt = False
    y_o, g_o = _bw(mod_, x_, mode_)
    mod_.self_ckpt = True
    y_n, g_n = _bw(mod_, x_, mode_)
    a_o = {n_ for n_, g_ in g_o.items() if g_ is not None}
    a_n = {n_ for n_, g_ in g_n.items() if g_ is not None}
    par_ok = max((float((g_n[n_] - g_o[n_]).norm())
                  - (1e-6 + 1e-6 * float(g_o[n_].norm())))
                 for n_ in a_o) if a_o else -1.0
    return torch.equal(y_o, y_n) and a_o == a_n and par_ok <= 0.0, a_n


sp3 = SectionedProjector(Wf.clone(), (q_r, k_r, v_r, z_r), v_head_rows=32,
                         plan_q=plan_row_modes(q_r, IN, 4, 0.08),
                         plan_z=plan_row_modes(z_r, IN, 4, 0.08),
                         mast=mast2, morm=MoRMWritePolicy(3, mast2, 2), topk=2)
ok_sp, n_sp = _ckpt_eq(sp3, xs, "soft")
rl4 = ResonantLinear(torch.randn(128, IN) / math.sqrt(IN),
                     plan_row_modes(128, IN, 4, 0.08),
                     AntennaMast(IN, sketch_dim=48, seed=5), topk=2)
ok_rl, _ = _ckpt_eq(rl4, xs, "soft")
moe3 = PrismMoeMLP(gate, up, down, assign, E, rank=RANK, seed=27, k_topk=K)
with torch.no_grad():      # non-trivial fields so the completion path runs
    moe3.alpha.add_(torch.randn(E, RANK) * 0.3)
    moe3.beta.add_(torch.randn(E, RANK) * 0.3)
    moe3.gains.add_(torch.randn(E) * 0.2)
    moe3.router.weight.add_(torch.randn(E, d) * 0.1)
ok_moe, _ = _ckpt_eq(moe3, h, "soft")
# telemetry contract: a grad-mode forward WITHOUT _store_aux stores nothing;
# with it, aux is stored. SectionedProjector.telemetry is value-transparent.
moe3.self_ckpt = False
moe3._store_aux = False
_ = moe3(h, mode="soft")
aux_off = getattr(moe3, "last_aux", None)
moe3._store_aux = True
_ = moe3(h, mode="soft")
aux_on = moe3.last_aux
moe3._store_aux = False
moe3.self_ckpt = True
sp3.telemetry = True
y_ton = sp3(xs.clone(), mode="topk")
t_on = sp3.last_out is not None
sp3.telemetry = False
y_toff = sp3(xs.clone(), mode="topk")
t_off = sp3.last_out is None
sp3.telemetry = True
with torch.no_grad():
    sp3.eval()
    _ = sp3(xs, mode="soft")
    m_ev = sp3.last_mask
    sp3.train()
    _ = sp3(xs, mode="soft")
    m_tr = sp3.last_mask
check("C12 self-checkpointing ON == OFF (3 families, frozen input); "
      "telemetry diet value-transparent",
      ok_sp and ok_rl and ok_moe and aux_off is None and aux_on is not None
      and "p" in aux_on and torch.equal(y_ton, y_toff)
      and t_on and t_off and torch.equal(m_ev, m_tr),
      f"params-with-grad sp={len(n_sp)}; aux gated; masks bitwise")

n_pass = sum(1 for _, ok_, _ in RESULTS if ok_)
print(f"\n==== verification suite: {n_pass}/{len(RESULTS)} checks passed ====")
assert n_pass == len(RESULTS), "verification FAILED — see the FAIL lines above."
print("All load-bearing contracts verified. Proceeding to the real transformation.")


## MoRM 2.0 verification — the M1-M10 contracts

Every theorem above is executable. This cell runs the full contract suite on a synthetic GDN stream (CPU, ~40 s at demo size) plus the policy-stack smoke test (critic ridge fit on exact labels, elastic forward + controller, BetaGate bitwise-exact mode, decode scheduler). The v1 contracts C1-C12 in the previous cell are unchanged and still pass — `MoRMWritePolicy` is kept as the legacy reference implementation.


In [ ]:
# ============================================================================
# MoRM 2.0 verification — the 10 load-bearing contracts (M1-M10) + policy
# smoke test. Pure torch, CPU, self-contained. Raises AssertionError on any
# violated contract.
# ============================================================================
print("=== MoRM 2.0 contracts (M1-M10) ===")
MORM2_REPORT = verify_morm2(verbose=True)

# ---- policy-stack smoke test (ridge fit on exact labels -> elastic forward
# ---- -> BetaGate bitwise-exact mode -> decode scheduler) --------------------
import math as _m
class _MastSmoke:
    """AntennaMast stand-in (the real mast is created in calibration)."""
    def __init__(self, S=64, d=32, seed=0):
        g = torch.Generator().manual_seed(seed)
        self.S, self.d = S, d
        self.R = torch.randn(S, d, generator=g) / _m.sqrt(d)
    def __call__(self, h):
        return F.silu(h @ self.R.t())

torch.manual_seed(0)
_sH, _sdk, _sdv, _sd = 6, 16, 12, 32
_mast_s = _MastSmoke()
_h_s = torch.randn(256, _sd)
_phi_s = _mast_s(_h_s)
_q_s = F.normalize(torch.randn(256, _sH, _sdk), dim=-1)
_k_s = F.normalize(torch.randn(256, _sH, _sdk), dim=-1)
_v_s = torch.randn(256, _sH, _sdv)
_b_s = torch.sigmoid(torch.randn(256, _sH) + 1.0)
_lam_s = torch.full((256, _sH), 0.96)
_, _, _d_s = gdn_sequential(_q_s, _k_s, _v_s, _b_s, _lam_s, "plain",
                            _sdk ** -0.5, return_deltas=True)
_lab_s = write_influence_labels(_q_s, _k_s, _d_s, _b_s, _lam_s,
                                scale=_sdk ** -0.5, window=256)["labels"]
_pol_s = MoRM2Policy(_sH, _mast_s, m_bar=3, m_max=4)
_pol_s.ridge_init(_phi_s, None, _lab_s)
with torch.no_grad():
    _r2 = float(1 - ((_pol_s.scores(_phi_s) - _lab_s).pow(2).sum()
                     / _lab_s.pow(2).sum()))
_pol_s.eval()
for _ in range(200):
    _mk = _pol_s(_phi_s)
    _pol_s.controller_update(float((_mk > 0.5).sum(-1).float().mean()))
_mk = _pol_s(_phi_s)
print(f"policy smoke: critic R^2 {_r2:.2f} | elastic mean m "
      f"{float((_mk > 0.5).sum(-1).float().mean()):.2f} (target 3) | "
      f"threshold {float(_pol_s.rate_threshold):.3e}")
assert _r2 > 0.3 and abs(float((_mk > 0.5).sum(-1).float().mean()) - 3) < 1.0
print("MoRM 2.0 verification: ALL CONTRACTS PASS")


## v9 contracts — the new formulas, asserted on CPU toys (U1–U12)

Seconds on CPU. Every v9 upgrade has an executable check and every check owns a switch: if a contract fails, **that formula falls back to its v8 behaviour (loudly)** instead of stopping a multi-hour run — or silently training with a broken formula. Contracts never raise.

| | What is asserted | Switch on failure |
|---|---|---|
| U1 | Lawson–Hanson NNLS satisfies the KKT conditions (c ≥ 0, ∇ ≥ 0, c·∇ = 0) | `FFN_CAL_MODE="v8"` |
| U2 | the calibration objective equals the error of the module that is actually served (k = 1, 2); the router is validated; NNLS gains beat unit gains | `FFN_CAL_MODE="v8"` |
| U3 | neuron importance == brute-force E[(a_r w_r)²]; shared-core ranking | `FFN_CAL_MODE="v8"` |
| U4 | Partition-KL: coarse-graining identity, data-processing lower bound, sensitivity to support mass, finite gradients | `KL_MODE="support"` |
| U5 | EMRD orders routers by oracle energy and gives k = 1 a gradient; group energies are exact | `LAMBDA_EMRD=0` |
| U6 | LTD is 0 for the dense twin, 1 for a 2× output, target detached | `LAMBDA_LTD=0` |
| U7 | HeadFunnel: `U_r = W P_r`, Cauchy–Schwarz certificate, κ fit never loses recall, shortlist rows exact, `active_params` formula | `FUNNEL_USE_RHO=False` |
| U8 | budget solver: trims only what the budget demands, the plan re-evaluates to its own total, an impossible target is reported infeasible | — |
| U9 | the torchao crash root cause reproduced on a toy graph (a module with two parents is visited twice); the id-dedup filter fixes it | — |
| U10 | census of the live module graph == the waterfall that planned it (toy transformer built from the real classes, both sketch accountings) | — |


In [ ]:
# ============================================================================
# v9 contracts — the new formulas, asserted numerically on CPU toys (seconds).
# Each contract is independent and SAFE: a failing formula flips its own switch back to the
# v8 behaviour (loudly) instead of stopping the run or — worse — silently training with it.
#   U1  Lawson-Hanson NNLS: KKT optimality of the gain solver
#   U2  serving-consistent gains: calibration objective == the served module error (k = 1, 2)
#   U3  output-energy importance == brute-force E[(a_r w_r)^2]; shared core ranking
#   U4  Partition-KL: coarse-grain identity, data-processing bound, mass sensitivity, finite grads
#   U5  EMRD: ordering, k=1 router gradient, oracle group energies == brute force
#   U6  LTD: zero / unit / detached target
#   U7  HeadFunnel: exact projection, Cauchy-Schwarz certificate, kappa fit, exact shortlist rows
#   U8  budget solver: honest trims, plan re-evaluates to its own total, infeasible is reported
#   U9  torchao crash root cause: a module with two parents is visited twice; the id-dedup filter once
#   U10 census(module walk) == waterfall on a toy transformer built from the REAL classes
#   U11 MoRM critic intercept: per-head level recovered (log-R^2), more influence mass at an equal budget
#   U12 shadow execution (v10): projection identity, grad path == served path, error vs zeroing, router gradient
# ============================================================================
def _run_v9_contracts():
    out = []
    flips = []

    def contract(name, fn, on_fail=None):
        try:
            ok, detail = fn()
        except Exception as e:                       # a contract that crashes is a failed contract
            ok, detail = False, f"{type(e).__name__}: {str(e)[:200]}"
        out.append((name, bool(ok), detail))
        print(f"[{'PASS' if ok else 'FAIL'}] {name} {detail}")
        if not ok and on_fail is not None:
            flips.append(on_fail())
        return ok

    g_ = globals()

    def flip(var, value):
        def _do():
            old = g_.get(var)
            g_[var] = value
            return f"{var}: {old!r} -> {value!r}"
        return _do

    torch.manual_seed(1234)

    # ----------------------------------------------------------------- U1 NNLS (KKT)
    def u1():
        rng = torch.Generator().manual_seed(7)
        worst, n_act = 0.0, 0
        for n in (4, 9, 16):
            M = torch.randn(3 * n, n, generator=rng).double()
            A = (M.t() @ M).numpy()
            b = (M.t() @ torch.randn(3 * n, generator=rng).double()).numpy()
            c = nnls_normal(A, b)
            grad = A @ c - b                         # KKT: c >= 0, grad >= 0, c * grad == 0
            scale = max(float(abs(b).max()), 1.0)
            viol = max(float((-c).max()), float((-grad).max()), float(abs(c * grad).max()))
            worst = max(worst, viol / scale)
            n_act += int((c == 0).sum())
        return worst < 1e-7 and n_act > 0, f"max KKT violation {worst:.1e} (constraints active on {n_act} coords)"
    contract("U1 NNLS gain solver satisfies the KKT conditions", u1, flip("FFN_CAL_MODE", "v8"))

    # ----------------------------------------------------------------- shared toy FFN
    def toy_ffn(seed, d=48, D=192, E=8, shared=16, T=768, rank=8, k=1):
        g = torch.Generator().manual_seed(seed)
        C = torch.randn(E, d, generator=g)
        C = C / C.norm(dim=1, keepdim=True) * 3.0                    # planted cluster centres
        cid = torch.randint(0, E, (T,), generator=g)
        H = C[cid] + 0.4 * torch.randn(T, d, generator=g)
        owner = torch.full((D,), -1, dtype=torch.long)
        owner[shared:] = torch.arange(D - shared) % E                # planted expert of every routed neuron
        Wg = 0.15 * torch.randn(d, D, generator=g)
        Wu = 0.15 * torch.randn(d, D, generator=g)
        for r in range(shared, D):
            Wg[:, r] += 0.8 * C[int(owner[r])] / 3.0
            Wu[:, r] += 0.8 * C[int(owner[r])] / 3.0
        Wd = 0.2 * torch.randn(d, D, generator=g)
        moe = PrismMoeMLP(Wg, Wu, Wd, owner, E, rank=rank, seed=seed, k_topk=k)
        moe.eval()
        return moe, H, (Wg, Wu, Wd, owner)

    # ----------------------------------------------------------------- U2 serving consistency
    def u2():
        notes, ok_all = [], True
        for k in (1, 2):
            moe, H, _ = toy_ffn(11 + k, k=k)
            with torch.no_grad():
                cal = calibrate_ffn_layer(moe, H, k, router_mode="auto", seed=3)
                y_top = moe(H, mode="topk")
                y_d = moe.dense_forward(H)
                y_s = moe.shared_forward(H, use_completion=False)
                err_mod = float((y_top.float() - y_d.float()).pow(2).sum())
                tt = float((y_d.float() - y_s.float()).pow(2).sum())
                rel_mod = err_mod / max(tt, 1e-30)
            rel_cal = cal["rel_err_all"]
            consistent = abs(rel_mod - rel_cal) <= 2e-3 * max(rel_cal, 1e-3) + 1e-6
            better = cal["rel_err"] <= 1.05 * cal["rel_err_prior"] + 1e-4
            routed = cal["capture"] >= cal["capture_proto"] - 1e-6 and cal["capture_all"] >= 0.5 * cal["oracle"]
            ok = consistent and better and routed
            ok_all &= ok
            notes.append(f"k={k}: router={cal['router']} capture {cal['capture_all']:.2f}/{cal['oracle']:.2f} "
                         f"served rel.err module {rel_mod:.4f} vs calibration {rel_cal:.4f} "
                         f"(unit gains {cal['rel_err_prior']:.4f})")
        return ok_all, " | ".join(notes)
    contract("U2 calibration objective == the served module error; router validated; gains beat unit gains",
             u2, flip("FFN_CAL_MODE", "v8"))

    # ----------------------------------------------------------------- U3 importance
    def u3():
        moe, H, (Wg, Wu, Wd, owner) = toy_ffn(5)
        with torch.no_grad():
            A = F.silu(H @ Wg) * (H @ Wu)                            # [T, D] activations
            imp = ffn_importance((A * A).mean(0), Wd)
            brute = torch.stack([((A[:, r:r + 1] * Wd[:, r][None, :]) ** 2).sum(-1).mean()
                                 for r in range(A.shape[1])])
            same = float((imp - brute).abs().max() / brute.abs().max())
            S_ = 16
            top_imp = set(imp.topk(S_).indices.tolist())
            top_brute = set(brute.topk(S_).indices.tolist())
            top_mean = set(A.mean(0).topk(S_).indices.tolist())      # v8: signed mean activation
        return same < 1e-5 and top_imp == top_brute, \
            f"identity rel {same:.1e}; core == brute force; overlap with v8 signed-mean core {len(top_imp & top_mean)}/{S_}"
    contract("U3 neuron importance == E[a_r^2] ||W_down[:, r]||^2 (brute force); core ranking", u3,
             flip("FFN_CAL_MODE", "v8"))

    # ----------------------------------------------------------------- U4 Partition-KL
    def u4():
        V_, K_ = 40, 8
        g = torch.Generator().manual_seed(3)
        tl = 2.0 * torch.randn(64, V_, generator=g)
        sl = tl + 0.7 * torch.randn(64, V_, generator=g)
        t_full, s_full = tl.log_softmax(-1), sl.log_softmax(-1)
        ids = t_full.topk(K_, dim=-1).indices
        t_lp, s_lp = t_full.gather(-1, ids), s_full.gather(-1, ids)
        pk = partition_kl(t_lp, s_lp)
        P = torch.cat([t_lp.exp(), (1 - t_lp.exp().sum(-1, keepdim=True)).clamp_min(1e-12)], -1)
        Q = torch.cat([s_lp.exp(), (1 - s_lp.exp().sum(-1, keepdim=True)).clamp_min(1e-12)], -1)
        brute = (P * (P.log() - Q.log())).sum(-1)
        full_kl = (t_full.exp() * (t_full - s_full)).sum(-1)
        ident = float((pk - brute).abs().max())
        bound = float((pk - full_kl).max())                         # data processing: pk <= full KL
        s_shift = s_lp + math.log(0.5)                              # halve the support mass, keep the shape
        blind = float((kl_support(t_lp, s_shift) - kl_support(t_lp, s_lp)).abs().max())
        sees = float((partition_kl(t_lp, s_shift) - pk).mean())
        sat = (s_lp + 40.0).detach().requires_grad_(True)           # support mass -> 1: stable?
        sat_lp = sat - sat.logsumexp(-1, keepdim=True) * 0.0 - 1e-9
        partition_kl(t_lp, torch.minimum(sat_lp, torch.zeros(()))).mean().backward()
        g_ok = bool(torch.isfinite(sat.grad).all())
        s_req = s_lp.detach().requires_grad_(True)
        partition_kl(t_lp, s_req).mean().backward()
        g_ok &= bool(torch.isfinite(s_req.grad).all())
        ok = ident < 1e-4 and bound < 1e-5 and blind < 1e-5 and sees > 1e-3 and g_ok
        return ok, (f"coarse-grain identity {ident:.1e}; <= full KL (max excess {bound:.1e}); v8 KL blind to "
                    f"support mass ({blind:.1e}) while Partition-KL +{sees:.3f}; grads finite {g_ok}")
    contract("U4 Partition-KL: identity, lower bound, mass sensitivity, finite gradients", u4,
             flip("KL_MODE", "support"))

    # ----------------------------------------------------------------- U5 EMRD + oracle energies
    def u5():
        T_, E_ = 256, 8
        g = torch.Generator().manual_seed(9)
        en = torch.rand(T_, E_, generator=g) ** 4
        best, worst = en.argmax(-1), en.argmin(-1)

        def conc(idx):
            p = torch.full((T_, E_), 0.02)
            p[torch.arange(T_), idx] = 1 - 0.02 * (E_ - 1)
            return p
        l_good, l_uni, l_bad = (float(emrd_loss(p, en, 1)) for p in
                                (conc(best), torch.full((T_, E_), 1.0 / E_), conc(worst)))
        lg = torch.zeros(T_, E_, requires_grad=True)
        emrd_loss(lg.softmax(-1), en, 1).backward()
        gnorm = float(lg.grad.abs().sum())
        act = torch.randn(50, 30)
        dn2 = torch.rand(30) + 0.1
        ra = torch.randint(0, 5, (30,))                              # 5 = the "shared" group
        got = oracle_group_energy(act, dn2, ra, 4)
        brute = torch.stack([(act[:, ra == gi] ** 2 * dn2[ra == gi]).sum(-1) for gi in range(5)], 1)
        e_err = float((got - brute).abs().max())
        ok = l_good < l_uni < l_bad and gnorm > 0 and e_err < 1e-5 and got.shape == (50, 5)
        return ok, (f"loss good {l_good:.3f} < uniform {l_uni:.3f} < bad {l_bad:.3f}; "
                    f"k=1 router gradient {gnorm:.2e}; group energies == brute force ({e_err:.1e})")
    contract("U5 EMRD orders routers by oracle energy and gives k=1 a gradient; group energies exact", u5,
             flip("LAMBDA_EMRD", 0.0))

    # ----------------------------------------------------------------- U6 LTD
    def u6():
        y_d = torch.randn(32, 16).requires_grad_(True)
        y = torch.randn(32, 16, requires_grad=True)
        z0 = float(ltd_loss(y_d.detach().clone(), y_d))
        one = float(ltd_loss(2.0 * y_d.detach(), y_d))
        ltd_loss(y, y_d).backward()
        return abs(z0) < 1e-9 and abs(one - 1.0) < 1e-5 and y_d.grad is None and y.grad is not None, \
            f"ltd(y_d)={z0:.1e}, ltd(2 y_d)={one:.6f}, dense twin detached={y_d.grad is None}"
    contract("U6 layer-local dense-twin loss: 0 / 1 / detached target", u6, flip("LAMBDA_LTD", 0.0))

    # ----------------------------------------------------------------- U7 HeadFunnel
    def u7():
        V_, d_, r_, sl_ = 300, 32, 8, 48
        g = torch.Generator().manual_seed(21)
        decay = torch.linspace(1.0, 0.05, d_)
        W = torch.randn(V_, d_, generator=g) * decay[None, :]
        hf = HeadFunnel(W.clone(), rank=r_, shortlist=sl_, use_rho=True)
        hh = torch.randn(64, d_, generator=g) * decay[None, :].sqrt()
        with torch.no_grad():
            hp = hh @ hf.P_r
            z, zt = hh @ W.t(), hp @ hf.U_r.t()
            s = (hh.pow(2).sum(-1) - hp.pow(2).sum(-1)).clamp_min(0).sqrt()
            viol = float(((z - zt).abs() - s[:, None] * hf.rho[None, :]).max())      # Cauchy-Schwarz
            proj = float((hf.U_r - W @ hf.P_r).abs().max())                          # U_r = W P_r (unscaled)
            orth = float((hf.P_r.t() @ hf.P_r - torch.eye(r_)).abs().max())
            true_top = z.topk(16, dim=-1).indices
            res = hf.fit_kappa(hh, true_top)
            r0, rk = res[0.0], res[hf.kappa]
            got, sl = hf(hh, mode="funnel")
            exact = float((got - z.gather(1, sl)).abs().max())
        ok = (viol < 1e-3 and proj < 1e-4 and orth < 1e-4 and rk >= r0 - 1e-9 and exact < 1e-4
              and hf.active_params() == V_ * r_ + d_ * r_ + sl_ * d_ + V_)
        return ok, (f"certificate violation {viol:.1e}; U_r=W P_r ({proj:.1e}); orthonormal ({orth:.1e}); "
                    f"recall kappa=0 {r0:.3f} -> kappa={hf.kappa} {rk:.3f}; shortlist rows exact ({exact:.1e})")
    contract("U7 HeadFunnel: exact projection, Cauchy-Schwarz certificate, kappa fit never loses recall", u7,
             flip("FUNNEL_USE_RHO", False))

    # ----------------------------------------------------------------- U8 solver honesty
    def u8():
        ss = dict(q_rows=6144, k_rows=1024, v_rows=1024, gate_rows=1024, o_in=6144,
                  dn_q_rows=2048, dn_k_rows=2048, dn_v_rows=6144, dn_z_rows=6144,
                  dn_ba_rows=96, dn_v_head_rows=128, dn_out_in=6144)
        geo = (5120, 17408, 64, 48, 16, 248320)
        sol = solve_budget_v2(*geo, ss, 3.0e9, verbose=False, shared_ffn=512, m_write_max=26,
                              shortlist_max=8192)
        p, w = sol["plan"], sol["waterfall"]
        re = active_waterfall(*geo[:5], geo[5], ss, E_ffn=p["E_ffn"], k_ffn=p["k_ffn"],
                              shared_ffn=p["shared_ffn"], rank_zssi=64, mast_S=p["mast_S"],
                              lin_blocks=p["lin_blocks"], lin_topk=p["lin_topk"],
                              carrier_frac=p["carrier_frac"], m_write=p["m_write"],
                              funnel_rank=p["funnel_rank"], funnel_shortlist=p["funnel_shortlist"])
        bad = solve_budget_v2(*geo, ss, 1.0e9, verbose=False, shared_ffn=512, m_write_max=26,
                              shortlist_max=8192)
        app = mast_reads(5120, 64, 48, 16, ss, 512, "applications")
        dis = mast_reads(5120, 64, 48, 16, ss, 512, "distinct")
        ok = (p["feasible"] and w["TOTAL_ACTIVE"] <= 3.0e9 and re["TOTAL_ACTIVE"] == w["TOTAL_ACTIVE"]
              and p["m_write"] <= 26 and p["funnel_shortlist"] <= 8192 and not bad["plan"]["feasible"]
              and app > 20 * dis and (17408 - p["shared_ffn"]) % p["E_ffn"] == 0)
        return ok, (f"{w['TOTAL_ACTIVE'] / 1e9:.4f}B <= 3.0B | shared {p['shared_ffn']} mast {p['mast_S']} "
                    f"m_write {p['m_write']} V_s {p['funnel_shortlist']} | trims {p['trims']} | 1.0B target "
                    f"reported infeasible: {not bad['plan']['feasible']} | sketch reads {app / 1e6:.0f}M "
                    f"(per layer & width) vs {dis / 1e6:.1f}M (per matrix)")
    contract("U8 budget solver: trims only what the budget demands, plan == its own waterfall, infeasible reported",
             u8)

    # ----------------------------------------------------------------- toy transformer (real classes)
    d_, D_, E_, SH_, RK_, K_ = 64, 256, 8, 32, 16, 2
    S_, B_, TOPK_, CF_ = 48, 8, 1, 0.05
    H_, VH_, MW_ = 6, 16, 4
    DN = dict(q=32, k=32, v=H_ * VH_, z=H_ * VH_, out_in=H_ * VH_)
    FA = dict(q=64, k=32, v=32, gate=32, o_in=48)
    V_, R_, SL_ = 512, 16, 64
    ss_toy = dict(q_rows=FA["q"], k_rows=FA["k"], v_rows=FA["v"], gate_rows=FA["gate"], o_in=FA["o_in"],
                  dn_q_rows=DN["q"], dn_k_rows=DN["k"], dn_v_rows=DN["v"], dn_z_rows=DN["z"],
                  dn_ba_rows=2 * H_, dn_v_head_rows=VH_, dn_out_in=DN["out_in"])

    class _ToyNorm(nn.Module):
        def __init__(self, d):
            super().__init__()
            self.weight = nn.Parameter(torch.ones(d), requires_grad=False)

    class _ToyVGate(nn.Module):                   # same attribute contract as the surgery cell's VGateShim
        def __init__(self, full_param, rl_gate):
            super().__init__()
            self.register_parameter("weight", full_param)
            self.v_rows = full_param.shape[0] // 2
            self.rl_gate = rl_gate
            self.register_buffer("v_w", full_param.data[:self.v_rows], persistent=False)

    class _ToyDN(nn.Module):
        def __init__(self, mast, sd):
            super().__init__()
            rn = lambda *s: torch.randn(*s) / math.sqrt(s[-1])      # noqa: E731
            policy = MoRM2Policy(n_heads=H_, mast=mast, m_bar=MW_, m_max=MW_ + 1)
            nq, nk, nv = DN["q"], DN["k"], DN["v"]
            self.in_proj_qkv = SectionedProjector(
                rn(nq + nk + nv, d_), (nq, nk, nv, 0), v_head_rows=VH_,
                plan_q=plan_row_modes(nq, d_, B_, CF_), plan_z=None, mast=mast, morm=policy, topk=TOPK_)
            self.in_proj_z = ResonantLinear(rn(DN["z"], d_), plan_row_modes(DN["z"], d_, B_, CF_), mast,
                                            topk=TOPK_)
            self.in_proj_b = BetaGate(nn.Linear(d_, H_, bias=False), H_, policy, b_first=True, b_only=True)
            self.in_proj_a = nn.Linear(d_, H_, bias=False)
            self.conv1d = nn.Conv1d(nq + nk + nv, nq + nk + nv, 4, groups=nq + nk + nv, bias=False)
            self.out_proj = ResonantLinear(rn(d_, DN["out_in"]), plan_row_modes(d_, DN["out_in"], B_, CF_),
                                           mast, topk=TOPK_)

    class _ToyFA(nn.Module):
        def __init__(self, mast, sd):
            super().__init__()
            rn = lambda *s: torch.randn(*s) / math.sqrt(s[-1])      # noqa: E731
            self.q_proj = ResonantLinear(rn(FA["q"], d_), plan_row_modes(FA["q"], d_, B_, CF_), mast, topk=TOPK_)
            self.k_proj = nn.Linear(d_, FA["k"], bias=False)
            vfull = nn.Parameter(rn(FA["v"] + FA["gate"], d_), requires_grad=False)
            gate = ResonantLinear(vfull.data[FA["v"]:], plan_row_modes(FA["gate"], d_, B_, CF_), mast, topk=TOPK_)
            self.v_proj = _ToyVGate(vfull, gate)
            self.o_proj = ResonantLinear(rn(d_, FA["o_in"]), plan_row_modes(d_, FA["o_in"], B_, CF_), mast,
                                         topk=TOPK_)

    class _ToyLayer(nn.Module):
        def __init__(self, li, linear, mast):
            super().__init__()
            if linear:
                self.linear_attn = _ToyDN(mast, li)
            else:
                self.self_attn = _ToyFA(mast, li)
            assign = torch.full((D_,), -1, dtype=torch.long)
            assign[SH_:] = torch.arange(D_ - SH_) % E_
            self.mlp = PrismMoeMLP(torch.randn(d_, D_) * 0.1, torch.randn(d_, D_) * 0.1,
                                   torch.randn(d_, D_) * 0.1, assign, E_, rank=RK_, seed=li, k_topk=K_)
            self.input_layernorm = _ToyNorm(d_)
            self.post_attention_layernorm = _ToyNorm(d_)

    class _ToyModel(nn.Module):
        def __init__(self, mast):
            super().__init__()
            self.embed_tokens = nn.Embedding(V_, d_)
            self.layers = nn.ModuleList([_ToyLayer(i, i < 3, mast) for i in range(4)])
            self.norm = _ToyNorm(d_)

    # ----------------------------------------------------------------- U9 the torchao crash, in miniature
    def u9():
        mast = AntennaMast(d_, sketch_dim=S_, seed=0)
        toy = _ToyModel(mast)

        def walk_like_quantize(mod, flt, fqn=""):          # torchao.quantize_ recurses via named_children
            hits = []                                      # and calls filter_fn on EVERY visit
            if flt(mod, fqn):
                hits.append(id(mod))
            for n, c in mod.named_children():
                hits += walk_like_quantize(c, flt, f"{fqn}.{n}" if fqn else n)
            return hits

        def naive(m, fqn):                                 # the v8 filter
            return isinstance(m, nn.Linear) and not any(t in fqn.lower() for t in ("lm_head", "router"))
        unique = {id(m) for _, m in toy.named_modules() if naive(m, _)}
        v8_hits = walk_like_quantize(toy, naive)

        done = set()

        def deduped(m, fqn):                               # the v9 filter (cell 27): same eligibility, once per module
            k = id(m)
            if not naive(m, fqn) or k in done:
                return False
            done.add(k)
            return True
        v9_hits = walk_like_quantize(toy, deduped)
        twice = len(v8_hits) - len(set(v8_hits))
        ok = twice >= 2 and len(v9_hits) == len(set(v9_hits)) == len(unique)
        return ok, (f"v8 filter visited {len(v8_hits)} times for {len(unique)} unique Linear modules "
                    f"({twice} repeat visits: the shared MoRM critics -> quantized twice -> NotImplementedError); "
                    f"v9 filter: {len(v9_hits)} visits, none repeated")
    contract("U9 torchao crash root cause reproduced structurally; the id-dedup filter fixes it", u9)

    # ----------------------------------------------------------------- U10 census == waterfall
    def u10():
        mast = AntennaMast(d_, sketch_dim=S_, seed=0)
        toy = _ToyModel(mast)
        fun = HeadFunnel(torch.randn(V_, d_), rank=R_, shortlist=SL_, use_rho=True)
        res = {}
        for mode in ("applications", "distinct"):
            cen = census_active(toy, d_, 4, funnel=fun, mast=mast, mast_mode=mode)
            wf = active_waterfall(d_, D_, 4, 3, 1, V_, ss_toy, E_ffn=E_, k_ffn=K_, shared_ffn=SH_,
                                  rank_zssi=RK_, mast_S=S_, lin_blocks=B_, lin_topk=TOPK_, carrier_frac=CF_,
                                  m_write=MW_, funnel_rank=R_, funnel_shortlist=SL_, funnel_rho=True,
                                  mast_mode=mode)
            res[mode] = (cen, wf)
        unclass = list(census_active.last["unclassified"])
        diffs = {}
        for mode, (cen, wf) in res.items():
            for fam_, v in wf.items():
                if cen.get(fam_, 0) != v:
                    diffs[f"{mode}:{fam_}"] = (cen.get(fam_, 0), v)
        if diffs:
            for k_, (c_, w_) in diffs.items():
                print(f"      census/waterfall mismatch {k_}: census {c_:,} vs waterfall {w_:,}")
        ok = not diffs and not unclass
        cen, wf = res["applications"]
        return ok, (f"toy total {cen['TOTAL_ACTIVE']:,} == {wf['TOTAL_ACTIVE']:,} (applications), "
                    f"{res['distinct'][0]['TOTAL_ACTIVE']:,} (distinct); shared MoRM policy counted once; "
                    f"unclassified params: {unclass}")
    contract("U10 census of the live module graph == the waterfall that planned it (toy transformer, real classes)",
             u10)

    # ----------------------------------------------------------------- U11 MoRM critic intercept
    def u11():
        torch.manual_seed(11)
        d11, S11, H11, N11, MB = 48, 32, 8, 3000, 3
        mast11 = AntennaMast(d11, sketch_dim=S11, seed=5)
        phi11 = mast11(torch.randn(N11, d11))
        level = torch.linspace(-4.0, -1.0, H11)[None, :]                    # per-head log-influence LEVEL (3 nats apart)
        slope = torch.randn(S11, H11) * 0.2
        logy = level + (phi11 - phi11.mean(0, keepdim=True)) @ slope + 0.15 * torch.randn(N11, H11)
        lab = logy.exp()
        res = {}
        for tag, cb in (("v8", False), ("v9", True)):
            pol = MoRM2Policy(H11, mast11, m_bar=MB, m_max=H11, critic_bias=cb)
            pol.ridge_init(phi11, None, lab)
            with torch.no_grad():
                lh = pol.scores(phi11).clamp_min(1e-12).log()
                r2 = float(1 - (lh - logy).pow(2).sum() / (logy - logy.mean()).pow(2).sum())
                sc = lh.exp()
                thr = float(sc.flatten().quantile(1.0 - MB / H11))             # same mean write budget for both
                cap = float(((sc > thr).float() * lab).sum() / lab.sum())
            res[tag] = (r2, cap, pol.critic_phi.bias is not None)
        (r8, c8, b8), (r9, c9, b9) = res["v8"], res["v9"]
        ok = (b9 and not b8 and r9 > 0.9 and r9 > r8 + 0.5 and c9 > c8 + 0.02)
        return ok, (f"pooled log-R^2 {r8:.2f} (v8 bias-free) -> {r9:.3f} (v9 intercept); influence mass captured at an "
                    f"equal budget ({MB}/{H11} heads/token) {c8:.3f} -> {c9:.3f}")
    contract("U11 MoRM critic carries each head's mean log-influence (intercept) and captures more influence mass",
             u11, on_fail=flip("MORM2_CRITIC_BIAS", False))

    # ----------------------------------------------------------------- U12 shadow execution
    def u12():
        torch.manual_seed(12)
        d12, o12, S12, N12, r12 = 40, 96, 24, 400, 12
        mast12 = AntennaMast(d12, sketch_dim=S12, seed=7)
        # a site whose outputs have spectral structure (low-rank signal + noise), like a real projection
        W = (torch.randn(o12, 10) @ torch.randn(10, d12)) / math.sqrt(d12) + 0.15 * torch.randn(o12, d12) / math.sqrt(d12)
        X = torch.randn(N12, d12)
        Y = X @ W.t()
        U, cap, res = fit_shadow_basis(Y, r12)
        plan = plan_row_modes(o12, d12, 8, 0.02, row_energy=res.pow(2).mean(0))
        rl = ResonantLinear(W.clone(), plan, mast12, topk=1)
        rl.attach_shadow(U)
        with torch.no_grad():
            rl.router.weight.copy_(ridge_router_init(mast12(X), _block_energy(res, plan["block_idx"])))
            rl.gains.copy_(1.0 + 0.1 * torch.randn(rl.B))
        xt = torch.randn(64, d12)
        y_dense = xt @ W.t()
        rl.eval()
        with torch.no_grad():
            y_srv = rl(xt, mode="topk")                          # true sparse path (no grad)
            e_shadow = float((y_srv - y_dense).pow(2).sum() / y_dense.pow(2).sum())
            sU, sM = rl.sh_U, rl.sh_M
            rl.sh_U = rl.sh_M = None
            y_v9 = rl(xt, mode="topk")
            e_zero = float((y_v9 - y_dense).pow(2).sum() / y_dense.pow(2).sum())
            rl.sh_U, rl.sh_M = sU, sM
            proj = float((rl._shadow(xt) - (y_dense @ U) @ U.t()).abs().max())   # S(x) == U U^T (W x)
        rl.train()
        y_g = rl._forward_2d(xt, "topk")                         # grad-mode supergraph, straight-through router
        same = float((y_g.detach() - y_srv).abs().max())
        loss = (y_g - y_dense).pow(2).mean()
        loss.backward()
        g_router = float(rl.router.weight.grad.abs().sum()) if rl.router.weight.grad is not None else 0.0
        ok = (proj < 1e-4 and same < 1e-4 and e_shadow < 0.5 * e_zero and g_router > 0 and cap > 0.5)
        return ok, (f"S(x) == U U^T W x ({proj:.1e}); grad-mode == served ({same:.1e}); served rel.err "
                    f"{e_zero:.3f} (zeroed rows, v9) -> {e_shadow:.3f} (shadow); held-out energy captured {cap:.3f}; "
                    f"top-1 router gradient {g_router:.2e} (v9: none)")
    contract("U12 shadow execution: exact projection, train path == served path, error falls, the router learns",
             u12, on_fail=flip("SHADOW_RANK_LIN", 0))

    n_ok = sum(1 for _, ok, _ in out if ok)
    print(f"\n==== v9 contracts: {n_ok}/{len(out)} passed ====")
    for f_ in flips:
        print(f"  [fallback engaged] {f_}")
    if not flips and n_ok == len(out):
        print("All v9 formulas verified; every switch stays on.")
    return out


V9_CONTRACTS = _run_v9_contracts()


## Load the dense model — census, section introspection, the 3B solver

1. **Waterfall** — exact parameter census by role from the real weights.
2. **Section introspection** — the DeltaNet fused projection's `[q|k|v|z]` boundaries and the full-attention `v|gate` split are resolved **dynamically and never fatally** (rules R1-R5 in the code): discovery uses multiple independent signals, every claimed split is cross-checked against a second shape, surgery runs only on name-resolved sites, and anything unresolved degrades to its exact dense cost in the budget.
3. **The 3B solver (v9: honest and consumed)** — `solve_budget_v2` runs the strict touched-weights waterfall over the resolved section sizes and spends the budget in quality-per-param order (head shortlist → memory freshness → projection coverage → carriers → FFN top-k). When a request does not fit it walks a **trim ladder** (sketch size → shared core → shortlist → write heads), prints every trim, and the notebook **re-binds** `SHARED_SIZE`, `MAST_S`, `FUNNEL_SHORTLIST`, `M_WRITE`, `LIN_BLOCKS`, … from the plan — so what is built is what was priced. `RUN_MODE="full"` raises when the target cannot be met; the demo minimizes and reports.

This is where the old *~10.5B floor* of "only FFNs can be sparse" is broken: the RESONA families lift it.


In [ ]:
# Load the dense model — waterfall, section introspection, 3B budget solver
from transformers import AutoConfig, AutoTokenizer, AutoProcessor
import inspect, re

cfg = AutoConfig.from_pretrained(MODEL_ID)
tok = AutoTokenizer.from_pretrained(MODEL_ID)
proc = None
try:
    proc = AutoProcessor.from_pretrained(MODEL_ID)   # multimodal processor
except Exception as e:
    print("Processor load skipped:", e)

tc = getattr(cfg, "text_config", cfg)
D_INT   = tc.intermediate_size
N_LAY   = tc.num_hidden_layers
HID     = tc.hidden_size
VOCAB   = tc.vocab_size
N_LIN   = sum(1 for t in getattr(tc, "layer_types", []) if t == "linear_attention")
N_FULL  = sum(1 for t in getattr(tc, "layer_types", []) if t == "full_attention")
print(f"hidden={HID} | layers={N_LAY} ({N_LIN} DeltaNet + {N_FULL} full-attn) "
      f"| intermediate={D_INT} | vocab={VOCAB} "
      f"| ctx={getattr(tc,'max_position_embeddings','?')}")

# ----------------------- load the dense model -------------------------------
from transformers import AutoModelForImageTextToText, AutoModelForCausalLM
import torch

# ---- v8.0.3 OOM guard: re-run safety + fail-fast capacity check -----------
release_previous_run()
if RUN_MODE == "full" and torch.cuda.is_available() and not globals().get("FORCE_LOAD", False):
    _vfree, _vtot = vram_gib()
    _NEED_FREE_GIB = 60.0        # 27B bf16 (~52 GiB) + calibration/training headroom
    if _vfree < _NEED_FREE_GIB:
        raise RuntimeError(
            f"Not enough free GPU memory for RUN_MODE='full': {_vfree:.0f} GiB free of "
            f"{_vtot:.0f} GiB (need >= {_NEED_FREE_GIB:.0f} GiB free; the bf16 weights alone are ~52 GiB).\n"
            "  - Colab: Runtime > Change runtime type > A100 with the 'High-RAM' option (80 GB), "
            "or an H100 / G4.   (40GB A100이면 High-RAM(80GB)으로 바꾸세요)\n"
            "  - A 40 GB A100 / L4 / T4 cannot hold this model: use RUN_MODE='demo' there.\n"
            "  - If a previous run still holds the GPU: Runtime > Restart session, run from the top.\n"
            "  - To skip this check anyway: set FORCE_LOAD = True before this cell.")
LOAD_KW = dict(dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32)
if RUN_MODE == "full":
    LOAD_KW.update(device_map="auto", low_cpu_mem_usage=True)
else:
    LOAD_KW.update(device_map={"": "cuda" if torch.cuda.is_available() else "cpu"})

_vl_err = None
try:
    model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **LOAD_KW)
except Exception as e:
    if is_oom(e):
        raise                      # a second full load can only OOM harder
    _vl_err = str(e)
if _vl_err is not None:            # handled OUTSIDE the except block: the half-loaded
    print(f"VL loader failed ({_vl_err}); falling back to text-only CausalLM")   # VL attempt is
    free_cuda("before text-only fallback")                                      # releasable now
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **LOAD_KW)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)
print("Loaded", MODEL_ID, f"({sum(p.numel() for p in model.parameters())/1e9:.2f}B params)")
free_cuda(f"after {MODEL_ID} load")   # baseline census before calibration

# --------------------------- param waterfall --------------------------------
from collections import defaultdict
def waterfall(m):
    buckets = defaultdict(int)
    def bucketize(name):
        ln = name.lower()
        if "visual" in ln or "vision" in ln or "merger" in ln: return "vision_tower"
        if "mtp" in ln: return "mtp_block"
        if "embed" in ln: return "embeddings"
        if "lm_head" in ln: return "lm_head"
        if ".mlp." in ln: return "ffn"
        if "attn" in ln or "linear_attn" in ln or "delta" in ln or "conv" in ln: return "attention"
        if "norm" in ln: return "norms"
        return "other"
    for name, p in m.named_parameters():
        buckets[bucketize(name)] += p.numel()
    return dict(buckets)

wf = waterfall(model)
total_dense = sum(wf.values())
print(f"\n{'bucket':<14}{'params':>14}{'%':>8}")
for k, v in sorted(wf.items(), key=lambda x: -x[1]):
    print(f"{k:<14}{v:>14,}{100*v/total_dense:>7.1f}%")
print(f"{'TOTAL':<14}{total_dense:>14,}")

# ============================================================================
# SECTION INTROSPECTION — multi-signal, never-fatal
# ============================================================================
#   R1 DISCOVER DeltaNet modules by multiple independent signals (class-name
#      regex, attribute fingerprints, fused-shape signature) — never one.
#   R2 RESOLVE sections dynamically: attribute-name match first, then a
#      config-consistent shape search; every size tagged measured/inferred.
#   R3 CROSS-CHECK every claimed split (v|gate, fused qkvz) against a second
#      shape; inconsistent splits are DISABLED — never guessed.
#   R4 SURGERY ONLY on name-resolved sites; shape-only resolutions still
#      feed the budget waterfall (their savings are honestly removed).
#   R5 NO DEAD-END ASSERTS: anything unresolved degrades gracefully to its
#      exact dense cost, with a diagnostic dump of the offending module.

_EXCLUDE_PAT = re.compile(
    r"(visual|vision|image|patch_embed|mtp|multi_modal|mm_|proj_encoder)")
_DN_CLASS_PAT = re.compile(
    r"(deltanet|delta_?net|gated_?delta|\bgdn\b|linear_?attention|linear_?attn"
    r"|linearized|recurrent|retent|rwkv|mamba|state_?space)", re.I)
_DN_ATTRS = ("in_proj_qkvz", "in_proj_qkv", "in_proj_ba", "in_proj_z",
             "in_proj", "qkvz", "conv1d", "short_conv", "A_log", "a_log",
             "dt_bias", "l2norm", "decay", "gdn")
_FUSED_ATTR_NAMES = ("in_proj_qkvz", "in_proj_qkv", "qkvz_proj", "in_proj",
                     "Wqkvz", "qkvz", "in_proj_qkvy")
_BA_ATTR_NAMES = ("in_proj_ba", "ba_proj", "in_proj_ba_g", "ba")
_OUT_ATTR_NAMES = ("out_proj", "o_proj", "wo", "out")
_CONV_ATTR_NAMES = ("conv1d", "short_conv", "conv", "depthwise_conv")


def _excluded(name):
    return bool(_EXCLUDE_PAT.search(name.lower()))


def _own_2d(mod):
    out = []
    for n, p in mod.named_parameters(recurse=False):
        if p.ndim == 2:
            out.append((n, tuple(p.shape)))
    return out


def _iter_2d(mod):
    """2D weights at this granularity: own params + each direct child
    Linear's weight (fused projections are child Linears in practice)."""
    for n, p in _own_2d(mod):
        yield (mod, n, dict(mod.named_parameters(recurse=False))[n])
    for cn, child in mod.named_children():
        W = getattr(child, "weight", None)
        if isinstance(W, torch.Tensor) and W.ndim == 2:
            yield (child, f"{cn}.weight", W)


def find_delta_nets_v3(model, hints=None):
    """R1 — multi-signal DeltaNet discovery (never a single attribute)."""
    hints = hints or {}
    n_k, n_v = hints.get("n_k"), hints.get("n_v")
    fused_rows = (2 * n_k + 2 * n_v) if (n_k and n_v) else None
    qkv_rows = (2 * n_k + n_v) if (n_k and n_v) else None   # split layout
    found = []
    for name, mod in model.named_modules():
        if _excluded(name):
            continue
        signals = []
        if _DN_CLASS_PAT.search(type(mod).__name__):
            signals.append("class:" + type(mod).__name__)
        probe = {n for n, _ in mod.named_parameters(recurse=False)}
        probe |= {n for n, _ in mod.named_children()}
        hits = sorted({a for a in _DN_ATTRS if a in probe})
        inproj_hit = any(a.startswith("in_proj") or a == "qkvz" for a in hits)
        if hits:
            signals.append("attrs:" + ",".join(hits[:4]))
        for _, _, W in _iter_2d(mod):
            if fused_rows is not None and W.shape[0] == fused_rows:
                signals.append(f"fused-shape:{tuple(W.shape)}")
                break
            if qkv_rows is not None and W.shape[0] == qkv_rows:
                signals.append(f"qkv-shape:{tuple(W.shape)}")
                break
        strong = bool(signals) and (
            any(s.startswith("class:") for s in signals)
            or len(signals) >= 2 or inproj_hit)
        if strong:
            found.append({"name": name, "mod": mod, "signals": signals})
    found.sort(key=lambda r: r["name"].count("."))       # deepest wins
    kept, kept_paths = [], []
    for r in found:
        if any(r["name"].startswith(p + ".") for p in kept_paths):
            continue
        kept.append(r)
        kept_paths.append(r["name"])
    return kept


def _resolve_by_name(mod, names):
    for nm in names:
        child = getattr(mod, nm, None)
        if child is None:
            continue
        W = getattr(child, "weight", None)
        if isinstance(W, torch.Tensor) and W.ndim == 2:
            return (mod, nm, W, "name")
        if isinstance(child, torch.Tensor) and child.ndim == 2:
            return (mod, nm, child, "name")
    return None


def _resolve_by_shape(mod, rows, exclude_names=()):
    hits = [(o, n, t) for o, n, t in _iter_2d(mod)
            if t.shape[0] == rows and n not in exclude_names]
    if len(hits) == 1:
        o, n, t = hits[0]
        return (o, n, t, "shape")
    return None


def resolve_dn_sections(mod, hints):
    """R2 — resolve fused [q|k|v|z] (+ ba/out/conv) of one DeltaNet module."""
    notes = []
    sec = {"notes": notes}
    d = hints.get("d")
    n_k, n_v = hints.get("n_k"), hints.get("n_v")
    v_head_rows = hints.get("v_head_rows")

    ref = _resolve_by_name(mod, _FUSED_ATTR_NAMES)
    if ref is None:
        rows_hint = (2 * n_k + 2 * n_v) if (n_k and n_v) else None
        if rows_hint:
            ref = _resolve_by_shape(mod, rows_hint)
            if ref:
                notes.append("fused proj resolved by SHAPE only — surgery "
                             "disabled for this site (R4)")
    if ref:
        owner, attr, W, prov = ref
        sec["qkvz"] = (owner, attr, W, prov)
        R = W.shape[0]
        sec["split"] = False
        sec["z"] = None
        if n_k and n_v and R == 2 * n_k + 2 * n_v:
            # fused [q|k|v|z] projection (qwen3_next-style GDN)
            q_rows = k_rows = n_k
            v_rows = z_rows = n_v
            sec["provenance_sections"] = "measured(config-consistent)"
        elif n_k and n_v and R == 2 * n_k + n_v:
            # The qwen3_5-family GDN (Qwen3.5/3.6/3.8, which Qwen3.8-27B
            # belongs to) stores SPLIT projections: in_proj_qkv [q|k|v] +
            # in_proj_z [z] (+ in_proj_b / in_proj_a). Resolve the z
            # sibling by name/shape and record the SPLIT layout: sections
            # stay MEASURED, surgery wires in_proj_qkv (q routed, k dense,
            # v MoRM) and in_proj_z (routed) separately, each preserving
            # the stock module interface and state-dict key.
            q_rows = k_rows = n_k
            v_rows = n_v
            zref = _resolve_by_name(mod, ("in_proj_z",))
            if zref is None:
                zref = _resolve_by_shape(mod, n_v,
                                         exclude_names=("in_proj_qkv",
                                                        "in_proj_qkvz"))
            if zref is not None and tuple(zref[2].shape) == (n_v, W.shape[1]):
                z_rows = n_v
                sec["split"] = True
                sec["z"] = zref
                sec["provenance_sections"] = \
                    "measured(split q|k|v + separate z; qwen3_5 family)"
            else:
                z_rows = 0                    # no z sibling: 3-section site
                sec["provenance_sections"] = "measured(split q|k|v, z absent)"
                notes.append("in_proj_qkv split layout without in_proj_z — "
                             "z routed rows disabled for this site (R5)")
        elif R % 4 == 0:
            q_rows = k_rows = v_rows = z_rows = R // 4
            sec["provenance_sections"] = "inferred(quarter-split)"
            notes.append(f"fused rows {R} != 2*n_k+2*n_v — assumed q=k=v=z=R/4")
        else:
            q_rows = k_rows = R // 4
            v_rows = z_rows = R - 2 * (R // 4)
            sec["provenance_sections"] = "inferred(partial-split)"
        sec.update(q_rows=q_rows, k_rows=k_rows, v_rows=v_rows, z_rows=z_rows)
    else:
        sec["qkvz"] = None
        sec["split"] = False
        sec["z"] = None
        notes.append("fused qkvz projection NOT resolved")

    ref_ba = _resolve_by_name(mod, _BA_ATTR_NAMES)
    if ref_ba is None and sec.get("v_rows") and v_head_rows:
        ref_ba = _resolve_by_shape(mod, 2 * (sec["v_rows"] // v_head_rows))
    sec["ba"] = ref_ba
    if ref_ba is not None:
        sec["ba_rows"] = ref_ba[2].shape[0]
        # v8 (MoRM 2.0): fused [b|a] layout for the write-enable gate.
        # qwen3_5 convention: b (per-head write gates, H values) FIRST,
        # a (per-head log-decay) SECOND — BetaGate wraps in_proj_ba with
        # b_first=True; the Theorem-7 anchor test fails loudly if a target
        # model ever deviates from this layout.
        if sec.get("v_head_rows"):
            sec["b_rows"] = sec["v_rows"] // sec["v_head_rows"]
            sec["a_rows"] = max(0, int(ref_ba[2].shape[0]) - sec["b_rows"])
    else:
        # split b/a siblings (qwen3_5: in_proj_b [H,d] + in_proj_a [H,d])
        _nvh = (sec.get("v_rows") or 0) // v_head_rows if v_head_rows else 0
        _bref = _resolve_by_name(mod, ("in_proj_b",))
        _aref = _resolve_by_name(mod, ("in_proj_a",))
        _found = [nm for nm, r in (("in_proj_b", _bref),
                                   ("in_proj_a", _aref)) if r is not None]
        sec["ba_rows"] = _nvh * len(_found) if _found else None
        # v8.0.2 [P2]: record the SPLIT b-projection reference. qwen3_5 has
        # NO fused in_proj_ba, so sec["ba"] stays None here — and without
        # this reference the surgery cell would silently skip the BetaGate
        # install, leaving MoRM 2.0 INERT (critic fitted, the mask never
        # reaching a write). in_proj_a (the decay pre-activation) is left
        # dense on purpose: a drives decay, not writes — the exact skip
        # lever is beta alone (Proposition F).
        if _bref is not None and _nvh and _bref[2].shape[0] == _nvh:
            sec["b"] = _bref
            sec["b_rows"] = _nvh
            if _aref is not None and _aref[2].shape[0] == _nvh:
                sec["a"] = _aref
                sec["a_rows"] = _nvh

    ref_out = _resolve_by_name(mod, _OUT_ATTR_NAMES)
    if ref_out is None and d and sec.get("v_rows"):
        ref_out = _resolve_by_shape(mod, d)
    sec["out"] = ref_out
    sec["out_in"] = ref_out[2].shape[1] if ref_out else None
    ref_conv = _resolve_by_name(mod, _CONV_ATTR_NAMES)
    sec["conv"] = ref_conv

    if v_head_rows and sec.get("v_rows") and sec["v_rows"] % v_head_rows == 0:
        sec["v_head_rows"] = v_head_rows
    elif sec.get("v_rows"):
        sec["v_head_rows"] = next((c for c in (128, 64)
                                   if sec["v_rows"] % c == 0), None)
    else:
        sec["v_head_rows"] = None

    sec["name_resolved"] = bool(sec.get("qkvz")) and sec["qkvz"][3] == "name"
    if sec.get("split"):
        # z rows live in the SIBLING weight (sec["z"]): q+k+v must fill the
        # qkv projection exactly and z must fill the sibling exactly (R3).
        ok_shape = (sec.get("q_rows") and sec.get("v_rows")
                    and (sec["q_rows"] + sec["k_rows"] + sec["v_rows"])
                    == sec["qkvz"][2].shape[0]
                    and sec["z_rows"] == sec["z"][2].shape[0])
    else:
        ok_shape = (sec.get("q_rows") and sec.get("v_rows")
                    and (sec["q_rows"] + sec["k_rows"] + sec["v_rows"]
                         + sec["z_rows"]) == sec["qkvz"][2].shape[0])
    if sec.get("qkvz") and not ok_shape:
        sec["name_resolved"] = False
        notes.append("section sum != fused rows — surgery disabled (R3)")
    sec["morm_ok"] = bool(sec.get("v_head_rows") and sec.get("v_rows"))
    return sec


def _diag_dump(model, max_layers=1):
    lines = ["--- DIAGNOSTIC DUMP (first decoder layer) ---"]
    layers = getattr(getattr(model, "model", model), "layers", None)
    if layers is None:
        layers = [m for _, m in model.named_children()]
    for i, layer in enumerate(list(layers)[:max_layers]):
        lines.append(f"[layer {i}] class={type(layer).__name__}")
        for cn, child in layer.named_children():
            shapes = ", ".join(f"{n}{tuple(p.shape)}"
                               for n, p in _own_2d(child)[:6])
            lines.append(f"    .{cn}: {type(child).__name__} {shapes}")
    return "\n".join(lines)


def build_section_table(model, tc, n_lin_expected=0):
    """tc may be a config object OR a dict. Returns (SS, DNS_REF, GATES,
    report_lines, FAS). Families that cannot be resolved degrade, never
    crash (R5)."""
    rep = []
    _get = (lambda k, dv=None: tc.get(k, dv)) if isinstance(tc, dict) \
        else (lambda k, dv=None: getattr(tc, k, dv))
    hints = dict(
        d=_get("hidden_size"),
        n_k=((_get("linear_num_key_heads") or 0)
             * (_get("linear_key_head_dim") or 0) or None),
        n_v=((_get("linear_num_value_heads") or 0)
             * (_get("linear_value_head_dim") or 0) or None),
        v_head_rows=_get("linear_value_head_dim"),
    )
    cands = find_delta_nets_v3(model, hints)
    rep.append(f"DeltaNet candidates discovered: {len(cands)} "
               f"(config says {n_lin_expected})")
    for c in cands[:3]:
        rep.append(f"  {c['name']}: {c['signals']}")

    DNS_REF, SS, unresolved = [], {}, []
    for c in cands:
        sec = resolve_dn_sections(c["mod"], hints)
        if sec.get("qkvz") and sec["name_resolved"] and sec.get("q_rows"):
            DNS_REF.append((c["name"], c["mod"], sec))
        elif sec.get("qkvz"):
            unresolved.append((c["name"], sec))
    if cands and not DNS_REF:
        rep.append("!! DeltaNet modules exist but NO surgery-capable site — "
                   "their exact dense cost stays in the waterfall (R5).")
        rep.extend(_diag_dump(model).splitlines())
    elif len(cands) < n_lin_expected:
        rep.append(f"!! resolved {len(DNS_REF)}/{n_lin_expected} DeltaNet "
                   "sites — unresolved ones keep dense cost (R5).")
        for nm, sec in unresolved[:2]:
            rep.append(f"  unresolved {nm}: {sec['notes']}")

    if DNS_REF:
        sec0 = DNS_REF[0][2]
        SS.update(dn_q_rows=sec0["q_rows"], dn_k_rows=sec0["k_rows"],
                  dn_v_rows=sec0["v_rows"], dn_z_rows=sec0["z_rows"],
                  dn_ba_rows=sec0.get("ba_rows") or 0,
                  dn_v_head_rows=sec0.get("v_head_rows") or 0,
                  dn_out_in=sec0.get("out_in") or 0)
        rep.append(f"DN sections: q={SS['dn_q_rows']} k={SS['dn_k_rows']} "
                   f"v={SS['dn_v_rows']} z={SS['dn_z_rows']} "
                   f"vhead={SS['dn_v_head_rows']} "
                   f"({sec0.get('provenance_sections', '?')})")
    else:
        SS.update(dn_q_rows=0, dn_k_rows=0, dn_v_rows=0, dn_z_rows=0,
                  dn_ba_rows=0, dn_v_head_rows=0, dn_out_in=0)

    FAS = [(n, m) for n, m in model.named_modules()
           if hasattr(m, "q_proj") and hasattr(m, "o_proj")
           and not _excluded(n)]
    rep.append(f"full-attention layers discovered: {len(FAS)}")
    VGATE_OK = True
    if FAS:
        fa0 = FAS[0][1]
        q_rows = fa0.q_proj.weight.shape[0]
        k_rows = fa0.k_proj.weight.shape[0]
        v_raw = fa0.v_proj.weight.shape[0]
        o_in = fa0.o_proj.weight.shape[1]
        gate_fused = bool(_get("attn_output_gate", False))
        v_rows, gate_rows = (v_raw // 2, v_raw // 2) if gate_fused \
            else (v_raw, 0)
        # R3 cross-check: the v|gate split is only trusted when o_proj agrees
        if gate_fused and o_in != v_rows:
            VGATE_OK = False
            rep.append(f"!! v|gate split INCONSISTENT: o_in={o_in} != "
                       f"v_rows={v_rows} — gate surgery disabled, v_proj "
                       f"kept dense at {v_raw} rows (R3).")
            gate_rows, v_rows = 0, v_raw
        SS.update(q_rows=q_rows, k_rows=k_rows, v_rows=v_rows,
                  gate_rows=gate_rows, o_in=o_in)
        rep.append("full-attn sizes: " + str({k: SS[k] for k in
                   ("q_rows", "k_rows", "v_rows", "gate_rows", "o_in")}))
    else:
        SS.update(q_rows=0, k_rows=0, v_rows=0, gate_rows=0, o_in=0)

    GATES = {"dn_surgery": len(DNS_REF) > 0, "vgate_surgery": VGATE_OK}
    return SS, DNS_REF, GATES, rep, FAS


def surgery_ss(SS, GATES):
    """Solver-facing SS: disabled families contribute their exact dense cost."""
    if GATES.get("dn_surgery", False):
        return dict(SS)
    out = dict(SS)
    for k in ("dn_q_rows", "dn_k_rows", "dn_v_rows", "dn_z_rows"):
        out[k] = 0
    out["dn_v_head_rows"] = 1
    return out


SS, DNS_REF, GATES, INTRO_REP, FAS = build_section_table(model, tc, N_LIN)
for _ln in INTRO_REP:
    print(_ln)

# dynamic need_keys: only what the ENABLED surgery paths really require.
missing = []
if GATES["dn_surgery"]:
    _sec0 = DNS_REF[0][2]
    missing += ["dn_" + k for k in ("q_rows", "k_rows", "v_rows", "z_rows")
                if not _sec0.get(k)]
if GATES["vgate_surgery"]:
    missing += [k for k in ("q_rows", "k_rows", "v_rows", "gate_rows", "o_in")
                if not SS.get(k)]
if missing:
    print("!! section sizes incomplete:", missing,
          "— affected families keep their exact dense cost (R5)")
else:
    print("dynamic need_keys satisfied")

SS_SOLVER = surgery_ss(SS, GATES)

# ======================= THE 3B BUDGET SOLVER (v9: honest, and its plan is CONSUMED) ===========
# v8 solved for a 192-neuron shared core ... and then BUILT SHARED_SIZE=512 (+0.2B) with the
# config's own carrier / top-k / write cap: the model that was actually assembled drew ~3.15B
# while the log said 2.95B, hidden behind `assert act <= TARGET * 1.10`.  Now the solver is
# handed the REQUESTED geometry (shared core, write cap, head-shortlist cap, sketch width),
# trims only what the budget demands, and the pipeline RE-BINDS SHARED_SIZE / MAST_S /
# LIN_TOPK / CARRIER_FRAC / FUNNEL_SHORTLIST / M_WRITE from the plan it gets back.
_morm_site = bool(GATES["dn_surgery"] and DNS_REF[0][2].get("morm_ok"))
if _morm_site:
    _nh = DNS_REF[0][2]["v_rows"] // DNS_REF[0][2]["v_head_rows"]
    _m_cap = _nh if M_WRITE_FRAC is None else max(1, int(M_WRITE_FRAC * _nh))
else:
    _nh, _m_cap = 0, None
_solve_kw = dict(d=HID, D_ffn=D_INT, n_layers=N_LAY,
                 n_linear_attn=N_LIN, n_full_attn=N_FULL,
                 vocab=VOCAB, ss=SS_SOLVER, mast_S=MAST_S, rank_zssi=COMPLETION_RANK,
                 shared_ffn=SHARED_SIZE, e_ffn=FFN_EXPERTS, funnel_rank=FUNNEL_RANK,
                 m_write_max=_m_cap, shortlist_max=FUNNEL_SHORTLIST,
                 mast_S_min=MAST_S_MIN, mast_mode=MAST_ACCOUNTING,
                 funnel_rho=FUNNEL_USE_RHO,
                 # v10: the plan starts from the REQUESTED geometry and carries the shadows
                 k_ffn0=FFN_TOPK, lin_blocks0=LIN_BLOCKS, carrier0=CARRIER_FRAC,
                 m_write0=(M_WRITE_FLOOR if (SHADOW_V and SHADOW_RANK_LIN and _morm_site) else None),
                 shadow_lin=SHADOW_RANK_LIN, shadow_ffn=SHADOW_RANK_FFN,
                 shadow_v=bool(SHADOW_V and SHADOW_RANK_LIN and _morm_site))
if RUN_MODE == "full":
    sol = solve_budget_v2(target_active=TARGET_ACTIVE, **_solve_kw)
    PLAN, WF = sol["plan"], sol["waterfall"]
    if not PLAN["feasible"]:
        _msg = (f"3B mandate cannot be met: even the trimmed floor plan draws "
                f"{WF['TOTAL_ACTIVE']/1e9:.3f}B > {TARGET_ACTIVE/1e9:.2f}B "
                f"({MAST_ACCOUNTING} sketch accounting). Lower SHARED_SIZE / MAST_S_MIN, "
                f"or set MAST_ACCOUNTING='distinct' (each sketch matrix counted once), "
                f"or raise TARGET_ACTIVE.")
        if ENFORCE_BUDGET:
            raise RuntimeError(_msg)
        print("!! " + _msg + "  (ENFORCE_BUDGET=False: continuing)")
    # ---- the plan is CONSUMED: every knob the solver trimmed or spent is re-bound here ----
    SHARED_SIZE, MAST_S = PLAN["shared_ffn"], PLAN["mast_S"]
    LIN_BLOCKS, LIN_TOPK = PLAN["lin_blocks"], PLAN["lin_topk"]
    CARRIER_FRAC = PLAN["carrier_frac"]
    FUNNEL_SHORTLIST, FUNNEL_RANK = PLAN["funnel_shortlist"], PLAN["funnel_rank"]
    SHADOW_RANK_LIN = PLAN["shadow_lin"]
    for _t in PLAN["trims"]:
        print(f"  [budget] trimmed to fit {TARGET_ACTIVE/1e9:.2f}B: {_t}")
else:
    sol = solve_budget_v2(target_active=float("inf"), **_solve_kw)
    PLAN, WF = sol["plan"], sol["waterfall"]
    print(f"\ndemo minimal plan active: {WF['TOTAL_ACTIVE']/1e6:.1f}M "
          f"({100*WF['TOTAL_ACTIVE']/total_dense:.1f}% of dense)")

# reconcile FFN E with balanced-fission divisibility
feasible = [E for E, _ in CANDIDATES
            if (D_INT - SHARED_SIZE) % E == 0 and E <= PLAN["E_ffn"]]
E_CHOSEN = max(feasible) if feasible else max(E for E, _ in CANDIDATES
                                              if (D_INT - SHARED_SIZE) % E == 0)
K_CHOSEN = PLAN["k_ffn"]
M_E      = (D_INT - SHARED_SIZE) // E_CHOSEN
if _morm_site:
    M_WRITE = max(1, min(PLAN["m_write"], _nh)) if RUN_MODE == "full" else (
        _nh if M_WRITE_FRAC is None else max(1, int(M_WRITE_FRAC * _nh)))
else:
    M_WRITE = 1          # inert: MoRM is only installed on morm_ok sites
FUNNEL_ON = True


def _realized_waterfall(**ov):
    """Waterfall of the geometry that is ACTUALLY about to be built (not the solver's idea of it)."""
    kw = dict(E_ffn=E_CHOSEN, k_ffn=K_CHOSEN, shared_ffn=SHARED_SIZE,
              rank_zssi=COMPLETION_RANK, mast_S=MAST_S, lin_blocks=LIN_BLOCKS,
              lin_topk=LIN_TOPK, carrier_frac=CARRIER_FRAC,
              m_write=(M_WRITE if _morm_site else PLAN["m_write"]),
              funnel_rank=FUNNEL_RANK, funnel_shortlist=FUNNEL_SHORTLIST,
              funnel_rho=FUNNEL_USE_RHO, mast_mode=MAST_ACCOUNTING,
              shadow_lin=SHADOW_RANK_LIN, shadow_ffn=SHADOW_RANK_FFN,
              shadow_v=bool(SHADOW_V and SHADOW_RANK_LIN and _morm_site))
    kw.update(ov)
    return active_waterfall(HID, D_INT, N_LAY, N_LIN, N_FULL, VOCAB, SS_SOLVER, **kw)


WF = _realized_waterfall()           # what is about to be BUILT (full and demo alike)
if RUN_MODE != "full":
    print(f"demo realized geometry draws {WF['TOTAL_ACTIVE']/1e6:.1f}M active (no budget is enforced in demo)")
if RUN_MODE == "full":
    _alt = "distinct" if MAST_ACCOUNTING == "applications" else "applications"
    _WF_alt = _realized_waterfall(mast_mode=_alt)
    if WF["TOTAL_ACTIVE"] > TARGET_ACTIVE:
        _msg = (f"realized geometry draws {WF['TOTAL_ACTIVE']/1e9:.3f}B > "
                f"{TARGET_ACTIVE/1e9:.2f}B (E reconcile changed the plan)")
        if ENFORCE_BUDGET:
            raise RuntimeError(_msg)
        print("!! " + _msg)
    print(f"\n===> BUDGET (waterfall, '{MAST_ACCOUNTING}' sketch accounting): "
          f"{WF['TOTAL_ACTIVE']/1e9:.3f}B <= {TARGET_ACTIVE/1e9:.2f}B  "
          f"[same geometry, '{_alt}' convention: {_WF_alt['TOTAL_ACTIVE']/1e9:.3f}B] <===")
    print("     the census of the REAL module graph re-checks this after the surgery cell")
print(f"\nresolved: E={E_CHOSEN} k={K_CHOSEN} m_e={M_E} shared={SHARED_SIZE} "
      f"| dn_surgery={GATES['dn_surgery']} vgate={GATES['vgate_surgery']} "
      f"| MoRM m_write={M_WRITE} | mast S={MAST_S} lin_topk={LIN_TOPK} carrier={CARRIER_FRAC} "
      f"| funnel r={FUNNEL_RANK} V_s={FUNNEL_SHORTLIST} "
      f"| v10 shadows: lin rank {SHADOW_RANK_LIN}, ffn rank {SHADOW_RANK_FFN}, shadow-v {bool(SHADOW_V and SHADOW_RANK_LIN)} "
      f"| blocks {LIN_BLOCKS}")

# ------------- capture the ORIGINAL dense logits (the anchor) ----------------
PROBE_IDS = tok("The theory of resonant computation states that", return_tensors="pt").input_ids[:, :32]
DEV = next(model.parameters()).device
def _probe_logits():
    with torch.no_grad():
        return model(input_ids=PROBE_IDS.to(DEV)).logits.float().clone()


# v8.0.4: if the fla Triton kernel cannot run on this GPU / Triton build, the very first
# forward (this one) fails. Switch every Gated-DeltaNet layer to the PyTorch reference
# kernel and go on — slower, but the run survives. (Handled outside the except block.)
_fla_err = None
try:
    ORIG_LOGITS = _probe_logits()
except Exception as _e:
    if not globals().get("FLA_ACTIVE") or is_oom(_e):
        raise
    _fla_err = f"{type(_e).__name__}: {str(_e)[:200]}"
if _fla_err is not None:
    import inspect as _insp_f
    _gm = next((_insp_f.getmodule(type(m)) for m in model.modules()
                if "GatedDeltaNet" in type(m).__name__), None)
    for _fn in ("torch_chunk_gated_delta_rule", "torch_recurrent_gated_delta_rule"):
        _f = getattr(_gm, _fn, None)
        if callable(_f):
            setattr(_gm, _fn, _insp_f.unwrap(_f))
    FLA_ACTIVE = False
    free_cuda("fla fallback")
    print(f"[fla] Triton GDN kernel failed here ({_fla_err}) -> every Gated-DeltaNet layer now "
          "uses the PyTorch reference kernel (slower, same math). Set USE_FLA = False next time.")
    ORIG_LOGITS = _probe_logits()
print("anchor logits captured:", tuple(ORIG_LOGITS.shape))


## Calibration — reading the spectra (3 streaming passes + WIK labels)

**Pass 1 & 2 (FFN):** per-neuron duty cycles **and second moments** E[a²] (the importance of a neuron is its output energy, not its signed mean), Johnson–Lindenstrauss co-firing sketches ($O(Ds)$ memory), and **stratified** stashes of layer inputs — block *i* of *n* contributes $\lfloor N(i+1)/n\rfloor-\lfloor Ni/n\rfloor$ rows, so router and gain fits (and their held-out validation) see the whole corpus mix instead of its first block.

**Pass 3 (the linear spectra):** every routed Linear site (attention $q$, $o$, the fused $v|gate$ gate-half, DeltaNet $q/z$ sections and $out\_proj$) records, **for free** via forward hooks, (a) the **per-row output energy** $E_i = \sum_t (w_i x_t)^2$ — the hook reads the module's own output, so no extra matmul is needed — and (b) a stratified subsample of *(mast features, output rows)* pairs for the ridge router warm-start and per-block energy calibration. This is literally Theorem 5's certificate data being collected while the model runs.

**Also here:** the **Spectral Head Funnel** is fitted (exact rank-r projection of the real `lm_head`), $\kappa$ of the residual-aware ranking is fitted on half of the probe blocks, and the **shortlist recall** is reported on the other half — held-out, before any training.


In [ ]:
# Calibration — three streaming passes (FFN spectra + linear spectra + funnel) + WIK labels (v9: stratified stashes)
from datasets import load_dataset

# ---- v8.0.4 [DATA]: a domain-balanced corpus ----------------------------------
# v8.0.2 calibrated AND distilled on 1% of wikitext only: encyclopedic English.
# Expert specialization (spectra, duty cycles, router prototypes) and the student's
# distillation then only ever "saw" that one register, and 16 epochs over 384
# blocks invited memorization. Now the same pipeline sees a mix of web / Korean /
# code / chat text (sources are STREAMED, a few MB each; any source that fails to
# load is skipped and wikitext fills the gap, so the worst case is the old corpus).
# Blocks are shuffled, so the calibration blocks (the first ones) and the held-out
# tail both carry the whole mixture. Set CORPUS_MIX = False to get wikitext only.
CORPUS_MIX = globals().get("CORPUS_MIX", RUN_MODE == "full")
MIX_TOTAL_BLOCKS = globals().get("MIX_TOTAL_BLOCKS", 1152)        # ~1.2M tokens
MIX_SHARES = globals().get("MIX_SHARES", {"fineweb-edu": 0.30, "ko-wiki": 0.15,
                                           "code": 0.15, "chat": 0.25})  # wikitext = the rest
MIX_DOC_CAP = 4 * SEQ_LEN        # one document contributes <= 4 blocks (breadth over depth)


def _wikitext_ids():
    try:
        ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="train[:1%]")
    except Exception:
        ds = load_dataset("wikitext", "wikitext-103-raw-v1", split="train[:1%]")
    text = "\n\n".join(t for t in ds["text"] if t.strip())
    return tok(text, return_tensors=None, add_special_tokens=False)["input_ids"]


def _stream_texts(name):
    """Lazily yield raw documents of one source."""
    if name == "fineweb-edu":
        ds = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT",
                          split="train", streaming=True)
        for r in ds:
            yield r["text"]
    elif name == "ko-wiki":
        ds = load_dataset("wikimedia/wikipedia", "20231101.ko", split="train", streaming=True)
        for r in ds:
            yield r["text"]
    elif name == "code":
        ds = load_dataset("codeparrot/codeparrot-clean", split="train", streaming=True)
        for r in ds:
            yield r["content"]
    elif name == "chat":
        ds = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft", streaming=True)
        for r in ds:
            msgs = r["messages"]
            try:
                yield tok.apply_chat_template(msgs, tokenize=False)
            except Exception:
                yield "\n".join(f"{m['role']}: {m['content']}" for m in msgs)
    else:
        raise ValueError(f"unknown corpus source {name!r}")


def _blocks_from(name, n_blocks, max_docs=20000):
    """Tokenize a stream, EOS-join documents, cut into SEQ_LEN blocks."""
    need_tok = n_blocks * SEQ_LEN
    eos = tok.eos_token_id if tok.eos_token_id is not None else 0
    ids, docs = [], 0
    for t in _stream_texts(name):
        if not t or not t.strip():
            continue
        ids.extend(tok(t, add_special_tokens=False)["input_ids"][:MIX_DOC_CAP])
        ids.append(eos)
        docs += 1
        if len(ids) >= need_tok or docs >= max_docs:
            break
    return [ids[i * SEQ_LEN:(i + 1) * SEQ_LEN] for i in range(len(ids) // SEQ_LEN)][:n_blocks]


CORPUS_COMPOSITION = {}
if CORPUS_MIX:
    _parts = {}
    for _src, _share in MIX_SHARES.items():
        _want = int(MIX_TOTAL_BLOCKS * _share)
        try:
            _parts[_src] = _blocks_from(_src, _want)
        except Exception as _e:
            print(f"  corpus source '{_src}' unavailable ({type(_e).__name__}: "
                  f"{str(_e)[:90]}) — skipped")
            _parts[_src] = []
        print(f"  corpus {_src:<12s}: {len(_parts[_src]):4d}/{_want} blocks")
    _wiki_ids = _wikitext_ids()
    _wiki_blocks = [_wiki_ids[i * SEQ_LEN:(i + 1) * SEQ_LEN]
                    for i in range(len(_wiki_ids) // SEQ_LEN)]
    _wiki_want = max(0, MIX_TOTAL_BLOCKS - sum(len(v) for v in _parts.values()))
    _parts["wikitext"] = _wiki_blocks[:_wiki_want]
    print(f"  corpus {'wikitext':<12s}: {len(_parts['wikitext']):4d}/{_wiki_want} blocks "
          "(fills whatever the other sources did not provide)")
    _all = [b for v in _parts.values() for b in v]
    random.Random(0).shuffle(_all)
    ids_all = [t for b in _all for t in b]
    CORPUS_COMPOSITION = {s: len(v) for s, v in _parts.items()}
    del _parts, _wiki_ids, _wiki_blocks, _all
else:
    ids_all = _wikitext_ids()

n_blocks = max(2, CAL_TOKENS // SEQ_LEN)
blocks = torch.tensor([ids_all[i*SEQ_LEN:(i+1)*SEQ_LEN]
                       for i in range(min(n_blocks, len(ids_all)//SEQ_LEN))])
print(f"calibration corpus: {blocks.shape[0]} blocks x {SEQ_LEN} tokens")

# deterministic subsamples across sessions (the hooks below draw
# randperm from the global RNG)
torch.manual_seed(0)

# ------------------------- discover every dense FFN -------------------------
def discover_ffns(m):
    found = []
    for name, mod in m.named_modules():
        if hasattr(mod, "gate_proj") and hasattr(mod, "up_proj") and hasattr(mod, "down_proj"):
            if "visual" in name.lower() or "vision" in name.lower():
                continue
            found.append((name, mod))
    return found

FFNS = discover_ffns(model)
FFNS_MAP = dict(FFNS)
print(f"discovered {len(FFNS)} dense SwiGLU FFNs")
STREAM_DTYPE = torch.float32

# ---- v9 [STRATIFIED STASHES] -------------------------------------------------------------
# v8 kept `torch.cat([old, new])[:N]`: once block 0 had filled a stash, every later block was
# truncated away, so the router fits, the gain fits and their validation split only ever saw
# ONE block of the corpus (~0.1% of the mixture).  Now block i of nb contributes exactly
#     floor(N (i+1) / nb) - floor(N i / nb)      rows   (sum over blocks = N, evenly spread)
# and the stashes are plain concatenations (no truncation).
_STASH_BLK = {"i": 0, "n": 1}


def _quota(n_total: int, avail: int) -> int:
    i, nb = _STASH_BLK["i"], max(1, _STASH_BLK["n"])
    return int(min(avail, (n_total * (i + 1)) // nb - (n_total * i) // nb))


def _stash_cat(old, new):
    """Append `new` (tensor or tuple of tensors) to the stash `old` (None = empty)."""
    if old is None:
        return new
    if torch.is_tensor(new):
        return torch.cat([old, new])
    return tuple(torch.cat([a, b]) for a, b in zip(old, new))


# ---------------- Pass 1: FFN duty cycles ------------------------
print("Pass 1/3: FFN duty cycles ...")
stats = {name: {"sum": None, "sq": None, "count": 0} for name, _ in FFNS}
def make_pass1_hook(name):
    gp, up_ = FFNS_MAP[name].gate_proj, FFNS_MAP[name].up_proj
    def hook(mod, args, kwargs, out):
        h = args[0] if args else kwargs["hidden_states"]
        with torch.no_grad():
            a = F.silu(gp(h)) * up_(h)
            a = a.reshape(-1, a.shape[-1]).to(STREAM_DTYPE)
            s = stats[name]
            s["sum"] = a.sum(0) if s["sum"] is None else s["sum"] + a.sum(0)
            s["sq"]  = (a*a).sum(0) if s["sq"] is None else s["sq"] + (a*a).sum(0)
            s["count"] += a.shape[0]
    return hook
handles = [mod.register_forward_hook(make_pass1_hook(name), with_kwargs=True)
           for name, mod in FFNS]
with torch.no_grad():
    for i in range(blocks.shape[0]):
        _STASH_BLK.update(i=i, n=blocks.shape[0])
        _ = model(input_ids=blocks[i:i+1].to(DEV))
for h_ in handles: h_.remove()
mean = {n: (s["sum"]/s["count"]) for n, s in stats.items()}
std  = {n: ((s["sq"]/s["count"] - mean[n]**2).clamp_min(1e-12)).sqrt() for n, s in stats.items()}
duty = {n: (s["sum"]/s["count"]) for n, s in stats.items()}
sqmean = {n: (s["sq"]/s["count"]) for n, s in stats.items()}   # v9: E[a^2] per neuron (duty is the SIGNED mean)

# ---------------- Pass 2: FFN sketches + stratified stashes -----------------
print("Pass 2/3: FFN JL sketches + stratified stashes ...")
sketch = {name: None for name, _ in FFNS}
h_sub  = {name: None for name, _ in FFNS}
H_SUB_N   = GAIN_CAL_TOKENS                                                  # linear-site / WIK stashes
H_SUB_FFN = FFN_CAL_TOKENS if FFN_CAL_MODE == "v9" else GAIN_CAL_TOKENS      # FFN calibration rows / layer
def make_pass2_hook(name):
    gp, up_ = FFNS_MAP[name].gate_proj, FFNS_MAP[name].up_proj
    def hook(mod, args, kwargs, out):
        h = args[0] if args else kwargs["hidden_states"]
        with torch.no_grad():
            a = F.silu(gp(h)) * up_(h)
            a = a.reshape(-1, a.shape[-1]).to(STREAM_DTYPE)
            hz = h.reshape(-1, h.shape[-1]).to(STREAM_DTYPE)
            Az = (a - mean[name]) / std[name]
            Phi = torch.randn(Az.shape[0], SKETCH_DIM, device=Az.device,
                              dtype=Az.dtype) / math.sqrt(SKETCH_DIM)
            Sb = Az.t() @ Phi
            sketch[name] = Sb if sketch[name] is None else sketch[name] + Sb
            take = _quota(H_SUB_FFN, Az.shape[0])          # v9: this block's share of the stash
            if take > 0:
                sel = torch.randperm(Az.shape[0], device=a.device)[:take]
                h_sub[name] = _stash_cat(h_sub[name], hz[sel].to(torch.bfloat16).cpu())
    return hook
handles = [mod.register_forward_hook(make_pass2_hook(name), with_kwargs=True)
           for name, mod in FFNS]
with torch.no_grad():
    for i in range(blocks.shape[0]):
        _STASH_BLK.update(i=i, n=blocks.shape[0])
        _ = model(input_ids=blocks[i:i+1].to(DEV))
for h_ in handles: h_.remove()
sketch = {n: s / s.norm(dim=1, keepdim=True).clamp_min(1e-8) for n, s in sketch.items()}

# ------------- Pass 3 (NEW): the LINEAR spectra + mast stash ----------------
print("Pass 3/3: linear-site row energies + mast stashes ...")
MAST = AntennaMast(HID, sketch_dim=MAST_S, seed=0).to(DEV)
LINEAR_SITES = []       # (name, module, kind)  kind: whole | sectioned
for name, mod in FAS:
    LINEAR_SITES.append((name + ".q_proj", mod.q_proj, "whole"))
    LINEAR_SITES.append((name + ".o_proj", mod.o_proj, "whole"))
    if SS.get("gate_rows", 0) > 0:
        LINEAR_SITES.append((name + ".v_gate", mod.v_proj, "vgate"))
for name, mod, sec in DNS_REF:
    if sec.get("out"):
        _o_owner, _o_attr, _oW, _prov = sec["out"]
        _o_mod = getattr(mod, _o_attr, None)
        if isinstance(_o_mod, torch.nn.Module):
            LINEAR_SITES.append((f"{name}.{_o_attr}", _o_mod, "whole"))
print(f"routed whole-linear sites: {sum(1 for _,_,k in LINEAR_SITES if k != 'sectioned')} "
      f"| sectioned DeltaNet in_proj: {len(DNS_REF)}")

row_energy = {name: None for name, _, _ in LINEAR_SITES}
site_stash = {name: None for name, _, _ in LINEAR_SITES}   # (phi, out) subsample
dn_row_energy = {(name, part): None for name, _, _ in DNS_REF
                 for part in ("main", "z")}
dn_stash = {(name, part): None for name, _, _ in DNS_REF
            for part in ("main", "z")}

def site_hook(name, kind):
    def hook(mod, args, kwargs, out):
        h = args[0] if args else kwargs.get("hidden_states")
        with torch.no_grad():
            o = out[0] if isinstance(out, tuple) else out       # [B, L, rows]
            o2 = o.reshape(-1, o.shape[-1]).to(STREAM_DTYPE)
            if kind == "vgate":                                 # gate = 2nd half
                o2 = o2[:, o2.shape[-1]//2:]
            re_ = (o2*o2).sum(0)                                # [rows]
            row_energy[name] = re_ if row_energy[name] is None \
                               else row_energy[name] + re_
            take = _quota(H_SUB_N, o2.shape[0])             # v9: stratified over blocks
            if take > 0:
                sel = torch.randperm(o2.shape[0], device=o2.device)[:take]
                phi = MAST(h.reshape(-1, h.shape[-1])[sel.to(h.device)].to(DEV))   # row-wise: phi of the kept rows only
                st = (phi.to(torch.bfloat16).cpu(),
                      o2[sel].to(torch.bfloat16).cpu())
                site_stash[name] = _stash_cat(site_stash[name], st)
    return hook

def dn_hook(name, part="main"):
    """Split-layout sites hook in_proj_qkv AND in_proj_z separately
    (part 'main' / 'z'); energies are token-SUMS so they merge cleanly into
    the virtual [q|k|v|z] order after the pass, while stashes stay per-part
    (each hook draws its own token subsample — phi/out pairing must never be
    mixed across parts)."""
    def hook(mod, args, kwargs, out):
        h = args[0] if args else kwargs.get("hidden_states")
        with torch.no_grad():
            o = out[0] if isinstance(out, tuple) else out
            o2 = o.reshape(-1, o.shape[-1]).to(STREAM_DTYPE)
            re_ = (o2*o2).sum(0)
            key = (name, part)
            dn_row_energy[key] = re_ if dn_row_energy[key] is None \
                                  else dn_row_energy[key] + re_
            take = _quota(H_SUB_N, o2.shape[0])             # v9: stratified over blocks
            if take > 0:
                sel = torch.randperm(o2.shape[0], device=o2.device)[:take]
                phi = MAST(h.reshape(-1, h.shape[-1])[sel.to(h.device)].to(DEV))
                st = (phi.to(torch.bfloat16).cpu(),
                      o2[sel].to(torch.bfloat16).cpu())
                dn_stash[key] = _stash_cat(dn_stash[key], st)
    return hook

handles = []
for name, mod, kind in LINEAR_SITES:
    if kind == "whole":
        handles.append(mod.register_forward_hook(site_hook(name, "whole"),
                                                 with_kwargs=True))
    elif kind == "vgate":
        handles.append(mod.register_forward_hook(site_hook(name, "vgate"),
                                                 with_kwargs=True))
for name, mod, sec in DNS_REF:
    _fq = sec.get("qkvz")
    if _fq is None:
        continue
    _fused_mod = getattr(mod, _fq[1], None)
    if isinstance(_fused_mod, torch.nn.Module):
        handles.append(_fused_mod.register_forward_hook(dn_hook(name, "main"),
                                                        with_kwargs=True))
    if sec.get("split") and sec.get("z") is not None:      # qwen3_5 split
        _z_mod = getattr(mod, sec["z"][1], None)
        if isinstance(_z_mod, torch.nn.Module):
            handles.append(_z_mod.register_forward_hook(dn_hook(name, "z"),
                                                        with_kwargs=True))
with torch.no_grad():
    for i in range(blocks.shape[0]):
        _STASH_BLK.update(i=i, n=blocks.shape[0])
        _ = model(input_ids=blocks[i:i+1].to(DEV))
for h_ in handles: h_.remove()

# ---- merge per-part energies into the virtual [q|k|v|z] order ----------
# (row energies are token sums -> merge is exact; stashes stay per-part and
# are exposed as dn_stash[name] / dn_stash[name + "#z"]).
for name, mod, sec in DNS_REF:
    main_e = dn_row_energy.pop((name, "main"), None)
    z_e = dn_row_energy.pop((name, "z"), None)
    if main_e is None:
        dn_row_energy[name] = None
    else:
        dn_row_energy[name] = torch.cat([main_e, z_e], 0) \
            if z_e is not None else main_e
    if dn_stash.get((name, "main")) is not None:
        dn_stash[name] = dn_stash.pop((name, "main"))
    if dn_stash.get((name, "z")) is not None:
        dn_stash[name + "#z"] = dn_stash.pop((name, "z"))
print("linear spectra collected for",
      len([k for k, v in row_energy.items() if v is not None]), "whole sites +",
      len([k for k, v in dn_row_energy.items() if v is not None]), "fused sections")

# ---------------- fit the Spectral Head Funnel (v9) + held-out recall -------------
lm_w = model.get_parameter("lm_head.weight")
funnel = HeadFunnel(lm_w.detach(), rank=FUNNEL_RANK, shortlist=FUNNEL_SHORTLIST,
                    use_rho=FUNNEL_USE_RHO)
# v9: kappa (residual-aware ranking) is FITTED on blocks spread over the corpus (even blocks of
# the probe set) and the recall is REPORTED on the other half (odd blocks) — held-out.
_fb = min(max(2, FUNNEL_FIT_BLOCKS), blocks.shape[0])
_fidx = sorted({int(round(x)) for x in torch.linspace(0, blocks.shape[0] - 1, _fb).tolist()})
_fit_h, _fit_t, _val_h, _val_t = [], [], [], []
with torch.no_grad():
    for n_, i in enumerate(_fidx):
        b = blocks[i:i+1, :FUNNEL_FIT_TOK].to(DEV)
        out = model(input_ids=b, output_hidden_states=True)
        h_last = out.hidden_states[-1][0]                     # [L, d]  (final-norm output)
        ref = h_last @ lm_w.to(DEV).t()                       # [L, V] exact logits
        if n_ == 0:                                           # exercise the real forward path once
            got, sl = funnel(h_last, mode="funnel", ref_logits=ref.float())
            del got, sl
        top = ref.float().topk(min(64, ref.shape[-1]), dim=-1).indices
        (_fit_h if n_ % 2 == 0 else _val_h).append(h_last.clone())
        (_fit_t if n_ % 2 == 0 else _val_t).append(top)
        del out, h_last, ref, top
_fit_h, _fit_t = torch.cat(_fit_h), torch.cat(_fit_t)
_val_h, _val_t = ((torch.cat(_val_h), torch.cat(_val_t)) if _val_h else (_fit_h, _fit_t))
FUNNEL_FIT = funnel.fit_kappa(_fit_h, _fit_t)                 # {kappa: recall}; kappa=0 = plain proxy
FUNNEL_KAPPA = funnel.kappa
FUNNEL_RECALL = funnel.shortlist_recall(_val_h, _val_t)                 # held-out, fitted kappa
FUNNEL_RECALL_K0 = funnel.shortlist_recall(_val_h, _val_t, 0.0)         # held-out, plain rank-r proxy
del b, _fit_h, _fit_t, _val_h, _val_t
print(f"\nHeadFunnel fitted: r={funnel.rank} V_s={funnel.shortlist} "
      f"| residual-aware kappa={FUNNEL_KAPPA} "
      f"| held-out shortlist top-64 recall: {FUNNEL_RECALL:.4f} "
      f"(plain proxy kappa=0: {FUNNEL_RECALL_K0:.4f}; exact shortlist rows by construction)")

# ---------------- Pass 4 (v8): MoRM 2.0 WIK influence labels ----------------
# THE WRITE INFLUENCE KERNEL (Theorem B): the exact first-order counterfactual
# value of every (token, head) memory write — from ONE capture of the stock
# GDN kernel's own (q, k, v, beta, decay), zero ablation runs. These labels
# supervise the MoRM 2.0 critic; MoRM 1.0's v-energy heuristic is the
# degenerate no-read-overlap special case.
#
# Capture: a module-level shim around the modeling module's kernel symbol
# (version-tolerant: kwargs captured wholesale; zero-copy: references to
# the kernel's live inputs; restored in `finally`). The per-call site is
# identified by a forward-pre-hook marker on each DNS_REF module (robust to
# vision/MTP side-calls: one capture per site per forward, shape-guarded).
# Lambda inference is PROBED against the kernel's own outputs (my reference
# oracle must reproduce the stock o to < 2e-2 rel) before any label is
# trusted. Loud fallback ladder, never silent: symbol not found | probe
# fails -> v-ENERGY labels (v1 semantics) in the surgery cell + warning.
print("Pass 4/4: MoRM 2.0 WIK labels (stock-kernel capture) ...")
MORM2_STASH = {}            # dn name -> {"phi": [N,S], "labels": [N,H]} cpu fp32
MORM2_PROBE = {"ok": None, "rel": None, "formula": None, "calls": 0}
_morm2_caps, _morm2_seen, _morm2_current = [], set(), [None]
_morm2_hstack = {}          # dn name -> last input hidden [T, d] (cpu)

def _morm2_hhook(name):
    def hook(mod, args, kwargs):
        _morm2_current[0] = name
        h = args[0] if args else kwargs.get("hidden_states")
        if h is not None:
            _morm2_hstack[name] = h.reshape(-1, h.shape[-1]).detach().float().cpu()
    return hook

def _morm2_wrap(orig):
    def _w(*a, **kw):
        out = orig(*a, **kw)
        site = _morm2_current[0]
        if site in _morm2_seen:            # one capture per site per forward
            return out                     # (skips MTP/vision re-calls)
        try:
            # v8.0.2 [G2]: qwen3_5's forward calls the kernel with POSITIONAL
            # (query, key, value) + keyword (g=, beta=); the fla/qwen3_next
            # era used (q, k, v) names. Accept both conventions.
            q = kw.get("q", kw.get("query", a[0] if len(a) > 0 else None))
            k = kw.get("k", kw.get("key", a[1] if len(a) > 1 else None))
            v = kw.get("v", kw.get("value", a[2] if len(a) > 2 else None))
            beta = kw.get("beta", a[3] if len(a) > 3 else None)
            g = kw.get("g", kw.get("decay", a[4] if len(a) > 4 else None))
            if None not in (q, k, v, beta, g) and q.shape[:2] == k.shape[:2]:
                _morm2_seen.add(site)
                _morm2_caps.append(dict(
                    site=site, q=q, k=k, v=v, beta=beta, g=g,
                    A_log=kw.get("A_log", None), dt_bias=kw.get("dt_bias", None),
                    scale=kw.get("scale", None),
                    l2=bool(kw.get("use_qk_l2norm_in_kernel", False)),
                    o=out[0] if isinstance(out, tuple) else out))
        except Exception:
            pass
        return out
    _w._morm2_wrap = True
    return _w

import inspect as _insp
_morm2_patches = []         # (module, attr, original)
# v8.0.2 [G1]: KERNEL SYMBOL DRIFT — the root cause of the user's
# "WIK labels UNAVAILABLE (probe failed or no captures)". transformers
# 5.8's qwen3_5 modeling module defines its GDN kernels as
# torch_chunk_gated_delta_rule / torch_recurrent_gated_delta_rule (fla-
# dispatching wrappers); the legacy names chunk_gated_delta_rule /
# fused_recurrent_gated_delta_rule belong to the fla/qwen3_next era and
# simply do not exist there, so the old loop installed ZERO wraps and
# Pass 4 captured nothing. Probe BOTH families and wrap EVERY symbol
# found (calibration takes the chunk path; single-token decode the
# recurrent one), each exactly once, guarded against double-wrap.
_MORM2_KERNEL_SYMS = ("torch_chunk_gated_delta_rule",
                      "torch_recurrent_gated_delta_rule",
                      "chunk_gated_delta_rule",
                      "fused_recurrent_gated_delta_rule")
def _torch_ref(f):
    """v8.0.4: capture on the PyTorch REFERENCE kernel, whatever the module symbol
    dispatches to. With fla-core installed the symbol runs the Triton kernel in
    training / teacher caching; the WIK probe, however, was validated against the
    reference's numerics, so the capture pass calls that twin directly (the
    wrapper chain keeps it reachable through __wrapped__). A no-op when there is
    nothing to unwrap."""
    try:
        g = _insp.unwrap(f)
        return g if callable(g) else f
    except Exception:
        return f


for _name, _mod, _sec in DNS_REF:
    _mm = _insp.getmodule(type(_mod))
    if _mm is None:
        continue
    for _fn in _MORM2_KERNEL_SYMS:
        _orig = getattr(_mm, _fn, None)
        if callable(_orig) and not getattr(_orig, "_morm2_wrap", False):
            setattr(_mm, _fn, _morm2_wrap(_torch_ref(_orig)))
            _morm2_patches.append((_mm, _fn, _orig))
if _morm2_patches:
    _morm2_mod = _morm2_patches[0][0].__name__
    print(f"  WIK kernel hook: {len(_morm2_patches)} symbol(s) "
          f"{[fn for _, fn, _ in _morm2_patches]} in {_morm2_mod}")
else:
    _diag = _insp.getmodule(type(DNS_REF[0][1])) if DNS_REF else None
    _avail = sorted(a for a in (dir(_diag) if _diag is not None else [])
                    if "delta" in a.lower() or "recurrent" in a.lower())
    print(f"  !! no GDN kernel symbol found in the modeling module "
          f"(delta/recurrent attrs: {_avail}) — WIK capture disabled")
MORM2_PROBE["calls"] = len(_morm2_patches)

def _morm2_lam(g, A_log, dt_bias, formula="auto"):
    """Per-head decay in (0,1) from the captured gate ingredients."""
    # v8.0.2 [G3]: qwen3_5's layer computes the log-decay BEFORE the kernel
    # (g = -exp(A_log) * softplus(a + dt_bias), entries <= 0) and the kernel
    # multiplies the state by exp(g) each step — so for the CAPTURED g the
    # decay multiplier is exp(g) directly ("logdecay"). "gdn" expects the
    # raw pre-activation plus A_log/dt_bias kernel kwargs (qwen3_5 never
    # passes those); "gla" is the sigmoid family. The probe validates
    # whichever it selects against the stock kernel's own output.
    if formula == "logdecay":
        return torch.exp(g.float())
    if A_log is not None and dt_bias is not None and formula in ("auto", "gdn"):
        return torch.exp(-torch.exp(A_log.float())
                         * F.softplus(g.float() + dt_bias.float()))
    return torch.sigmoid(g.float())


def _morm2_g2(g):
    """v8.0.2 [G5]: normalize a captured gate tensor to [T,H] or [T,H,dk].

    qwen3_5/fla hand the kernel g as [B,T,H] (3-D); some kernels keep a
    per-key-dim resolution [B,T,H,dk] (4-D). The old reshape(T,H,-1)
    turned the 3-D case into [T,H,1], which the oracle's lam_to_full
    asserts away — every formula then failed and Pass 4 degraded."""
    if g is None:
        return None
    if g.dim() == 2:
        return g.float()
    if g.dim() == 3:                                   # [B, T, H]
        return g.reshape(-1, g.shape[-1]).float()
    if g.dim() == 4:                                   # [B, T, H, dk]
        return g.reshape(-1, g.shape[-3], g.shape[-2]).float()
    return g.float()

def _morm2_probe_formulas(cap):
    """Validate (formula, l2, scale) by reproducing the stock kernel's o
    with the morm2 reference oracle. Returns (lam, rel_err) of the best."""
    q = cap["q"].reshape(-1, cap["q"].shape[-2], cap["q"].shape[-1]).float()
    T_, H_, dk_ = q.shape
    k_ = cap["k"].reshape(T_, H_, -1).float()
    v_ = cap["v"].reshape(T_, H_, -1).float()
    b_ = cap["beta"].reshape(-1, H_).float().clamp(1e-6, 1.0)
    # v8.0.2 [G4]: normalize g's dimensionality (see _morm2_g2) — the
    # old reshape made the 3-D [B,T,H] capture into [T,H,1] and every
    # formula died in the oracle's shape assert.
    g_ = _morm2_g2(cap["g"]) if cap["g"] is not None else None
    if g_ is None:
        return None, float("inf")
    o_ref = cap["o"].reshape(T_, H_, -1).float()
    best = (None, float("inf"))
    for formula in ("logdecay", "gdn", "gla"):
        try:
            lam = _morm2_lam(g_, cap["A_log"], cap["dt_bias"], formula)
            if lam.dim() == 2:
                lam3 = lam[:, :, None].expand(T_, H_, dk_)
            else:
                lam3 = lam
            if cap["l2"]:
                qq, kk = F.normalize(q, dim=-1), F.normalize(k_, dim=-1)
            else:
                qq, kk = q, k_
            sc = float(cap["scale"]) if cap["scale"] else dk_ ** -0.5
            o_my, _ = gdn_sequential(qq, kk, v_, b_, lam3, "delta", sc)
            rel = float((o_my - o_ref).norm() / o_ref.norm().clamp_min(1e-9))
            if rel < best[1]:
                best = (formula, rel)
        except Exception:
            continue
    return best

def _morm2_labels_for(cap, formula):
    """v8.0.2 [G6]: WIK labels for one captured call, computed on the
    ACTUAL deposited write values. The stock GDN kernel (the fla WY form
    and transformers' torch twin alike) runs the DELTA rule: a write
    deposits d_s = v_s - S~_s^T k_s, NOT the raw v_s — Theorem B's
    first-order counterfactual must therefore be labelled on d_s. (The
    v1/v-energy family priced writes on ||v_s||, which mis-prices every
    write the state already predicts — the "no-read-overlap special
    case".) One sequential pass of the M1-validated oracle — the SAME
    reference whose output the lambda probe just certified against the
    stock kernel — yields the exact deposited values; the influence
    kernel itself stays closed-form and windowed."""
    q = cap["q"].reshape(-1, cap["q"].shape[-2], cap["q"].shape[-1]).float()
    T_, H_, dk_ = q.shape
    k_ = cap["k"].reshape(T_, H_, -1).float()
    v_ = cap["v"].reshape(T_, H_, -1).float()
    b_ = cap["beta"].reshape(-1, H_).float().clamp(1e-6, 1.0)
    g_ = _morm2_g2(cap["g"])
    lam = _morm2_lam(g_, cap["A_log"], cap["dt_bias"], formula)
    if cap["l2"]:
        q, k_ = F.normalize(q, dim=-1), F.normalize(k_, dim=-1)
    sc = float(cap["scale"]) if cap["scale"] else dk_ ** -0.5
    _, _, deltas = gdn_sequential(q, k_, v_, b_, lam, "delta", sc,
                                  return_deltas=True)
    lab = write_influence_labels(
        q, k_, deltas, b_, lam, scale=sc,
        window=min(MORM2_WIK_WINDOW, T_), chunk=64)["labels"]
    del deltas
    return lab, T_

if _morm2_patches:
    import gc                                   # v8.0.2 [G7]: hygiene pass
    _morm2_hh = [mod.register_forward_pre_hook(_morm2_hhook(name),
                                               with_kwargs=True)
                 for name, mod, sec in DNS_REF]
    _nblocks = min(MORM2_LABEL_BLOCKS, blocks.shape[0])
    # v8.0.2 [G7a]: sample ACROSS blocks. The old min(H_SUB_N, T_) rows per
    # block + concat-then-truncate kept only block 0's rows — 15/16 of the
    # (now delta-form) label compute was discarded and the ridge warm start
    # saw a single text block.
    _per_blk = max(64, H_SUB_N // max(1, _nblocks))
    try:
        with torch.no_grad():
            for bi in range(_nblocks):
                _morm2_caps.clear(); _morm2_seen.clear()
                _ = model(input_ids=blocks[bi:bi+1].to(DEV))
                for cap in _morm2_caps:
                    if MORM2_PROBE["ok"] is None:
                        f_, rel_ = _morm2_probe_formulas(cap)
                        MORM2_PROBE.update(formula=f_, rel=rel_,
                                           ok=(f_ is not None and rel_ < 2e-2))
                        print(f"  WIK lambda probe: formula={f_} "
                              f"rel={rel_:.1e} -> "
                              f"{'TRUSTED' if MORM2_PROBE['ok'] else 'FALLBACK'}")
                    if not MORM2_PROBE["ok"]:
                        break
                    name_c = cap["site"]
                    try:
                        lab, T_ = _morm2_labels_for(cap,
                                                    MORM2_PROBE["formula"])
                    except torch.cuda.OutOfMemoryError:
                        # v8.0.2 [G7b] OOM GUARD: the label pass may NEVER
                        # kill the pipeline — keep every stash row already
                        # earned, flush the allocator, stop the block loop
                        # loudly (a warm start on fewer blocks is strictly
                        # better than a crashed run).
                        print(f"  !! WIK label OOM at block {bi} "
                              f"({name_c}) — keeping "
                              f"{sum(len(s['labels']) for s in MORM2_STASH.values())} "
                              f"stash rows earned so far")
                        torch.cuda.empty_cache()
                        break
                    # v8.0.2 [G7c] OOM GUARD: release this capture's GPU
                    # references the moment its labels are stashed — the
                    # processing peak stays flat at ~one site's transients
                    # instead of accumulating all sites' (q, k, v, g, beta,
                    # o) across the whole loop.
                    for _k2 in ("q", "k", "v", "beta", "g", "o"):
                        cap[_k2] = None
                    h_c = _morm2_hstack.get(name_c)
                    if h_c is None or h_c.shape[0] != T_:
                        continue
                    phi_c = MAST(h_c.to(DEV)).float().cpu()
                    take = min(max(1, _per_blk), T_)
                    sel = torch.randperm(T_)[:take]
                    st = MORM2_STASH.setdefault(
                        name_c, {"phi": [], "labels": []})
                    st["phi"].append(phi_c[sel])
                    st["labels"].append(lab[sel].cpu())
                if not MORM2_PROBE["ok"]:
                    break
                gc.collect()
                torch.cuda.empty_cache()   # per-block allocator hygiene
    finally:
        for h_ in _morm2_hh:
            h_.remove()
        for _mm, _fn, _orig in _morm2_patches:
            setattr(_mm, _fn, _orig)
        _morm2_caps.clear()
        _morm2_hstack.clear()         # ~1 GiB of CPU hidden-state copies
    for st in MORM2_STASH.values():
        st["phi"] = torch.cat(st["phi"])[:H_SUB_N]
        st["labels"] = torch.cat(st["labels"])[:H_SUB_N]
if not (MORM2_PROBE["ok"] and MORM2_STASH):
    print("  !! MoRM 2.0 WIK labels UNAVAILABLE (probe failed or no captures) "
          "-> the surgery cell falls back to v-ENERGY labels (v1 semantics); "
          "MoRM 2.0 elastic scheduling + certificates still run, only the "
          "critic warm start degrades to the energy heuristic")
print(f"  WIK labels stashed for {len(MORM2_STASH)} DN sites "
      f"(probe rel {MORM2_PROBE['rel'] if MORM2_PROBE['rel'] else 'n/a'})")

print("calibration complete.")
free_cuda("calibration complete")   # sketches stay resident (2.3 GiB, needed by surgery); transients released


## Resonant surgery — fission + resonant linears + MoRM + funnel

Per FFN layer: **importance-ranked shared core** ($E[a^2]\,\lVert W_{down}[:,r]\rVert^2$) + balanced spectral experts + ZSSI zero-init completions + a **validated router** (prototype vs ridge on oracle energy shares, picked on held-out tokens) + **NNLS gains** fitted against the module that is actually served ($y_{dense}-y_{shared}$) — with **the HF weight convention handled by adopting transposed views** (zero-copy, mathematically exact).

Per Linear site: `plan_row_modes` partitions output rows into carrier + equal blocks ordered by measured energy; `ResonantLinear` adopts the weight; the ridge router warm-starts from the Pass-3 stash.

Per DeltaNet: the fused `in_proj_qkvz` becomes a `SectionedProjector` — **same input/output interface** so the stock layer code runs unchanged — with $q/z$ routed, $k$ dense, $v$ MoRM-write-sparse. The MoRM mask zeroes $v$ rows of non-selected heads, which *exactly* zeroes their rank-1 writes — no kernel surgery.

**Then the anchor test:** `set_world('dense_teacher')` must reproduce `ORIG_LOGITS` captured before surgery — Theorem 7 on the real model. Any drift fails the notebook.

**Then the census (v9):** `census_active` walks the built graph and compares every family with the solver's plan; with `ENFORCE_BUDGET` a build above the target stops here, not hours later.


In [ ]:
# Surgery — fission, resonant linears, MoRM, funnel + the Theorem-7 anchor (v9: validated routers, serving-consistent gains, census)
# ---- v8.0.4: re-runnable surgery ----------------------------------------------
# A failed / interrupted surgery used to leave the model half-rewired and the
# RESONA_* registries half-filled; pressing Run again then stacked a second layer
# of wrappers on the first (a BetaGate around a BetaGate -> the mask applied
# twice, with two different policies). Every module swap below goes through
# _swap(), which remembers the ORIGINAL once. The cell starts by restoring all of
# them and clearing every registry, so running it twice == running it once.
_SURG_ORIG = globals().get("_SURG_ORIG") or {}


def _swap(parent, attr, new):
    key = (id(parent), attr)
    if key not in _SURG_ORIG:
        _SURG_ORIG[key] = (parent, attr, getattr(parent, attr, None))
    setattr(parent, attr, new)


def _unwind_surgery():
    n = 0
    for parent, attr, orig in list(_SURG_ORIG.values()):      # (1) registry of swaps
        if orig is not None:
            setattr(parent, attr, orig)
            n += 1
    _SURG_ORIG.clear()
    for _nm, _dm, _sc in (globals().get("DNS_REF") or []):    # (2) swaps made by a pre-v8.0.4
        _ba = _sc.get("ba") if _sc.get("ba") is not None else _sc.get("b")   # run (no registry):
        if _ba is None:                                       # a BetaGate must never be wrapped twice
            continue
        _cur = getattr(_ba[0], _ba[1], None)
        while isinstance(_cur, BetaGate):
            _cur = _cur.inner
            setattr(_ba[0], _ba[1], _cur)
            n += 1
    for _lst in (PRISM_MODULES, RESONA_LINEARS, RESONA_SECTIONED,   # (3) registries
                 RESONA_MORM, RESONA_FUNNEL):
        _lst.clear()
    if n:
        print(f"[surgery re-run] restored {n} module(s) rewired by the previous attempt")
        free_cuda("surgery unwound")


_unwind_surgery()

assign_store = {}
RESONA_MORM_GATES = []   # (name, BetaGate) — v8 write-enable levers
SHADOW_REPORT = {"lin": [], "dn_q": [], "dn_v": [], "dn_z": [], "ffn": []}   # v10: held-out quality of every shadow

# ---------------- 1) FFN fission (v1 machinery, weight fix applied) ---------
print("fissioning FFNs ...")
CAL_REPORT = {}      # v9: per-layer calibration report (router choice, capture, serving error)
for li, (name, mlp) in enumerate(FFNS):
    S_l = sketch[name].to(DEV)
    # v9: the always-on core = the neurons with the largest OUTPUT energy E[a^2] * ||W_down col||^2
    # (v8 ranked by the SIGNED mean activation: blind to magnitude, and a sign-flipping neuron cancels)
    imp_l = (ffn_importance(sqmean[name].to(DEV), mlp.down_proj.weight.data)
             if FFN_CAL_MODE == "v9" else duty[name].to(DEV))
    fus = fission_layer(imp_l, S_l, E_CHOSEN, SHARED_SIZE, seed=li, min_expert_size=8)
    assign = fus["assign"]; assign_store[name] = assign.cpu()
    # HF stores gate/up as [D, d]; PrismMoeMLP expects [d, D] — adopt
    # TRANSPOSED VIEWS (zero-copy, exact math).
    prism = PrismMoeMLP(mlp.gate_proj.weight.data.t(),
                        mlp.up_proj.weight.data.t(),
                        mlp.down_proj.weight.data,
                        assign, E_CHOSEN, rank=COMPLETION_RANK, seed=li,
                        k_topk=K_CHOSEN).to(DEV)
    hs = h_sub[name].to(DEV, dtype=prism.W_gate.dtype)
    cal = None
    if FFN_CAL_MODE == "v9":
        try:
            cal = calibrate_ffn_layer(prism, hs, K_CHOSEN, router_mode=FFN_ROUTER_MODE, seed=li)
            CAL_REPORT[name] = cal
        except Exception as _e:            # a new formula must never be the reason a long run dies
            if is_oom(_e):
                raise
            print(f"  !! v9 FFN calibration failed at {name} ({type(_e).__name__}: "
                  f"{str(_e)[:140]}) -> v8 prototype router + Adam gains for this layer")
            cal = None
    if cal is None:                        # ---- v8 calibration (fallback / FFN_CAL_MODE='v8') ----
        asub = F.silu(hs.to(STREAM_DTYPE) @ prism.W_gate.to(STREAM_DTYPE)) * \
               (hs.to(STREAM_DTYPE) @ prism.W_up.to(STREAM_DTYPE))
        e_routed = asub[:, prism.expert_idx.reshape(-1)].pow(2)
        energy = e_routed.reshape(asub.shape[0], E_CHOSEN, prism.m_e).sum(-1)
        Wr = router_prototypes(hs.to(STREAM_DTYPE), energy)
        logits0 = hs.to(STREAM_DTYPE) @ Wr.t()
        # v9: the calibration clamp now equals the SERVING clamp (forward: clamp(0.25, 4); v8 fitted with (0.5, 8))
        tau = (logits0.std(dim=-1).mean() / max(0.5 * math.log(E_CHOSEN), 1e-3)).clamp(0.25, 4.0)
        with torch.no_grad():
            prism.router.weight.copy_(Wr.to(prism.router.weight.dtype))
            prism.log_tau.fill_(math.log(tau))
            y_e = prism.expert_outputs(hs, use_completion=False)
            y_dense_sub = prism.dense_forward(hs.to(prism.W_gate.dtype))
            topk_idx = torch.topk(torch.softmax(
                prism.router(hs.to(prism.router.weight.dtype)) / tau, -1),
                                  K_CHOSEN, dim=-1).indices
        c = calibrate_gains(y_e, y_dense_sub, topk_idx, steps=GAIN_CAL_STEPS, lr=0.05)
        with torch.no_grad():
            prism.gains.copy_(c.to(prism.gains.dtype))
    if SHADOW_RANK_FFN > 0:
        try:
            SHADOW_REPORT["ffn"].append(fit_ffn_shadow(prism, hs, K_CHOSEN, SHADOW_RANK_FFN, seed=li))
        except Exception as _e:        # a new formula must never be the reason a long run dies
            if is_oom(_e):
                raise
            print(f"  !! FFN shadow fit failed at {name} ({type(_e).__name__}: {str(_e)[:140]}) -> no shadow here")
    parent = model.get_submodule(name.rpartition(".")[0])
    prism.default_mode = "soft"
    _swap(parent, name.rpartition(".")[2], prism)
    PRISM_MODULES.append((name, prism))
    sketch[name] = sketch[name].cpu()
    if li % max(1, len(FFNS)//6) == 0:
        print(f"  fissioned [{li+1}/{len(FFNS)}] {name}")
    # v8.0.3: drop this layer's calibration temporaries NOW. The loop variables
    # otherwise keep layer li's [T, E, d] expert stack (~2.7 GiB at E=256) alive
    # while layer li+1 builds ITS stack (2x peak).
    y_e = y_dense_sub = asub = e_routed = energy = logits0 = hs = topk_idx = None
    if li % 8 == 7:
        torch.cuda.empty_cache()   # keep fragmentation flat across 64 sites

if CAL_REPORT:
    _R = list(CAL_REPORT.values())
    def _mean(key):
        v = [r[key] for r in _R if r.get(key) is not None]
        return sum(v) / len(v) if v else float("nan")
    _picks = {}
    for r in _R:
        _picks[r["router"]] = _picks.get(r["router"], 0) + 1
    print(f"  FFN calibration v9 ({len(_R)} layers): routers {_picks} | held-out oracle-energy capture "
          f"{_mean('capture'):.3f} (prototype {_mean('capture_proto'):.3f}, ceiling {_mean('oracle'):.3f}) "
          f"| expert-load CV {_mean('load_cv'):.2f} | serving rel.err {_mean('rel_err'):.3f} "
          f"(unit gains {_mean('rel_err_prior'):.3f})")

# ---------------- 2) resonant whole-Linears ---------------------------------
print("installing ResonantLinears ...")
for name, mod, kind in LINEAR_SITES:
    if kind == "sectioned" or row_energy.get(name) is None:
        continue
    W0 = mod.weight.data
    if kind == "vgate":
        W_eff = W0[W0.shape[0]//2:]                 # route ONLY the gate half
        re_eff = row_energy[name]                   # hook already sliced the half
    else:
        W_eff, re_eff = W0, row_energy[name]
    phi_s, out_s = site_stash[name]
    phi_s = phi_s.to(DEV).float()
    U_sh = res_s = None
    if SHADOW_RANK_LIN > 0 and kind != "vgate":
        try:
            U_sh, _cap, res_s = fit_shadow_basis(out_s, SHADOW_RANK_LIN, device=DEV)
        except Exception as _e:
            if is_oom(_e):
                raise
            print(f"  !! shadow fit failed at {name} ({type(_e).__name__}: {str(_e)[:120]}) -> v9 path here")
            U_sh = res_s = None
    if U_sh is not None:
        SHADOW_REPORT["lin"].append(_cap)
        re_eff = res_s.pow(2).mean(0)          # v10: the row plan sees the RESIDUAL energy
    plan = plan_row_modes(W_eff.shape[0], W_eff.shape[1], LIN_BLOCKS,
                          CARRIER_FRAC, row_energy=re_eff)
    B = plan["block_idx"].shape[0]
    blk_idx = plan["block_idx"].reshape(-1)
    # v8.0.4: gather the routed columns WHERE THE STASH LIVES (CPU), then move only that
    # [N, B*rows] slice — blk_idx comes from the GPU-side plan, and indexing a CPU tensor
    # with a CUDA index is a RuntimeError.
    E_tok = out_s.index_select(1, blk_idx.to(out_s.device)).to(DEV).float().pow(2)
    E_tok = E_tok.view(out_s.shape[0], B, -1).sum(-1)            # [T, B]
    rl = ResonantLinear(W_eff, plan, MAST, calib_feats=phi_s,
                        calib_energy=E_tok, topk=LIN_TOPK).to(DEV)
    if U_sh is not None:
        rl.attach_shadow(U_sh)
        with torch.no_grad():              # router: which block does the shadow explain WORST?
            rl.router.weight.copy_(ridge_router_init(
                phi_s, _block_energy(res_s, plan["block_idx"])).to(rl.router.weight.dtype))
    rl.router_st = bool(ROUTER_ST)
    del res_s, U_sh
    rl.default_mode = "dense"
    if kind == "vgate":
        # name is a VIRTUAL path "<attn>.v_gate"; the stock attribute is v_proj.
        attn_pfx = name[:-len(".v_gate")]
        attn_mod = model.get_submodule(attn_pfx)
        class VGateShim(torch.nn.Module):
            """dense value half + resonant gate half, stock v_proj interface.

            ADOPTS the original Parameter as `.weight` (persistent, exact
            state-dict key, zero-copy) and derives the value half as a
            non-persistent view; dense mode re-dispatches the EXACT
            original single GEMM (a row-split GEMM picks a different
            kernel and drifts ~1e-2 in bf16 — the Theorem-7 anchor needs
            the bitwise path)."""
            def __init__(self, full_param, rl_gate):
                super().__init__()
                self.register_parameter("weight", full_param)
                self.v_rows = full_param.shape[0] // 2
                self.rl_gate = rl_gate
                self.register_buffer("v_w", full_param.data[:self.v_rows],
                                     persistent=False)
            def forward(self, x, mode=None):
                m = mode or self.rl_gate.default_mode
                if m == "dense":
                    return F.linear(x, self.weight)
                v = F.linear(x, self.v_w)
                g = self.rl_gate(x, mode=m)
                return torch.cat([v, g], dim=-1)
        _swap(attn_mod, "v_proj", VGateShim(mod.weight, rl))
        # sidecar path points at the REAL weight location (v_proj)
        RESONA_LINEARS.append((attn_pfx + ".v_proj(gate)", rl))
    else:
        parent = model.get_submodule(name.rpartition(".")[0])
        _swap(parent, name.rpartition(".")[2], rl)
        RESONA_LINEARS.append((name, rl))

# ---------------- 3) DeltaNet sectioned projectors + MoRM -------------------
print("installing SectionedProjectors + MoRM ...")
if GATES.get("dn_surgery") and DNS_REF:
    for name, mod, sec in DNS_REF:
        n_q, n_k_ = sec["q_rows"], sec["k_rows"]
        n_v_, n_z = sec["v_rows"], sec["z_rows"]
        re_ = dn_row_energy.get(name)
        stash = dn_stash.get(name)
        if re_ is None or stash is None:
            print(f"  !! no calibration data for {name} — site kept dense (R5)")
            continue
        q_re, z_re = re_[:n_q], re_[n_q + n_k_ + n_v_:]
        owner, attr, W_fused, prov = sec["qkvz"]
        # SPLIT LAYOUT (qwen3_5 family: in_proj_qkv [q|k|v] + in_proj_z):
        # the SectionedProjector owns q|k|v ONLY (z_rows=0 -> its output is
        # exactly the stock [q|k|v] interface of in_proj_qkv); the z
        # projection gets its own ResonantLinear at in_proj_z, preserving
        # that module's interface and state-dict key. Fused layout
        # (qwen3_next-style in_proj_qkvz): the 4-section projector.
        split = bool(sec.get("split")) and sec.get("z") is not None
        stash_z = dn_stash.get(name + "#z") if split else None
        if split and (stash_z is None or z_re.numel() != n_z):
            print(f"  !! z-sibling calibration missing for {name} — "
                  "in_proj_z kept dense (R5)")
            split = False
            n_z = 0
        _phi_dn, _out_dn = stash
        _Uq = _Uv = _Uz = _res_q = _res_zf = None
        if SHADOW_RANK_LIN > 0:
            try:
                _Uq, _c, _res_q = fit_shadow_basis(_out_dn[:, :n_q], SHADOW_RANK_LIN, device=DEV)
                if _Uq is not None:
                    SHADOW_REPORT["dn_q"].append(_c)
                    q_re = _res_q.pow(2).mean(0)
                if SHADOW_V and sec.get("morm_ok"):
                    _Uv, _c, _ = fit_shadow_basis(_out_dn[:, n_q + n_k_:n_q + n_k_ + n_v_],
                                                 SHADOW_RANK_LIN, device=DEV)
                    if _Uv is not None:
                        SHADOW_REPORT["dn_v"].append(_c)
                if not split and n_z > 0:
                    _Uz, _c, _res_zf = fit_shadow_basis(_out_dn[:, n_q + n_k_ + n_v_:],
                                                        SHADOW_RANK_LIN, device=DEV)
                    if _Uz is not None:
                        SHADOW_REPORT["dn_z"].append(_c)
                        z_re = _res_zf.pow(2).mean(0)
            except Exception as _e:
                if is_oom(_e):
                    raise
                print(f"  !! shadow fit failed at {name} ({type(_e).__name__}: {str(_e)[:120]}) -> v9 path here")
                _Uq = _Uv = _Uz = _res_q = _res_zf = None
        plan_q = plan_row_modes(n_q, W_fused.shape[1], LIN_BLOCKS,
                                CARRIER_FRAC, row_energy=q_re)
        plan_z = None if (split or n_z == 0) else \
            plan_row_modes(n_z, W_fused.shape[1], LIN_BLOCKS,
                           CARRIER_FRAC, row_energy=z_re)
        morm = None
        if sec.get("morm_ok"):
            n_heads = n_v_ // sec["v_head_rows"]
            # v8 (MoRM 2.0): influence critic + elastic budget. m_bar is the
            # solver's m_write (same AVERAGE cost as v7's fixed top-m); the
            # per-token count is now token-adaptive (E[m] = m_bar).
            # v8.0.1 [F1]: `MORM2_M_MAX` never existed in any cell (the
            # config knob is MORM2_M_MAX_FRAC — a FRACTION of n_heads). Derive
            # the ABSOLUTE per-token cap from the fraction, clamped into
            # [1, n_heads], then lifted to >= m_bar so the exported m_max
            # certificate can never undercut the mean budget (the controller
            # clamps internally too — this keeps every consumer and the
            # export metadata honest). For Qwen3.8-27B (H=48): 0.75*48 -> 36.
            _m_bar = min(M_WRITE, n_heads)
            _m_max = max(_m_bar, min(max(1, int(MORM2_M_MAX_FRAC * n_heads)),
                                     n_heads))
            morm = MoRM2Policy(n_heads=n_heads, mast=MAST,
                               m_bar=_m_bar,
                               m_max=_m_max,
                               controller_lr=MORM2_CTRL_LR,
                               ste_tau=MORM2_STE_TAU,
                               critic_bias=MORM2_CRITIC_BIAS).to(DEV)
            st4 = MORM2_STASH.get(name)
            if st4 is not None and len(st4["labels"]):
                # supervised warm start on the EXACT influence labels
                morm.ridge_init(st4["phi"].to(DEV), None,
                                st4["labels"].to(DEV))
                morm._crit_fit = (st4["phi"][:512].to(DEV),
                                  st4["labels"][:512].to(DEV))
                _wsrc = "exact WIK influence labels"
            else:
                # fallback: v1 energy warm start (loudly reported upstream)
                phi_s, out_s = stash
                phi_s = phi_s.to(DEV).float()
                # v8.0.1 [F3a] OOM GUARD: slice the v-columns from the CPU
                # stash BEFORE the device move — only [N, n_v_] crosses to
                # the GPU, never the fused [N, q+k+v] block (the fallback
                # path's biggest transient is now bounded by the v-section).
                v_re = out_s[:, n_q+n_k_:n_q+n_k_+n_v_].to(DEV).float().pow(2)
                v_head_E = v_re.view(v_re.shape[0], -1,
                                     sec["v_head_rows"]).sum(-1)
                morm.ridge_init(phi_s, None, v_head_E.clamp_min(1e-12))
                morm._crit_fit = None
                _wsrc = "v-energy labels (fallback)"
            print(f"  MoRM 2.0 {name.rpartition('.')[2] if '.' in name else name}: "
                  f"m_bar={morm.m_bar}/{n_heads} warm-start={_wsrc}")
        sp_mod = SectionedProjector(W_fused,
                                    (n_q, n_k_, n_v_, 0 if split else n_z),
                                    v_head_rows=sec.get("v_head_rows") or 1,
                                    plan_q=plan_q, plan_z=plan_z,
                                    mast=MAST, morm=morm,
                                    topk=LIN_TOPK).to(DEV)
        # v10: attach the shadows and CALIBRATE the q/z routers (v9 left them at zero: with zero logits
        # top-1 always picked block 0 — the 'routed' q section was a fixed 1/16 of the rows)
        with torch.no_grad():
            _phi_g = _phi_dn.to(DEV).float()
            for _tag, _U, _res, _lo in (("q", _Uq, _res_q, 0), ("z", _Uz, _res_zf, n_q + n_k_ + n_v_)):
                if not hasattr(sp_mod, f"{_tag}_B"):
                    continue
                _blk = getattr(sp_mod, f"{_tag}_blocks")
                if _U is not None:
                    sp_mod.attach_shadow(_tag, _U)
                    _E = _block_energy(_res, _blk)
                else:
                    _nr = getattr(sp_mod, f"{_tag}_rows")
                    _E = _block_energy(_out_dn[:, _lo:_lo + _nr], _blk).to(DEV)
                _rw = getattr(sp_mod, f"{_tag}_router").weight
                _rw.copy_(ridge_router_init(_phi_g, _E).to(_rw.dtype))
            if _Uv is not None:
                sp_mod.attach_shadow("v", _Uv)
        sp_mod.router_st = bool(ROUTER_ST)
        del _Uq, _Uv, _Uz, _res_q, _res_zf
        sp_mod.default_mode = "dense"
        _swap(owner, attr, sp_mod)        # NAME-RESOLVED attribute (R4)
        RESONA_SECTIONED.append((name + "." + attr, sp_mod))
        # v8: the write-enable lever — BetaGate wraps the stock in_proj_ba
        # so skipped heads carry beta ~ 0 EXACTLY in the stock kernel
        # (Proposition F; the v7 v-mask leaks deletion writes there). The
        # exact-mode anchor stays bitwise: mask == 1 reproduces the stock
        # tensor. Fails LOUDLY (never silently) to v1 semantics on any
        # wiring surprise.
        # v8.0.2 [P4]: the write lever resolves the FUSED in_proj_ba when
        # the layout has one, and the SPLIT in_proj_b (qwen3_5) when it
        # does not — sec["b"] is recorded by the (v8.0.2) census cell. The
        # old code only knew the fused ref and SILENTLY skipped the split
        # layout, leaving MoRM 2.0 inert: critic fitted, certificates
        # exported, and the mask never reaching a single write.
        _ba_sec = sec.get("ba") if sec.get("ba") is not None else sec.get("b")
        if morm is not None and _ba_sec is not None:
            try:
                _bo, _battr, _bW, _bprov = _ba_sec
                _b_only = sec.get("ba") is None      # split: in_proj_b alone
                _ba_mod = getattr(_bo, _battr, None)
                while isinstance(_ba_mod, BetaGate):     # v8.0.4: never wrap a gate in a gate
                    _ba_mod = _ba_mod.inner
                if isinstance(_ba_mod, torch.nn.Module) and \
                        getattr(_ba_mod, "weight", None) is not None:
                    # v8.0.2 [P4]: qwen3_5 split layout — BetaGate wraps
                    # in_proj_b alone (b_only=True gates the whole [H]
                    # output; in_proj_a stays dense — a drives decay, not
                    # writes; the exact skip lever is beta, Prop F).
                    gate = BetaGate(_ba_mod, morm.H, morm,
                                    b_first=True,
                                    b_only=_b_only,
                                    logit_floor=MORM2_BETA_LOGIT_FLOOR,
                                    A_log=getattr(mod, "A_log", None),
                                    dt_bias=getattr(mod, "dt_bias", None)
                                    ).to(DEV)
                    _swap(_bo, _battr, gate)
                    # v10: with a v shadow all heads write — the gate must not delete those writes
                    gate.shadow_writes = bool(getattr(sp_mod, "v_sh_U", None) is not None)
                    RESONA_MORM_GATES.append((name + "." + _battr, gate))
            except Exception as _e:
                print(f"  !! BetaGate install failed for {name} ({_e}) — "
                      "keeping v1 v-mask semantics (beta path unmasked)")
        if split:
            _zo, _za, W_z, _zp = sec["z"]
            phi_z, out_z = stash_z
            phi_z = phi_z.to(DEV).float()
            _Uzl = _res_z = None
            if SHADOW_RANK_LIN > 0:
                try:
                    _Uzl, _c, _res_z = fit_shadow_basis(out_z, SHADOW_RANK_LIN, device=DEV)
                except Exception as _e:
                    if is_oom(_e):
                        raise
                    print(f"  !! z shadow fit failed at {name} ({type(_e).__name__}) -> v9 path here")
                    _Uzl = _res_z = None
                if _Uzl is not None:
                    SHADOW_REPORT["dn_z"].append(_c)
                    z_re = _res_z.pow(2).mean(0)
            plan_zl = plan_row_modes(n_z, W_z.shape[1], LIN_BLOCKS,
                                     CARRIER_FRAC, row_energy=z_re)
            B_z = plan_zl["block_idx"].shape[0]
            zblk = plan_zl["block_idx"].reshape(-1)
            # v8.0.1 [F3b] OOM GUARD: gather only the routed z block
            # columns CPU-side first; the full [N, n_z] stash never lands
            # on the GPU.
            # v8.0.4 BUGFIX: zblk is a CUDA index (plan_row_modes ran on the GPU) and out_z a
            # CPU stash: `out_z[:, zblk]` raised "indices should be either on cpu or on the
            # same device as the indexed tensor". Index on the stash's own device.
            E_z = out_z.index_select(1, zblk.to(out_z.device)).to(DEV).float().pow(2)
            E_z = E_z.view(out_z.shape[0], B_z, -1).sum(-1)      # [T, B]
            rl_z = ResonantLinear(W_z, plan_zl, MAST, calib_feats=phi_z,
                                  calib_energy=E_z, topk=LIN_TOPK).to(DEV)
            if _Uzl is not None:
                rl_z.attach_shadow(_Uzl)
                with torch.no_grad():
                    rl_z.router.weight.copy_(ridge_router_init(
                        phi_z, _block_energy(_res_z, plan_zl["block_idx"])).to(rl_z.router.weight.dtype))
            rl_z.router_st = bool(ROUTER_ST)
            del _Uzl, _res_z
            rl_z.default_mode = "dense"
            _swap(_zo, _za, rl_z)       # stock in_proj_z interface + key kept
            RESONA_LINEARS.append((name + "." + _za, rl_z))
        if morm is not None:
            RESONA_MORM.append((name, morm))
        # v8.0.1 [F3c] OOM GUARD: per-site allocator hygiene for the DN loop
        # (the FFN loop clears every 8 sites; this loop never did — the
        # surgery transients of 48 fused sites no longer accumulate between
        # installs). no-op on CPU.
        torch.cuda.empty_cache()
    print(f"  sectioned {len(RESONA_SECTIONED)} fused in_proj sites "
          f"| MoRM active on {len(RESONA_MORM)}")
else:
    print("no surgery-capable DeltaNet sites (introspection GATES) — MoRM "
          "validated synthetically in the verification suite; the DeltaNet "
          "family keeps its exact dense cost in the waterfall (R5)")

def _sh_mean(v):
    return sum(v) / len(v) if v else float("nan")
if any(SHADOW_REPORT.values()):
    _f = SHADOW_REPORT["ffn"]
    print(f"v10 SHADOWS (held-out, per site): output energy captured by the rank-{SHADOW_RANK_LIN} shadow — "
          f"routed linears {_sh_mean(SHADOW_REPORT['lin']):.3f} ({len(SHADOW_REPORT['lin'])} sites) | "
          f"DeltaNet q {_sh_mean(SHADOW_REPORT['dn_q']):.3f} | v {_sh_mean(SHADOW_REPORT['dn_v']):.3f} | "
          f"z {_sh_mean(SHADOW_REPORT['dn_z']):.3f}")
    if _f:
        _kinds = {}
        for _r in _f:
            _kinds[_r["choice"]] = _kinds.get(_r["choice"], 0) + 1
        print(f"   FFN shadow ({len(_f)} layers): share of the served FFN error left after the shadow "
              f"{_sh_mean([r['rel_after'] for r in _f]):.3f} (1.000 = no gain; bias only "
              f"{_sh_mean([r['rel_bias_only'] for r in _f]):.3f}) | kept: {_kinds}")
    print("   (v9 zeroed all of this: 'captured' is energy that used to be pure error on every unselected row)")

# ---------------- 4) funnel registered (wired only in NATIVE serving) -------
RESONA_FUNNEL.append(("lm_head(funnel)", funnel))
free_cuda("surgery complete")   # release the per-site surgery transients

# ---------------- 5) THE ANCHOR TEST — Theorem 7 on the real model ----------
def measured_active(detail: bool = False):
    """v9: touched-weights CENSUS of the live module graph (one text token, serving semantics).
    A walk over every module (see census_active): embedding lookup, norms, the dense k/v/in_proj_a
    sites, DeltaNet conv taps of the selected channels, router reads, antenna-mast applications
    (per layer / per width) and the head funnel are all inside TOTAL_ACTIVE. detail=True -> family dict.
    (v8 summed three registries by hand and then added funnel + embedding on top.)"""
    c = census_active(model, HID, N_LAY, funnel=funnel, mast=MAST, mast_mode=MAST_ACCOUNTING)
    return c if detail else c["TOTAL_ACTIVE"]

print("\n=== THE ANCHOR TEST (Theorem 7): dense_teacher == original dense ===")
with torch.no_grad():
    set_world("dense_teacher")
    anchor = model(input_ids=PROBE_IDS.to(DEV)).logits.float()
    rel_anchor = float((anchor - ORIG_LOGITS).norm() / ORIG_LOGITS.norm())
print(f"rel deviation vs pre-surgery dense logits: {rel_anchor:.2e} "
      f"({'PASS — exact anchor' if rel_anchor < 1e-4 else 'FAIL'})")
assert rel_anchor < 1e-4, "Theorem 7 anchor violated — inspect surgery"

with torch.no_grad():
    set_world("exact_resonant")
    exact2 = model(input_ids=PROBE_IDS.to(DEV)).logits.float()
    rel_exact = float((exact2 - ORIG_LOGITS).norm() / ORIG_LOGITS.norm())
with torch.no_grad():
    set_world("serving")
    serv = model(input_ids=PROBE_IDS.to(DEV)).logits.float()
    rel_serv = float((serv - ORIG_LOGITS).norm() / ORIG_LOGITS.norm())
set_world("soft_train")
print(f"exact_resonant (Thm 7 resonant path): rel {rel_exact:.2e} "
      f"({'PASS' if rel_exact < 1e-4 else 'FAIL'})")
print(f"serving (all families sparse):        rel {rel_serv:.4f} "
      "(the certified deviation that Spectrum Consolidation closes)")
del anchor, exact2, serv   # release the probe-logits transients

CENSUS = measured_active(detail=True)
act = CENSUS["TOTAL_ACTIVE"]
print(f"\nmeasured active (census of the live module graph, '{MAST_ACCOUNTING}' sketch accounting): "
      f"{act/1e9:.3f}B  (budget waterfall {WF['TOTAL_ACTIVE']/1e9:.3f}B)")
for _k, _v in CENSUS.items():
    if _k != "TOTAL_ACTIVE" and (_v or WF.get(_k)):
        print(f"   {_k:<18}{_v:>16,}   waterfall {WF.get(_k, 0):>16,}")
if census_active.last["unclassified"]:
    print("   !! unclassified 2-D+ parameters (charged in full):",
          census_active.last["unclassified"][:6])
if RUN_MODE == "full" and TARGET_ACTIVE:
    if act > TARGET_ACTIVE:
        _msg = (f"CENSUS {act/1e9:.3f}B > {TARGET_ACTIVE/1e9:.2f}B: the graph that was built is larger than "
                f"the plan (families above). Fix the family that exceeds its waterfall value, or set "
                f"ENFORCE_BUDGET=False to continue with a measured, reported overshoot.")
        if ENFORCE_BUDGET:
            raise AssertionError(_msg)
        print("!! " + _msg)
    else:
        print(f"===> 3B MANDATE (census): {act/1e9:.3f}B <= {TARGET_ACTIVE/1e9:.2f}B <===")
print("surgery complete.")


## The free teacher — caching dense logits from *inside* the student

Because of **Corollary 1.1**, switching every PRISM layer to `uniform` mode turns the transformed model into **the original dense teacher exactly**, with zero extra memory. We stream it over the distillation corpus once, caching only the **top-64 logits** per position (ids + log-probs) to Google Drive — megabytes instead of terabytes of full-vocabulary logits.

We also record **before-training perplexities** here on a **disjoint held-out tail** of the corpus — the baseline every later number is judged against. (This cell runs *before* any quantization so the teacher is the pristine bf16 dense model.)


In [ ]:
# Teacher cache (the free in-place teacher) + before-training PPL
# v8.0.4 [DATA]: teach on (almost) the whole corpus. v8.0.2 cached only 6x the calibration
# blocks (384) and then trained 6400 micro-batches on them (~17 epochs: memorization).
# The held-out tail stays 1/16 of the teaching set and never overlaps it.
TEACH_BLOCKS_MAX = globals().get("TEACH_BLOCKS_MAX", 1024)
if RUN_MODE == "full":
    TEACH_BLOCKS_N = max(blocks.shape[0] * 6,
                         min(TEACH_BLOCKS_MAX, int((len(ids_all) // SEQ_LEN) / (1 + 1 / 16))))
else:
    TEACH_BLOCKS_N = 32
HOLD_N = max(2, TEACH_BLOCKS_N // 16)
need = TEACH_BLOCKS_N + HOLD_N
if len(ids_all) < need * SEQ_LEN:
    TEACH_BLOCKS_N = max(4, len(ids_all) // SEQ_LEN - HOLD_N)
    print(f"corpus smaller than requested; teaching on {TEACH_BLOCKS_N} blocks")
teach_blocks = torch.tensor([ids_all[i*SEQ_LEN:(i+1)*SEQ_LEN]
                             for i in range(TEACH_BLOCKS_N)])
tail0 = TEACH_BLOCKS_N * SEQ_LEN
heldout = torch.tensor([ids_all[(tail0 + j*SEQ_LEN):(tail0 + (j+1)*SEQ_LEN)]
                        for j in range(HOLD_N)])
if heldout.numel() == 0:      # degenerate corpora: fall back to calibration tail
    heldout = blocks[-2:].clone()
    print("WARNING: held-out overlaps calibration (tiny corpus)")

DEV = next(model.parameters()).device
V = model.config.text_config.vocab_size if hasattr(model.config, "text_config") \
    else model.config.vocab_size

def set_mode(mode):
    """Shim mapping v1 mode names onto the set_world presets."""
    if mode == "uniform":
        set_world("dense_teacher")          # == the original dense model (Thm 7)
    elif mode == "topk":
        set_world("serving")
    elif mode == "soft":
        set_world("soft_train")
    else:
        raise ValueError(mode)

@torch.no_grad()
def cache_teacher(shard_path):
    set_mode("uniform")                       # == the original dense model (Thm 1)

    def _run(bs):                             # bs = pure batching knob
        shards = []
        for i0 in range(0, teach_blocks.shape[0], bs):
            batch = teach_blocks[i0:i0+bs].to(DEV)
            logits = model(input_ids=batch).logits                # [B, L, V]
            B, L, _ = logits.shape
            ids_k = torch.empty(B, L, TEACHER_TOPK, dtype=torch.long)
            lp_k  = torch.empty(B, L, TEACHER_TOPK, dtype=torch.float16)
            for t0 in range(0, L, 512):                           # chunked fp32 softmax
                lg = logits[:, t0:t0+512].float()
                lsm = lg.log_softmax(-1)
                tv, ti = lsm.topk(TEACHER_TOPK, dim=-1)
                ids_k[:, t0:t0+512] = ti.cpu()
                lp_k[:, t0:t0+512]  = tv.cpu().half()
                del lg, lsm
            shards.append({"input_ids": batch.cpu(),
                           "teacher_ids": ids_k, "teacher_logprobs": lp_k})
            del logits
        return shards

    # v8.0.3: on OOM halve the micro-batch (identical output — bs only batches)
    shards = oom_backoff(_run, [2, 1] if RUN_MODE == "full" else [8, 4, 2, 1],
                         tag="teacher cache")
    torch.save(shards, shard_path)
    set_mode("soft")
    return shard_path

# v8.0.4: the cache file is keyed by WHAT it contains (model, sequence length, top-k and the
# exact token blocks). The old fixed name was silently reused from Drive even after the corpus
# changed, which would have trained on stale data while evaluating on new data.
import hashlib
_sig = hashlib.sha1(f"{MODEL_ID}|{SEQ_LEN}|{TEACHER_TOPK}|{TEACH_BLOCKS_N}".encode()
                    + teach_blocks.numpy().tobytes()).hexdigest()[:10]
TEACHER_SHARD = os.path.join(CACHE_DIR, f"teacher_top{TEACHER_TOPK}_{_sig}.pt")
print(f"teacher set: {TEACH_BLOCKS_N} blocks x {SEQ_LEN} tokens, held-out {HOLD_N} blocks "
      f"-> cache {os.path.basename(TEACHER_SHARD)}")
if os.path.exists(TEACHER_SHARD) and not os.environ.get("PRISM_RECACHE"):
    print("teacher cache found on Drive — reusing")
else:
    t0 = time.time(); cache_teacher(TEACHER_SHARD)
    print(f"teacher cached -> {TEACHER_SHARD} ({time.time()-t0:.0f}s)")

# --------------------- before-training perplexities -------------------------
@torch.no_grad()
def ppl(blocks_, mode):
    # Chunked NLL: CE is a per-token sum, so chunking over the sequence
    # axis is EXACT and bounds the fp32 transient to one [B, 256, V]
    # slice (~250 MiB).
    set_mode(mode)

    def _run(bs):                             # bs = pure batching knob
        nll, ntok = 0.0, 0
        for i0 in range(0, blocks_.shape[0], bs):
            b = blocks_[i0:i0+bs].to(DEV)
            out = model(input_ids=b).logits
            lg = out[:, :-1]; tg = b[:, 1:]
            V = lg.shape[-1]
            for c0 in range(0, lg.shape[1], 256):
                c1 = min(c0 + 256, lg.shape[1])
                lgc = lg[:, c0:c1].float()
                nll += F.cross_entropy(lgc.reshape(-1, V), tg[:, c0:c1].reshape(-1),
                                       reduction="sum").item()
                del lgc
            ntok += tg.numel()
            del out, lg
        return nll, ntok

    # v8.0.3: on OOM halve the micro-batch (identical output)
    nll, ntok = oom_backoff(_run, [2, 1] if RUN_MODE == "full" else [8, 4, 2, 1],
                            tag=f"ppl[{mode}]")
    set_mode("soft")
    return math.exp(nll / ntok)

PPL = {}
PPL["teacher_dense"]     = ppl(heldout, "uniform")
PPL["student_topk@init"] = ppl(heldout, "topk")
PPL["student_soft@init"] = ppl(heldout, "soft")     # v9.1: decides whether the soft phase is worth running
torch.save(PPL, os.path.join(CACHE_DIR, "ppl_before.pt"))
free_cuda("teacher cache + pre-training PPL")   # phase boundary
print({k: round(v, 3) for k, v in PPL.items()})
print("NOTE: all numbers share the same substrate. dense -> served top-k is the sparsity damage that training "
      "must close;\n      ln(PPL) is the cross-entropy: a served PPL in the thousands means the calibrated "
      "student starts far from the teacher\n      (expect a long recovery; raise STEPS / lower the sparsity).")


## Precision plan (torchao ≥ 0.16 — int8 is opt-in since v9)

The dense substrate never takes gradients, so it is a natural quantization target — *in principle*. On the surgered model, surgery adopts almost every weight as a raw `Parameter` inside the RESONA modules, and torchao only rewrites plain `nn.Linear` weights, so int8 reaches ≈ 0.1–0.2B of 27B (< 1 %). v8 paid for that with a crash (the MoRM critics shared by two parents were visited twice and an `Int8Tensor` was re-quantized) and with quantization of trainable critics and of the bit-exact BetaGate anchor.

| `PRECISION_PLAN` | Substrate during training | Notes |
|---|---|---|
| `"bf16"` (default) | bf16 | simplest, best fidelity, exact export |
| `"int8_frozen"` | int8 weight-only on **frozen plain bf16 `nn.Linear`s only** | id-deduplicated filter (each module offered once), protect list (lm_head, routers, critics / policy, vision, MTP, wrappers), failures fall back to bf16 |
| `"fp8_train"` (H100, sm90+) | torchao float8 training | speed win; off by default |

Routers, gains and ZSSI fields always stay bf16/fp32. `restore_precision()` puts the original bf16 tensors back **losslessly** (kept on the CPU) before export — v8's `dequantize_` shipped int8 round-trip noise.


In [ ]:
# Precision plan (v9) — bf16 by default; int8 / fp8 are guarded opt-ins
#
# WHY v8 CRASHED HERE  (torchao: "Int8Tensor dispatch ... aten.view ... arg_types=(Int8Tensor, list)")
#   After the surgery the MoRM policy is a child of TWO modules (SectionedProjector.morm and
#   BetaGate.policy). quantize_ walks named_children() recursively, so it met the policy's critic
#   nn.Linear twice: the first visit turned its weight into an Int8Tensor, the second one tried to
#   quantize that Int8Tensor again (Int8Tensor.from_hp -> weight.view(list)) -> NotImplementedError.
#   Re-running the cell fails the same way. The v8 filter also aimed at TRAINABLE critics and at the
#   BetaGate anchor weight that must stay bit-exact (Theorem 7).
# THE FIX
#   (1) a shared module is quantized at most once (id-dedup in the filter) and only if it is a plain
#       frozen bf16 tensor; (2) trainable / anchor-critical modules are protected by name AND by
#       requires_grad; (3) the original bf16 weights are kept on the CPU, so restore_precision()
#       (called by the export cell) is LOSSLESS — v8's dequantize_ exported int8 round-trip noise;
#       (4) the whole step is guarded: any failure restores bf16 and the run goes on.
# AND A NUMBER WORTH KNOWING: after the surgery almost every weight is an adopted raw Parameter,
#   not an nn.Linear, so int8 can only reach k_proj, the plain v_proj, in_proj_a ... — a fraction
#   of a percent of the 27B. It saves ~0.2 GiB and costs export noise: default OFF (bf16).
PRECISION_PLAN = globals().get("PRECISION_PLAN", "bf16")      # "bf16" | "int8_frozen" | "fp8_train"
assert PRECISION_PLAN in ("bf16", "int8_frozen", "fp8_train"), PRECISION_PLAN
USE_INT8_SUBSTRATE = False          # legacy flag (read by the export cell) — True only if int8 was applied
USE_FP8_TRAINING = False
_QUANTIZED = []                     # [(module, original bf16 weight on the CPU)]
_PROTECT = ("lm_head", "router", "visual", "vision", "merger", "mtp",
            "critic", "policy", "morm", ".inner")


def skip_predicate(mod, name):
    ln = name.lower()
    return any(tag in ln for tag in _PROTECT)       # routers / heads / BetaGate anchor / MTP / vision


def _eligible_linears():
    """Unique, frozen, plain-bf16 nn.Linear modules the int8 plan may touch."""
    out, seen = [], set()
    for fqn, m in model.named_modules():            # named_modules() yields a shared module ONCE
        if id(m) in seen or not isinstance(m, torch.nn.Linear):
            continue
        seen.add(id(m))
        w = m.weight
        if skip_predicate(m, fqn) or w.requires_grad:
            continue                                # trainable heads / protected modules
        if type(w.data) is not torch.Tensor or w.dtype != torch.bfloat16:
            continue                                # already quantized (cell re-run) / not bf16
        out.append((fqn, m))
    return out


def restore_precision():
    """Put the ORIGINAL bf16 weights back (lossless). Safe to call any number of times."""
    n = 0
    for mod, w0 in _QUANTIZED:
        mod.weight = torch.nn.Parameter(w0.to(mod.weight.device), requires_grad=False)
        n += 1
    _QUANTIZED.clear()
    if n:
        print(f"restored {n} bf16 weights (lossless)")
    return n


_ELIG = _eligible_linears()
_n_elig = sum(m.weight.numel() for _, m in _ELIG)
_n_all = sum(p.numel() for p in model.parameters())
print(f"precision plan: {PRECISION_PLAN} | int8-eligible frozen nn.Linear: {len(_ELIG)} modules, "
      f"{_n_elig/1e9:.3f}B of {_n_all/1e9:.2f}B parameters ({100*_n_elig/max(_n_all, 1):.2f}%) "
      f"-> would save ~{_n_elig/2**30:.2f} GiB")

if PRECISION_PLAN == "int8_frozen" and _ELIG:
    try:
        from torchao.quantization import quantize_
        try:
            from torchao.quantization import Int8WeightOnlyConfig as QCfg
        except ImportError:                          # <= 0.15 legacy API
            from torchao.quantization import int8_weight_only as QCfg
        _ids, _done = {id(m) for _, m in _ELIG}, set()

        def _filter(mod, fqn):
            k = id(mod)
            if k not in _ids or k in _done:          # a module reached via two parents counts once
                return False
            _done.add(k)
            return True

        for _, m in _ELIG:                           # keep the originals BEFORE quantize_ touches them
            _QUANTIZED.append((m, m.weight.detach().cpu().clone()))
        quantize_(model, QCfg(), filter_fn=_filter)
        USE_INT8_SUBSTRATE = True
        print(f"torchao int8 weight-only applied to {len(_done)} frozen Linear modules (each exactly once)")
    except Exception as _e:
        restore_precision()
        USE_INT8_SUBSTRATE = False
        print(f"int8 substrate failed ({type(_e).__name__}: {str(_e)[:160]}) -> bf16 restored, continuing")
elif PRECISION_PLAN == "bf16":
    print("substrate stays bf16 (default: exact export, nothing to quantize that matters)")

if PRECISION_PLAN == "fp8_train":
    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 9:
        try:
            from torchao.float8 import convert_to_float8_training

            def _fp8_filter(m, fqn):
                return (isinstance(m, torch.nn.Linear) and not skip_predicate(m, fqn)
                        and m.in_features % 16 == 0 and m.out_features % 16 == 0)
            try:
                convert_to_float8_training(model, module_filter_fn=_fp8_filter)
            except TypeError:                        # older torchao spelled it filter_fn
                convert_to_float8_training(model, filter_fn=_fp8_filter)
            USE_FP8_TRAINING = True
            print("FP8 training enabled (torchao.float8) on the remaining nn.Linear modules")
        except Exception as e:
            print("FP8 setup skipped:", e)
    else:
        print("fp8_train needs an sm90+ GPU (H100 / Blackwell) -> staying bf16")

x = heldout[0, :64].to(next(model.parameters()).device)
with torch.no_grad():
    set_mode("uniform"); _ = model(input_ids=x[None])
    set_mode("topk");    _ = model(input_ids=x[None])
    set_mode("soft")
del x, _                     # release the forward transients
free_cuda("post-precision forward")   # phase boundary
print("post-precision forward OK")


## Training — Spectrum Consolidation: Sinkhorn-OT self-distillation

The teacher is the student's `dense_teacher` world (Corollary 1.1 — zero extra memory); the student deforms continuously away from the Theorem-7 anchor toward the sparse serving semantics. The RESONA trainables share the same optimizer:

| Ingredient | Value (demo / full) | Role |
|---|---|---|
| CE loss | on raw text | keeps the LM head calibrated |
| **Partition-KL** (v9) | top-64 cached logits + tail-mass bit, λ=1.0 | transfers dense capability; unlike v8's renormalised KL it also matches the probability mass outside the teacher's support |
| Sinkhorn-OT balance | KL(Π\* \|\| p), λ=0.01 | projected load balancing for the FFN fields |
| Router z-loss | λ=1e-4 | logit stability |
| Zero-sum penalty | λ=0.05 | Theorem 2: antisymmetric growth around the dense anchor |
| **EMRD** (v9) | −log Σ pₑ·min(1, k·shareₑ), λ=0.05 | router distillation from the oracle energies — the only router signal when k = 1 |
| **LTD** (v9) | ‖y − y_dense‖² / ‖y_dense‖² per layer, λ=0.25, hard phase only | a dense local target for all 64 layers |
| Schedule | cosine, **warmup_steps** (never the deprecated warmup_ratio) | — |
| Annealing | last 25% of steps: ALL families on the hard serving path | train the exact operator deployment runs |
| Trainables | FFN routers/gains/ZSSI fields/τ **+ resonant-linear routers & block gains + section q/z routers & gains + MoRM write-policy heads + v-head gains** | < 0.5% of parameters; substrate frozen |
| Head funnel | frozen by construction (SVD) — recall measured in the calibration cell | serving-time only |

`LossPartsCallback` prints the loss components every 25 steps (CE / KL / balance / EMRD / LTD), so a term that misbehaves is visible immediately. MoRM's write-policy trains through the straight-through hard top-m mask: the dense-write teacher provides the signal for *which memory slabs deserved the write*. Soft-mode linears start exactly at the dense anchor (softmax×B weighting), so training is a continuous deformation — no quality cliff at any point of the trajectory.


In [ ]:
# Training — Sinkhorn-OT self-distillation (Spectrum Consolidation) + v9 Partition-KL / EMRD / LTD
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments, TrainerCallback

class PrismDistillSet(Dataset):
    """Per-SEQUENCE items: every stored sequence is a teaching sample —
    items are (shard i, row j) pairs over all stored batches."""
    def __init__(self, shards):
        self.data = shards
        self.index = [(i, j) for i, d in enumerate(shards)
                      for j in range(d["input_ids"].shape[0])]
    def __len__(self): return len(self.index)
    def __getitem__(self, k):
        i, j = self.index[k]
        d = self.data[i]
        return {"input_ids": d["input_ids"][j].long(),
                "teacher_ids": d["teacher_ids"][j].long(),
                "teacher_logprobs": d["teacher_logprobs"][j].float()}

def collate(feats):
    return {"input_ids": torch.stack([f["input_ids"] for f in feats]),
            "teacher_ids": torch.stack([f["teacher_ids"] for f in feats]),
            "teacher_logprobs": torch.stack([f["teacher_logprobs"] for f in feats])}

class _ChunkedHeadLoss(torch.autograd.Function):
    """The head GEMM itself, chunked (the Liger / Cut-Cross-Entropy design,
    cf. arXiv:2410.10989 / arXiv:2411.09009).

    DistillTrainer swaps lm_head for an Identity during the training
    forward, so the model returns HIDDEN states [B, L, d] (20 MiB), and
    THIS Function computes
        logits_chunk = hidden[:, c0:c1] @ W_head.t()   (bf16, [B*ch, V])
    chunk-by-chunk, cast to fp32 for the loss arithmetic; backward rebuilds
    the exact per-token gradient per chunk and contracts it back through
    W_head into a [B, L, d] hidden gradient — the exact bf16 GEMM the stock
    lm_head backward would have run. The full-vocab tensor NEVER exists, in
    forward OR backward (chunking the arithmetic of an already-materialized
    tensor cannot shrink what backward keeps). Peak transient: two
    [B, ch, V] fp32 chunks (~0.5 GiB at ch=256, B=1), genuinely governed by
    LOSS_CHUNK.

    Math is IDENTICAL per token: every logit is an independent dot product
    over d, so row-blocking the GEMM changes only cuBLAS tile selection
    (bf16 last-bit drift); the per-token gradient formula is unchanged:
      d/dlogits = g_nll*(softmax - onehot(label))
                + sum_k g_slog[k]*(onehot(tid_k) - softmax)"""
    @staticmethod
    def forward(ctx, hidden, W, labels, tid, chunk):
        B, L, d = hidden.shape
        V = W.shape[0]
        K = tid.shape[-1]
        nll = torch.empty(B, L, dtype=torch.float32, device=hidden.device)
        slog = torch.empty(B, L, K, dtype=torch.float32, device=hidden.device)
        with torch.no_grad():
            for c0 in range(0, L, chunk):
                c1 = min(c0 + chunk, L)
                hc = hidden[:, c0:c1].reshape(-1, d).to(W.dtype)   # [B*ch, d]
                lg = (hc @ W.t()).float()                           # [B*ch, V]
                lse = lg.logsumexp(-1)                              # [B*ch]
                lab = labels[:, c0:c1].reshape(-1)
                nll[:, c0:c1] = (lse - lg.gather(-1, lab[:, None])
                                 .squeeze(-1)).reshape(B, c1 - c0)
                tidc = tid[:, c0:c1].reshape(-1, K)
                slog[:, c0:c1] = (lg.gather(-1, tidc) - lse[:, None]) \
                    .reshape(B, c1 - c0, K)
                del hc, lg, lse, lab, tidc
        ctx.save_for_backward(hidden, W, labels, tid)
        ctx.chunk = int(chunk)
        return nll, slog

    @staticmethod
    def backward(ctx, g_nll, g_slog):
        hidden, W, labels, tid = ctx.saved_tensors
        B, L, d = hidden.shape
        K = tid.shape[-1]
        gh = torch.empty_like(hidden)          # [B, L, d] — 20 MiB, not [B, L, V]
        with torch.no_grad():
            for c0 in range(0, L, ctx.chunk):
                c1 = min(c0 + ctx.chunk, L)
                hc = hidden[:, c0:c1].reshape(-1, d).to(W.dtype)
                lg = (hc @ W.t()).float()                           # [B*ch, V]
                p = lg.softmax(-1)                                  # [B*ch, V]
                g = g_nll[:, c0:c1].reshape(-1, 1).to(p.dtype) * p
                lab = labels[:, c0:c1].reshape(-1)
                g.scatter_add_(-1, lab[:, None],
                               (-g_nll[:, c0:c1].reshape(-1)).to(p.dtype)[:, None])
                if g_slog is not None:
                    tidc = tid[:, c0:c1].reshape(-1, K)
                    gs = g_slog[:, c0:c1].reshape(-1, K).to(p.dtype)
                    g.scatter_add_(-1, tidc, gs)
                    g -= gs.sum(-1, keepdim=True) * p
                gh[:, c0:c1] = (g.to(W.dtype) @ W).reshape(B, c1 - c0, d) \
                    .to(hidden.dtype)
                del hc, lg, p, g
        return gh, None, None, None, None


_IDENTITY = torch.nn.Identity()
_PHASE = {"hard": False}      # v9: True once AnnealCallback switched every family to the serving path
_LP = {"n": 0, "sum": {}}     # v9: running sums of every loss term (detached, on the GPU; printed by LossPartsCallback)


def _legacy_head_loss(logits, labels, tid):
    """Fallback for wrappers without a swappable lm_head attribute (and
    the return_outputs/eval shape): chunked arithmetic on the materialized
    logits, as plain autograd — mathematically identical to
    _ChunkedHeadLoss; only the memory profile differs (full-vocab tensors
    exist). The stock Qwen3.8 wrapper always takes the headless path."""
    lg = logits.float()
    lse = lg.logsumexp(-1)
    nll = lse - lg.gather(-1, labels[..., None]).squeeze(-1)
    slog = lg.gather(-1, tid) - lse[..., None]
    return nll, slog


class DistillTrainer(Trainer):
    def _move_model_to_device(self, model, device):
        """No blanket model.to(device): the substrate is already resident on
        `device`; the pre-training CPU-parking of the text-inert subtrees
        (vision tower / MTP) must stand — they are never executed by the
        text-only distillation."""
        return

    # ---- v8.0.3: per-micro-batch OOM shield --------------------------------
    # One CUDA OOM inside ONE micro-batch used to abort the optimizer step (and,
    # after a single recovery, the run). Now that micro-batch is retried on a
    # causal PREFIX (1/2, then 1/4 of the tokens — every layer of this model is
    # causal, so a prefix is a valid, shorter training sample) and only skipped
    # if even that does not fit. Every OOM drops the half-built gradient
    # accumulation window (partial backward grads would be biased toward the
    # last layers).
    _oom_skips = 0           # consecutive micro-batches lost to OOM
    _oom_shrinks = 0         # micro-batches that only fit as a shorter prefix

    @staticmethod
    def _prefix(inputs, frac):
        L = inputs["input_ids"].shape[1]
        n = max(64, int(L * frac))
        out = {}
        for k, v in inputs.items():
            if torch.is_tensor(v) and v.dim() >= 2 and v.shape[1] == L:
                out[k] = v[:, :n]
            else:
                out[k] = v
        return out

    def training_step(self, model, inputs, *args, **kwargs):
        for frac in (1.0, 0.5, 0.25):
            cur = inputs if frac == 1.0 else self._prefix(inputs, frac)
            failed = False
            try:
                loss = super().training_step(model, cur, *args, **kwargs)
            except RuntimeError as e:
                if not is_oom(e):
                    raise
                failed = True
            if not failed:
                DistillTrainer._oom_skips = 0
                if frac < 1.0:
                    DistillTrainer._oom_shrinks += 1
                    print(f"[oom-shield] micro-batch trained on a {frac:g}-length prefix "
                          f"({DistillTrainer._oom_shrinks} so far)")
                return loss
            # ---- outside the except block: the dead graph is releasable ----
            model.zero_grad(set_to_none=True)
            for _m in (m for _, m in PRISM_MODULES):
                _m.last_aux = None
            free_cuda(f"OOM in training_step (length x{frac:g})")
        DistillTrainer._oom_skips += 1
        print(f"[oom-shield] micro-batch skipped: OOM even at 1/4 length "
              f"({DistillTrainer._oom_skips} in a row)")
        if DistillTrainer._oom_skips >= 4:
            raise torch.cuda.OutOfMemoryError(
                "training_step: 4 consecutive micro-batches OOM even at 1/4 length "
                "- see the [mem ...] lines above")
        return torch.zeros((), device=self.args.device)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Headless forward: lm_head is swapped for an Identity so out.logits
        # IS the hidden state [B, L, d] (20 MiB); the vocab GEMM happens
        # per-chunk inside _ChunkedHeadLoss (exception-safe swap, restored
        # in `finally`).
        #
        # Telemetry contract (the +24.2 GiB OOM class, fixed): the aux
        # side-channel is switched ON only for THIS forward; every value is
        # consumed here and the attributes cleared before returning — the
        # checkpoint recomputes that run during backward store NOTHING, so
        # each recomputed layer graph dies with its own backward.
        _prism_mods = [m for _, m in PRISM_MODULES]
        _hard = bool(_PHASE["hard"])
        for _m in _prism_mods:
            _m._store_aux = True
            # v9 side channels: detached oracle energies / dense twin, computed under no_grad inside the
            # module (so the checkpoint recompute during backward stays consistent whatever these flags are)
            _m.side_en = LAMBDA_EMRD > 0
            _m.side_ltd = (LAMBDA_LTD > 0) and _hard
        _lm = getattr(model, "lm_head", None)
        _headless = (not return_outputs) and (_lm is not None) \
            and (getattr(_lm, "weight", None) is not None)
        if _headless:
            model.lm_head = _IDENTITY
        try:
            out = model(input_ids=inputs["input_ids"])
            labels = inputs["input_ids"][:, 1:]
            tid = inputs["teacher_ids"][:, :-1]      # align with hidden[:, :-1]
            tlp = inputs["teacher_logprobs"][:, :-1]
            if _headless:
                hidden = out.logits[:, :-1]          # [B, L-1, d] VIEW (20 MiB)
                nll_tok, slog = _ChunkedHeadLoss.apply(
                    hidden, _lm.weight, labels, tid, LOSS_CHUNK)
            else:
                nll_tok, slog = _legacy_head_loss(out.logits[:, :-1],
                                                  labels, tid)
            ce = nll_tok.mean()
            if KL_MODE == "partition":
                # v9 PARTITION-KL: KL of the coarse-grained distributions on {each teacher top-K id} U
                # {everything else}. Chain rule: KL_part = P_S * KL_support + KL_Bernoulli(P_S || Q_S), so
                # (unlike the v8 KL renormalised inside the support) it also sees the probability MASS the
                # student keeps on the teacher's support; still a lower bound of the full-vocab KL.
                kl = partition_kl(tlp, slog).mean()
            else:                                   # v8: both sides renormalised inside the support
                kl = kl_support(tlp, slog).mean()
            bal = z = zs = emrd = ltd = nll_tok.new_zeros((), dtype=torch.float32)
            n = n_en = n_ltd = 0
            for m in _prism_mods:
                aux = getattr(m, "last_aux", None)
                m.last_aux = None          # consume-and-clear
                if aux is not None and "p" in aux:
                    p, lg = aux["p"], aux["logits"]
                    Pplan = sinkhorn_balanced(lg.detach().float(), n_iter=3)
                    bal = bal + (Pplan * (Pplan.clamp_min(1e-9).log()
                                          - p.float().clamp_min(1e-9).log())).sum(-1).mean()
                    z = z + torch.logsumexp(lg.float(), -1).pow(2).mean()
                    zs = zs + zero_sum_penalty(m.alpha) + zero_sum_penalty(m.beta)
                    n += 1
                    if "en" in aux:
                        # v9 EMRD: -log of the router mass on experts that own the oracle output energy.
                        # With k=1 the served gate renormalises to 1, so the output loss sends the router's
                        # CHOICE no gradient at all — this term is its supervision.
                        emrd = emrd + emrd_loss(p, aux["en"], K_CHOSEN)
                        n_en += 1
                    if "y_dense" in aux and "y" in aux:
                        # v9 LTD (hard phase): ||y - y_dense||^2 / ||y_dense||^2 against this layer's own
                        # dense twin on the same input — a dense local target for all 64 FFNs.
                        ltd = ltd + ltd_loss(aux["y"], aux["y_dense"])
                        n_ltd += 1
            # v9.1: LAMBDA_CE = 0 by default -> PURE TEACHER DISTILLATION. The corpus only supplies the inputs;
            # every target is the dense Qwen3.8 itself (its next-token distribution here, its own FFN outputs in
            # LTD). The hard-label CE (the corpus' next token) is still computed and logged, but not trained on:
            # fitting a small corpus' text would pull a strong model towards that text.
            _aux_reg = (LAMBDA_BAL * bal + LAMBDA_Z * z + LAMBDA_ZS * zs) / max(n, 1)
            loss = LAMBDA_CE * ce + LAMBDA_KL * kl + _aux_reg
            if n_en:
                loss = loss + LAMBDA_EMRD * emrd / n_en
            if n_ltd:
                loss = loss + LAMBDA_LTD * ltd / n_ltd
            for _k, _v in (("ce", ce), ("kl", kl), ("bal", bal / max(n, 1)), ("z", z / max(n, 1)),
                           ("emrd", emrd / max(n_en, 1)), ("ltd", ltd / max(n_ltd, 1)),
                           ("reg(w)", _aux_reg)):
                _LP["sum"][_k] = _LP["sum"].get(_k, 0.0) + _v.detach().float()
            _LP["n"] += 1
            # v8 (MoRM 2.0): critic regression on the EXACT influence labels
            # (log-space MSE against the calibration WIK stash; the distill
            # STE gradient flows separately through the mask sites).
            if MORM2_LAMBDA_CRITIC > 0:
                crit, nc = 0.0, 0
                for _nm, _m in RESONA_MORM:
                    fit = getattr(_m, "_crit_fit", None)
                    if not isinstance(_m, MoRM2Policy) or fit is None:
                        continue
                    phi_c, lab_c = fit
                    idx = torch.randint(0, phi_c.shape[0],
                                        (min(256, phi_c.shape[0]),),
                                        device=phi_c.device)
                    Lh = _m.scores(phi_c[idx]).clamp_min(1e-12).log()
                    crit = crit + (Lh - lab_c[idx].clamp_min(1e-12).log()
                                   ).pow(2).mean()
                    nc += 1
                if nc:
                    loss = loss + MORM2_LAMBDA_CRITIC * crit / nc
                    _LP["sum"]["crit"] = _LP["sum"].get("crit", 0.0) + (crit / nc).detach().float()
            _LP["sum"]["total"] = _LP["sum"].get("total", 0.0) + loss.detach().float()
            return (loss, out) if return_outputs else loss
        finally:
            for _m in _prism_mods:
                _m._store_aux = False
                _m.side_en = False
                _m.side_ltd = False
            if _headless:
                model.lm_head = _lm


class MoRM2Callback(TrainerCallback):
    """v8 elastic-budget governance for MoRM 2.0 policies: anneals m_bar
    from exact (all writes — pure distillation, the critic learns from the
    label regression) down to the solver's target by mid-anneal, and feeds
    each policy's rate controller the observed mean write count of the last
    training forward (Theorem D in the loop)."""
    def on_step_end(self, args, state, control, **kw):
        if not RESONA_MORM or not isinstance(RESONA_MORM[0][1], MoRM2Policy):
            return
        frac = min(1.0, _abs_step(state)
                   / max(1, 0.5 * ANNEAL_FRAC * STEPS))      # v9: absolute step (resume-safe)
        for _nm, m in RESONA_MORM:
            if not isinstance(m, MoRM2Policy):
                continue
            tgt = max(1, int(round(m.H + (min(M_WRITE, m.H) - m.H) * frac)))
            m.set_budget(tgt, m_max=(max(tgt, int(m.m_max)) if m.m_max is not None else None))
            if m.last_m is not None:
                m.controller_update(float(m.last_m.float().mean()))


class AnnealCallback(TrainerCallback):
    """Last ANNEAL_FRAC of training: hard serving path for ALL families
    (FFN top-k + resonant top-k linears + MoRM write-sparsity). The switch
    reads the ABSOLUTE optimizer step from the LR scheduler (resume-safe:
    a per-call TrainerState counter restarts at 0 on every resumed
    trainer.train())."""
    def __init__(self): self.switched = False
    def on_step_end(self, args, state, control, **kw):
        sched = getattr(self, "sched", None)
        abs_step = int(sched.last_epoch) if sched is not None else state.global_step
        if not self.switched and abs_step >= ANNEAL_FRAC * STEPS:
            set_world("serving")
            _PHASE["hard"] = True        # v9: LTD (layer-local dense twin) switches on with the hard path
            self.switched = True
            print(f"\n[anneal] absolute step {abs_step}: switched ALL families "
                  "to the hard serving path")

_CKPT_EXEC = [0, 0]   # (checkpointed, plain) decoder layer-calls — a flag
#                      can be present while nothing executes; an execution
#                      counter cannot lie. Printed by the [vram] pulse.


class VRAMPulse(TrainerCallback):
    """Observe, don't govern: with the telemetry leak fixed the heap
    simply returns to the baseline between steps — this pulse proves it
    every 25 steps (alloc/peak/free + the checkpoint execution counter),
    and the leak canary (>+5 GiB over baseline at a step boundary, well
    above the optimizer's ~1.2 GiB band) turns any future pin into a
    one-line diagnosis."""

    def __init__(self, base_gib: float):
        self.base = float(base_gib)
        self.tick = 0

    def on_step_end(self, args, state, control, **kw):
        self.tick += 1
        if not torch.cuda.is_available() or self.tick % 25 != 0:
            return
        alloc = torch.cuda.memory_allocated() / 2**30
        peak = torch.cuda.max_memory_allocated() / 2**30
        free = torch.cuda.mem_get_info()[0] / 2**30
        print(f"[vram step {_abs_step(state)}] alloc {alloc:.1f} GiB | "
              f"peak {peak:.1f} GiB | free {free:.1f} GiB | ckpt-exec "
              f"{_CKPT_EXEC[0]}")
        if alloc - self.base > 5.0:
            print(f"[vram step {_abs_step(state)}] WARNING: "
                  f"{alloc - self.base:+.1f} GiB over baseline at a step "
                  "boundary — something is pinning a step graph")


def _abs_step(state):
    """ABSOLUTE optimizer step. TrainerState.global_step restarts at 0 on every trainer.train() — i.e. after each
    OOM recovery — while the LR scheduler (restored from the light checkpoint) carries the true step. Everything
    schedule-like (MoRM anneal, light checkpoints, logs) must read this, not state.global_step."""
    sched = globals().get("scheduler")
    return int(sched.last_epoch) if sched is not None else int(state.global_step)


class LossPartsCallback(TrainerCallback):
    """v9: running mean of every loss term, printed every `every` optimizer steps — one host sync per print.
    ce/kl/bal/z/emrd/ltd/crit are RAW (unweighted) values, reg(w) is the weighted bal+z+zs sum, and
    total is what the Trainer shows as 'Training Loss'. ce is logged even when LAMBDA_CE = 0."""
    every = 5      # same cadence as the Trainer's 'Training Loss' column

    def on_step_end(self, args, state, control, **kw):
        st = _abs_step(state)
        if _LP["n"] and st % self.every == 0:
            parts = {k: float(v) / _LP["n"] for k, v in _LP["sum"].items()}
            print(f"[loss parts step {st}] "
                  + " | ".join(f"{k} {v:.4f}" for k, v in parts.items())
                  + f" | {'hard' if _PHASE['hard'] else 'soft'} path")
            _LP["n"], _LP["sum"] = 0, {}


train_ds = PrismDistillSet(torch.load(TEACHER_SHARD, weights_only=False))

# frozen-substrate leak check: if the dense substrate ever became
# trainable, AdamW fp32 states alone (27B x 12 B/param) would demand ~300
# GiB and OOM at optimizer construction. Fail with a diagnosis instead.
_n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert _n_trainable < 500_000_000, (
    f"{_n_trainable/1e6:.0f}M trainable params — the frozen substrate leaked "
    f"into the optimizer (AdamW fp32 states ≈ {_n_trainable*12/2**30:.0f} GiB). "
    "Check the surgery cells' requires_grad wiring.")
print(f"trainable parameters: {_n_trainable/1e6:.1f}M "
      f"(AdamW fp32 states ≈ {_n_trainable*12/2**30:.2f} GiB)")

# ============================================================================
# ============================================================================
# Pre-training baseline reclaim: free every pre-training resident, then
# name what is left.
#   (a) probe/recall/surgery leftovers bound in the notebook namespace,
#   (b) per-module telemetry caches pinning the LAST no-grad forward,
#   (c) calibration dicts that stayed CUDA-resident,
#   (d) stale teacher-cache tensors restored onto CUDA by torch.load —
#       forced back to host RAM,
#   (e) text-inert subtrees (vision tower / merger / MTP) CPU-parked for
#       the whole train() — restored right after training.
# ============================================================================
# ============================================================================
def _reclaim():
    for _name in ("anchor", "exact2", "serv", "out", "h_last", "ref",
                  "got", "sl", "b", "x",
                  # surgery-cell locals that pin the last calibration
                  # forward's tensors (e.g. the y_e [T, E, d] stacks)
                  "hs", "asub", "e_routed", "energy", "logits0", "topk_idx",
                  "y_e", "y_dense_sub", "Wr", "tau", "fus", "c",
                  "cal", "imp_l", "S_l"):
        _g = globals()
        if _name in _g:
            del _g[_name]
    # defensively resolve the stash dicts (missing names skip)
    for _dname in ("sketch", "stats", "mean", "std", "duty", "sqmean", "row_energy",
                   "dn_row_energy", "h_sub", "site_stash", "assign_store"):
        _d = globals().get(_dname)
        if not isinstance(_d, dict):
            continue
        for _k in list(_d):
            _t = _d[_k]
            if torch.is_tensor(_t) and _t.is_cuda:
                _d[_k] = _t.cpu()
    for _shard in train_ds.data:
        for _k in list(_shard):
            _t = _shard[_k]
            if torch.is_tensor(_t) and _t.is_cuda:
                _shard[_k] = _t.cpu()
    for _nm, _m in PRISM_MODULES + RESONA_LINEARS + RESONA_SECTIONED \
            + RESONA_MORM:
        for _attr in ("last_out", "last_mask", "last_aux", "last_cert"):
            if getattr(_m, _attr, None) is not None:
                setattr(_m, _attr, None)
_reclaim(); del _reclaim
gc.collect()

def cuda_census(tag: str = "", top: int = 15) -> float:
    """Name every live CUDA tensor group: aggregates bytes by (shape,
    dtype) over the GC heap (shared-storage views are counted once per
    wrapper, so the total is an UPPER bound — good enough to name the
    holder of any multi-GiB block). If a phase OOMs, call cuda_census()
    there and the culprit is NAMED, not guessed."""
    groups = {}
    for obj in gc.get_objects():
        try:
            if torch.is_tensor(obj) and obj.is_cuda:
                key = (tuple(obj.shape), str(obj.dtype))
                groups[key] = groups.get(key, 0) \
                    + obj.numel() * obj.element_size()
        except Exception:
            pass
    total = float(sum(groups.values()))
    print(f"[census{(' ' + tag) if tag else ''}] {total / 2**30:.2f} GiB live "
          f"in {len(groups)} groups (upper bound; views double-counted)")
    for (_shape, _dt), _nb in sorted(groups.items(),
                                     key=lambda kv: -kv[1])[:top]:
        print(f"    {str(_shape):>30s}  {_dt:>15s}  {_nb / 2**30:8.3f} GiB")
    return total

_PARK_PAT = re.compile(r"(visual|vision|merger|multi_modal|mm_|mtp)", re.I)
def _park_inert(model) -> list:
    """Top-2-level subtrees the TEXT-ONLY distillation never executes.
    Parking moves storage only — weights are untouched, so state_dict(),
    the export, and the Theorem-7 anchors are all unaffected. Restored by
    the epilogue at the end of this cell."""
    parked = []
    for _name, _mod in list(model.named_children()):
        if _PARK_PAT.search(_name):
            parked.append((_name, _mod))
            continue
        for _c_name, _c_mod in list(_mod.named_children()):
            if _PARK_PAT.search(_c_name):
                parked.append((f"{_name}.{_c_name}", _c_mod))
    return parked
_PARKED = _park_inert(model)
del _park_inert
_before = torch.cuda.mem_get_info()[0] if torch.cuda.is_available() else 0
for _name, _mod in _PARKED:
    _mod.to("cpu")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
_after = torch.cuda.mem_get_info()[0] if torch.cuda.is_available() else 0
print(f"{__PRISM_VER__} parked {len(_PARKED)} text-inert subtree(s) "
      f"{[_n for _n, _ in _PARKED]} -> freed "
      f"{(_after - _before) / 2**30:.2f} GiB (restored after training)")
del _before, _after


# leak-canary baseline: the heap the training phase starts from
# (post-parking); the optimizer adds its ~1.2 GiB at the first step and
# stays — the [vram] canary tolerates that band.
_base_gib = (torch.cuda.memory_allocated() / 2**30
             if torch.cuda.is_available() else 0.0)

groups = [
    {"params": [p for _, m in PRISM_MODULES for p in (m.U, m.P)], "lr": LR_FIELDS},
    {"params": [p for _, m in PRISM_MODULES for p in (m.alpha, m.beta,
                                                      m.alpha_shared, m.beta_shared)],
     "lr": LR_FIELDS},
    {"params": [m.router.weight for _, m in PRISM_MODULES], "lr": LR_ROUTER},
    {"params": [m.gains for _, m in PRISM_MODULES], "lr": LR_ROUTER * 0.1},
    {"params": [m.log_tau for _, m in PRISM_MODULES], "lr": LR_TAU},
    # RESONA trainables: resonant-linear routers/gains, section q/z
    # routers/gains, MoRM write-policy heads, v-head gains
    {"params": [m.router.weight for _, m in RESONA_LINEARS], "lr": LR_ROUTER},
    {"params": [m.gains for _, m in RESONA_LINEARS], "lr": LR_ROUTER * 0.1},
    {"params": [getattr(m, f"{t}_router").weight
                for _, m in RESONA_SECTIONED for t in ("q", "z")
                if hasattr(m, f"{t}_B")], "lr": LR_ROUTER},
    {"params": [getattr(m, f"{t}_gains")
                for _, m in RESONA_SECTIONED for t in ("q", "z")
                if hasattr(m, f"{t}_B")], "lr": LR_ROUTER * 0.1},
    {"params": [m.v_gains for _, m in RESONA_SECTIONED], "lr": LR_ROUTER * 0.1},
    {"params": [p for _, m in RESONA_MORM
                for p in (([m.critic_phi.weight, m.critic_obs.weight]
                           + ([m.critic_phi.bias] if m.critic_phi.bias is not None else []))
                          if isinstance(m, MoRM2Policy)
                          else [m.router.weight])], "lr": LR_ROUTER},
]
args = TrainingArguments(
    output_dir=os.path.join(DRIVE_DIR, "ckpt"),
    max_steps=STEPS,
    warmup_steps=WARMUP_STEPS,          # warmup_ratio is DEPRECATED -> warmup_steps
    lr_scheduler_type="cosine",
    bf16=(torch.cuda.is_available()),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    logging_steps=5, save_strategy="no", report_to=[],
    remove_unused_columns=False, optim="adamw_torch",
    weight_decay=0.0, max_grad_norm=1.0, seed=0,
)
model.config.use_cache = False
# also kill KV-cache building on the TEXT config (VL wrappers hide the
# decoder's flag one level down — a silent per-step activation tax).
for _cfg in (model.config, getattr(model.config, "text_config", None)):
    if _cfg is not None:
        _cfg.use_cache = False

from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

# 8-bit PAGED optimizer: blockwise 8-bit states shrink the fp32 twin
# states (1.18 GiB -> ~0.3 GiB) and PAGED states spill to host RAM under
# allocator pressure instead of OOMing. Fallback chain (no bitsandbytes:
# plain AdamW, foreach OFF — no transient per-group tensor lists). The
# training signal, per-group LRs and the cosine schedule are UNCHANGED.
_OPT_GROUPS = [{"params": g["params"], "lr": g["lr"]} for g in groups]
try:
    import bitsandbytes as _bnb
    _bnb_err = ""
except Exception as _e:
    _bnb, _bnb_err = None, str(_e)
if _bnb is not None and hasattr(_bnb.optim, "PagedAdamW8bit"):
    optimizer = _bnb.optim.PagedAdamW8bit(_OPT_GROUPS, weight_decay=0.0)
    _opt_name = "PagedAdamW8bit (bitsandbytes)"
elif _bnb is not None and hasattr(_bnb.optim, "AdamW8bit"):
    optimizer = _bnb.optim.AdamW8bit(_OPT_GROUPS, weight_decay=0.0)
    _opt_name = "AdamW8bit (bitsandbytes)"
else:
    optimizer = AdamW(_OPT_GROUPS, weight_decay=0.0, foreach=False)
    _opt_name = "AdamW (torch, foreach=False)" + \
        (f" [bnb unavailable: {_bnb_err}]" if _bnb is None else "")
print(f"[{__PRISM_VER__}] optimizer: {_opt_name}")
scheduler = get_cosine_schedule_with_warmup(optimizer,
                                            num_warmup_steps=WARMUP_STEPS,
                                            num_training_steps=STEPS)

# ============================================================================
# the v5.x memory-governance machinery (65 GiB governor target,
# allocator guard cap, dry-run probe, 3-level self-heal ladder,
# census-as-retry loop) is DELETED — the RESONA-core telemetry fix removed
# the actual leak, and those knobs never touched it.
# ============================================================================

VER = __PRISM_VER__
LIGHT_CKPT_EVERY = 25
# version-tagged: a state saved under a different memory plan must never
# resume into this one
LIGHT_CKPT = os.path.join(CACHE_DIR, "v70_train_state.pt")
# v8.0.4: this file exists ONLY for the in-run OOM recovery below. One left on Drive by an
# EARLIER run would be "recovered" as if it were this run's state (old weights + old step).
if os.path.exists(LIGHT_CKPT):
    os.replace(LIGHT_CKPT, LIGHT_CKPT + ".prev")
    print("[light-ckpt] checkpoint from an earlier run moved to .prev — this run starts clean")

def _save_light(step, mdl, opt, sched):
    tmp = LIGHT_CKPT + ".tmp"
    torch.save({"step": int(step),
                "trainables": {n: p.detach().cpu()
                               for n, p in mdl.named_parameters()
                               if p.requires_grad},
                "opt": opt.state_dict(), "sched": sched.state_dict()}, tmp)
    os.replace(tmp, LIGHT_CKPT)               # atomic — never a torn file
    print(f"[light-ckpt] step {int(step)} saved "
          f"({os.path.getsize(LIGHT_CKPT) / 2**20:.0f} MiB)")

def _load_light(mdl, opt, sched):
    ck = torch.load(LIGHT_CKPT, map_location="cpu", weights_only=False)
    with torch.no_grad():
        for n, p in mdl.named_parameters():
            if p.requires_grad and n in ck["trainables"]:
                p.copy_(ck["trainables"][n].to(p.device, p.dtype))
    opt.load_state_dict(ck["opt"])
    sched.load_state_dict(ck["sched"])
    return int(ck["step"])

class _LightCkptCallback(TrainerCallback):
    """L2 — resumable state every LIGHT_CKPT_EVERY global steps."""
    def __init__(self):
        self._last = -1
    def on_step_end(self, args, state, control, **kw):
        st = _abs_step(state)       # v9: ABSOLUTE step — a checkpoint written after a recovery must say so
        if st % LIGHT_CKPT_EVERY == 0 and st > self._last and st < STEPS:
            self._last = st
            try:
                _save_light(st, model, optimizer, scheduler)
            except Exception as e:            # NEVER kill training over a save
                print(f"[light-ckpt] save failed (non-fatal): {e}")

def _build_trainer(bs, accum):
    args.per_device_train_batch_size = bs
    args.gradient_accumulation_steps = accum
    _anneal = AnnealCallback()
    _trainer = DistillTrainer(model=model, args=args, train_dataset=train_ds,
                              data_collator=collate,
                              callbacks=[_anneal, MoRM2Callback(),
                                         VRAMPulse(_base_gib),
                                         _LightCkptCallback(),
                                         LossPartsCallback()],
                              optimizers=(optimizer, scheduler))
    _anneal.sched = scheduler               # absolute-step anneal (resume-safe)
    return _trainer

# NO allocator cap: expandable_segments is already active (the setup cell
# set PYTORCH_CUDA_ALLOC_CONF before torch ever touched the GPU) and the
# lean step footprint leaves >20 GiB of true headroom.

# ---------------- engage + VERIFY HF layer checkpointing ---------------------
# The flag alone is not trusted: it is engaged here AND counted; if it
# reaches no module, every RESONA module still self-checkpoints and the
# at-OOM census names the rest.
try:
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False})
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()   # frozen embeddings: keep the
        # checkpoint graph differentiable from token 0 (also lets layer-1
        # self-checkpointing engage immediately)
except Exception as _gc_err:
    print(f"[{VER}] WARNING: gradient_checkpointing_enable failed "
          f"({_gc_err}) — module self-checkpointing (L1) is the bound")
_n_gc = sum(1 for _m in model.modules()
            if getattr(_m, "_gradient_checkpointing_func", None) is not None)
print(f"[{VER}] HF gradient checkpointing engaged on {_n_gc} module(s)")
if _n_gc == 0:
    print(f"[{VER}] NOTE: HF checkpointing reached no module — every RESONA "
          "module still self-checkpoints (L1); the stock GDN/full-attn "
          "cores are the residual consumers named by the at-OOM census")

# TRUST EXECUTION, NOT FLAGS: a present HF checkpointing flag proves
# nothing about the call path (the wrap decision lives in the container's
# forward loop, which the VL/PRISM surgery does not have to execute the way
# the flag walk assumed). Every decoder layer forward is therefore wrapped
# UNCONDITIONALLY with torch.utils.checkpoint(use_reentrant=False) — the
# per-layer saved set is the [B, L, d] input BY CONSTRUCTION — and the wrap
# counts its own EXECUTIONS (_CKPT_EXEC, printed by the [vram] pulse; zero
# executions under grad would be a loud stop, never a silent bare run).
def _text_tower(m):
    """Locate the text decoder: the largest .layers ModuleList not living
    under a parked inert subtree (visual/vision/merger/mtp). VL wrapper:
    model.model.language_model; plain LM: model.model. Returns (path, mod)
    or (None, None)."""
    _best = (0, None, None)
    for _p, _mod in m.named_modules():
        _ls = getattr(_mod, "layers", None)
        if (isinstance(_ls, torch.nn.ModuleList) and len(_ls) > _best[0]
                and len(_ls) >= 8 and not _PARK_PAT.search(_p)):
            _best = (len(_ls), _p, _mod)
    return _best[1], _best[2]

_tt_path, _tt = _text_tower(model)
_FORCED_CKPT = []          # (layer, original forward) — restored post-training
if _tt is not None:
    _tt_layers = list(_tt.layers)
    _tt_flagged = sum(
        1 for _ly in _tt_layers
        if getattr(_ly, "_gradient_checkpointing_func", None) is not None
        and getattr(_ly, "gradient_checkpointing", False))
    print(f"[{VER}] text-decoder checkpoint audit ({_tt_path}): "
          f"{_tt_flagged}/{len(_tt_layers)} layers carry the HF flag "
          "(informational only — the wrap below runs regardless)")
    from torch.utils.checkpoint import checkpoint as _ck_fn
    for _ly in _tt_layers:
        _orig_fwd = _ly.forward
        def _ckpt_fwd(*a, __ly=_ly, __f=_orig_fwd, **kw):
            if __ly.training and torch.is_grad_enabled():
                _CKPT_EXEC[0] += 1
                return _ck_fn(__f, *a, use_reentrant=False, **kw)
            _CKPT_EXEC[1] += 1
            return __f(*a, **kw)
        _ly.forward = _ckpt_fwd
        _FORCED_CKPT.append((_ly, _orig_fwd))
    print(f"[{VER}] #30 FORCE-wrapped {len(_FORCED_CKPT)} decoder layer "
          "forwards UNCONDITIONALLY (torch.utils.checkpoint, "
          "use_reentrant=False) — per-layer saved set is the [B, L, d] "
          "input BY CONSTRUCTION; values unchanged (the suite's "
          "self-checkpoint contract, RNG preserved); _CKPT_EXEC counts "
          "every execution — "
          "printed by the [vram] pulse every 25 steps")
else:
    print(f"[{VER}] #30 NOTE: no text decoder .layers found to wrap "
          "(unexpected topology — L1 module self-checkpointing stands)")

# -------- telemetry diet for the training phase ------------------------------
# SectionedProjector.last_out pins [T, q+k+v+z] tensors per GDN layer
# across the whole fwd+bwd and the trainer never reads it (compute_loss
# reads PRISM_MODULES[].last_aux only); restored after training.
for _, _m in RESONA_SECTIONED:
    _m.telemetry = False
print(f"[{VER}] #28 telemetry diet: last_out disabled on "
      f"{len(RESONA_SECTIONED)} SectionedProjector(s) for training "
      "(restored after)")

t0 = time.time()
free_cuda("before training")   # phase boundary
cuda_census(f"before training ({VER})")   # names every live CUDA block
if torch.cuda.is_available():
    print(f"[{VER} budget] baseline {_base_gib:.1f} GiB | free "
          f"{torch.cuda.mem_get_info()[0] / 2**30:.1f} GiB — expected "
          "steady-state peak ≈ baseline + ~3 GiB (optimizer 1.2 + one-step "
          "transients). No cap, no governor, no dry-run: step 1 is the "
          "measurement, the [vram] pulse verifies it, the OOM handler "
          "below survives the rest.")

# self-check: chunked-head path vs full-GEMM oracle (CPU mocks; the
# notebook refuses to train before it passes).
_dsc, _Vsc, _Ksc = 32, 97, 6
_hin = torch.randn(2, 17, _dsc, requires_grad=True)
_Wsc = torch.randn(_Vsc, _dsc)                      # the frozen head
_lab = torch.randint(0, _Vsc, (2, 16))
_tid = torch.randint(0, _Vsc, (2, 16, _Ksc))
_hv = _hin[:, :-1]
_nsc, _ssc = _ChunkedHeadLoss.apply(_hv, _Wsc, _lab, _tid, 7)
_lgsc = (_hv.reshape(-1, _dsc) @ _Wsc.t()).reshape(2, 16, _Vsc)
_lse = _lgsc.logsumexp(-1)
_nref = _lse - _lgsc.gather(-1, _lab[..., None]).squeeze(-1)
_sref = _lgsc.gather(-1, _tid) - _lse[..., None]
assert float((_nsc - _nref).detach().abs().max()) < 1e-5, "chunked-head nll drift"
assert float((_ssc - _sref).detach().abs().max()) < 1e-5, "chunked-head slog drift"
_gn = torch.randn_like(_nsc); _gs = torch.randn_like(_ssc) * 0.1
(_gn * _nsc + (_gs * _ssc).sum(-1)).sum().backward()
_gh = _hin.grad.clone(); _hin.grad = None
(_gn * _nref + (_gs * _sref).sum(-1)).sum().backward()
_gd = float((_gh - _hin.grad).norm()
            / (_hin.grad.norm() + 1e-12))
assert _gd < 1e-5, "chunked-head hidden-grad drift"
print(f"[{VER} self-check] chunked-head loss == full-GEMM oracle: "
      f"nll max|d| {float((_nsc - _nref).detach().abs().max()):.2e}, "
      f"slog max|d| {float((_ssc - _sref).detach().abs().max()):.2e}, "
      f"hidden-grad rel {_gd:.2e} (fp32 mocks, chunk=7)")
del _dsc, _Vsc, _Ksc, _hin, _Wsc, _lab, _tid, _hv, _nsc, _ssc, _lgsc, _lse
del _nref, _sref, _gn, _gs, _gh, _gd

import traceback

# ============================================================================
# ============================================================================
# Training loop — plain run + ONE clean OOM recovery.
#   * no dry-run probe (it doubled the OOM surface and leaked its own graph)
#   * the handler keeps STRINGS only (`traceback.format_exc()`), never the
#     exception object: cleaning up INSIDE the except block pins the dead
#     graph via the very frames being handled — and `raise ... from e`
#     chains the pinned graph into the final error. Both avoided here.
#   * after one recovery (heap scrub + light-ckpt resume) a second OOM
#     raises with the census naming every live block.
# ============================================================================
# ============================================================================
# ---- v9.1 TRAINING PATH ------------------------------------------------------------------------------
# 'soft_then_hard' (v8): differentiable all-block path first, hard serving path for the last part.
# 'hard': train the function that is SERVED from step 0 (KL to the teacher + LTD + EMRD).
# 'auto': measured, not assumed — the soft path weights blocks by softmax(router)*B*gain, which equals the
#   dense model only for an untrained router. With calibrated routers/gains it can start FAR from the
#   teacher (a loss above ln(vocab) ~ 12.4 is that symptom). If the soft student is not better than the
#   served student on held-out text, the soft phase is skipped.
_ps, _pt = PPL.get("student_soft@init"), PPL.get("student_topk@init")
_path = TRAIN_PATH
if _path == "auto":
    _path = "hard" if (_ps is None or _pt is None or not (_ps < _pt)) else "soft_then_hard"
print(f"[{VER}] training path: {_path} (TRAIN_PATH={TRAIN_PATH!r}; held-out PPL at init: "
      f"dense {PPL.get('teacher_dense', float('nan')):.2f} | served top-k {_pt if _pt is not None else float('nan'):.2f} | "
      f"soft {_ps if _ps is not None else float('nan'):.2f}) | LAMBDA_CE={LAMBDA_CE} LAMBDA_KL={LAMBDA_KL}")
if _path == "hard":
    ANNEAL_FRAC = 0.0                 # MoRM budget at its target from step 1; no soft phase
    set_world("serving")
    _PHASE["hard"] = True
else:
    set_world("soft_train")
    _PHASE["hard"] = False
_MAX_OOM_RECOVERIES = 3     # v8.0.3: was ONE; each recovery also halves LOSS_CHUNK
_done, _oom_retries = 0, 0
while _done < STEPS:
    trainer = _build_trainer(BATCH_SIZE, GRAD_ACCUM)
    if _done > 0:
        args.max_steps = STEPS - _done        # continue the remaining steps
        print(f"[{VER} resume] {STEPS - _done} steps left")
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    try:
        trainer.train()
        _done = STEPS
    except RuntimeError as _e:
        if not is_oom(_e):
            raise
        _err = traceback.format_exc()   # STRING only — the exception dies here
        _oom_retries += 1
        if _oom_retries > _MAX_OOM_RECOVERIES:
            cuda_census("at-OOM (final)")
            raise RuntimeError(
                f"{VER}: training OOMed again after {_MAX_OOM_RECOVERIES} clean recoveries. "
                "The census above names every live block — paste it with "
                f"the last [vram] line.\n{_err}") from None
    else:
        break
    # ---- cleanup AFTER the except block: the OOM exception and its
    # graph-pinning traceback have been released by now ----
    del trainer
    optimizer.zero_grad(set_to_none=True)
    for _m in (m for _, m in PRISM_MODULES):
        _m.last_aux = None
    LOSS_CHUNK = max(64, LOSS_CHUNK // 2)   # smaller vocab-GEMM chunks (math-neutral)
    print(f"[{VER} OOM recovery] LOSS_CHUNK -> {LOSS_CHUNK}")
    free_cuda(f"OOM recovery {_oom_retries}/{_MAX_OOM_RECOVERIES}")
    _done = _load_light(model, optimizer, scheduler) \
        if os.path.exists(LIGHT_CKPT) else 0
    print(f"[{VER} OOM recovery] heap back to "
          f"{torch.cuda.memory_allocated() / 2**30:.1f} GiB (baseline "
          f"{_base_gib:.1f}); resuming from step {_done}")

print(f"training wall time: {(time.time()-t0)/60:.1f} min")
free_cuda("after training")    # release optimizer/activation transients

# restore what the training phase borrowed: the forced decoder
# checkpoints and the SectionedProjector telemetry diet.
for _ly, _f in _FORCED_CKPT:            # identity math, restored
    _ly.forward = _f                     # so later phases see stock modules
_FORCED_CKPT.clear()
for _, _m in RESONA_SECTIONED:           # telemetry back on
    _m.telemetry = True

# bring the parked text-inert subtrees home: the export, the parity
# reload, and vLLM all expect the full VL model resident again (parking
# moved storage only — weight-neutral).
for _name, _mod in _PARKED:
    _mod.to(DEV)
_PARKED.clear()
free_cuda(f"{__PRISM_VER__} parked subtrees restored")
model.config.use_cache = True


## Post-training verification & evaluation

Four measurements, all on **held-out** data:

1. **Perplexity table** — dense teacher (uniform mode) vs. PRISM top-k student, before and after training. The gap the training had to close is $\Delta = \mathrm{PPL}_{topk} - \mathrm{PPL}_{teacher}$.
2. **Serving parity re-check** — re-assert Theorem 3 on the *trained* fields (materialization must still be exact after learning).
3. **Routing health** — expert load balance (CV = std/mean over experts) and gate entropy: healthy PRISM routing is near-balanced (the Sinkhorn target) with high entropy, no dead experts.
4. **Cost accounting (v9: census)** — one walk over the live module graph gives the active parameters per family next to the solver's plan; the strict 3B mandate is checked on the *trained* model; COMPAT vs NATIVE active parameters; the head-funnel quality on the trained student (probability mass and argmax inside the shortlist); plus a wall-clock forward benchmark of the training-path top-k vs. uniform (the *serving* speedup is measured by vLLM in the next section).


In [ ]:
# Evaluation — PPL table, parity re-check, routing health, census-based cost (v9)
PPL_AFTER = {}
PPL_AFTER["teacher_dense"]   = ppl(heldout, "uniform")
PPL_AFTER["student_topk"]    = ppl(heldout, "topk")
PPL_AFTER["student_soft"]    = ppl(heldout, "soft")
PPL_BEFORE = torch.load(os.path.join(CACHE_DIR, "ppl_before.pt"), weights_only=False)

print("\n=========================== PERPLEXITY ===========================")
print(f"{'model path':<26}{'before':>10}{'after':>10}")
print(f"{'dense teacher (uniform)':<26}{PPL_BEFORE['teacher_dense']:>10.3f}"
      f"{PPL_AFTER['teacher_dense']:>10.3f}")
print(f"{'PRISM top-k (serving)':<26}{PPL_BEFORE['student_topk@init']:>10.3f}"
      f"{PPL_AFTER['student_topk']:>10.3f}")
gap0 = PPL_BEFORE["student_topk@init"] - PPL_BEFORE["teacher_dense"]
gap1 = PPL_AFTER["student_topk"] - PPL_AFTER["teacher_dense"]
recovered = 100 * (1 - gap1 / gap0) if gap0 > 0 else 100.0
if gap0 <= 0:
    _rec_txt = "(the sparse student was not worse than the dense teacher at init — nothing to recover)"
elif recovered >= 0:
    _rec_txt = f"({recovered:.1f}% of the sparsity damage recovered)"
else:
    _rec_txt = (f"(WORSE than at init by {-recovered:.1f}% — training did not close the gap; "
                "see the [loss parts] lines above)")
print(f"\ninit routing gap {gap0:.3f} -> {gap1:.3f} PPL points " + _rec_txt)

# ------------------- Theorem 3 re-check on trained fields -------------------
worst = 0.0
x = heldout[0, :64].to(next(model.parameters()).device)
with torch.no_grad():
    set_mode("topk")
    y_mod = model(input_ids=x[None]).logits
    set_mode("soft")
h0 = x
for li, (name, m) in enumerate(PRISM_MODULES[:3] if RUN_MODE == "full" else PRISM_MODULES):
    with torch.no_grad():
        for e in (0, m.E // 2):
            W = m.materialize_expert(e)
            hs = torch.randn(8, m.d_model, device=x.device, dtype=m.W_gate.dtype)
            y_ref = m.expert_outputs(hs, True)[:, e, :] * m.gains[e]
            y_mat = (F.silu(hs @ W["gate_proj"].t()) *
                     (hs @ W["up_proj"].t())) @ W["down_proj"].t()
            worst = max(worst, float((y_mat - y_ref).norm() / y_ref.norm()))
# Tolerance = 2 ULPs of the STORAGE dtype: runtime fuses the fp32 correction
# AFTER the gate GEMM, materialization rounds it INTO the bf16 weight — both
# are valid roundings of the same math and cannot be bitwise-equal in bf16
# (in fp32 they agree to ~1e-7, matching the unit suite).  A drift beyond
# 2 ULPs would mean a REAL fusion bug (wrong slicing/gain), not rounding.
_T3_SUB = PRISM_MODULES[0][1].W_gate.dtype if PRISM_MODULES else torch.float32
_T3_TOL = max(2.0 * torch.finfo(_T3_SUB).eps, 1e-7)
print(f"\nTheorem 3 re-check after training: worst materialization rel-err {worst:.2e}"
      f"  ({'PASS' if worst < _T3_TOL else 'FAIL'} — tol {_T3_TOL:.1e} = 2 ULP of {_T3_SUB})")

# ------------------------------ routing health ------------------------------
loads, entropies = [], []
with torch.no_grad():
    set_mode("topk")
    for i in range(min(4, heldout.shape[0])):
        b = heldout[i:i+1].to(next(model.parameters()).device)
        _ = model(input_ids=b)
        for name, m in PRISM_MODULES:
            if hasattr(m, "last_aux") and "load" in m.last_aux:
                loads.append(m.last_aux["load"].float().cpu())
                p_ = m.last_aux["p"]
                entropies.append(float(-(p_ * p_.clamp_min(1e-9).log()).sum(-1).mean()))
    set_mode("soft")
L = torch.stack(loads).mean(0)
cv = float(L.std() / L.mean())
dead = int((L == 0).sum())
maxent = math.log(E_CHOSEN)
print(f"\nrouting health: load CV={cv:.3f} (0=perfect balance) | dead experts: {dead}"
      f" | mean gate entropy {sum(entropies)/len(entropies):.2f} / {maxent:.2f}")
top = L.topk(min(8, len(L))).values
bar = lambda v: "#" * max(1, int(40 * v / L.max()))
print("busiest experts:", "  ".join(f"{float(v):.0f}" for v in top))

# --------------------------- MoRM 2.0 telemetry (v8) ------------------------
if RESONA_MORM and isinstance(RESONA_MORM[0][1], MoRM2Policy):
    print("\n=================== MoRM 2.0 (oracle-scheduled writes) ============")
    for name, m in RESONA_MORM[:6]:
        fit = getattr(m, "_crit_fit", None)
        r2 = cap = orc = None
        if fit is not None:
            with torch.no_grad():
                # v9: the critic predicts LOG influence -> judge it there (pooled over heads, so each head's
                # level counts), and by what the policy is for: the share of the influence mass that its
                # elastic write set captures (oracle = the best top-m with the same mean m).
                lh = m.scores(fit[0]).clamp_min(1e-12).log()
                ly = fit[1].clamp_min(1e-12).log()
                r2 = float(1 - (lh - ly).pow(2).sum() / (ly - ly.mean()).pow(2).sum().clamp_min(1e-9))
                _thr = float(m.rate_threshold)
                if _thr > 0:
                    msk = (lh.exp() > _thr).to(fit[1].dtype)
                    tot_ = fit[1].sum().clamp_min(1e-12)
                    cap = float((msk * fit[1]).sum() / tot_)
                    _mb = max(1, min(int(round(float(msk.sum(-1).mean()))), fit[1].shape[-1]))
                    orc = float(fit[1].gather(-1, fit[1].topk(_mb, dim=-1).indices).sum() / tot_)
        mm = (float(m.last_m.float().mean())
              if m.last_m is not None else float("nan"))
        print(f"  {name.split('.')[-2] if '.' in name else name:<28} "
              f"m_bar={m.m_bar:>3} | train mean m={mm:.2f} | "
              f"thr={float(m.rate_threshold):.3e} | "
              f"critic log-R^2={'n/a' if r2 is None else f'{r2:.3f}'}"
              + ("" if cap is None else f" | captured influence {cap:.3f} (oracle top-m {orc:.3f})"))
    print("  (critic log-R^2 / captured influence: measured on the calibration label stash, i.e. in-sample)")
    # scheduler simulation on the calibration labels: deferral + staleness
    _any = next(iter(MORM2_STASH.values()), None) if MORM2_STASH else None
    if _any is not None:
        lab = _any["labels"].to(DEV)
        T_, H_ = lab.shape
        sched_sim = schedule_writes(
            lab, torch.randn(T_, H_, 8, device=DEV),
            torch.randn(T_, H_, 8, device=DEV),
            torch.ones(T_, H_, device=DEV) * 0.7,
            torch.full((T_, H_), 0.97, device=DEV),
            m_bar=max(1, min(M_WRITE, H_)), staleness_k=MORM2_STALENESS_K,
            m_max=H_)
        st = sched_sim["stats"]
        print(f"  scheduler sim (label-driven): {st.summary()}")
        print(f"  => elastic writes hit the same mean budget as v7's fixed "
              f"top-m while deferral recovers late-critical writes "
              f"(contracts M6/M7/M8 in the MoRM 2.0 verification cell)")

# ------------------------------ cost accounting -----------------------------
# v9: ONE census of the live module graph (see census_active): embedding lookup, norms, head funnel,
# sketch applications, router reads and every dense site the surgery left behind are all inside
# TOTAL_ACTIVE (v8 added funnel / embedding / norms to a hand-picked registry sum).
set_world("serving")
CEN = measured_active(detail=True)
act_native = CEN["TOTAL_ACTIVE"]
set_world("compat_parity")
# COMPAT, same text-token convention as the census: one embedding ROW (not the table), the dense head
# (tied checkpoints: the embedding table IS the head), dense attention/DeltaNet, sparse FFN fields.
# The vision tower and the MTP draft head are idle for a plain text token (v8 charged them in full).
_head_dense = wf.get("lm_head", 0) or wf.get("embeddings", 0)
act_compat = (HID + _head_dense + wf.get("attention", 0) + wf.get("norms", 0)
              + N_LAY * 3 * HID * SHARED_SIZE + N_LAY * K_CHOSEN * 3 * HID * M_E
              + N_LAY * E_CHOSEN * HID)
set_world("soft_train")
print("\n==================== ACTIVE-PARAMETER WATERFALL ====================")
print(f"{'family':<26}{'plan (waterfall)':>20}{'census (built)':>20}")
for k, v in WF.items():
    if k != "TOTAL_ACTIVE":
        print(f"{k:<26}{v/1e9:>19.3f}B{CEN.get(k, 0)/1e9:>19.3f}B")
print(f"{'TOTAL (' + MAST_ACCOUNTING + ' sketch)':<26}{WF['TOTAL_ACTIVE']/1e9:>19.3f}B{act_native/1e9:>19.3f}B")
if census_active.last["unclassified"]:
    print("!! unclassified 2-D+ parameters were charged in full:", census_active.last["unclassified"][:6])
print(f"\nCOMPAT artifact (stock vLLM): ~{act_compat/1e9:.2f}B active "
      f"(FFN-sparse; attention/head dense by stock-arch necessity)")
print(f"NATIVE artifact (RESONA execution): ~{act_native/1e9:.3f}B active "
      f"| dense touched: {total_dense/1e9:.2f}B "
      f"| compression {total_dense/max(act_native,1):.1f}x")
if RUN_MODE == "full" and TARGET_ACTIVE:
    if act_native > TARGET_ACTIVE:
        _msg = (f"3B mandate violated at eval: census {act_native/1e9:.3f}B > {TARGET_ACTIVE/1e9:.2f}B "
                f"(families above)")
        if ENFORCE_BUDGET:
            raise AssertionError(_msg)
        print("!! " + _msg)
    else:
        print(f"===> 3B MANDATE (census, after training): {act_native/1e9:.3f}B <= {TARGET_ACTIVE/1e9:.2f}B <===")
print(f"\nfunnel shortlist recall (serving head, held-out): {FUNNEL_RECALL:.4f} "
      f"(plain rank-r proxy {FUNNEL_RECALL_K0:.4f}; residual-aware kappa={FUNNEL_KAPPA})")

# head funnel on the TRAINED student: how much of the exact next-token distribution survives the shortlist
try:
    _fq_mass, _fq_top1, _fq_n = 0.0, 0.0, 0
    _lw = model.get_parameter("lm_head.weight")
    with torch.no_grad():
        set_mode("topk")
        for _i in range(min(4, heldout.shape[0])):
            _o = model(input_ids=heldout[_i:_i+1, :256].to(DEV), output_hidden_states=True)
            _h = _o.hidden_states[-1][0]
            _p = (_h @ _lw.to(_h.device).t()).float().softmax(-1)               # [L, V] exact
            _sl = funnel._proxy_scores(_h.to(funnel.weight.dtype)).topk(funnel.shortlist, dim=-1).indices
            _fq_mass += float(_p.gather(-1, _sl).sum())
            _fq_top1 += float((_sl == _p.argmax(-1, keepdim=True)).any(-1).float().sum())
            _fq_n += int(_h.shape[0])
            del _o, _h, _p, _sl
        set_mode("soft")
    FUNNEL_QUALITY = {"mass_kept": _fq_mass / max(_fq_n, 1), "top1_kept": _fq_top1 / max(_fq_n, 1)}
    print(f"head funnel on the trained student (held-out): next-token probability mass inside the shortlist "
          f"{FUNNEL_QUALITY['mass_kept']:.5f} | argmax inside {FUNNEL_QUALITY['top1_kept']:.5f}")
except Exception as _e:
    set_mode("soft")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"(funnel quality probe skipped: {type(_e).__name__}: {str(_e)[:120]})")

if torch.cuda.is_available():
    import time as _t
    xb = heldout[0, :256].to(DEV)[None]
    for mode in ("uniform", "topk"):
        set_mode(mode)
        with torch.no_grad():
            _ = model(input_ids=xb); torch.cuda.synchronize()
            t0 = _t.time()
            for _ in range(3): _ = model(input_ids=xb)
            torch.cuda.synchronize()
        dt = (_t.time() - t0) / 3
        print(f"training-path forward ({mode:>7}): {dt*1000:.0f} ms "
              f"({256/dt:,.0f} tok/s)")
    set_mode("soft")
    print("(serving-path speed is measured by vLLM below — its grouped MoE "
          "kernels are the real deployment target)")
free_cuda("evaluation complete")   # phase boundary before the export


## Compile-to-serving — export as `Qwen3_5MoeForConditionalGeneration`

Everything PRISM does differently lives in the **weights and the training**; at inference the model is a bog-standard fine-grained MoE, exactly like the official `Qwen3.5-35B-A3B`:

| PRISM object | Exported as |
|---|---|
| fragment ⊕ ZSSI completion $A_e, B_e$ | expert `gate_proj` / `up_proj` (fused, Theorem 3) |
| calibrated gain $c_e$ | expert `down_proj` columns (fused) |
| shared core ⊕ shared field | `shared_expert.{gate,up,down}_proj` |
| prototype-resonant router | `mlp.gate` Linear — a plain affine map |
| vision tower, embeddings, 262K RoPE | **copied unchanged** (MTP is config-carried — the HF checkpoint has no separate MTP weights) |

We discover the reference key names **at runtime from a meta-instantiated `Qwen3_5MoeForConditionalGeneration`** (no hardcoded names), stream the checkpoint to Drive shard-by-shard (no 56GB RAM spike), then reload and verify parity.


In [ ]:
# Export — COMPAT (stock Qwen3_5Moe) + NATIVE (RESONA sidecar + launcher) (v9)
import gc
import shutil
from safetensors.torch import save_file

# ---------------------------- 1) restore the bf16 substrate (lossless) ------
# v9: weights quantized by the opt-in PRECISION_PLAN='int8_frozen' are put back from the bf16 originals
# that were kept at quantization time — no int8 round-trip noise ends up in the shipped checkpoint.
if globals().get("_QUANTIZED"):
    restore_precision()

# ---------------------------- 2) MoE config ---------------------------------
# Rebuild the export config as a REAL Qwen3_5Moe config. Works for BOTH the
# VL 27B (vision tower carried over) and text-only checkpoints.
moe_cfg_src = AutoConfig.from_pretrained(MODEL_ID)
IS_VL = getattr(moe_cfg_src, "vision_config", None) is not None and \
    not getattr(moe_cfg_src, "language_model_only", False)
from transformers.models.qwen3_5_moe.configuration_qwen3_5_moe import (
    Qwen3_5MoeConfig as _QCfg, Qwen3_5MoeTextConfig as _QTxtCfg)
_src_tc = getattr(moe_cfg_src, "text_config", moe_cfg_src)
tc = _QTxtCfg.from_dict(_src_tc.to_dict())          # carries EVERY text dim
tc.model_type = "qwen3_5_moe_text"
tc.num_experts = E_CHOSEN
tc.num_experts_per_tok = K_CHOSEN
tc.moe_intermediate_size = M_E
tc.shared_expert_intermediate_size = SHARED_SIZE
tc.mlp_only_layers = []
tc.router_aux_loss_coef = 0.001
# preserve the source rope convention EXACTLY (partial rotary, theta, type):
# the reference text config has its own defaults (partial 0.25!) that would
# silently change the attention math if the source didn't specify it.
_src_rope = (getattr(_src_tc, "rope_parameters", None)
             or getattr(_src_tc, "rope_scaling", None))
if _src_rope:
    tc.rope_parameters = dict(_src_rope)
    tc.rope_scaling = dict(_src_rope)
tc.architectures = ["Qwen3_5MoeForConditionalGeneration" if IS_VL
                    else "Qwen3_5MoeForCausalLM"]
if IS_VL:
    moe_cfg = _QCfg(vision_config=moe_cfg_src.vision_config, text_config=tc)
    moe_cfg.architectures = tc.architectures
    moe_cfg.model_type = "qwen3_5_moe"
else:
    moe_cfg = tc                                   # text-only artifact
# NOTE: the real Qwen3_5MoeTopKRouter ALWAYS renormalizes top-k weights
# (softmax -> top-k -> divide by sum) - exactly PRISM's gate semantics -
# so no extra config switch is needed.
moe_cfg.name_or_path = ""

os.makedirs(EXPORT_DIR, exist_ok=True)
moe_cfg.save_pretrained(EXPORT_DIR)
tok.save_pretrained(EXPORT_DIR)
if proc is not None:
    try: proc.save_pretrained(EXPORT_DIR)
    except Exception as e: print("processor save skipped:", e)
try:
    model.generation_config.save_pretrained(EXPORT_DIR)
except Exception:
    pass
print("config/tokenizer/processor written. Context preserved:",
      getattr(tc, "max_position_embeddings", "?"), "tokens")

# ------------------- 3) reference key discovery (meta) ----------------------
import transformers as _tf
from transformers import AutoModelForCausalLM as _ALM
from transformers import AutoModelForImageTextToText as _AVL
_AM = _AVL if IS_VL else _ALM
with torch.device("meta"):
    ref = _AM.from_config(moe_cfg)
ref_sd = ref.state_dict()          # names + shapes on meta (costs nothing)
del ref
PFX2PRISM = {name.rpartition(".")[0]: m for name, m in PRISM_MODULES}
print(f"reference has {len(ref_sd)} tensors; we own {len(PFX2PRISM)} MoE layers")

import re
import glob as _glob
# REAL Qwen3_5Moe layout (validated against transformers 5.17 source):
#   mlp.experts.gate_up_proj  [E, 2*m_e, d]  (no .weight suffix - raw Parameter)
#   mlp.experts.down_proj     [E, d, m_e]
#   mlp.shared_expert.{gate,up,down}_proj.weight (separate)
#   mlp.shared_expert_gate.weight [1, d]  (sigmoid gate on the shared expert)
#   mlp.gate.weight [E, d]  (router; softmax -> top-k -> renormalize, like ours)
RX_EFUSED = re.compile(r"^(?P<pfx>.*)\.mlp\.experts\.(?P<proj>gate_up_proj|down_proj)$")
RX_SHARED = re.compile(r"^(?P<pfx>.*)\.mlp\.shared_expert\.(?P<proj>gate_proj|up_proj|down_proj)\.weight$")
RX_SGATE  = re.compile(r"^(?P<pfx>.*)\.mlp\.shared_expert_gate\.weight$")
RX_ROUTER = re.compile(r"^(?P<pfx>.*)\.mlp\.gate\.weight$")
# DeltaNet in_proj: the student stores a FUSED in_proj_qkvz [q|k|v|z]; the
# 5.17 reference stores UNFUSED in_proj_qkv / in_proj_z / in_proj_b / in_proj_a.
RX_DN_QKV = re.compile(r"^(?P<pfx>.*)\.linear_attn\.in_proj_qkv\.weight$")
RX_DN_Z   = re.compile(r"^(?P<pfx>.*)\.linear_attn\.in_proj_z\.weight$")
RX_DN_B   = re.compile(r"^(?P<pfx>.*)\.linear_attn\.in_proj_b\.weight$")
RX_NORM_ZC = re.compile(
    r"^(?P<rest>.*\.(?:input_layernorm|post_attention_layernorm|q_norm|k_norm)"
    r"|model\.norm)\.weight$")
RX_DN_A   = re.compile(r"^(?P<pfx>.*)\.linear_attn\.in_proj_a\.weight$")
_DN_KDIM = (int(getattr(tc, "linear_num_key_heads", 0))
            * int(getattr(tc, "linear_key_head_dim", 0)))
_DN_VDIM = (int(getattr(tc, "linear_num_value_heads", 0))
            * int(getattr(tc, "linear_value_head_dim", 0)))
_DN_NVH  = int(getattr(tc, "linear_num_value_heads", 0))
dense_sd = model.state_dict()      # hoisted once (not per-tensor)
# The reference multiplies the shared expert by sigmoid(shared_expert_gate(h))
# where shared_expert_gate is a [1, d] LINEAR (no bias): output = w . h.
# PRISM has no such gate. EXACT solution: pin w = 0  ->  sigmoid(0) = 0.5
# for EVERY token, and pre-divide the shared down_proj by 0.5 (x2). Any
# non-zero pin (e.g. w = 8) gives sigmoid(8 * sum(h)) which flips to ~0 for
# negative sums and silently kills the shared expert.
SHARED_GATE_B = 0.0
SHARED_GATE_CORR = 1.0 / float(torch.sigmoid(torch.tensor(SHARED_GATE_B)))

def deq(t):
    for attr in ("dequantize",):
        if hasattr(t, attr):
            try: return t.dequantize()
            except Exception: pass
    return t

_FUSED_CACHE: dict = {}          # pfx -> {"gate_up_proj", "down_proj", "uses"}
# Materialized fused experts are ~0.7 GiB (bf16) per layer: caching them
# for the WHOLE reference walk would pile ~44 GiB on top of the resident
# model. Every materialized tensor moves to CPU immediately, and a layer's
# cache entry is EVICTED once both of its expert tensors are consumed.
def _fuse_get(prism, pfx):
    ent = _FUSED_CACHE.get(pfx)
    if ent is None:
        fused = prism.materialize_experts_fused()      # ~1.2 GiB GPU transient
        ent = {"gate_up_proj": fused["gate_up_proj"].cpu(),
               "down_proj": fused["down_proj"].cpu(), "uses": 0}
        _FUSED_CACHE[pfx] = ent
        del fused
        torch.cuda.empty_cache()        # return the materialize transients
    return ent


def resolve(kname, meta_shape):
    m = RX_EFUSED.match(kname)
    if m:                                   # fused 3-D experts (real layout)
        prism = PFX2PRISM.get(m.group("pfx"))
        if prism is None: raise KeyError(f"no PRISM layer for {kname}")
        ent = _fuse_get(prism, m.group("pfx"))
        ent["uses"] += 1
        t = ent[m.group("proj")]
        if ent["uses"] >= 2:               # both expert tensors consumed -> evict
            _FUSED_CACHE.pop(m.group("pfx"), None)
        assert tuple(t.shape) == tuple(meta_shape), (kname, t.shape, meta_shape)
        return t
    m = RX_SHARED.match(kname)
    if m:                                   # shared expert (separate projections)
        prism = PFX2PRISM.get(m.group("pfx"))
        if prism is None: raise KeyError(f"no PRISM layer for {kname}")
        t = prism.materialize_shared()[m.group("proj")]
        if m.group("proj") == "down_proj":  # absorb the sigmoid(shared_gate)
            t = t * SHARED_GATE_CORR
        assert tuple(t.shape) == tuple(meta_shape), (kname, t.shape, meta_shape)
        return t
    m = RX_SGATE.match(kname)
    if m:                                   # pin the shared-expert sigmoid gate
        prism = PFX2PRISM.get(m.group("pfx"))
        d_in = prism.d_model
        return torch.zeros(1, d_in, dtype=torch.bfloat16)   # sigmoid(0)=0.5
    m = RX_ROUTER.match(kname)
    if m:                                   # prototype-resonant router (affine)
        prism = PFX2PRISM.get(m.group("pfx"))
        if prism is None: raise KeyError(f"no PRISM layer for {kname}")
        # fold the temperature EXACTLY: softmax(W h / tau) == softmax((W/tau) h)
        tau = torch.exp(prism.log_tau.detach()).clamp(0.25, 4.0)
        return prism.router.weight.detach() / tau
    m = RX_DN_QKV.match(kname)
    if m:                                   # in_proj_qkv: student key first
        p = m.group("pfx")
        # the split-layout surgery (qwen3_5 family) PRESERVES the stock
        # key — the SectionedProjector adopts the original [q|k|v] weight —
        # so the direct hit is the exact original projection. The fused-slice
        # path remains for qwen3_next-style students (in_proj_qkvz).
        t = deq(dense_sd[kname]) if kname in dense_sd else None
        if t is None:
            Wf = dense_sd.get(p + ".linear_attn.in_proj_qkvz.weight")
            if Wf is None:
                raise KeyError(f"no qkv weight for {kname} (tried {kname} and "
                               f"{p}.linear_attn.in_proj_qkvz.weight)")
            t = deq(Wf[: 2 * _DN_KDIM + _DN_VDIM])
        assert tuple(t.shape) == tuple(meta_shape), (kname, t.shape, meta_shape)
        return t
    m = RX_DN_Z.match(kname)
    if m:                                   # in_proj_z: student key first
        p = m.group("pfx")
        # the split-layout z sibling is routed by its own ResonantLinear,
        # which ADOPTS the original weight — direct key hit. Fused fallback
        # slices z rows out of a qwen3_next-style in_proj_qkvz student.
        t = deq(dense_sd[kname]) if kname in dense_sd else None
        if t is None:
            Wf = dense_sd.get(p + ".linear_attn.in_proj_qkvz.weight")
            if Wf is None:
                raise KeyError(f"no z weight for {kname} (tried {kname} and "
                               f"{p}.linear_attn.in_proj_qkvz.weight)")
            t = deq(Wf[2 * _DN_KDIM + _DN_VDIM:])
        assert tuple(t.shape) == tuple(meta_shape), (kname, t.shape, meta_shape)
        return t
    m = (RX_DN_B.match(kname) or RX_DN_A.match(kname))
    if m:                                   # b/a: student key first, then ba
        p = m.group("pfx")
        # split b/a siblings are untouched by surgery -> direct keys;
        # fused students keep the in_proj_ba half-slice fallback.
        if kname in dense_sd:
            t = deq(dense_sd[kname])
        elif p + ".linear_attn.in_proj_ba.weight" in dense_sd:
            Wba = dense_sd[p + ".linear_attn.in_proj_ba.weight"]
            half = 0 if kname.endswith("in_proj_b.weight") else 1
            t = deq(Wba[half * _DN_NVH:(half + 1) * _DN_NVH])
        else:
            raise KeyError(f"no b/a weight for {kname} (tried {kname} and "
                           f"{p}.linear_attn.in_proj_ba.weight)")
        assert tuple(t.shape) == tuple(meta_shape), (kname, t.shape, meta_shape)
        return t
    # Qwen3.5-family RMSNorm uses ZERO-CENTERED gamma: forward = x * (1 + w).
    # Our student norms store the EFFECTIVE gain directly -> emit (w - 1).
    # (Covers input/post_attention layernorms, attn q/k norms, final model.norm.
    #  The DeltaNet gated norm uses plain gamma -> NOT matched here on purpose.)
    m = RX_NORM_ZC.match(kname)
    if m:
        return deq(dense_sd[kname]).float() - 1.0
    if kname in dense_sd:
        return deq(dense_sd[kname])
    # transformers sometimes strips an inner wrapper prefix — try both
    alt = kname.replace(".model.", ".", 1)
    if alt in dense_sd: return deq(dense_sd[alt])
    raise KeyError(f"UNMAPPED reference tensor: {kname} {tuple(meta_shape)}")

# ------------------- 4) stream shards straight to Drive ---------------------
SHARD_BYTES = 4 * 1024**3
shard, shard_idx, weight_map, total, idx_n = {}, {}, {}, 0, 0

def flush():
    global shard, shard_idx, idx_n
    if not shard: return
    idx_n += 1
    fname = f"model-{idx_n:05d}-of-XXXXX.safetensors"
    save_file({k: v.contiguous().clone().to(torch.bfloat16).cpu()
               for k, v in shard.items()},
              os.path.join(EXPORT_DIR, fname), metadata={"format": "pt"})
    for k in shard: weight_map[k] = fname
    print(f"  wrote {fname} ({sum(v.numel()*2 for v in shard.values())/1e9:.2f} GB)")
    shard = {}

t0 = time.time()
for kname, meta_t in ref_sd.items():
    t = resolve(kname, meta_t.shape)
    shard[kname] = t
    total += t.numel() * 2
    if sum(v.numel() * v.element_size() for v in shard.values()) > SHARD_BYTES:
        flush()
flush()
# fix shard filenames now that the count is known
for i, f in enumerate(sorted(_glob.glob(os.path.join(EXPORT_DIR, "model-*.safetensors")))):
    final = f"model-{i+1:05d}-of-{idx_n:05d}.safetensors"
    os.rename(f, os.path.join(EXPORT_DIR, final))
    for k in weight_map:
        if weight_map[k] == os.path.basename(f): weight_map[k] = final
from json import dump as _dump
_dump({"metadata": {"total_size": total}, "weight_map": weight_map},
      open(os.path.join(EXPORT_DIR, "model.safetensors.index.json"), "w"), indent=2)
print(f"export complete: {total/1e9:.2f} GB in {idx_n} shards "
      f"({time.time()-t0:.0f}s) -> {EXPORT_DIR}")

# v8.0.3 [OOM-1]: RELEASE every reference the export walk still holds.
# `dense_sd = model.state_dict()` aliases the ORIGINAL GPU storage of every
# weight. After `model.to("cpu")` (parity reload below, vLLM later) the
# Parameters point at CPU copies but dense_sd kept ~50 GiB pinned on the GPU:
# empty_cache() cannot free it, so the parity reload and vLLM both OOMed, and
# the CPU fallback pushed the VM past its host RAM.
dense_sd = None
_FUSED_CACHE.clear()
shard = {}
ref_sd = None
free_cuda("export references released")

# ------------------------- 5) reload + parity check -------------------------
# Reload parity without doubling the footprint: compute the student side
# FIRST, park the student (+ mast + funnel) on CPU RAM, give the GPU to the
# reloaded artifact, compare on CPU, then swap back.
print("reloading exported model for the parity check ...")
xs = heldout[0, :64].to(DEV)[None]
with torch.no_grad():
    set_world("compat_parity")
    y_ours = model(input_ids=xs).logits.float().cpu()   # survives the swap
    set_world("soft_train")

# v8.0.3 [OOM-2]: the swap parks the whole student (~50 GiB) in HOST RAM. If the
# VM cannot hold it, do not try — parking plus a CPU reload (2 x ~50 GiB) kills
# the Colab session ("crashed after using all available RAM"). The checkpoint
# on Drive is already complete; the parity check is a convenience.
_student_gib = sum(p.numel() * p.element_size() for p in model.parameters()) / 2**30
_ram_av, _ram_tot = host_ram_gib()
rel = argmax_match = float("nan")
if _ram_av < _student_gib + 8.0:
    print(f"[parity skipped] parking the student needs ~{_student_gib:.0f} GiB host RAM "
          f"(+8 GiB slack) but only {_ram_av:.0f}/{_ram_tot:.0f} GiB is free. The exported "
          "checkpoint is complete; to verify it, load EXPORT_DIR in a fresh session and "
          "compare logits.")
else:
    model.to("cpu")
    MAST.to("cpu")
    funnel.to("cpu")
    free_cuda("student parked for parity reload")
    reloaded, y_ref2, _gpu_err = None, None, None
    try:
        reloaded = _AM.from_pretrained(EXPORT_DIR, dtype=torch.bfloat16,
                                       device_map={"": DEV} if torch.cuda.is_available() else "cpu")
    except Exception as e:
        _gpu_err = str(e)[:200]
    if _gpu_err is not None:       # handled OUTSIDE the except block (partial load releasable)
        free_cuda("parity reload failed")
        print(f"[parity skipped] reload failed ({_gpu_err}). A CPU retry would add another "
              f"~{_student_gib:.0f} GiB next to the parked student and crash the VM, so it is not attempted.")
    else:
        reloaded.eval()
        with torch.no_grad():
            y_ref2 = reloaded(input_ids=xs.to(reloaded.device)).logits.float().cpu()
        del reloaded
    gc.collect(); torch.cuda.empty_cache()
    model.to(DEV); MAST.to(DEV); funnel.to(DEV)
    if y_ref2 is not None:
        rel = float((y_ours - y_ref2).norm() / y_ours.norm())
        print(f"PARITY student(top-k) vs exported+reloaded: rel diff {rel:.4f} "
              f"({'PASS' if rel < 0.05 else 'FAIL — inspect expert mapping'})  "
              "(bf16 numerics + kernel order explain the residual)")
        argmax_match = float((y_ours.argmax(-1) == y_ref2.argmax(-1)).float().mean())
        print(f"argmax agreement over positions: {argmax_match*100:.1f}%")
free_cuda("export parity complete")

# ============================================================================
# 6) NATIVE artifact: the full ~3B-active RESONA execution package
#    (compat weights + execution sidecar + self-contained launcher)
# ============================================================================
os.makedirs(NATIVE_DIR, exist_ok=True)
sidecar = {"mast_R": MAST.R.detach().cpu(),
           "mast_S": MAST_S,
           "mast_seed": int(getattr(MAST, "seed", 0)),
           "mast_d_model": int(MAST.d_model),
           "plan": {k: v for k, v in PLAN.items()},
           "waterfall": WF,
           "funnel_recall": FUNNEL_RECALL,
           "funnel_kappa": float(funnel.kappa),
           "mast_accounting": MAST_ACCOUNTING,
           "census": {k_: int(v_) for k_, v_ in (globals().get("CEN") or {}).items()},
           "sites": {"linear": [], "sectioned": []}}
for name, m in RESONA_LINEARS:
    sidecar["sites"]["linear"].append({
        "path": name, "kind": "linear",
        "carrier_idx": m.carrier_idx.cpu(), "block_idx": m.block_idx.cpu(),
        "topk": int(m.topk),
        "router": m.router.weight.detach().cpu(),
        "gains": m.gains.detach().cpu(),
        "sh_U": (m.sh_U.detach().cpu() if m.sh_U is not None else None),      # v10 shadow
        "sh_M": (m.sh_M.detach().cpu() if m.sh_M is not None else None)})
for name, m in RESONA_SECTIONED:
    entry = {"path": name, "kind": "sectioned",
             "sec_bounds": m.sec_bounds.cpu(),
             "v_head_rows": int(m.v_head_rows), "topk": int(m.topk),
             "v_gains": m.v_gains.detach().cpu(),
             "morm_m": (int(m.morm.m_bar)
                        if isinstance(m.morm, MoRM2Policy)
                        else (int(m.morm.m) if m.morm is not None else None)),
             "morm_router": (m.morm.critic_phi.weight.detach().cpu()
                             if isinstance(m.morm, MoRM2Policy)
                             else (m.morm.router.weight.detach().cpu()
                                   if m.morm is not None else None)),
             # v8: the full MoRM 2.0 execution state for the native launcher
             "morm2": ({"critic_obs": m.morm.critic_obs.weight.detach().cpu(),
                        "critic_bias": (m.morm.critic_phi.bias.detach().cpu()
                                        if m.morm.critic_phi.bias is not None else None),
                        "threshold": float(m.morm.rate_threshold),
                        "controller": m.morm.controller.state_dict(),
                        "m_bar": int(m.morm.m_bar),
                        "m_max": int(m.morm.controller.m_max if m.morm.controller.m_max is not None
                                     else m.morm.H),
                        "staleness_k": int(MORM2_STALENESS_K),
                        "logit_floor": float(MORM2_BETA_LOGIT_FLOOR)}
                       if isinstance(m.morm, MoRM2Policy) else None)}
    for t in ("q", "z"):
        if hasattr(m, f"{t}_B"):
            entry[f"{t}_carrier"] = getattr(m, f"{t}_carrier").cpu()
            entry[f"{t}_blocks"] = getattr(m, f"{t}_blocks").cpu()
            entry[f"{t}_router"] = getattr(m, f"{t}_router").weight.detach().cpu()
            entry[f"{t}_gains"] = getattr(m, f"{t}_gains").detach().cpu()
    for t in ("q", "v", "z"):                                             # v10 shadows
        if getattr(m, f"{t}_sh_U", None) is not None:
            entry[f"{t}_sh_U"] = getattr(m, f"{t}_sh_U").detach().cpu()
            entry[f"{t}_sh_M"] = getattr(m, f"{t}_sh_M").detach().cpu()
    sidecar["sites"]["sectioned"].append(entry)
sidecar["ffn_shadow"] = [{"path": name, "U": m.sh_U.detach().cpu(), "M": m.sh_M.detach().cpu(),
                          "b": m.sh_b.detach().cpu()}
                         for name, m in PRISM_MODULES if getattr(m, "sh_U", None) is not None]
torch.save(sidecar, os.path.join(NATIVE_DIR, "resona_sidecar.pt"))
print(f"native sidecar saved: {len(sidecar['sites']['linear'])} linear sites, "
      f"{len(sidecar['sites']['sectioned'])} sectioned sites, "
      f"funnel recall {FUNNEL_RECALL}")

RESONA_KERNELS = r"""
# resona_kernels.py — minimal runtime for the RESONA native execution.
# (Compact twin of the notebook core: AntennaMast, ResonantLinear,
#  SectionedProjector with MoRM, HeadFunnel. Weights come from the COMPAT
#  checkpoint; geometry comes from resona_sidecar.pt.)
import math, torch, torch.nn as nn, torch.nn.functional as F

class AntennaMast(nn.Module):
    # phi(h) = silu(R_d h): one sketch per input width; widths != d_model get
    # a DETERMINISTIC sibling derived from the master seed (matches the
    # calibration-time mast bit-for-bit, no extra serialization).
    def __init__(self, R, seed=0, d_model=None):
        super().__init__()
        self.register_buffer("R", R)
        self.seed = int(seed)
        self.d_model = int(d_model) if d_model is not None else R.shape[1]
        self.S = R.shape[0]
    def _R_for(self, d_in, device):
        if d_in == self.d_model:
            return self.R
        key = f"R{d_in}"
        R = getattr(self, key, None)
        if R is None or R.device != device:
            g = torch.Generator().manual_seed(self.seed * 1000003 + d_in)
            R = torch.randn(self.S, d_in, generator=g) / math.sqrt(d_in)
            self.register_buffer(key, R.to(device=device, dtype=self.R.dtype))
        return getattr(self, key)
    def forward(self, h):
        R = self._R_for(h.shape[-1], h.device)
        return F.silu(h.to(R.dtype) @ R.t())

class ResonantLinear(nn.Module):
    def __init__(self, weight, carrier_idx, block_idx, router_w, gains, mast, topk=1, sh_U=None, sh_M=None):
        super().__init__()
        self.register_buffer("weight", weight)
        # v10 shadow (or None), kept in the weight's dtype / device
        self.register_buffer("sh_U", None if sh_U is None else sh_U.to(device=weight.device, dtype=weight.dtype))
        self.register_buffer("sh_M", None if sh_M is None else sh_M.to(device=weight.device, dtype=weight.dtype))
        self.register_buffer("carrier_idx", carrier_idx)
        self.register_buffer("block_idx", block_idx)
        self.router = nn.Linear(router_w.shape[1], router_w.shape[0], bias=False)
        self.router.weight.data = router_w
        self.gains = nn.Parameter(gains)
        self.mast = mast; self.topk = topk; self.default_mode = "topk"
    def forward(self, x, mode=None):
        mode = mode or self.default_mode
        lead = x.shape[:-1]; T = x.numel() // self.weight.shape[1]
        x2 = x.reshape(T, -1)
        if mode == "dense":
            return F.linear(x2, self.weight).reshape(*lead, self.weight.shape[0])
        if mode == "full":
            y = x2.new_zeros(T, self.weight.shape[0])
            if self.carrier_idx.numel():
                y.index_copy_(1, self.carrier_idx, F.linear(x2, self.weight[self.carrier_idx]))
            Wb = self.weight[self.block_idx.reshape(-1)].reshape(self.block_idx.shape[0], -1, self.weight.shape[1])
            y.index_copy_(1, self.block_idx.reshape(-1),
                          torch.einsum("ti,bri->btr", x2, Wb).permute(1,0,2).reshape(T, -1))
            return y.reshape(*lead, -1)
        logits = self.router(self.mast(x2))
        topv, topi = logits.topk(self.topk, dim=-1)
        w = self.gains[topi]          # v9: the notebook's serving gate is the block GAIN (no softmax weight)
        # chunked over T: the Wsel gather [T*k, r, d] is unbounded (A100 OOM class)
        r_b = self.block_idx.shape[1]
        c = max(1, (512 << 20) // max(1, self.topk * r_b * self.weight.shape[1] * self.weight.element_size()))
        ys_parts, ti_parts = [], []
        for i in range(0, T, c):
            ti = topi[i:i + c]
            Ws = self.weight[self.block_idx[ti.reshape(-1)]].reshape(-1, r_b, self.weight.shape[1])
            xr = x2[i:i + c].repeat_interleave(self.topk, 0).unsqueeze(-1)
            ys_parts.append(torch.bmm(Ws, xr).squeeze(-1).reshape(-1, self.topk, r_b))
            ti_parts.append(ti)
        topi = torch.cat(ti_parts, 0)
        ys = torch.cat(ys_parts, 0) * w.unsqueeze(-1)
        # v10: unselected rows take the shadow value U (M x); exact rows replace it
        shadow = self.sh_U is not None
        y = ((x2 @ self.sh_M.t()) @ self.sh_U.t()) if shadow else x2.new_zeros(T, self.weight.shape[0])
        if self.carrier_idx.numel():
            y.index_copy_(1, self.carrier_idx, F.linear(x2, self.weight[self.carrier_idx]))
        if shadow:
            y.scatter_(1, self.block_idx[topi].reshape(T, -1), ys.reshape(T, -1).to(y.dtype))
        else:
            y.scatter_add_(1, self.block_idx[topi].reshape(T, -1), ys.reshape(T, -1).to(y.dtype))
        return y.reshape(*lead, -1)

class SectionedProjector(nn.Module):
    def __init__(self, weight, entry, mast):
        super().__init__()
        self.register_buffer("weight", weight)
        self.register_buffer("sec_bounds", entry["sec_bounds"])
        self.v_head_rows = entry["v_head_rows"]; self.topk = entry["topk"]
        self.default_mode = "topk"
        b = self.sec_bounds
        self.q_rows, self.k_rows = int(b[1]), int(b[2]-b[1])
        self.v_rows, self.z_rows = int(b[3]-b[2]), int(b[4]-b[3])
        self.n_v_heads = self.v_rows // self.v_head_rows
        for t in ("q", "z"):
            if f"{t}_blocks" in entry:
                self.register_buffer(f"{t}_carrier", entry[f"{t}_carrier"])   # (nn.Buffer needs torch >= 2.5)
                self.register_buffer(f"{t}_blocks", entry[f"{t}_blocks"])
                r = nn.Linear(entry[f"{t}_router"].shape[1], entry[f"{t}_router"].shape[0], bias=False)
                r.weight.data = entry[f"{t}_router"]; setattr(self, f"{t}_router", r)
                setattr(self, f"{t}_gains", nn.Parameter(entry[f"{t}_gains"]))
                setattr(self, f"{t}_B", int(entry[f"{t}_blocks"].shape[0]))
        # v8.0.2 [P5]: MoRM 2.0 exports (morm2 entry) run the elastic
        # twin; legacy v1 exports keep their fixed top-m unchanged.
        if entry.get("morm2") is not None:
            m2 = entry["morm2"]
            self.morm = MoRM2PolicyLite(entry["morm_router"],
                                        int(m2["m_bar"]), int(m2["m_max"]),
                                        float(m2["threshold"]), mast,
                                        critic_b=m2.get("critic_bias"))
        elif entry.get("morm_router") is not None:
            self.morm = MoRMPolicy(entry["morm_router"], int(entry["morm_m"]), mast)
        else:
            self.morm = None
        self.v_gains = nn.Parameter(entry["v_gains"])
        self.mast = mast
        for t in ("q", "v", "z"):                       # v10 shadows (absent in older exports)
            for nm in (f"{t}_sh_U", f"{t}_sh_M"):
                b_ = entry.get(nm)
                self.register_buffer(nm, None if b_ is None else b_.to(device=weight.device, dtype=weight.dtype))
    def _routed(self, x, tag, lo, hi, mode):
        W = self.weight[lo:hi]; T = x.shape[0]
        if mode == "dense":
            return F.linear(x, W)
        carrier = getattr(self, f"{tag}_carrier"); blocks = getattr(self, f"{tag}_blocks")
        B, r = getattr(self, f"{tag}_B"), blocks.shape[1]
        if mode == "full":
            y = x.new_zeros(T, hi-lo)
            if carrier.numel(): y.index_copy_(1, carrier, F.linear(x, W[carrier]))
            Wb = W[blocks.reshape(-1)].reshape(B, r, W.shape[1])
            y.index_copy_(1, blocks.reshape(-1), torch.einsum("ti,bri->btr", x, Wb).permute(1,0,2).reshape(T,-1))
            return y
        logits = getattr(self, f"{tag}_router")(self.mast(x))
        topv, topi = logits.topk(self.topk, dim=-1)
        w = getattr(self, f"{tag}_gains")[topi]     # v9: gain only (see ResonantLinear)
        # chunked over T: the Wsel gather [T*k, r, d] is unbounded (A100 OOM class)
        k_sel = topi.shape[1]
        c = max(1, (512 << 20) // max(1, k_sel * r * W.shape[1] * W.element_size()))
        ys_parts, ti_parts = [], []
        for i in range(0, T, c):
            ti = topi[i:i + c]
            Ws = W[blocks[ti.reshape(-1)]].reshape(-1, r, W.shape[1])
            xr = x[i:i + c].repeat_interleave(k_sel, 0).unsqueeze(-1)
            ys_parts.append(torch.bmm(Ws, xr).squeeze(-1).reshape(-1, k_sel, r))
            ti_parts.append(ti)
        topi = torch.cat(ti_parts, 0)
        ys = torch.cat(ys_parts, 0) * w.unsqueeze(-1)
        sU = getattr(self, f"{tag}_sh_U")
        y = ((x @ getattr(self, f"{tag}_sh_M").t()) @ sU.t()) if sU is not None else x.new_zeros(T, hi-lo)
        if carrier.numel(): y.index_copy_(1, carrier, F.linear(x, W[carrier]))
        if sU is not None:
            y.scatter_(1, blocks[topi].reshape(T, -1), ys.reshape(T, -1).to(y.dtype))
        else:
            y.scatter_add_(1, blocks[topi].reshape(T, -1), ys.reshape(T, -1).to(y.dtype))
        return y
    def forward(self, x, mode=None):
        mode = mode or self.default_mode
        # v8.0.2 [P6]: the stock GDN layer calls in_proj_qkv on [B, T, d];
        # the routed path assumed 2D [T, d] (the launcher was never run
        # end-to-end). Flatten any leading dims — ResonantLinear's
        # convention — and restore them at the return.
        lead = x.shape[:-1]
        x = x.reshape(-1, x.shape[-1])
        b0,b1,b2,b3,b4 = [int(v) for v in self.sec_bounds]
        if mode == "dense":
            return F.linear(x, self.weight).reshape(*lead, -1)
        q = self._routed(x, "q", b0, b1, mode) if hasattr(self, "q_B") else F.linear(x, self.weight[b0:b1])
        k = F.linear(x, self.weight[b1:b2])
        z = self._routed(x, "z", b3, b4, mode) if hasattr(self, "z_B") else F.linear(x, self.weight[b3:b4])
        mask = self.morm(self.mast(x)) if (self.morm is not None and mode != "full") \
               else torch.ones(x.shape[0], self.n_v_heads, device=x.device, dtype=x.dtype)
        Wv = self.weight[b2:b3].reshape(self.n_v_heads, self.v_head_rows, self.weight.shape[1])
        sel = (mask > 0.5).nonzero(as_tuple=False)
        if self.v_sh_U is not None and mode != "full":    # v10: every head writes its shadow value
            v = (x @ self.v_sh_M.t()) @ self.v_sh_U.t()
        else:
            v = x.new_zeros(x.shape[0], self.v_rows)
        if sel.shape[0]:
            # segmented per-head GEMM — tokens grouped by head, ONE bmm over
            # weight VIEWS: same selected rows as a chunked gather, no
            # [N, hr, d] transient, ~60x less weight traffic.
            hidx, tidx = sel[:, 1], sel[:, 0]
            order = torch.argsort(hidx, stable=True)           # group by head
            h_s, t_s = hidx[order], tidx[order]
            counts = torch.bincount(h_s, minlength=self.n_v_heads)
            starts = torch.cumsum(counts, 0) - counts
            pos = torch.arange(sel.shape[0], device=x.device) - starts[h_s]
            max_n = int(counts.max())
            pad_t = torch.zeros(self.n_v_heads, max_n, dtype=torch.long,
                                device=x.device)
            pad_t[h_s, pos] = t_s
            X_pad = x[pad_t]                                   # [H, max_n, d]
            V_pad = torch.bmm(X_pad, Wv.transpose(1, 2))       # [H, max_n, hr]
            vs = V_pad[h_s, pos]                               # [N, hr]
            vs = vs * self.v_gains.to(x.dtype)[h_s][:, None]
            hr = self.v_head_rows
            rows = h_s[:,None]*hr + torch.arange(hr, device=x.device)[None,:]
            v[t_s[:,None].expand(-1,hr).reshape(-1), rows.reshape(-1)] = vs.reshape(-1).to(v.dtype)
        return torch.cat([q,k,v,z], dim=-1).reshape(*lead, -1)

class MoRMPolicy(nn.Module):
    def __init__(self, router_w, m, mast):
        super().__init__()
        self.router = nn.Linear(router_w.shape[1], router_w.shape[0], bias=False)
        self.router.weight.data = router_w
        self.m = m; self.mast = mast
    def forward(self, phi):
        s = self.router(phi)
        topv, topi = s.topk(self.m, dim=-1)
        return torch.zeros_like(s).scatter_(-1, topi, 1.0)

class MoRM2PolicyLite(nn.Module):
    # v8.0.2 [P5]: the MoRM 2.0 serving twin — influence critic with the
    # ELASTIC threshold (the converged rate_threshold exported from
    # training; no controller dynamics at decode) and the m_max burst cap.
    # Replaces the v1 fixed top-m on every export that carries a morm2
    # entry. (Comments, not docstrings: this code lives inside the
    # RESONA_KERNELS r-string.)
    def __init__(self, critic_w, m_bar, m_max, threshold, mast, critic_b=None):
        super().__init__()
        # v9: the critic has a per-head intercept (critic_b); older exports have none
        self.router = nn.Linear(critic_w.shape[1], critic_w.shape[0], bias=critic_b is not None)
        self.router.weight.data = critic_w
        if critic_b is not None:               # nested in the sidecar dict: move it next to the weight
            self.router.bias.data = critic_b.to(device=critic_w.device, dtype=critic_w.dtype)
        self.m_bar, self.m_max = int(m_bar), int(m_max)
        self.rate_threshold = float(threshold)
        self.mast = mast
    def forward(self, phi):
        L = self.router(phi).exp()                 # predicted influence
        thr = self.rate_threshold
        if not (thr > 0.0):                        # untrained export: seed
            thr = float(L.flatten().quantile(
                max(0.0, 1.0 - self.m_bar / L.shape[-1])).clamp_min(1e-12)) * 0.5
        hard = (L > thr).to(L.dtype)
        if self.m_max and self.m_max < L.shape[-1]:
            over = hard.sum(-1, keepdim=True) > self.m_max
            if bool(over.any()):
                ti = L.topk(self.m_max, dim=-1).indices
                capped = torch.zeros_like(L).scatter_(-1, ti, 1.0)
                hard = torch.where(over, capped * (L > thr).to(L.dtype), hard)
        return hard

class BetaGateLite(nn.Module):
    # v8.0.2 [P5]: the serving write-enable lever (Proposition F).
    # Wraps the stock in_proj_b (split qwen3_5, b_only) or in_proj_ba
    # (fused, [b|a]) and rewrites masked heads' b logits to the floor —
    # beta ~ 0 means the write is EXACTLY absent in the delta-rule kernel.
    # Replaces the v7-era v-mask, which that kernel turns into deletion
    # writes (-beta k k^T S~) instead of skips.
    def __init__(self, inner, policy, n_heads, logit_floor=-60.0, b_first=True):
        super().__init__()
        self.inner = inner
        self.register_parameter("weight", inner.weight)   # stock state-dict key
        self.policy = policy
        self.logit_floor = float(logit_floor)
        self.b_first = bool(b_first)
        self.b_only = bool(getattr(inner, "out_features", 0) == n_heads)
    def forward(self, x):
        out = self.inner(x)
        out2 = out.reshape(-1, out.shape[-1])
        H = out2.shape[-1] if self.b_only else out2.shape[-1] // 2
        phi = self.policy.mast(x.reshape(-1, x.shape[-1]))
        mask = self.policy(phi)
        mf = (mask.reshape(-1, H) if mask.dim() == 2 else mask).to(out2.dtype)
        b = out2 if self.b_only else (out2[:, :H] if self.b_first else out2[:, H:])
        b = mf * b + (1.0 - mf) * self.logit_floor
        if self.b_only:
            gated = b
        else:
            other = out2[:, H:] if self.b_first else out2[:, :H]
            gated = torch.cat([b, other], dim=-1) if self.b_first \
                else torch.cat([other, b], dim=-1)
        return gated.reshape(*out.shape[:-1], -1)

class FunnelHead(nn.Module):
    # v9: exact rank-r projection proxy (U_r = W P_r) + residual-aware ranking
    #     score_i = (h P_r) . U_r[i]  +  kappa * ||(I - P_r P_r^T) h|| * rho_i ,   rho_i = ||(I - P_r P_r^T) w_i||
    # (Cauchy-Schwarz: |z_i - proxy_i| <= rho_i * ||(I - P_r P_r^T) h||; kappa in [0, 1] was FITTED by recall in
    # the notebook). The shortlist rows are evaluated EXACTLY.
    def __init__(self, weight, U_r, P_r, shortlist, rho=None, kappa=0.0):
        super().__init__()
        self.register_buffer("weight", weight)
        self.register_buffer("U_r", U_r); self.register_buffer("P_r", P_r)
        self.register_buffer("rho", rho)
        self.shortlist = int(shortlist); self.kappa = float(kappa)
    def forward(self, h):
        # any leading shape ([T, d] or [B, L, d]): flatten, run, restore
        lead = h.shape[:-1]
        h = h.reshape(-1, h.shape[-1])
        # Chunked over T: weight[sl] is [t, V_s, d] (~84 MB/token on the 27B
        # head, bf16) — scoring is per-token, so chunking is mathematically
        # exact and bounds the gather to ~512 MiB (A100 OOM regression).
        per_tok = self.shortlist * self.weight.shape[1] * self.weight.element_size()
        c = max(1, (512 << 20) // max(1, per_tok))
        sl_parts, ex_parts = [], []
        for i in range(0, h.shape[0], c):
            hc = h[i:i + c].to(self.weight.dtype)
            hp = hc @ self.P_r
            pc = hp @ self.U_r.t()
            if self.kappa and self.rho is not None:
                s = (hc.float().pow(2).sum(-1) - hp.float().pow(2).sum(-1)).clamp_min(0).sqrt()
                pc = pc + (self.kappa * s)[:, None].to(pc.dtype) * self.rho[None, :].to(pc.dtype)
            slc = pc.topk(self.shortlist, dim=-1).indices
            rows = self.weight[slc]
            ex_parts.append(torch.bmm(rows, hc.unsqueeze(-1)).squeeze(-1).to(h.dtype))
            sl_parts.append(slc)
        sl = torch.cat(sl_parts, 0)
        full = h.new_full((h.shape[0], self.weight.shape[0]), float("-inf"))
        full.scatter_(1, sl, torch.cat(ex_parts, 0).to(full.dtype))
        return full.reshape(*lead, full.shape[-1])
"""

SERVE_NATIVE = r"""
# serve_native.py — RESONA native launcher (full ~3B-active execution).
# Usage:  python serve_native.py --dir <NATIVE_DIR> [--prompt "..."] [--chat]
# Loads the COMPAT checkpoint (standard HF Qwen3_5Moe weights), wires the
# RESONA execution from resona_sidecar.pt, serves greedy/temperature chat.
import argparse, json, os, torch
from resona_kernels import (AntennaMast, ResonantLinear, SectionedProjector,
                            FunnelHead, MoRM2PolicyLite, BetaGateLite)
from transformers import AutoConfig, AutoModelForImageTextToText, AutoTokenizer

def wire(model, sidecar, device):
    mast = AntennaMast(sidecar["mast_R"].to(device),
                       seed=sidecar.get("mast_seed", 0),
                       d_model=sidecar.get("mast_d_model"))
    named_p = dict(model.named_parameters())
    named_m = dict(model.named_modules())
    for entry in sidecar["sites"]["linear"]:
        p = entry["path"]
        if "(gate)" in p:
            # '<attn>.v_proj(gate)': replace v_proj with (dense v half + resonant gate)
            attn_pfx = p.split("(gate)")[0]
            attn = named_m[attn_pfx.rpartition(".")[0]]
            W = named_p[attn_pfx + ".weight"].data
            rl = ResonantLinear(W[W.shape[0]//2:], entry["carrier_idx"].to(device),
                                entry["block_idx"].to(device),
                                entry["router"].to(device),
                                entry["gains"].to(device), mast, entry["topk"])
            class VGateShim(torch.nn.Module):
                def __init__(self, v_w, rl_gate):
                    super().__init__()
                    self.register_buffer("v_w", v_w); self.rl_gate = rl_gate
                def forward(self, x):
                    return torch.cat([torch.nn.functional.linear(x, self.v_w),
                                      self.rl_gate(x)], dim=-1)
            setattr(attn, attn_pfx.rpartition(".")[2], VGateShim(W[:W.shape[0]//2], rl))
        else:
            parent = model.get_submodule(p.rpartition(".")[0])
            W = named_p[p + ".weight"].data
            _sU, _sM = entry.get("sh_U"), entry.get("sh_M")
            rl = ResonantLinear(W, entry["carrier_idx"].to(device),
                                entry["block_idx"].to(device),
                                entry["router"].to(device),
                                entry["gains"].to(device), mast, entry["topk"],
                                sh_U=(_sU.to(device) if _sU is not None else None),
                                sh_M=(_sM.to(device) if _sM is not None else None))
            setattr(parent, p.rpartition(".")[2], rl)
    for entry in sidecar["sites"]["sectioned"]:
        # entry["path"] ends with the ACTUAL projection attribute
        # (in_proj_qkvz fused OR in_proj_qkv split) — resolve generically.
        p = entry["path"]
        parent = model.get_submodule(p.rpartition(".")[0])
        attr = p.rpartition(".")[2]
        W = getattr(parent, attr).weight.data
        clean = {k: (v.to(device) if torch.is_tensor(v) else v)
                 for k, v in entry.items() if k != "path"}
        sp = SectionedProjector(W, clean, mast)
        setattr(parent, attr, sp)
        # v8.0.2 [P5]: install the WRITE lever (the exact skip, Prop F) —
        # qwen3_5: in_proj_b (split, b_only); fused families: in_proj_ba.
        # The compact projector's per-head v-GEMM sparsity stays (a masked
        # head's v is never read once beta ~ 0); the WRITE semantics move
        # to the b gate. v1 exports (no morm2 entry) keep the old v-mask.
        # v10: with a v shadow every head writes — no write gate (the mask only picks the EXACT heads)
        if entry.get("morm2") is not None and entry.get("v_sh_U") is None:
            for b_attr in ("in_proj_b", "in_proj_ba"):
                b_mod = getattr(parent, b_attr, None)
                if isinstance(b_mod, torch.nn.Module) and \
                        getattr(b_mod, "weight", None) is not None:
                    # v9: ONE policy per site, shared by the v-GEMM mask and the write gate. v8 built a second
                    # one from the raw (CPU) sidecar tensors: a device mismatch on GPU, and two masks to keep
                    # in sync. sp.morm already lives on `device`.
                    pol = sp.morm
                    setattr(parent, b_attr,
                            BetaGateLite(b_mod, pol, n_heads=sp.n_v_heads,
                                         logit_floor=float(
                                             entry["morm2"]["logit_floor"]),
                                         b_first=True))
                    break
    # v10 FFN shadow: served FFN output += b + U (M h)
    class ShadowMLP(torch.nn.Module):
        def __init__(self, inner, U, M, b):
            super().__init__()
            self.inner = inner
            self.register_buffer("U", U); self.register_buffer("M", M); self.register_buffer("b", b)
        def forward(self, h, *a, **k):
            out = self.inner(h, *a, **k)
            s = ((h.to(self.M.dtype) @ self.M.t()) @ self.U.t() + self.b).to(h.dtype)
            return (out[0] + s,) + tuple(out[1:]) if isinstance(out, tuple) else out + s
    for e in (sidecar.get("ffn_shadow") or []):
        parent = model.get_submodule(e["path"].rpartition(".")[0])
        attr = e["path"].rpartition(".")[2]
        setattr(parent, attr, ShadowMLP(getattr(parent, attr), e["U"].to(device), e["M"].to(device),
                                        e["b"].to(device)))
    head_w = named_p["lm_head.weight"].data
    _rho = sidecar.get("funnel_rho")
    model.lm_head = FunnelHead(head_w, sidecar["funnel_U_r"].to(device),
                               sidecar["funnel_P_r"].to(device),
                               int(sidecar["funnel_shortlist"]),
                               rho=(_rho.to(device) if _rho is not None else None),
                               kappa=float(sidecar.get("funnel_kappa", 0.0)))
    return model

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dir", required=True)
    ap.add_argument("--prompt", default="Explain resonant execution in one paragraph.")
    ap.add_argument("--max-new", type=int, default=256)
    ap.add_argument("--temperature", type=float, default=0.7)
    args = ap.parse_args()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    sidecar = torch.load(os.path.join(args.dir, "resona_sidecar.pt"), weights_only=False)
    from transformers import AutoModelForCausalLM
    cfg = AutoConfig.from_pretrained(args.dir)
    _AM = (AutoModelForImageTextToText
           if getattr(cfg, "vision_config", None) is not None else AutoModelForCausalLM)
    model = _AM.from_pretrained(args.dir, dtype=torch.bfloat16, device_map=dev)
    model = wire(model, sidecar, dev); model.eval()
    tok = AutoTokenizer.from_pretrained(args.dir)
    ids = tok(args.prompt, return_tensors="pt").input_ids.to(dev)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=args.max_new,
                             do_sample=args.temperature > 0,
                             temperature=max(args.temperature, 1e-4))
    print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

if __name__ == "__main__":
    main()
"""

with open(os.path.join(NATIVE_DIR, "resona_kernels.py"), "w") as f:
    f.write(RESONA_KERNELS)
# funnel factors into the sidecar for the launcher
sidecar["funnel_U_r"] = funnel.U_r.detach().cpu()
sidecar["funnel_P_r"] = funnel.P_r.detach().cpu()
sidecar["funnel_rho"] = funnel.rho.detach().cpu() if funnel.rho is not None else None
sidecar["funnel_kappa"] = float(funnel.kappa)
sidecar["funnel_shortlist"] = int(funnel.shortlist)
torch.save(sidecar, os.path.join(NATIVE_DIR, "resona_sidecar.pt"))
with open(os.path.join(NATIVE_DIR, "serve_native.py"), "w") as f:
    f.write(SERVE_NATIVE)
_ACT_NAT = (globals().get("act_native") or WF["TOTAL_ACTIVE"]) / 1e9
_ACT_CMP = (globals().get("act_compat") or 0) / 1e9
with open(os.path.join(NATIVE_DIR, "README.md"), "w") as f:
    f.write(f"""# RESONA Native Artifact ({'Qwen3.8-27B-A3B' if RUN_MODE=='full' else 'demo'})

Two ways to serve:

1) **COMPAT (stock vLLM, ~{_ACT_CMP:.1f}B active)**: serve `../{os.path.basename(EXPORT_DIR)}`
   with plain `vllm serve` — it is a standard Qwen3_5MoeForConditionalGeneration.

2) **NATIVE (~{_ACT_NAT:.2f}B active by census, the 3B mandate)**: this directory.
   `pip install transformers accelerate torch>=2.4`
   `python serve_native.py --dir . --prompt "Hello"`
   The launcher loads the COMPAT checkpoint for weights, wires the RESONA
   execution (resonant linears, MoRM write-sparse memory, head funnel) from
   `resona_sidecar.pt`, and runs generation. Research-grade reference launcher:
   correctness is anchored by the notebook's Theorem-7 checks; throughput work
   (grouped kernels, vLLM plugin registration) is the documented next step.

Accounting (strict touched weights per text token, '{MAST_ACCOUNTING}' sketch accounting): the antenna
sketch is charged at every (layer, input width) where it is evaluated; router reads, the dense k/v/in_proj_a
sites, DeltaNet conv taps of the selected channels and the head funnel (rank-r projection + residual norms
+ the exact shortlist rows) are all inside the number. NOTE: the reference launcher recomputes the sketch at
EVERY routed site, so its own cost is higher than the budget's, which assumes one evaluation per layer and
width — sharing phi within a layer is the documented optimisation.

Funnel shortlist recall: {FUNNEL_RECALL} (residual-aware kappa={FUNNEL_KAPPA})
Solved plan: {json.dumps(PLAN)}
""")
print(f"native package written -> {NATIVE_DIR}")
import py_compile
py_compile.compile(os.path.join(NATIVE_DIR, "resona_kernels.py"), doraise=True)
py_compile.compile(os.path.join(NATIVE_DIR, "serve_native.py"), doraise=True)
print("launcher syntax check: PASS")


## Serve it — two artifacts, both real

**Artifact A — COMPAT (stock vLLM, zero custom code).** `EXPORT_DIR` is a **standard `Qwen3_5MoeForConditionalGeneration`** checkpoint — vLLM ships this architecture (verified: `vllm/model_executor/models/registry.py` exposes `Qwen3_5MoeForConditionalGeneration`, `Qwen3_5MoeForCausalLM`, `Qwen3_5MTP`). FFN runs sparse (our fission); attention/head run dense (stock-arch necessity) → **≈ 9.1B active** on the 27B (text token) — already 3× cheaper than dense and the safest deployment.

**Artifact B — NATIVE (the 3B mandate).** `NATIVE_DIR` adds `resona_sidecar.pt` + a self-contained launcher (`resona_kernels.py`, `serve_native.py`): the full RESONA execution — resonant linears, MoRM write-sparse memory, spectral head funnel — at **≈ 2.99B active** (strict touched-weights, census of the built graph). It loads Artifact A for weights and wires the sparse geometry from the sidecar (including the funnel's `rho` / `kappa`):

```bash
python serve_native.py --dir NATIVE_DIR --prompt "Explain MoE routing."
```

The reference launcher is correctness-first: it recomputes the sketch phi at every routed site, whereas the budget (and a production kernel) evaluates it once per layer and input width — see the README written next to the launcher.

**Preserved by construction (both artifacts):** 262,144-token context (`max_position_embeddings` + mRoPE untouched), multimodal input (vision tower + processor copied), MTP draft carried in config, and the 131K-class output budget is a *serving-time* setting (`max_tokens` / `max_new_tokens`).


In [ ]:
# Serving — vLLM smoke test (offline LLM API)
# vLLM's gpu_memory_utilization is a fraction of TOTAL VRAM and the engine
# profiles FREE memory at startup: park the whole student world on CPU RAM
# first, run vLLM, tear it down (del + gc + empty_cache), then restore.
# Also: installing vLLM mid-session can swap the torch wheel underneath an
# already-imported torch — detect it and degrade to guidance instead of an
# obscure CUDA error.
INSTALL_VLLM = True   # set False if already installed / to skip the heavy install
if INSTALL_VLLM:
    print("installing vLLM (this can take a few minutes) ...")
    _torch_before = torch.__version__
    get_ipython().system('pip install -q -U vllm 2>&1 | tail -1')
    try:
        import importlib.metadata as _md
        _torch_after = _md.version("torch")
        if _torch_after != _torch_before:
            print(f"!! vLLM install moved torch {_torch_before} -> "
                  f"{_torch_after}. The in-session torch stays {_torch_before}; "
                  "if vLLM fails below, use Runtime > Restart and re-run ONLY "
                  "the serving cells — the exported checkpoint on Drive is "
                  "already complete.")
    except Exception:
        pass

llm = None
_prev_alloc_conf = None
_parked_for_vllm = False
try:
    from vllm import LLM, SamplingParams
    # -- park the student world on CPU (trainee + mast + funnel) ------------
    # v8.0.3 [OOM-3]: (a) parking needs ~50 GiB of HOST RAM — refuse instead of
    # crashing the VM; (b) vLLM measures gpu_memory_utilization against FREE
    # memory at start-up, so size it from what is really free (a few leftover
    # GiB used to abort engine start); (c) expandable_segments is a setting for
    # THIS process — the engine subprocess must not inherit it.
    _student_gib = sum(p.numel() * p.element_size() for p in model.parameters()) / 2**30
    _ram_av, _ram_tot = host_ram_gib()
    if torch.cuda.is_available() and _ram_av < _student_gib + 8.0:
        raise RuntimeError(f"not enough host RAM to park the student for vLLM "
                           f"({_ram_av:.0f}/{_ram_tot:.0f} GiB free, need ~{_student_gib + 8:.0f})")
    model.to("cpu"); MAST.to("cpu"); funnel.to("cpu")
    _parked_for_vllm = True
    gc.collect(); torch.cuda.empty_cache()
    free_cuda("student parked for vLLM")
    _vfree, _vtot = vram_gib()
    _frac = min(VLLM_GPU_FRAC, max(0.0, (_vfree - 2.0) / max(_vtot, 1e-9)))
    if _vtot and _frac < 0.5:
        raise RuntimeError(f"only {_vfree:.0f}/{_vtot:.0f} GiB of GPU memory is free after parking "
                           "the student — something still holds it (restart the session and "
                           "re-run the serving cells only; the exported checkpoint is complete)")
    print(f"vLLM gpu_memory_utilization = {_frac:.2f} (configured {VLLM_GPU_FRAC}, "
          f"{_vfree:.0f} GiB free)")
    _prev_alloc_conf = os.environ.pop("PYTORCH_CUDA_ALLOC_CONF", None)
    llm = LLM(model=EXPORT_DIR, max_model_len=8192,      # smoke-test length
              gpu_memory_utilization=_frac, enforce_eager=True)
    msgs = [[{"role": "user", "content":
              "In one sentence, explain what a mixture-of-experts layer is."}]]
    out = llm.chat(msgs, SamplingParams(temperature=0.7, max_tokens=128))
    print("TEXT GENERATION OK:\n", out[0].outputs[0].text[:400])

    # multimodal smoke test (the vision tower survived the transformation)
    try:
        img_msgs = [[{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url":
             "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/rabbit.png"}},
            {"type": "text", "text": "What is in this image? One sentence."}]}]]
        out_img = llm.chat(img_msgs, SamplingParams(temperature=0.2, max_tokens=64))
        print("\nIMAGE GENERATION OK:\n", out_img[0].outputs[0].text[:300])
    except Exception as e:
        print("image test skipped (CPU-only runtime or vLLM version):", e)
except Exception as e:
    print("vLLM offline test skipped:", e)
    print("-> the checkpoint is still a standard MoE; serve it with the CLI below.")
finally:
    # -- tear vLLM down and hand the GPU back to the student -----------------
    if _prev_alloc_conf is not None:
        os.environ["PYTORCH_CUDA_ALLOC_CONF"] = _prev_alloc_conf
    if llm is not None:
        # v8.0.3: dropping the Python handle does not always stop the EngineCore
        # subprocess promptly, and until it exits the GPU stays claimed (the
        # student would then be left on host RAM and every later cell crawls).
        try:
            llm.llm_engine.engine_core.shutdown()
        except Exception:
            pass
        del llm
    try:
        from vllm.distributed.parallel_state import (
            destroy_model_parallel, destroy_distributed_environment)
        destroy_model_parallel(); destroy_distributed_environment()
    except Exception:
        pass
    gc.collect(); torch.cuda.empty_cache()
    if _parked_for_vllm:                     # only if the student was actually moved off the GPU
        if torch.cuda.is_available():
            _need = sum(p.numel() * p.element_size() for p in model.parameters())
            for _ in range(45):              # wait <= 90 s for the engine to release the GPU
                _free, _tot = torch.cuda.mem_get_info()
                if _free >= _need * 1.15:
                    break
                time.sleep(2)
            if _free >= _need * 1.15:
                model.to(DEV); MAST.to(DEV); funnel.to(DEV)
                free_cuda("student restored after vLLM")
            else:
                print(f"[vLLM teardown] only {_free/2**30:.0f} GiB free (student "
                      f"needs {_need/2**30:.0f} GiB) — student left on CPU; the "
                      "SLIPSTREAM cell below auto-adapts (it re-derives DEV and "
                      "builds its store from CPU weights just as safely).")
        else:
            model.to(DEV)   # CPU-only runtime: no-op


## SLIPSTREAM — the anti-throttle serving substrate

**MoRM 2.0 decode integration (v8).** The NATIVE decode loop drives each
GDN layer's writes through `MoRM2Decoder`: one `step()` per generated token
emits (k_eff, v_eff, beta_eff) for the stock kernel — elastic budgets,
deferred commits with exact decay compensation, and the charge ledger all
live in that per-layer scheduler state (serialized in
`resona_sidecar.pt["sites"]["sectioned"][i]["morm2"]`). The COMPAT artifact
is unaffected (stock vLLM executes dense attention semantics there).

**The throttle.** After RESONA surgery a decode step touches ~3B params' worth of bytes — but only if every touched byte is already in VRAM. With routed-expert slabs on disk, every routing miss is a synchronous disk read (~seconds per token, naive).

- **S1 (Lemma S1, router-margin stability):** reusing the previous token's routing is *certified safe* whenever the router margin $m_t > 2L\rho_{t+1}$ — a distribution-free per-step certificate, measured on the real routers below.
- **S2 (Prop S2, 2-bit cold tier):** slabs stored group-wise 2-bit (~0.28 B/param, 7.1× under bf16) with a worst-case Frobenius bound.
- **S3 (Theorem S3, no-stall):** if $(1-h)\,c/\mathrm{BW}_{disk} \le T_{comp}$, speculative prefetch hides **all** disk latency and decode runs at resident speed.

This cell holds the core, compact assertions, and the **on-model benchmark** (real 2-bit store, real routing schedule, Drive-class and NVMe-class bandwidth emulation, the Theorem-S3 table at 27B scale).


In [ ]:
# ============================================================================
# SLIPSTREAM core — 2-bit expert store · LRU hot set · speculative prefetch
# ============================================================================
# A converted 27B-A3B model still STORES 27B parameters; if the routed-
# expert slabs live on disk (Colab, offload, Drive), naive offload reads
# the whole touched footprint from storage on every miss — ~seconds per
# token. SLIPSTREAM removes the throttle with three verified mechanisms:
#
#   S1 Router-margin stability (Lemma S1): reusing the previous token's
#      routing is SAFE whenever the router margin m_t > 2*L*rho_{t+1} — a
#      distribution-free, per-step certificate (certified hit-rate floor).
#   S2 2-bit cold tier (Prop S2): routed slabs stored group-wise 2-bit
#      (~0.28 B/param, 7.1x under bf16) with a worst-case ||W-Q||_F bound:
#      ||W-Q||_F <= (1/3) sqrt(sum_g G max|W_g|^2), ||(W-Q)x|| <= bound*||x||.
#   S3 No-stall pipeline (Theorem S3): if (1-h)*cold_bytes / BW_disk <=
#      T_compute, one-step speculative prefetch hides ALL disk latency and
#      decode runs at the resident speed — tokens/sec becomes compute-bound,
#      not disk-bound.
# ============================================================================

import json
import os
import queue
import struct
import threading
import time
from collections import OrderedDict

import numpy as np
import torch
import torch.nn.functional as F

# ---------------------------------------------------------------------------
# S2 · BitPack2 — group-wise 2-bit weight storage
# ---------------------------------------------------------------------------
# Per group g of G weights: s_g = max|W_g| / 1.5, levels {±0.5, ±1.5}·s_g.
# Nearest-level rounding gives |w - q| <= s_g / 2 = max|W_g| / 3, hence
#   ||W - Q||_F <= (1/3) · sqrt( sum_g G · max|W_g|^2 )          (Prop S2)
# and ||(W-Q)x||_2 <= ||W-Q||_F · ||x||_2  (Cauchy-Schwarz).

_LUT2 = torch.tensor([-1.5, -0.5, 0.5, 1.5])


def pack2(W: torch.Tensor, group: int = 64) -> dict:
    # W can arrive on CUDA (the on-model store build gathers GPU weights)
    # — move to host FIRST (identical math, any device).
    Wf = W.detach().to(device="cpu", dtype=torch.float32).reshape(-1)
    n = Wf.numel()
    ng = (n + group - 1) // group
    pad = ng * group - n
    if pad:
        Wf = torch.cat([Wf, Wf.new_zeros(pad)])
    G = Wf.reshape(ng, group)
    m = G.abs().amax(dim=1, keepdim=True).clamp_min(1e-12)
    s = m / 1.5                                            # [ng, 1]
    idx = torch.clamp(torch.round(G / s + 1.5), 0, 3).to(torch.uint8)
    idx = idx.reshape(-1)
    codes = (idx[0::4] | (idx[1::4] << 2) | (idx[2::4] << 4)
             | (idx[3::4] << 6))
    return {"codes": codes.numpy().tobytes(),
            "scales": s.reshape(-1).to(torch.float16).numpy(),
            "numel": int(n), "group": int(group),
            "shape": tuple(int(x) for x in W.shape)}


def unpack2(pk: dict, device="cpu", dtype=torch.float32) -> torch.Tensor:
    n, g = pk["numel"], pk["group"]
    ng = len(pk["scales"])
    b = torch.frombuffer(bytearray(pk["codes"]), dtype=torch.uint8)
    idx = torch.stack([b & 3, (b >> 2) & 3, (b >> 4) & 3, (b >> 6) & 3],
                      dim=1).reshape(-1)[: ng * g]
    lut = _LUT2.to(device)
    s = torch.from_numpy(np.array(pk["scales"])).to(device).float()
    W = lut[idx.to(device).long()] * s.repeat_interleave(g)[: ng * g]
    return W[:n].reshape(pk["shape"]).to(dtype)


def pack2_bound(W: torch.Tensor, group: int = 64) -> float:
    """Prop S2 upper bound on ||W - Q||_F / ||W||_F (computable pre-quant):
    ||W-Q||_F <= (1/3)·sqrt( sum_g G · max|W_g|^2 )."""
    Wf = W.detach().to(device="cpu", dtype=torch.float32).reshape(-1)
    n = Wf.numel()
    ng = (n + group - 1) // group
    pad = ng * group - n
    if pad:
        Wf = torch.cat([Wf, Wf.new_zeros(pad)])
    m = Wf.reshape(ng, group).abs().amax(dim=1)
    denom = W.detach().float().norm().clamp_min(1e-12)
    return float(((1.0 / 3.0) * torch.sqrt(group * (m ** 2).sum())) / denom)


def pack2_bytes(numel: int, group: int = 64) -> int:
    ng = (numel + group - 1) // group
    return (numel + 3) // 4 + 2 * ng


# ---------------------------------------------------------------------------
# ExpertStore — per-(layer, expert) blob files, the ONLY cold-tier format
# ---------------------------------------------------------------------------
class ExpertStore:
    def __init__(self, root: str):
        self.root = root
        os.makedirs(root, exist_ok=True)
        self.index = {}                     # (layer, expert) -> meta
        self.bytes_written = 0

    def put_triple(self, layer, expert, gate, up, down, group=64):
        parts, metas, off = [], [], 0
        for name, W in (("gate", gate), ("up", up), ("down", down)):
            pk = pack2(W, group)
            sb = np.asarray(pk["scales"]).tobytes()
            cb = pk["codes"]
            metas.append({"name": name, "shape": list(pk["shape"]),
                          "numel": pk["numel"], "group": pk["group"],
                          "scales_off": off, "scales_len": len(sb),
                          "codes_off": off + len(sb),
                          "codes_len": len(cb)})
            parts += [sb, cb]
            off += len(sb) + len(cb)
        hdr = json.dumps({"layer": int(layer), "expert": int(expert),
                          "tensors": metas}).encode()
        blob = struct.pack("<I", len(hdr)) + hdr + b"".join(parts)
        path = os.path.join(self.root, f"l{layer}_e{expert}.bin")
        with open(path, "wb") as f:
            f.write(blob)
        self.index[(int(layer), int(expert))] = {
            "path": path, "bytes": len(blob),
            "gate_shape": metas[0]["shape"]}
        self.bytes_written += len(blob)
        return len(blob)

    def read_blob(self, layer, expert) -> bytes:
        meta = self.index[(int(layer), int(expert))]
        with open(meta["path"], "rb") as f:
            return f.read()

    @staticmethod
    def _split_blob(blob: bytes):
        (hl,) = struct.unpack_from("<I", blob, 0)
        hdr = json.loads(blob[4:4 + hl])
        return hdr, blob[4 + hl:]

    def unpack_triple(self, layer, expert, device="cpu",
                      dtype=torch.float32):
        blob = self.read_blob(layer, expert)
        hdr, payload = self._split_blob(blob)
        out = {}
        for t in hdr["tensors"]:
            scales = np.frombuffer(
                payload[t["scales_off"]:t["scales_off"] + t["scales_len"]],
                dtype=np.float16)
            codes = payload[t["codes_off"]:t["codes_off"] + t["codes_len"]]
            pk = {"codes": codes, "scales": scales, "numel": t["numel"],
                  "group": t["group"], "shape": tuple(t["shape"])}
            out[t["name"]] = unpack2(pk, device=device, dtype=dtype)
        return out["gate"], out["up"], out["down"]

    def raw_bytes_per_expert(self, numel_gate, numel_up, numel_down,
                             group=64, header_overhead=256):
        return (pack2_bytes(numel_gate, group)
                + pack2_bytes(numel_up, group)
                + pack2_bytes(numel_down, group) + header_overhead)


# ---------------------------------------------------------------------------
# SlipstreamCache — LRU hot set + certified counters
# ---------------------------------------------------------------------------
class SlipstreamCache:
    def __init__(self, store: ExpertStore, capacity: int = 64, device="cpu",
                 dtype=torch.float32, sim_disk_bw=None):
        self.store = store
        self.cap = capacity
        self.device = device
        self.dtype = dtype
        self.sim_disk_bw = sim_disk_bw          # bytes/sec, or None = raw
        self.entries = OrderedDict()            # (l, e) -> (Wg, Wu, Wd)
        self.hits = self.misses = 0
        self.bytes_disk = 0.0
        self.disk_time = 0.0

    def __contains__(self, key):
        return key in self.entries

    def get(self, layer, expert):
        key = (int(layer), int(expert))
        if key in self.entries:
            self.hits += 1
            self.entries.move_to_end(key)
            return self.entries[key]
        self.misses += 1
        t0 = time.perf_counter()
        blob = self.store.read_blob(*key)
        nb = len(blob)
        if self.sim_disk_bw:
            time.sleep(nb / self.sim_disk_bw)
        trip = self.store.unpack_triple(*key, device=self.device,
                                        dtype=self.dtype)
        dt = time.perf_counter() - t0
        self.bytes_disk += nb
        self.disk_time += dt
        self.entries[key] = trip
        if self.cap is not None:
            while len(self.entries) > self.cap:
                self.entries.popitem(last=False)
        return trip

    @property
    def hit_rate(self):
        tot = self.hits + self.misses
        return self.hits / tot if tot else float("nan")


# ---------------------------------------------------------------------------
# SpeculativePrefetcher — Lemma S1 made concurrent
# ---------------------------------------------------------------------------
class SpeculativePrefetcher(threading.Thread):
    """Prefetches the PREDICTED next-step experts (prediction = the previous
    token's routing; safety certified by Lemma S1) during the current step's
    compute window. `useful` = prefetches that turned a would-be sync miss
    into a hit."""

    def __init__(self, cache: SlipstreamCache, maxsize: int = 4096):
        super().__init__(daemon=True)
        self.cache = cache
        self.q = queue.Queue(maxsize=maxsize)
        self.submitted = self.useful = self.dropped = 0
        self.stopped = False

    def submit(self, keys):
        for k in keys:
            k = (int(k[0]), int(k[1]))
            if k in self.cache:
                continue
            try:
                self.q.put_nowait(k)
                self.submitted += 1
            except queue.Full:
                self.dropped += 1

    def run(self):
        while not self.stopped:
            try:
                key = self.q.get(timeout=0.01)
            except queue.Empty:
                continue
            was_cached = key in self.cache
            self.cache.get(*key)
            if not was_cached:
                self.useful += 1

    def stop(self):
        self.stopped = True


# ---------------------------------------------------------------------------
# Lemma S1 / Theorem S3 machinery
# ---------------------------------------------------------------------------
def router_lipschitz(W: torch.Tensor, tau: float = 1.0) -> float:
    """L for a linear router f(x) = tau · W x  (op-2 norm)."""
    return float(torch.linalg.matrix_norm(W.detach().float(), ord=2)) \
        * abs(float(tau))


def margins_and_radii(logits: torch.Tensor, inputs: torch.Tensor):
    """m_t = k-th minus (k+1)-th logit gap-precursor (a-b over ALL margins):
    here m_t = gap between top-1... we return the sorted-logit sequence so the
    caller picks k. rho_t = ||x_t - x_{t-1}||."""
    s = logits.sort(-1, descending=True).values
    rho = (inputs[1:].float() - inputs[:-1].float()).norm(dim=-1)
    return s, rho


def margin_certificate(logits: torch.Tensor, inputs: torch.Tensor, k: int,
                       L: float):
    """Certified reuse steps: t is certified iff
        m_t = f_(k)(x_t) - f_(k+1)(x_t) > 2 L rho_{t+1}
    Lemma S1 then guarantees S_{t+1} == S_t on EVERY certified step."""
    s, rho = margins_and_radii(logits, inputs)
    m = s[:, k - 1] - s[:, k]
    cert = m[:-1] > 2 * L * rho
    return cert, float(cert.float().mean().item())


def reuse_hit_rate(logits: torch.Tensor, k: int) -> float:
    """Set-equality hit rate of reusing the previous step's top-k experts.
    SET semantics (order within the top-k is irrelevant for the cache)."""
    idx = logits.argsort(-1, descending=True)[:, :k].sort(-1).values
    same = (idx[1:] == idx[:-1]).all(-1)
    return float(same.float().mean().item())


def throughput_table(total_model_b, active_b, ff_routed_b,
                     bw_vram, bw_disk, hit, comp_s=0.0):
    """Theorem S3 — steady-state decode, batch 1. Byte counts in BYTES
    (pass e.g. 27e9*2 for bf16). Returns tok/s per regime + no-stall flag.

      resident : T = max(comp, active / bw_vram)
      naive    : T = max(comp, active / bw_vram + ff_routed / bw_disk)
      slip     : T = max(comp, active / bw_vram) provided
                 (1 - hit) · ff_routed_2bit / bw_disk <= comp   (no-stall)
    """
    t_res = max(comp_s, active_b / bw_vram)
    t_naive = max(comp_s, active_b / bw_vram + ff_routed_b / bw_disk)
    cold_2bit = ff_routed_b / 8.0
    stall_s = (1.0 - hit) * cold_2bit / bw_disk
    no_stall = stall_s <= comp_s + active_b / bw_vram
    t_slip = max(comp_s, active_b / bw_vram)
    return {"resident_tps": 1.0 / t_res,
            "naive_tps": 1.0 / t_naive,
            "slipstream_tps": 1.0 / t_slip,
            "no_stall": bool(no_stall),
            "stall_s": stall_s,
            "cold_2bit_bytes": cold_2bit}


# ---------------------------------------------------------------------------
# Benchmark harness — resident vs naive-offload vs SLIPSTREAM
# ---------------------------------------------------------------------------
def bench_decode(schedule, mode, store, capacity=64, prefetch=True,
                 sim_compute_s=0.0, sim_disk_bw=None, device="cpu",
                 dtype=torch.float32, seed=0):
    """schedule: list of steps; step = list of (layer, [expert_ids]) — the
    REAL routing decisions measured on the converted model.

      resident   : every expert materialized once; zero disk I/O.
      naive      : disk read on EVERY use (capacity 0, no prefetch) — the
                   'read the whole model from disk' throttle.
      slipstream : LRU cache + Lemma-S1 speculative prefetch (previous step's
                   routing) + synchronous fallback on misprediction.
    """
    torch.manual_seed(seed)
    keys = []
    for step in schedule:
        for layer, ids in step:
            keys += [(int(layer), int(e)) for e in ids]
    uniq = list(dict.fromkeys(keys))

    if mode == "resident":
        pre = {}
        for k in uniq:
            pre[k] = store.unpack_triple(*k, device=device, dtype=dtype)
        wall0 = time.perf_counter()
        for step in schedule:
            for layer, ids in step:
                for e in ids:
                    _ = pre[(int(layer), int(e))]
            if sim_compute_s:
                time.sleep(sim_compute_s)
        wall = time.perf_counter() - wall0
        return {"mode": mode, "wall_s": wall,
                "tok_per_s": len(schedule) / wall, "disk_bytes": 0.0,
                "hit_rate": 1.0, "prefetch_useful": 0}

    cap = 0 if mode == "naive" else capacity
    cache = SlipstreamCache(store, capacity=cap, device=device,
                            dtype=dtype, sim_disk_bw=sim_disk_bw)
    pf = None
    if mode == "slipstream" and prefetch:
        pf = SpeculativePrefetcher(cache)
        pf.start()
    wall0 = time.perf_counter()
    prev_keys = []
    for step in schedule:
        if pf is not None and prev_keys:
            pf.submit(prev_keys)            # predict next step = last step
        if sim_compute_s:
            time.sleep(sim_compute_s)       # the compute window
        for layer, ids in step:
            for e in ids:
                _ = cache.get(layer, e)
        prev_keys = [(layer, e) for layer, ids in step for e in ids]
    wall = time.perf_counter() - wall0
    if pf is not None:
        pf.stop()
        pf.join(timeout=1.0)
    return {"mode": mode, "wall_s": wall,
            "tok_per_s": len(schedule) / wall,
            "disk_bytes": cache.bytes_disk,
            "hit_rate": cache.hit_rate,
            "prefetch_useful": pf.useful if pf else 0}


# ---------------------------------------------------------------------------
# Compact verification (asserts; the on-model benchmark below exercises the
# same machinery end-to-end on the real converted model).
# ---------------------------------------------------------------------------
import shutil
import tempfile

W_t = torch.randn(256, 192)
pk = pack2(W_t, group=64)
err_q = float((W_t - unpack2(pk)).norm() / W_t.norm())
bnd_q = pack2_bound(W_t, group=64)
assert err_q <= bnd_q, f"S2 round-trip {err_q:.4f} > bound {bnd_q:.4f}"
assert len(pk["codes"]) + 2 * len(pk["scales"]) == pack2_bytes(W_t.numel(), 64)
print(f"[slip] S2 2-bit round-trip within the Frobenius bound: "
      f"err {err_q:.4f} <= {bnd_q:.4f}")

# Lemma S1: zero violations on certified steps over a smooth random walk
# (the real regime: hidden states evolve smoothly, so rho_t is small).
Wr = torch.randn(8, 16) * 0.5
L_walk = router_lipschitz(Wr)
Xw = torch.empty(4000, 16)
Xw[0] = torch.randn(16)
for t_ in range(1, Xw.shape[0]):
    Xw[t_] = Xw[t_ - 1] + 0.01 * torch.randn(16)
logits_w = Xw @ Wr.t()
cert_w, frac_w = margin_certificate(logits_w, Xw, 2, L_walk)
idx_w = logits_w.argsort(-1, descending=True)[:, :2].sort(-1).values
changed_w = ~(idx_w[1:] == idx_w[:-1]).all(-1)
assert int((cert_w & changed_w).sum()) == 0, "Lemma S1 violation on a certified step"
assert frac_w > 0.5, "certificate trivially small"
reuse_w = reuse_hit_rate(logits_w, 2)
assert reuse_w >= frac_w - 1e-9, "measured reuse below the certified floor"
print(f"[slip] Lemma S1: 0 violations on {int(cert_w.sum())} certified steps "
      f"(floor {frac_w:.3f} <= reuse {reuse_w:.3f})")

# Theorem S3 regime ordering at 27B-scale numbers.
tbl = throughput_table(total_model_b=27.4e9 * 2, active_b=3.0e9 * 2,
                       ff_routed_b=1.4e9 * 2, bw_vram=1.0e12, bw_disk=7e9,
                       hit=0.97, comp_s=0.006)
assert tbl["naive_tps"] < 0.5 * tbl["slipstream_tps"] \
    and tbl["slipstream_tps"] >= 0.999 * tbl["resident_tps"] \
    and tbl["no_stall"] is True, "Theorem S3 ordering violated"
print(f"[slip] Theorem S3: naive {tbl['naive_tps']:.1f} << slipstream "
      f"{tbl['slipstream_tps']:.0f} ~= resident {tbl['resident_tps']:.0f} tok/s "
      f"(stall {tbl['stall_s'] * 1e3:.3f} ms/token)")


# ============================================================================
# ---------------------------------------------------------------------------
# On-model benchmark: real 2-bit store from the fissioned FFNs, real routing
# schedule replayed through the trained routers, Lemma-S1 certificate per
# layer, and the three-regime memory benchmark (Drive-class and NVMe-class
# bandwidth emulation) + the Theorem-S3 table at 27B scale.
# ---------------------------------------------------------------------------
# works from CPU or GPU weights (robust to the vLLM park/restore dance
# in the serving cell).
DEV = next(model.parameters()).device

print("building the 2-bit expert store from the fissioned FFNs ...")
STORE_DIR = os.path.join(tempfile.gettempdir(), "slipstream_store_v3")
shutil.rmtree(STORE_DIR, ignore_errors=True)
STORE = ExpertStore(STORE_DIR)

N_BUILD = len(PRISM_MODULES) if RUN_MODE == "full" \
    else min(6, len(PRISM_MODULES))
BUILD = PRISM_MODULES[:N_BUILD]
LID = {name: li for li, (name, _m) in enumerate(BUILD)}

blob_bytes_total = 0
routed_params_total = 0
for name, prism in BUILD:
    d, E, m_e = prism.d_model, prism.E, prism.m_e
    idx = prism.expert_idx                       # [E, m_e] columns per expert
    for e in range(E):
        cols = idx[e]
        blob_bytes_total += STORE.put_triple(
            LID[name], e,
            prism.W_gate[:, cols], prism.W_up[:, cols], prism.W_down[:, cols])
    routed_params_total += E * 3 * d * m_e
n_disk_params = routed_params_total
print(f"store: {len(STORE.index)} expert triples "
      f"({n_disk_params/1e6:.1f}M routed params) "
      f"| on-disk {blob_bytes_total/1e6:.1f} MB "
      f"| bf16 equivalent {2*n_disk_params/1e6:.1f} MB "
      f"({2*n_disk_params/max(blob_bytes_total,1):.1f}x)")

# ---- replay the real routers over the calibration stream -------------------
print("capturing real routing decisions ...")
cap_h = {name: [] for name, _ in BUILD}
handles = []
for name, prism in BUILD:
    def mk(nm):
        def hook(mod, args, kwargs, out):
            h = args[0] if args else kwargs.get("hidden_states")
            cap_h[nm].append(h.detach().reshape(-1, h.shape[-1]).cpu())
        return hook
    handles.append(prism.register_forward_hook(mk(name), with_kwargs=True))
with torch.no_grad():
    for i in range(min(max(2, blocks.shape[0]//4), 8)):
        _ = model(input_ids=blocks[i:i+1].to(DEV))
for h_ in handles:
    h_.remove()

sched, cert_fracs, reuse_rates = [], [], []
TOK_CAP = 512
for name, prism in BUILD:
    H = torch.cat(cap_h[name])[:TOK_CAP].to(prism.router.weight.device)   # [T, d] real router inputs
    logits = prism.router(H.float())             # temperature cancels in m/(L*rho)
    k = K_CHOSEN
    cert, cf = margin_certificate(logits, H, k, router_lipschitz(prism.router.weight))
    rr = reuse_hit_rate(logits, k)
    cert_fracs.append(cf)
    reuse_rates.append(rr)
    assert rr >= cf - 1e-9, f"Lemma S1 floor violated on {name}: {rr} < {cf}"
    ids = logits.argsort(-1, descending=True)[:, :k].cpu()
    for t in range(ids.shape[0]):
        if len(sched) <= t:
            sched.append([])
        sched[t].append((LID[name], ids[t].tolist()))
sched = sched[:min(len(sched), 256)]
print(f"schedule: {len(sched)} decode steps x {len(BUILD)} routed layers "
      f"x k={K_CHOSEN}")
print(f"Lemma S1 certificate per layer: mean {sum(cert_fracs)/len(cert_fracs):.3f} "
      f"| min {min(cert_fracs):.3f} | measured reuse mean "
      f"{sum(reuse_rates)/len(reuse_rates):.3f}")

# ---- benchmark the memory system -------------------------------------------
SIM_COMPUTE = 0.004        # 4 ms/token compute window (batch-1, 3B active)
for label, bw in (("emulated Drive (~8 MB/s)", 8e6),
                  ("emulated NVMe (~7 GB/s)", 7e9)):
    r_n = bench_decode(sched, "naive", STORE, sim_compute_s=SIM_COMPUTE,
                       sim_disk_bw=bw)
    r_s = bench_decode(sched, "slipstream", STORE,
                       capacity=max(8, 4*len(BUILD)*K_CHOSEN),
                       sim_compute_s=SIM_COMPUTE, sim_disk_bw=bw)
    r_r = bench_decode(sched, "resident", STORE, sim_compute_s=SIM_COMPUTE)
    print(f"\n--- storage: {label} ---")
    print(f"  {'mode':<11}{'tok/s':>9}   disk MB   hit-rate")
    for r in (r_n, r_s, r_r):
        hit_s = f"{r['hit_rate']:6.3f}" if r["mode"] != "resident" \
            else "  1.000"
        print(f"  {r['mode']:<11}{r['tok_per_s']:>9.1f}   "
              f"{r['disk_bytes']/1e6:8.2f}   {hit_s}")
    print(f"  speedup naive->slipstream: "
          f"{r_s['tok_per_s']/max(r_n['tok_per_s'],1e-9):.1f}x "
          f"| disk bytes cut: "
          f"{1 - r_s['disk_bytes']/max(r_n['disk_bytes'],1):.1%}")

# ---- Theorem S3 at REAL 27B scale ------------------------------------------
ff_routed_b = routed_params_total * 2 / max(N_BUILD, 1) * N_LAY   # scale up
hit = sum(reuse_rates)/max(len(reuse_rates), 1)
print("\n=== THEOREM S3 — steady-state decode table (27B program) ===")
for bw_v in (2e12, 1e12, 3e11):
    tps = {}
    for mode in ("resident", "naive", "slipstream"):
        t = throughput_table(total_dense*2, (globals().get("act_native") or WF["TOTAL_ACTIVE"])*2,
                             ff_routed_b, bw_v, 7e9, hit, comp_s=0.004)
        tps[mode] = t[f"{mode}_tps"]
    print(f"BWv={bw_v/1e12:.1f} TB/s | BWd=7 GB/s | hit={hit:.3f}: "
          f"resident {tps['resident']:.0f} tok/s | naive-offload "
          f"{tps['naive']:.1f} tok/s | SLIPSTREAM {tps['slipstream']:.0f} tok/s")
tchk = throughput_table(total_dense*2, (globals().get("act_native") or WF["TOTAL_ACTIVE"])*2, ff_routed_b,
                        1e12, 7e9, hit, comp_s=0.004)
print(f"no-stall (Thm S3): {tchk['no_stall']} | certified stall "
      f"{tchk['stall_s']*1e3:.2f} ms/token")
print("\nSLIPSTREAM on-model: the 27B program serves from disk at "
      "resident-class speed — the throttle is gone.")
free_cuda("SLIPSTREAM on-model complete")   # phase boundary


## Deployment guide

```bash
# ---- Artifact A: stock vLLM on the COMPAT checkpoint (zero custom code) ----
vllm serve /content/drive/MyDrive/PRISM-full/PRISM-Qwen3.8-27B-A3B \
    --max-model-len 262144 \          # full 262K context preserved from the base model
    --kv-cache-dtype fp8 \            # fit the 262K KV cache
    --tensor-parallel-size 2 \        # if you have >1 GPU
    --served-model-name prism-qwen3.8-27b-a3b
# then:
curl http://localhost:8000/v1/chat/completions -H 'Content-Type: application/json' -d '{
  "model": "prism-qwen3.8-27b-a3b",
  "messages": [{"role":"user","content":"Hello"}],
  "max_tokens": 131072 }'
# (long-context tip: test with --max-model-len 32768 first; vLLM flag names
#  move between releases — `vllm serve --help` is the source of truth)

# ---- Artifact B: the NATIVE ~3B-active launcher ----
python /content/drive/MyDrive/PRISM-full/PRISM-Qwen3.8-27B-A3B-RESONA-native/serve_native.py \
    --dir /content/drive/MyDrive/PRISM-full/PRISM-Qwen3.8-27B-A3B-RESONA-native
```

**Transformers fallback** (no vLLM needed):

```python
from transformers import AutoModelForImageTextToText, AutoProcessor
m = AutoModelForImageTextToText.from_pretrained(EXPORT_DIR, dtype="bfloat16", device_map="auto")
out = m.generate(**proc.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt"),
                 max_new_tokens=131072)
```

**Optional torchao-quantized serving variant:**

```python
from torchao.quantization import quantize_, Int4WeightOnlyConfig
m = AutoModelForImageTextToText.from_pretrained(EXPORT_DIR, dtype=torch.bfloat16)
quantize_(m.model.language_model, Int4WeightOnlyConfig())   # experts + attn, keep tower
m.save_pretrained(EXPORT_DIR + "-int4")
# vLLM: add  --quantization torchao   (verify support in your vLLM build)
```

### Scaling playbook

| Knob | Demo default | Recommended full | Why |
|---|---|---|---|
| distillation tokens | 32K (cached) | 1–4B tokens (`TEACH_BLOCKS_N`↑) | the routing gap closes with data, not steps |
| STEPS | 120 | 2,000–10,000 | cosine decay needs room after the serving anneal |
| LAMBDA_BAL | 0.01 | 0.01–0.05 if CV > 0.3 | keep the OT balance honest |
| top-k cache K | 64 | 64–256 | tighter KL fit on sharper teachers |
| E, k | auto (solver) | solver lands ~32/1 at 3B; 256/8 if you relax to A3B-style budgets | granularity ↔ quality trade-off |
| MoRM m_write | solver (strict accounting: 12 of 48 heads at the reference config) | sweep with eval; `MAST_ACCOUNTING="distinct"` buys more heads on paper by under-counting the sketch; `safety` lifts the cap | memory freshness ↔ savings |
| funnel (r, V_s) | solver | grow V_s until shortlist recall ≥ 99.5% | head quality ↔ savings |

**Known failure modes & mitigations:** (1) *load collapse* → raise LAMBDA_BAL, re-run gain calibration; (2) *PPL gap not closing* → more distillation tokens before more steps; (3) *OOM on 27B calibration* → lower GAIN_CAL_TOKENS, keep device_map auto offload; (4) *unmapped export key* → the exporter fails loudly with the exact tensor name — one-line regex fix mirroring your transformers version; (5) *vLLM arg drift* → `vllm serve --help`; (6) *MoRM degradation too high* → give the write heads more of the budget (smaller sketch / shortlist via the solver caps, or QUALITY_MODE='safety'), or relax the target; m_write = n_heads is exact but does not fit 3B; (7) *funnel recall < 99%* → raise FUNNEL_SHORTLIST / FUNNEL_RANK; (8) *a v9 contract fails* → its formula falls back to the v8 behaviour (the cell prints which); paste the `[FAIL]` line; (9) *census above the target* → `ENFORCE_BUDGET` stops the run; lower `SHARED_SIZE` / `M_WRITE_FRAC` / `FUNNEL_SHORTLIST` (or accept `MAST_ACCOUNTING="distinct"` knowingly).

**Productionization roadmap (native artifact):** (a) grouped-GEMM kernels for resonant row-blocks (the reference launcher is correctness-first); (b) a registered vLLM custom architecture (package `resona_kernels.py` as a plugin entry-point); (c) confidence-gated dynamic-k escalation spending the solver headroom; (d) write-residue compensation for MoRM (a tiny always-on state corrector distilled like the ZSSI fields).

### References (retrieved 2026-09-22)

1. Qwen3.8-27B model card & config — https://huggingface.co/Qwen/Qwen3.8-27B (released 2026-08-14, Apache-2.0); NVIDIA forum config notes (64 layers, hidden 5120, hybrid DeltaNet/full-attention)
2. Qwen3.8-27B coverage — thenewstack.io (Aug 14, 2026); simonwillison.net (Aug 16, 2026); ollama.com/library/qwen3.8
3. **GLM-5.3 (Z.ai, Aug 14 2026; open weights Aug 27)** — https://z.ai ; https://huggingface.co/zai-org/GLM-5.3 ; 744B/~40B active; **GLM-5.3-Flash 320B/18B active, MXFP8** — developers.cloudflare.com
4. **Qwen3-Next-80B-A3B (Alibaba's MoE playbook: 512 experts + 1 shared, top-10, hybrid attention)** — https://qwen.ai (Sep 10, 2025); huggingface.co/Qwen/Qwen3-Next-80B-A3B-Instruct
5. Official MoE sibling recipe — https://huggingface.co/Qwen/Qwen3.5-35B-A3B (`Qwen3_5MoeForConditionalGeneration`, 256 experts / top-8 / shared 512)
6. vLLM model registry (Qwen3_5Moe*) — https://github.com/vllm-project/vllm, `vllm/model_executor/models/registry.py`
7. torchao ≥ 0.16 / 0.17 docs — https://docs.pytorch.org/ao ; vLLM torchao docs — https://docs.vllm.ai
8. HF TrainingArguments (`warmup_steps`; `warmup_ratio` deprecated) — https://huggingface.co/docs/transformers
9. MoEfication: *Transformer Feed-forward Layers are Mixtures of Experts* (ACL 2022) — https://aclanthology.org/2022.findings-emnlp.284
10. Sparse Upcycling — Komatsuzaki et al.; **Drop-Upcycling** (ICR 2025) — https://arxiv.org/abs/2502.17573 ; NVIDIA D2S (Oct 2024)
11. *Upcycling Large Language Models into Mixture of Experts* (survey, 2025) — alphaxiv.org; *Mixture of Experts in Large Language Models* (review, Dec 2025) — arxiv.org
12. Dense-to-Dynamic-k (exploiting activation sparsity in any linear layer) — openreview.net
13. LG CVPR 2026 upcycling — lgresearch.ai; SIMoE (ACL 2025) — aclanthology.org
14. Community demand for a 3.8 A3B — huggingface.co discussions (Aug 17, 2026)

**License:** Qwen3.8-27B is Apache-2.0 — derivative checkpoints inherit it. This notebook is provided as-is for research; **the PRISM method (v9) is unvalidated beyond the verification suites in this notebook — treat results as a starting point, not a benchmark claim. The MoRM write-sparsity family is the most novel and the most experimental: always keep the `safety` mode one config away.**


## What was achieved

| axis | mechanism | certificate |
|---|---|---|
| compute-sparse | PRISM fission + RESONA execution | strict touched-weights census of the built graph ≤ 3.0B (plan == build, `ENFORCE_BUDGET`) |
| byte-sparse | SLIPSTREAM 2-bit cold tier | Prop-S2 Frobenius bound, measured under it |
| memory-sparse | MoRM 2.0 oracle-scheduled writes (WIK labels, elastic budgets, deferred write-back) | exact single-write influence (Thm B), E[m]=m_bar (Thm D), compensated commits (Thm C), staleness <= K (Thm E), beta-mask exactness (Prop F) |
| calibrated | oracle-energy routers, serving-consistent NNLS gains, Partition-KL / EMRD / LTD, certified head shortlist | contracts U1-U7, each with a v8 fallback switch |
| disk-native | LRU hot set + Lemma-S1 speculative prefetch | certified hit-rate floor → Theorem-S3 no-stall |

The conversion keeps its guarantees end-to-end: exact dense anchor at init (Theorem 7, asserted on real logits), total parameters preserved, 262K context and multimodality untouched, and two servable artifacts (stock-vLLM COMPAT ≈ 9.1B active; NATIVE ≈ 2.99B active). The same program naive offload serves at **~0.1–0.2 tok/s from storage** serves at resident-class speed under SLIPSTREAM, because the disk never sees more than $(1-h)\times$ the 2-bit expert slabs per token — and $h$ is certified before deployment, not hoped for after.
